In [ ]:
import os
import numpy as np
import pandas as pd
import random
import json
from itertools import product
from datetime import datetime
import traceback
from collections import Counter

# ======================================================================
# CONFIGURATION PARAMETERS
# ======================================================================

CONFIG = {
    'repetitions': 4,            # how many times each condition × SOA is repeated
    'num_participants': 5,       # default number of participants
    'included_conditions': {
        'inhalation': True,
        'exhalation': True,
        'baseline': True,
    },
    'catch_trial_percentage': 10,
    'soa_conditions_ms': [190, 400, 700, 1000, 1500],
    'jitter_options_ms': [100, 200, 300, 400, 500],
    'debug_mode': True,
    # CSV file with all the breath-hold timestamps
    'timestamps_csv': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_timestamps.csv",
    'paths': {
        'experiment_log_dir': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog",
    },
    'counterbalancing': {
        'max_sequence_length': 3,  # Avoid more than 3 of same trial type in a row
        'transition_balance': True, # Balance transitions between trial types
        'latin_square': True,       # Use latin square for participant sequence generation
        'balance_breathphase': True # Ensure even distribution across breath phases
    }
}

# Noise types for looming (these can be any labels you want)
LOOMING_STIMULI = ['blue', 'brown', 'pink', 'white']

# The boundaries of the segment we will replicate if needed
SEGMENT_START = "02:08.4"  # 13th hold ends
SEGMENT_END   = "04:56.4"  # 34th hold ends

# ======================================================================
# CLASS: PPSDesignGenerator
# ======================================================================

class PPSDesignGenerator:
    """Generates CSV design files for Peripersonal Space (PPS) experiments (no audio)."""
    
    def __init__(self, config=None):
        self.config = config or CONFIG
        
        # Prepare output directory
        self.experiment_log_dir = self.config['paths']['experiment_log_dir']
        os.makedirs(self.experiment_log_dir, exist_ok=True)
        
        # Calculate trial counts
        self.trial_counts = self._calculate_trial_counts()
        self.conditions = [cond for cond, inc in self.config['included_conditions'].items() if inc]
        
        # Internal property for the timestamps
        self.timestamps_df = None
        
        # Set seed for reproducibility across participants
        # But allow for per-participant variation
        self.base_seed = 42
        
        if self.config['debug_mode']:
            print("Initialized PPSDesignGenerator with:")
            print(f"  - Repetitions: {self.config['repetitions']}")
            print(f"  - Conditions: {self.conditions}")
            print(f"  - SOA values: {self.config['soa_conditions_ms']}")
            print(f"  - Total trials per participant: {sum(self.trial_counts.values())}")
            print(f"  - Output directory: {self.experiment_log_dir}")
    
    def _calculate_trial_counts(self):
        """Calculate how many trials for each condition (inhale/exhale/baseline) plus catch."""
        reps = self.config['repetitions']
        soa_count = len(self.config['soa_conditions_ms'])
        stimulus_types = LOOMING_STIMULI
        catch_percentage = self.config['catch_trial_percentage']
        
        trial_counts = {}
        for condition, include in self.config['included_conditions'].items():
            if include:
                # (# noise types) × (# SOA) × (reps)
                trial_counts[condition] = len(stimulus_types) * soa_count * reps
        
        regular_trials = sum(trial_counts.values())
        catch_trials = int(np.ceil(regular_trials * catch_percentage / 100))
        trial_counts['catch'] = catch_trials
        
        if self.config['debug_mode']:
            print("Trial count calculation:")
            for cond, count in trial_counts.items():
                print(f"  {cond}: {count} trials")
            print(f"  Total: {sum(trial_counts.values())} trials")
        
        return trial_counts
    
    def _expand_timestamps_if_needed(self):
        """
        Expand self.timestamps_df if we don't have enough rows to cover all trials.
        We replicate the segment from SEGMENT_START to SEGMENT_END, offsetting times
        and labeling each repeated block with repetition=1,2,3,... The original rows
        get repetition=0.
        """
        if self.timestamps_df is None or self.timestamps_df.empty:
            return
        
        # 1) Figure out how many total trials we have
        total_trials = sum(self.trial_counts.values())
        
        # 2) Check how many breath-hold rows are in the original CSV
        original_count = len(self.timestamps_df)
        if original_count >= total_trials:
            if self.config['debug_mode']:
                print(f"No expansion needed; CSV has {original_count} holds, need {total_trials}.")
            # Just set repetition=0 for all
            self.timestamps_df["repetition"] = 0
            return
        
        if self.config['debug_mode']:
            print(f"Expanding timestamps: have {original_count}, need {total_trials}.")
        
        # Sort the DataFrame by milliseconds to ensure order
        self.timestamps_df = self.timestamps_df.sort_values(by="milliseconds").reset_index(drop=True)
        
        # Identify the row indices for the segment we replicate
        seg_start_idx = self.timestamps_df.index[self.timestamps_df["timestamp"] == SEGMENT_START].tolist()
        seg_end_idx   = self.timestamps_df.index[self.timestamps_df["timestamp"] == SEGMENT_END].tolist()
        
        if not seg_start_idx or not seg_end_idx:
            raise ValueError(f"Could not find start='{SEGMENT_START}' or end='{SEGMENT_END}' in CSV.")
        
        start_i = seg_start_idx[0]
        end_i   = seg_end_idx[0]
        if end_i <= start_i:
            raise ValueError(f"Segment end index <= start index. Check your timestamps.")
        
        intro_df  = self.timestamps_df.iloc[:start_i].copy()
        repeat_df = self.timestamps_df.iloc[start_i:end_i+1].copy()
        outro_df  = self.timestamps_df.iloc[end_i+1:].copy()
        
        # Label original rows with repetition=0
        intro_df["repetition"]  = 0
        repeat_df["repetition"] = 0
        outro_df["repetition"]  = 0
        
        # Check how many we have if we keep the entire original CSV
        current_count = len(intro_df) + len(repeat_df) + len(outro_df)
        seg_len = len(repeat_df)
        
        if current_count >= total_trials:
            self.timestamps_df = pd.concat([intro_df, repeat_df, outro_df], ignore_index=True)
            return
        
        # Otherwise, replicate the repeat block until we have enough
        block_duration_ms = repeat_df["milliseconds"].iloc[-1] - repeat_df["milliseconds"].iloc[0]
        block_duration_samples = repeat_df["sample_index"].iloc[-1] - repeat_df["sample_index"].iloc[0]
        block_duration_sec = block_duration_ms / 1000.0
        
        def shift_timestamp(ts, offset_sec):
            if not isinstance(ts, str):
                return ts
            parts = ts.split(":")
            if len(parts) != 2:
                return ts
            mm = int(parts[0])
            ss = float(parts[1])
            total_s = mm * 60 + ss
            new_s = total_s + offset_sec
            new_mm = int(new_s // 60)
            new_ss = new_s % 60
            return f"{new_mm:02}:{new_ss:04.1f}"
        
        repeated_blocks = []
        offset_sec   = block_duration_sec
        offset_ms    = block_duration_ms
        offset_samps = block_duration_samples
        
        repetition_index = 1  # original = 0, so first copy = 1
        
        while current_count < total_trials:
            block_copy = repeat_df.copy()
            block_copy["timestamp"] = block_copy["timestamp"].apply(lambda t: shift_timestamp(t, offset_sec))
            block_copy["milliseconds"] = block_copy["milliseconds"] + offset_ms
            block_copy["sample_index"] = block_copy["sample_index"] + offset_samps
            block_copy["repetition"] = repetition_index
            repeated_blocks.append(block_copy)
            current_count += seg_len
            repetition_index += 1
            offset_sec   += block_duration_sec
            offset_ms    += block_duration_ms
            offset_samps += block_duration_samples
        
        self.timestamps_df = pd.concat([intro_df, repeat_df] + repeated_blocks + [outro_df], ignore_index=True)
        if self.config['debug_mode']:
            print(f"Final expanded CSV has {len(self.timestamps_df)} breath holds (needed {total_trials}).")
    
    def parse_timestamps(self):
        """Parse the box breathing timestamps CSV once, storing in self.timestamps_df, then expand if needed."""
        if self.timestamps_df is not None:
            return self.timestamps_df
        
        csv_path = self.config['timestamps_csv']
        if not os.path.exists(csv_path):
            print(f"ERROR: Timestamps CSV not found at {csv_path}")
            self.timestamps_df = pd.DataFrame()
            return self.timestamps_df
        
        try:
            df = pd.read_csv(csv_path)
            # Ensure 'breathphase' column is present
            if 'breathphase' not in df.columns:
                df['breathphase'] = ['inhale' if i % 2 == 0 else 'exhale' for i in range(len(df))]
            if 'time_seconds' not in df.columns:
                df['time_seconds'] = df['milliseconds'] / 1000.0
            df["repetition"] = 0
            self.timestamps_df = df
            
            if self.config['debug_mode']:
                print(f"Parsed {len(df)} timestamps from {csv_path}")
            
            self._expand_timestamps_if_needed()
            return self.timestamps_df
        except Exception as e:
            print(f"Error parsing timestamps CSV: {e}")
            traceback.print_exc()
            self.timestamps_df = pd.DataFrame()
            return self.timestamps_df
    
    def _generate_latin_square(self, factors, participant_id):
        """
        Generate a latin square ordering of factors for this participant.
        This ensures different participants get different factor orderings.
        """
        n = len(factors)
        base_sequence = list(range(n))
        rotation = participant_id % n
        sequence = base_sequence[rotation:] + base_sequence[:rotation]
        return [factors[i] for i in sequence]
    
    def _check_sequence_balance(self, sequence, max_length=3):
        """Check if a sequence has no more than max_length of same value in a row."""
        if not sequence:
            return True
        current_val = sequence[0]
        current_count = 1
        for val in sequence[1:]:
            if val == current_val:
                current_count += 1
                if current_count > max_length:
                    return False
            else:
                current_val = val
                current_count = 1
        return True
    
    def _check_transition_balance(self, sequence):
        """
        Check if transitions between values are balanced.
        e.g., A→B occurs similar number of times as B→A.
        """
        if len(sequence) < 3:
            return True
        transitions = {}
        for i in range(len(sequence) - 1):
            transition = (sequence[i], sequence[i+1])
            transitions[transition] = transitions.get(transition, 0) + 1
        for (a, b), count in transitions.items():
            reverse = (b, a)
            if reverse in transitions:
                if abs(transitions[reverse] - count) > 2:
                    return False
        return True
    
    def generate_counterbalanced_design(self, participant_id):
        """
        Generate a DataFrame with all trials (including baseline, catch, etc.)
        with enhanced counterbalancing of trial sequences and factor combinations.
        """
        random.seed(self.base_seed + participant_id)
        np.random.seed(self.base_seed + participant_id)
        
        soa_conditions = self.config['soa_conditions_ms']
        jitter_options = self.config['jitter_options_ms']
        stimulus_types = LOOMING_STIMULI
        all_trials = []
        
        if self.config['counterbalancing'].get('latin_square', True):
            ordered_soas = self._generate_latin_square(soa_conditions, participant_id)
            ordered_stimuli = self._generate_latin_square(stimulus_types, participant_id)
        else:
            ordered_soas = soa_conditions
            ordered_stimuli = stimulus_types
        
        for condition, count in self.trial_counts.items():
            if condition == 'catch' or count == 0:
                continue
            
            combinations_per_rep = []
            for rep in range(self.config['repetitions']):
                stim_soa_combinations = list(product(ordered_stimuli, ordered_soas))
                rep_seed = self.base_seed + participant_id * 100 + rep
                random.seed(rep_seed)
                random.shuffle(stim_soa_combinations)
                combinations_per_rep.append(stim_soa_combinations)
            all_combinations = [combo for rep_combos in combinations_per_rep for combo in rep_combos]
            for stim_type, soa in all_combinations:
                all_trials.append({
                    'participant_id': participant_id,
                    'trial_type': condition,  # inhalation, exhalation, baseline
                    'stimulus_type': stim_type,
                    'soa_value_ms': soa,
                    'jitter_ms': random.choice(jitter_options),
                    'is_tactile': True  # baseline & main trials have tactile
                })
        
        catch_count = self.trial_counts.get('catch', 0)
        if catch_count > 0:
            catch_combos = []
            stim_counts = {stim: 0 for stim in stimulus_types}
            soa_counts = {soa: 0 for soa in soa_conditions}
            while len(catch_combos) < catch_count:
                min_stim = min(stim_counts, key=stim_counts.get)
                min_soa = min(soa_counts, key=soa_counts.get)
                catch_combos.append((min_stim, min_soa))
                stim_counts[min_stim] += 1
                soa_counts[min_soa] += 1
            for stim_type, soa in catch_combos:
                all_trials.append({
                    'participant_id': participant_id,
                    'trial_type': 'catch',
                    'stimulus_type': stim_type,
                    'soa_value_ms': soa,
                    'jitter_ms': random.choice(jitter_options),
                    'is_tactile': False
                })
        
        design_df = pd.DataFrame(all_trials)
        max_attempts = 20
        best_sequence = None
        best_score = -1
        max_sequence = self.config['counterbalancing'].get('max_sequence_length', 3)
        
        for attempt in range(max_attempts):
            attempt_seed = self.base_seed + participant_id * 1000 + attempt
            random.seed(attempt_seed)
            shuffled_df = design_df.sample(frac=1).reset_index(drop=True)
            trial_sequence = shuffled_df['trial_type'].tolist()
            sequence_ok = self._check_sequence_balance(trial_sequence, max_sequence)
            transition_ok = (not self.config['counterbalancing'].get('transition_balance', True) or 
                             self._check_transition_balance(trial_sequence))
            score = 0
            if sequence_ok:
                score += 1
            if transition_ok:
                score += 1
            transition_counts = {}
            for i in range(len(trial_sequence) - 1):
                transition = (trial_sequence[i], trial_sequence[i+1])
                transition_counts[transition] = transition_counts.get(transition, 0) + 1
            values = list(transition_counts.values())
            entropy = -sum([(v/len(values)) * np.log(v/len(values)) for v in values])
            score += entropy
            if score > best_score:
                best_score = score
                best_sequence = shuffled_df
            if sequence_ok and transition_ok and entropy > 1.5:
                break
        
        if best_sequence is not None:
            design_df = best_sequence
        
        design_df['trial_number'] = range(1, len(design_df) + 1)
        
        if self.config['debug_mode']:
            trial_counts = design_df['trial_type'].value_counts()
            print(f"\nTrial type distribution:")
            print(trial_counts)
            trial_sequence = design_df['trial_type'].tolist()
            max_run = 1
            current_run = 1
            current_type = trial_sequence[0]
            for t in trial_sequence[1:]:
                if t == current_type:
                    current_run += 1
                    max_run = max(max_run, current_run)
                else:
                    current_type = t
                    current_run = 1
            print(f"Maximum consecutive trials of same type: {max_run}")
        
        return design_df
    
    def assign_breath_holds(self, design_df):
        """
        Assign each trial to an inhale or exhale row in self.timestamps_df.
        Now with enhanced counterbalancing of breath phases across conditions.
        """
        timestamps_df = self.parse_timestamps()  # ensures expansion is done
        if timestamps_df.empty:
            design_df['breathphase'] = None
            design_df['milliseconds'] = None
            design_df['jittered_ms'] = None
            design_df['soa_ms'] = None
            design_df['timestamp_original'] = None
            design_df['timestamp_after_jitter'] = None
            design_df['timestamp_with_soa'] = None
            return design_df
        
        skip_count = int(len(timestamps_df) * 0.1)
        usable = timestamps_df.iloc[skip_count:].copy()
        inhale_rows = usable[usable['breathphase'] == 'inhale']
        exhale_rows = usable[usable['breathphase'] == 'exhale']
        inhale_pool = inhale_rows.index.tolist()
        exhale_pool = exhale_rows.index.tolist()
        random.shuffle(inhale_pool)
        random.shuffle(exhale_pool)
        phase_counters = {}
        trial_assignments = {}
        
        for trial_type in design_df['trial_type'].unique():
            type_indices = design_df[design_df['trial_type'] == trial_type].index.tolist()
            if (trial_type, 'inhale') not in phase_counters:
                phase_counters[(trial_type, 'inhale')] = 0
            if (trial_type, 'exhale') not in phase_counters:
                phase_counters[(trial_type, 'exhale')] = 0
            for idx in type_indices:
                inhale_count = phase_counters[(trial_type, 'inhale')]
                exhale_count = phase_counters[(trial_type, 'exhale')]
                if trial_type == 'inhalation' and inhale_pool:
                    use_inhale = True
                elif trial_type == 'exhalation' and exhale_pool:
                    use_inhale = False
                else:
                    if inhale_count < exhale_count and inhale_pool:
                        use_inhale = True
                    elif exhale_count < inhale_count and exhale_pool:
                        use_inhale = False
                    else:
                        if not inhale_pool:
                            use_inhale = False
                        elif not exhale_pool:
                            use_inhale = True
                        else:
                            use_inhale = random.choice([True, False])
                if use_inhale and inhale_pool:
                    timestamp_idx = inhale_pool.pop(0)
                    row_ts = timestamps_df.loc[timestamp_idx]
                    trial_assignments[idx] = row_ts
                    phase_counters[(trial_type, 'inhale')] += 1
                elif exhale_pool:
                    timestamp_idx = exhale_pool.pop(0)
                    row_ts = timestamps_df.loc[timestamp_idx]
                    trial_assignments[idx] = row_ts
                    phase_counters[(trial_type, 'exhale')] += 1
                else:
                    raise ValueError(f"Ran out of timestamp rows for trial assignment. Need more timestamps!")
        
        design_df['breathphase'] = design_df.index.map(
            lambda i: trial_assignments[i]['breathphase'] if i in trial_assignments else None
        )
        design_df['milliseconds'] = design_df.index.map(
            lambda i: trial_assignments[i]['milliseconds'] if i in trial_assignments else None
        )
        design_df['jittered_ms'] = design_df.apply(
            lambda row: row['milliseconds'] + row['jitter_ms'] if pd.notnull(row['milliseconds']) else None,
            axis=1
        )
        design_df['soa_ms'] = design_df.apply(
            lambda row: row['jittered_ms'] + row['soa_value_ms']
            if (pd.notnull(row['jittered_ms']) and row['trial_type'] != 'catch') else None,
            axis=1
        )
        def ms_to_timestamp(ms):
            if pd.isnull(ms):
                return None
            m = int(ms // 60000)
            s = (ms % 60000) / 1000.0
            return f"{m:02}:{s:.1f}"
        
        design_df['timestamp_original'] = design_df['milliseconds'].apply(ms_to_timestamp)
        design_df['timestamp_after_jitter'] = design_df['jittered_ms'].apply(ms_to_timestamp)
        design_df['timestamp_with_soa'] = design_df['soa_ms'].apply(ms_to_timestamp)
        
        if self.config['debug_mode']:
            print("\nBreath phase allocation by trial type:")
            for trial_type in design_df['trial_type'].unique():
                type_df = design_df[design_df['trial_type'] == trial_type]
                phase_counts = type_df['breathphase'].value_counts()
                print(f"  {trial_type}: {phase_counts.to_dict()}")
        
        return design_df
    
    def finalize_design_csv(self, design_df):
        """
        1) Rename columns to final naming:
           - stimulus_type -> looming_stimulus_type
           - timestamp_original -> retentionphase_timestamp
           - timestamp_after_jitter -> looming_stimulus_timestamp
           - timestamp_with_soa -> tactile_stimulus_timestamp
        2) For baseline trials: set looming_stimulus_type='none', looming_stimulus_timestamp=NaN
        3) For catch trials: set tactile_stimulus_timestamp=NaN
        4) Return the final DataFrame (ready to save).
        """
        design_df.rename(columns={'stimulus_type': 'looming_stimulus_type'}, inplace=True)
        baseline_mask = (design_df['trial_type'] == 'baseline')
        design_df.loc[baseline_mask, 'looming_stimulus_type'] = 'none'
        design_df.rename(columns={
            'timestamp_original':      'retentionphase_timestamp',
            'timestamp_after_jitter':  'looming_stimulus_timestamp',
            'timestamp_with_soa':      'tactile_stimulus_timestamp'
        }, inplace=True)
        design_df.loc[baseline_mask, 'looming_stimulus_timestamp'] = np.nan
        catch_mask = (design_df['trial_type'] == 'catch')
        design_df.loc[catch_mask, 'tactile_stimulus_timestamp'] = np.nan
        design_df['expected_response'] = ~design_df['trial_type'].isin(['catch'])
        final_cols = [
            'participant_id',
            'trial_number',
            'trial_type',
            'breathphase',
            'retentionphase_timestamp',
            'looming_stimulus_timestamp',
            'tactile_stimulus_timestamp',
            'looming_stimulus_type',
            'soa_value_ms',
            'jitter_ms',
            'is_tactile',
            'expected_response'
        ]
        final_cols = [c for c in final_cols if c in design_df.columns]
        design_df = design_df[final_cols]
        self._validate_design(design_df)
        return design_df
    
    def _validate_design(self, design_df):
        """Check if the final design meets our counterbalancing criteria."""
        if not self.config['debug_mode']:
            return
        print("\n=== DESIGN VALIDATION ===")
        print("Trial type counts:")
        trial_counts = design_df['trial_type'].value_counts()
        print(trial_counts)
        print("\nStimulus distribution by trial type:")
        for trial_type in design_df['trial_type'].unique():
            if trial_type == 'baseline':
                continue
            type_df = design_df[design_df['trial_type'] == trial_type]
            stim_counts = type_df['looming_stimulus_type'].value_counts()
            print(f"  {trial_type}: {stim_counts.to_dict()}")
        print("\nSOA distribution by trial type:")
        for trial_type in design_df['trial_type'].unique():
            if trial_type == 'baseline':
                continue
            type_df = design_df[design_df['trial_type'] == trial_type]
            soa_counts = type_df['soa_value_ms'].value_counts()
            print(f"  {trial_type}: {soa_counts.to_dict()}")
        print("\nBreath phase distribution by trial type:")
        for trial_type in design_df['trial_type'].unique():
            type_df = design_df[design_df['trial_type'] == trial_type]
            phase_counts = type_df['breathphase'].value_counts()
            print(f"  {trial_type}: {phase_counts.to_dict()}")
        trial_sequence = design_df['trial_type'].tolist()
        consecutive_counts = {}
        max_consecutive = 1
        current_type = trial_sequence[0]
        current_count = 1
        for t in trial_sequence[1:]:
            if t == current_type:
                current_count += 1
                max_consecutive = max(max_consecutive, current_count)
            else:
                length_key = f"{current_type}_{current_count}"
                consecutive_counts[length_key] = consecutive_counts.get(length_key, 0) + 1
                current_type = t
                current_count = 1
        length_key = f"{current_type}_{current_count}"
        consecutive_counts[length_key] = consecutive_counts.get(length_key, 0) + 1
        print(f"\nMaximum consecutive trials of same type: {max_consecutive}")
        print("Distribution of consecutive runs:")
        for key, count in sorted(consecutive_counts.items()):
            print(f"  {key}: {count}")
        transitions = {}
        for i in range(len(trial_sequence) - 1):
            transition = (trial_sequence[i], trial_sequence[i+1])
            transitions[transition] = transitions.get(transition, 0) + 1
        print("\nTransition counts between trial types:")
        for transition, count in sorted(transitions.items()):
            print(f"  {transition[0]} → {transition[1]}: {count}")
    
    def process_participant(self, participant_id):
        """
        Generate the design, assign breath holds, finalize columns,
        and save as a CSV for one participant.
        """
        try:
            if self.config['debug_mode']:
                print(f"\n=== Processing participant {participant_id} ===")
            design_df = self.generate_counterbalanced_design(participant_id)
            design_df = self.assign_breath_holds(design_df)
            design_df = self.finalize_design_csv(design_df)
            out_path = os.path.join(self.experiment_log_dir, f"participant_{participant_id}_design.csv")
            design_df.to_csv(out_path, index=False)
            if self.config['debug_mode']:
                print(f"Saved participant {participant_id} design to {out_path}")
            return True, out_path
        except Exception as e:
            print(f"Error processing participant {participant_id}: {e}")
            traceback.print_exc()
            return False, None
    
    def run(self, participant_ids=None, num_participants=None):
        """
        Generate the design CSVs for multiple participants.
        """
        if participant_ids is None and num_participants is None:
            num_participants = self.config.get('num_participants', 5)
        if participant_ids is None:
            participant_ids = list(range(num_participants))
        if self.config['debug_mode']:
            print("=== Starting PPSDesignGenerator ===")
            print(f"Repetitions: {self.config['repetitions']}")
            print(f"Participants: {participant_ids}")
            print(f"Counterbalancing settings: {self.config['counterbalancing']}")
        results = {}
        for pid in participant_ids:
            success, path = self.process_participant(pid)
            results[pid] = {
                'success': success,
                'csv_path': path
            }
        return results

# ======================================================================
# MAIN
# ======================================================================

def main():
    generator = PPSDesignGenerator(CONFIG)
    results = generator.run()  # by default, uses num_participants=5
    print("\n=== Generation Summary ===")
    for pid, info in results.items():
        if info['success']:
            print(f"Participant {pid}: CSV generated at {info['csv_path']}")
        else:
            print(f"Participant {pid}: ERROR - no CSV generated.")

if __name__ == "__main__":
    main()


In [1]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
PPS Design Generator

This script generates experimental design files for a Peripersonal Space (PPS) experiment.
It creates CSV files with precise timing information for audio stimuli, using regular
8-second intervals for box breathing cycles.

Author: AI Assistant
"""

import os
import numpy as np
import pandas as pd
import random
import json
from itertools import product
from datetime import datetime
import traceback

# ======================================================================
# CONFIGURATION PARAMETERS
# ======================================================================

CONFIG = {
    'repetitions': 4,            # how many times each condition × SOA is repeated
    'num_participants': 5,       # default number of participants
    'included_conditions': {
        'inhalation': True,
        'exhalation': True,
        'baseline': True,
    },
    'catch_trial_percentage': 10,
    'soa_conditions_ms': [190, 400, 700, 1000, 1500],
    'jitter_options_ms': [100, 200, 300, 400, 500],
    'debug_mode': True,
    'paths': {
        'experiment_log_dir': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog",
    },
    'box_breathing': {
        'cycle_duration_sec': 8,      # Total duration of one breath cycle in seconds
        'intro_duration_sec': 60,     # Initial settling period before trials begin
        'outro_duration_sec': 30,     # Closing period after trials end
        'alternating_phases': True,   # Alternate between inhale and exhale phases
        'sample_rate': 48000,         # Audio sample rate
    }
}

# Noise types for looming (these can be any labels you want)
LOOMING_STIMULI = ['blue', 'brown', 'pink', 'white']

# ======================================================================
# CLASS: PPSDesignGenerator
# ======================================================================

class PPSDesignGenerator:
    """Generates CSV design files for Peripersonal Space (PPS) experiments."""
    
    def __init__(self, config=None):
        self.config = config or CONFIG
        
        # Prepare output directory
        self.experiment_log_dir = self.config['paths']['experiment_log_dir']
        os.makedirs(self.experiment_log_dir, exist_ok=True)
        
        # Calculate trial counts
        self.trial_counts = self._calculate_trial_counts()
        self.conditions = [cond for cond, inc in self.config['included_conditions'].items() if inc]
        
        # Set seed for reproducibility
        self.base_seed = 42
        
        if self.config['debug_mode']:
            print("Initialized PPSDesignGenerator with:")
            print(f"  - Repetitions: {self.config['repetitions']}")
            print(f"  - Conditions: {self.conditions}")
            print(f"  - SOA values: {self.config['soa_conditions_ms']}")
            print(f"  - Total trials per participant: {sum(self.trial_counts.values())}")
            print(f"  - Output directory: {self.experiment_log_dir}")
    
    def _calculate_trial_counts(self):
        """Calculate how many trials for each condition (inhale/exhale/baseline) plus catch."""
        reps = self.config['repetitions']
        soa_count = len(self.config['soa_conditions_ms'])
        stimulus_types = LOOMING_STIMULI
        catch_percentage = self.config['catch_trial_percentage']
        
        trial_counts = {}
        for condition, include in self.config['included_conditions'].items():
            if include:
                # (# noise types) × (# SOA) × (reps)
                trial_counts[condition] = len(stimulus_types) * soa_count * reps
        
        regular_trials = sum(trial_counts.values())
        catch_trials = int(np.ceil(regular_trials * catch_percentage / 100))
        trial_counts['catch'] = catch_trials
        
        if self.config['debug_mode']:
            print("Trial count calculation:")
            for cond, count in trial_counts.items():
                print(f"  {cond}: {count} trials")
            print(f"  Total: {sum(trial_counts.values())} trials")
        
        return trial_counts
    
    def generate_breath_timestamps(self):
        """
        Generate timestamps for breath cycles at regular 8-second intervals.
        Each timestamp represents a point where a stimulus could be inserted.
        IMPORTANT: The breath phase (inhale/exhale) strictly alternates as this
        is a fixed physiological cycle that cannot be reordered.
        
        Returns:
            DataFrame with columns: timestamp, milliseconds, sample_index, breathphase
        """
        # Calculate parameters
        total_trials = sum(self.trial_counts.values())
        cycle_sec = self.config['box_breathing']['cycle_duration_sec']
        intro_sec = self.config['box_breathing']['intro_duration_sec']
        outro_sec = self.config['box_breathing']['outro_duration_sec']
        sample_rate = self.config['box_breathing']['sample_rate']
        
        # We need enough timestamps for all trials with intro/outro period
        total_duration_sec = intro_sec + (total_trials * cycle_sec) + outro_sec
        
        # Generate timestamps at regular intervals
        timestamps = []
        
        # Start with intro period
        current_time_sec = intro_sec  # First trial starts after intro
        
        # Always start with inhale (physiologically correct)
        # Strict alternation: inhale, exhale, inhale, exhale...
        for i in range(total_trials):
            # Convert time to minutes:seconds format
            minutes = int(current_time_sec / 60)
            seconds = current_time_sec % 60
            timestamp = f"{minutes:02}:{seconds:.1f}"
            
            # Calculate milliseconds and sample index
            milliseconds = current_time_sec * 1000
            sample_index = int(current_time_sec * sample_rate)
            
            # Strict alternation of breath phases - this cannot be randomized
            # In box breathing, phases must follow the natural breath cycle
            breathphase = 'inhale' if i % 2 == 0 else 'exhale'
            
            # Add this timestamp
            timestamps.append({
                'timestamp': timestamp,
                'milliseconds': milliseconds,
                'sample_index': sample_index,
                'breathphase': breathphase,
                'trial_index': i
            })
            
            # Advance to next cycle
            current_time_sec += cycle_sec
        
        # Convert to DataFrame
        timestamps_df = pd.DataFrame(timestamps)
        
        if self.config['debug_mode']:
            print(f"Generated {len(timestamps_df)} breath timestamps")
            print(f"First timestamp: {timestamps_df['timestamp'].iloc[0]}")
            print(f"Last timestamp: {timestamps_df['timestamp'].iloc[-1]}")
            print(f"Total audio duration: {total_duration_sec} seconds")
            
            # Check breath phase alternation
            phases = timestamps_df['breathphase'].tolist()
            print(f"Breath phases: {phases[:10]}... (alternating inhale-exhale pattern)")
        
        return timestamps_df
    
    def generate_counterbalanced_design(self, participant_id):
        """
        Generate a counterbalanced design for one participant.
        """
        # Set seed based on participant ID for reproducibility
        random.seed(self.base_seed + participant_id)
        np.random.seed(self.base_seed + participant_id)
        
        soa_conditions = self.config['soa_conditions_ms']
        jitter_options = self.config['jitter_options_ms']
        stimulus_types = LOOMING_STIMULI
        
        all_trials = []
        
        # Build the main conditions (inhale, exhale, baseline)
        for condition, count in self.trial_counts.items():
            if condition == 'catch' or count == 0:
                continue
            
            # Create all combos of (noise type × SOA)
            combo_list = []
            
            # Generate balanced combinations for each stimulus type and SOA
            for stim_type in stimulus_types:
                for soa in soa_conditions:
                    for _ in range(self.config['repetitions']):
                        combo_list.append((stim_type, soa))
            
            # Shuffle the combinations
            random.shuffle(combo_list)
            
            # Create trial records
            for stim_type, soa in combo_list:
                all_trials.append({
                    'participant_id': participant_id,
                    'trial_type': condition,  # inhalation, exhalation, baseline
                    'stimulus_type': stim_type,
                    'soa_value_ms': soa,
                    'jitter_ms': random.choice(jitter_options),
                    'is_tactile': True  # baseline & main trials have tactile
                })
        
        # Add catch trials (looming only, no tactile)
        catch_count = self.trial_counts.get('catch', 0)
        if catch_count > 0:
            catch_combos = []
            
            # Ensure each stimulus type and SOA appears equally in catch trials
            stim_counts = {stim: 0 for stim in stimulus_types}
            soa_counts = {soa: 0 for soa in soa_conditions}
            
            while len(catch_combos) < catch_count:
                # Find stimulus type with fewest occurrences
                min_stim = min(stim_counts, key=stim_counts.get)
                # Find SOA with fewest occurrences
                min_soa = min(soa_counts, key=soa_counts.get)
                
                catch_combos.append((min_stim, min_soa))
                stim_counts[min_stim] += 1
                soa_counts[min_soa] += 1
            
            # Create catch trial entries
            for stim_type, soa in catch_combos:
                all_trials.append({
                    'participant_id': participant_id,
                    'trial_type': 'catch',
                    'stimulus_type': stim_type,
                    'soa_value_ms': soa,
                    'jitter_ms': random.choice(jitter_options),
                    'is_tactile': False
                })
        
        # Convert to DataFrame and shuffle
        design_df = pd.DataFrame(all_trials)
        design_df = design_df.sample(frac=1, random_state=participant_id*100).reset_index(drop=True)
        
        # Add a trial_number
        design_df['trial_number'] = range(1, len(design_df) + 1)
        
        # Check distributions for proper counterbalancing
        if self.config['debug_mode']:
            print("\nTrial type distribution:")
            print(design_df['trial_type'].value_counts())
            
            print("\nStimulus type distribution by trial type:")
            for trial_type in design_df['trial_type'].unique():
                if trial_type == 'baseline':
                    continue  # baseline has no looming stimuli
                subset = design_df[design_df['trial_type'] == trial_type]
                print(f"  {trial_type}: {subset['stimulus_type'].value_counts().to_dict()}")
            
            print("\nSOA distribution by trial type:")
            for trial_type in design_df['trial_type'].unique():
                if trial_type == 'baseline':
                    continue  # baseline has no SOA relationship
                subset = design_df[design_df['trial_type'] == trial_type]
                print(f"  {trial_type}: {subset['soa_value_ms'].value_counts().to_dict()}")
        
        return design_df
    
    def assign_breath_holds(self, design_df):
        """
        Assign trial types to fixed breath phases (inhale/exhale cycles).
        
        IMPORTANT: Unlike the previous implementation, we don't choose which breath
        phase to assign a trial to - rather, we choose which trial type goes into
        each predefined breath phase slot. The breath phases are fixed in their
        alternating pattern.
        """
        # Generate timestamps with fixed alternating breath phases
        timestamps_df = self.generate_breath_timestamps()
        
        # Split timestamp dataframe by breath phase
        inhale_timestamps = timestamps_df[timestamps_df['breathphase'] == 'inhale'].copy()
        exhale_timestamps = timestamps_df[timestamps_df['breathphase'] == 'exhale'].copy()
        
        # Split trials by preferred breath phase
        inhalation_trials = design_df[design_df['trial_type'] == 'inhalation'].copy()
        exhalation_trials = design_df[design_df['trial_type'] == 'exhalation'].copy()
        other_trials = design_df[~design_df['trial_type'].isin(['inhalation', 'exhalation'])].copy()
        
        # Shuffle the trials to randomize their order
        inhalation_trials = inhalation_trials.sample(frac=1).reset_index(drop=True)
        exhalation_trials = exhalation_trials.sample(frac=1).reset_index(drop=True)
        other_trials = other_trials.sample(frac=1).reset_index(drop=True)
        
        # Check if we have enough timestamps of each type
        if len(inhale_timestamps) < len(inhalation_trials):
            print(f"Warning: Not enough inhale phases ({len(inhale_timestamps)}) for all inhalation trials ({len(inhalation_trials)})")
        
        if len(exhale_timestamps) < len(exhalation_trials):
            print(f"Warning: Not enough exhale phases ({len(exhale_timestamps)}) for all exhalation trials ({len(exhalation_trials)})")
        
        # Prepare result dataframe
        result_rows = []
        
        # First, assign inhalation trials to inhale phases (as many as possible)
        used_inhale_indices = []
        for i, trial in enumerate(inhalation_trials.itertuples()):
            if i < len(inhale_timestamps):
                timestamp = inhale_timestamps.iloc[i]
                trial_dict = trial._asdict()
                trial_dict.update({
                    'breathphase': 'inhale',
                    'milliseconds': timestamp['milliseconds'],
                    'timestamp_original': timestamp['timestamp'],
                    'sample_index': timestamp['sample_index'],
                    'breath_cycle_index': timestamp['trial_index']
                })
                result_rows.append(trial_dict)
                used_inhale_indices.append(i)
            else:
                break  # Out of inhale phases
        
        # Then, assign exhalation trials to exhale phases (as many as possible)
        used_exhale_indices = []
        for i, trial in enumerate(exhalation_trials.itertuples()):
            if i < len(exhale_timestamps):
                timestamp = exhale_timestamps.iloc[i]
                trial_dict = trial._asdict()
                trial_dict.update({
                    'breathphase': 'exhale',
                    'milliseconds': timestamp['milliseconds'],
                    'timestamp_original': timestamp['timestamp'],
                    'sample_index': timestamp['sample_index'],
                    'breath_cycle_index': timestamp['trial_index']
                })
                result_rows.append(trial_dict)
                used_exhale_indices.append(i)
            else:
                break  # Out of exhale phases
        
        # Collect remaining trials that couldn't get their preferred breath phase
        remaining_inhalation = [row for i, row in enumerate(inhalation_trials.itertuples()) 
                               if i >= len(inhale_timestamps)]
        remaining_exhalation = [row for i, row in enumerate(exhalation_trials.itertuples()) 
                                if i >= len(exhale_timestamps)]
        all_remaining = remaining_inhalation + remaining_exhalation + list(other_trials.itertuples())
        random.shuffle(all_remaining)  # Mix them up
        
        # Assign remaining trials to any available timestamps
        unused_inhale = [timestamp for i, timestamp in enumerate(inhale_timestamps.itertuples()) 
                         if i >= len(used_inhale_indices)]
        unused_exhale = [timestamp for i, timestamp in enumerate(exhale_timestamps.itertuples()) 
                        if i >= len(used_exhale_indices)]
        
        # Combine all unused timestamps in their natural order
        all_unused = []
        for timestamp in timestamps_df.itertuples():
            is_inhale = timestamp.breathphase == 'inhale'
            inhale_idx = timestamp.trial_index // 2  # Convert to index within inhale/exhale group
            exhale_idx = timestamp.trial_index // 2
            
            if (is_inhale and inhale_idx >= len(used_inhale_indices)) or \
               (not is_inhale and exhale_idx >= len(used_exhale_indices)):
                all_unused.append(timestamp)
        
        # Sort by trial_index to maintain the natural breath cycle order
        all_unused.sort(key=lambda x: x.trial_index)
        
        # Assign remaining trials to the available timestamps
        for i, trial in enumerate(all_remaining):
            if i < len(all_unused):
                timestamp = all_unused[i]
                trial_dict = trial._asdict()
                trial_dict.update({
                    'breathphase': timestamp.breathphase,
                    'milliseconds': timestamp.milliseconds,
                    'timestamp_original': timestamp.timestamp,
                    'sample_index': timestamp.sample_index,
                    'breath_cycle_index': timestamp.trial_index
                })
                result_rows.append(trial_dict)
            else:
                raise ValueError(f"Not enough breath cycles for all trials! Need {len(all_remaining) - len(all_unused)} more.")
        
        # Convert to DataFrame
        design_with_timestamps = pd.DataFrame(result_rows)
        
        # Sort by the breath cycle index to maintain the natural order of breaths
        design_with_timestamps = design_with_timestamps.sort_values(by='breath_cycle_index').reset_index(drop=True)
        
        # Calculate jittered_ms = base + jitter
        design_with_timestamps['jittered_ms'] = design_with_timestamps['milliseconds'] + design_with_timestamps['jitter_ms']
        
        # Format jittered timestamp
        def ms_to_timestamp(ms):
            if pd.isna(ms):
                return None
            m = int(ms // 60000)
            s = (ms % 60000) / 1000.0
            return f"{m:02}:{s:.1f}"
        
        design_with_timestamps['timestamp_after_jitter'] = design_with_timestamps['jittered_ms'].apply(ms_to_timestamp)
        
        # For non-catch trials, add tactile timestamp (original + jitter + SOA)
        design_with_timestamps['soa_ms'] = None
        design_with_timestamps['timestamp_with_soa'] = None
        
        non_catch_mask = design_with_timestamps['trial_type'] != 'catch'
        design_with_timestamps.loc[non_catch_mask, 'soa_ms'] = (
            design_with_timestamps.loc[non_catch_mask, 'jittered_ms'] + 
            design_with_timestamps.loc[non_catch_mask, 'soa_value_ms']
        )
        
        design_with_timestamps.loc[non_catch_mask, 'timestamp_with_soa'] = (
            design_with_timestamps.loc[non_catch_mask, 'soa_ms'].apply(ms_to_timestamp)
        )
        
        if self.config['debug_mode']:
            # Verify the breathphase alternation is maintained
            phases = design_with_timestamps.sort_values(by='breath_cycle_index')['breathphase'].tolist()
            alternating = True
            for i in range(1, len(phases)):
                if phases[i] == phases[i-1]:
                    alternating = False
                    break
            
            if alternating:
                print("✓ Breath phases maintain perfect alternation (inhale-exhale-inhale-...)")
            else:
                print("⚠ WARNING: Breath phases do not alternate correctly!")
            
            # Show trial type distribution across breath phases
            print("\nTrial Type Distribution in Breath Phases:")
            pivot = pd.crosstab(design_with_timestamps['trial_type'], design_with_timestamps['breathphase'])
            print(pivot)
        
        return design_with_timestamps
        # Assign remaining trials to the available timestamps
        for i, trial in enumerate(all_remaining):
            if i < len(all_unused):
                timestamp = all_unused[i]
                trial_dict = trial._asdict()
                trial_dict.update({
                    'breathphase': timestamp.breathphase,
                    'milliseconds': timestamp.milliseconds,
                    'timestamp_original': timestamp.timestamp,
                    'sample_index': timestamp.sample_index,
                    'breath_cycle_index': timestamp.trial_index
                })
                result_rows.append(trial_dict)
            else:
                raise ValueError(f"Not enough breath cycles for all trials! Need {len(all_remaining) - len(all_unused)} more.")
        
        # Convert to DataFrame
        design_with_timestamps = pd.DataFrame(result_rows)
        
        # Sort by the breath cycle index to maintain the natural order of breaths
        design_with_timestamps = design_with_timestamps.sort_values(by='breath_cycle_index').reset_index(drop=True)
        
        # Calculate jittered_ms = base + jitter
        design_with_timestamps['jittered_ms'] = design_with_timestamps['milliseconds'] + design_with_timestamps['jitter_ms']
        
        # Calculate jittered_ms = base + jitter
        design_with_timestamps['jittered_ms'] = design_with_timestamps['milliseconds'] + design_with_timestamps['jitter_ms']
        
        # Format jittered timestamp
        def ms_to_timestamp(ms):
            m = int(ms // 60000)
            s = (ms % 60000) / 1000.0
            return f"{m:02}:{s:.1f}"
        
        design_with_timestamps['timestamp_after_jitter'] = design_with_timestamps['jittered_ms'].apply(ms_to_timestamp)
        
        # For non-catch trials, add tactile timestamp (original + jitter + SOA)
        design_with_timestamps['soa_ms'] = None
        design_with_timestamps['timestamp_with_soa'] = None
        
        non_catch_mask = design_with_timestamps['trial_type'] != 'catch'
        design_with_timestamps.loc[non_catch_mask, 'soa_ms'] = (
            design_with_timestamps.loc[non_catch_mask, 'jittered_ms'] + 
            design_with_timestamps.loc[non_catch_mask, 'soa_value_ms']
        )
        
        design_with_timestamps.loc[non_catch_mask, 'timestamp_with_soa'] = (
            design_with_timestamps.loc[non_catch_mask, 'soa_ms'].apply(ms_to_timestamp)
        )
        
        return design_with_timestamps
    
    def finalize_design_csv(self, design_df):
        """
        Finalize the design CSV with proper column names and special case handling.
        
        Args:
            design_df: DataFrame with raw design data including timestamps
            
        Returns:
            DataFrame with finalized column names and special case handling
        """
        # Rename columns
        design_df.rename(columns={
            'stimulus_type': 'looming_stimulus_type',
            'timestamp_original': 'retentionphase_timestamp',
            'timestamp_after_jitter': 'looming_stimulus_timestamp',
            'timestamp_with_soa': 'tactile_stimulus_timestamp'
        }, inplace=True)
        
        # Handle baseline trials (no looming stimulus)
        baseline_mask = (design_df['trial_type'] == 'baseline')
        design_df.loc[baseline_mask, 'looming_stimulus_type'] = 'none'
        design_df.loc[baseline_mask, 'looming_stimulus_timestamp'] = None
        
        # Handle catch trials (no tactile stimulus)
        catch_mask = (design_df['trial_type'] == 'catch')
        design_df.loc[catch_mask, 'tactile_stimulus_timestamp'] = None
        
        # Add expected response flag (true for all except catch)
        design_df['expected_response'] = ~design_df['trial_type'].isin(['catch'])
        

            'soa_value_ms',              # time between looming and tactile
            'jitter_ms',                 # random jitter added to base timestamp
            'sample_index',              # sample index of retention phase
            'is_tactile',                # whether trial has tactile component
            'expected_response'          # whether response is expected
        ]
        
        # Keep only columns that exist in the DataFrame
        final_cols = [col for col in final_cols if col in design_df.columns]
        design_df = design_df[final_cols]
        
        # Validate final design
        self._validate_design(design_df)
        
        return design_df
    
    def _validate_design(self, design_df):
        """Check if the final design meets counterbalancing criteria."""
        if not self.config['debug_mode']:
            return
            
        print("\n=== DESIGN VALIDATION ===")
        
        # 1. Check trial type distribution
        print("Trial type counts:")
        trial_counts = design_df['trial_type'].value_counts()
        print(trial_counts)
        
        # 2. Check stimulus type distribution within each trial type
        print("\nStimulus distribution by trial type:")
        for trial_type in design_df['trial_type'].unique():
            if trial_type == 'baseline':
                continue  # No looming in baseline
                
            type_df = design_df[design_df['trial_type'] == trial_type]
            stim_counts = type_df['looming_stimulus_type'].value_counts()
            print(f"  {trial_type}: {stim_counts.to_dict()}")
        
        # 3. Check SOA distribution within each trial type
        print("\nSOA distribution by trial type:")
        for trial_type in design_df['trial_type'].unique():
            if trial_type == 'baseline':
                continue  # No SOA relationship in baseline
                
            type_df = design_df[design_df['trial_type'] == trial_type]
            soa_counts = type_df['soa_value_ms'].value_counts()
            print(f"  {trial_type}: {soa_counts.to_dict()}")
        
        # 4. Check breath phase distribution within each trial type
        print("\nBreath phase distribution by trial type:")
        for trial_type in design_df['trial_type'].unique():
            type_df = design_df[design_df['trial_type'] == trial_type]
            phase_counts = type_df['breathphase'].value_counts()
            print(f"  {trial_type}: {phase_counts.to_dict()}")
        
        # 5. Check sequence properties
        trial_sequence = design_df['trial_type'].tolist()
        
        # Count runs of consecutive trial types
        consecutive_counts = {}
        current_type = trial_sequence[0]
        current_count = 1
        
        for t in trial_sequence[1:]:
            if t == current_type:
                current_count += 1
            else:
                # Record the run we just finished
                length_key = f"{current_type}_{current_count}"
                consecutive_counts[length_key] = consecutive_counts.get(length_key, 0) + 1
                # Reset for new type
                current_type = t
                current_count = 1
        
        # Record the final run
        length_key = f"{current_type}_{current_count}"
        consecutive_counts[length_key] = consecutive_counts.get(length_key, 0) + 1
        
        print("\nRuns of consecutive trials:")
        for key, count in sorted(consecutive_counts.items()):
            print(f"  {key}: {count}")
        
        # 6. Calculate transition probabilities between trial types
        transitions = {}
        for i in range(len(trial_sequence) - 1):
            transition = (trial_sequence[i], trial_sequence[i+1])
            transitions[transition] = transitions.get(transition, 0) + 1
        
        print("\nTransition counts between trial types:")
        for transition, count in sorted(transitions.items()):
            print(f"  {transition[0]} → {transition[1]}: {count}")
    
    def process_participant(self, participant_id):
        """
        Generate the design, assign breath holds, finalize columns,
        and save as a CSV for one participant.
        """
        try:
            if self.config['debug_mode']:
                print(f"\n=== Processing participant {participant_id} ===")
            
            # 1) Generate base design (counterbalanced combos)
            design_df = self.generate_counterbalanced_design(participant_id)
            
            # 2) Assign each trial to a breath hold (inhale or exhale)
            design_df = self.assign_breath_holds(design_df)
            
            # 3) Finalize columns (rename, set baseline/catch to NA, etc.)
            design_df = self.finalize_design_csv(design_df)
            
            # 4) Save CSV
            out_path = os.path.join(self.experiment_log_dir,
                                    f"participant_{participant_id}_design.csv")
            design_df.to_csv(out_path, index=False)
            
            if self.config['debug_mode']:
                print(f"Saved participant {participant_id} design to {out_path}")
            
            return True, out_path
        except Exception as e:
            print(f"Error processing participant {participant_id}: {e}")
            traceback.print_exc()
            return False, None
    
    def run(self, participant_ids=None, num_participants=None):
        """
        Generate the design CSVs for multiple participants.
        
        Args:
            participant_ids: List of specific participant IDs to process
            num_participants: Number of participants to generate if participant_ids not provided
            
        Returns:
            Dictionary mapping participant IDs to generation results
        """
        if participant_ids is None and num_participants is None:
            num_participants = self.config.get('num_participants', 5)
        
        if participant_ids is None:
            participant_ids = list(range(num_participants))
        
        if self.config['debug_mode']:
            print("=== Starting PPSDesignGenerator ===")
            print(f"Repetitions: {self.config['repetitions']}")
            print(f"Participants: {participant_ids}")
            print(f"Box breathing settings: {self.config['box_breathing']}")
        
        results = {}
        for pid in participant_ids:
            success, path = self.process_participant(pid)
            results[pid] = {
                'success': success,
                'csv_path': path
            }
        
        return results

# ======================================================================
# AUDIO GENERATION FUNCTIONS
# ======================================================================

def generate_audio_from_design(design_csv_path, output_dir, base_audio_path=None):
    """
    Generate audio files from a design CSV.
    
    This is a placeholder for audio generation functionality.
    The actual implementation would:
    1. Read the design CSV
    2. Create or load the base box breathing audio
    3. Insert looming and tactile stimuli at the specified timestamps
    4. Save the resulting audio files
    
    Args:
        design_csv_path: Path to the design CSV
        output_dir: Directory to save audio files
        base_audio_path: Optional path to base box breathing audio
        
    Returns:
        Tuple of (looming_audio_path, tactile_audio_path)
    """
    # This is just a placeholder - implement actual audio generation here
    print(f"Audio generation from {design_csv_path} not implemented yet")
    return None, None

# ======================================================================
# MAIN FUNCTION
# ======================================================================

def main():
    """Main function to run the design generator."""
    # Create the design generator
    generator = PPSDesignGenerator(CONFIG)
    
    # Run for all participants defined in CONFIG
    results = generator.run()
    
    # Print summary
    print("\n=== Generation Summary ===")
    for pid, info in results.items():
        if info['success']:
            print(f"Participant {pid}: CSV generated at {info['csv_path']}")
        else:
            print(f"Participant {pid}: ERROR - no CSV generated.")
    
    print("\nTo generate audio files from these designs, implement the audio generation function.")

if __name__ == "__main__":
    main()

IndentationError: unexpected indent (630239049.py, line 529)

In [2]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
PPS Design Generator

This script generates experimental design files for a Peripersonal Space (PPS) experiment.
It creates CSV files with precise timing information for audio stimuli, using regular
8-second intervals for box breathing cycles.

Author: AI Assistant
"""

import os
import numpy as np
import pandas as pd
import random
import json
from itertools import product
from datetime import datetime
import traceback

# ======================================================================
# CONFIGURATION PARAMETERS
# ======================================================================

CONFIG = {
    'repetitions': 4,            # how many times each condition × SOA is repeated
    'num_participants': 5,       # default number of participants
    'included_conditions': {
        'inhalation': True,
        'exhalation': True,
        'baseline': True,
    },
    'catch_trial_percentage': 10,
    'soa_conditions_ms': [190, 400, 700, 1000, 1500],
    'jitter_options_ms': [100, 200, 300, 400, 500],
    'debug_mode': True,
    'paths': {
        'experiment_log_dir': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog",
    },
    'box_breathing': {
        'cycle_duration_sec': 8,      # Total duration of one breath cycle in seconds
        'intro_duration_sec': 60,     # Initial settling period before trials begin
        'outro_duration_sec': 30,     # Closing period after trials end
        'alternating_phases': True,   # Alternate between inhale and exhale phases
        'sample_rate': 48000,         # Audio sample rate
    }
}

# Noise types for looming (these can be any labels you want)
LOOMING_STIMULI = ['blue', 'brown', 'pink', 'white']

# ======================================================================
# CLASS: PPSDesignGenerator
# ======================================================================

class PPSDesignGenerator:
    """Generates CSV design files for Peripersonal Space (PPS) experiments."""
    
    def __init__(self, config=None):
        self.config = config or CONFIG
        
        # Prepare output directory
        self.experiment_log_dir = self.config['paths']['experiment_log_dir']
        os.makedirs(self.experiment_log_dir, exist_ok=True)
        
        # Calculate trial counts
        self.trial_counts = self._calculate_trial_counts()
        self.conditions = [cond for cond, inc in self.config['included_conditions'].items() if inc]
        
        # Set seed for reproducibility
        self.base_seed = 42
        
        if self.config['debug_mode']:
            print("Initialized PPSDesignGenerator with:")
            print(f"  - Repetitions: {self.config['repetitions']}")
            print(f"  - Conditions: {self.conditions}")
            print(f"  - SOA values: {self.config['soa_conditions_ms']}")
            print(f"  - Total trials per participant: {sum(self.trial_counts.values())}")
            print(f"  - Output directory: {self.experiment_log_dir}")
    
    def _calculate_trial_counts(self):
        """Calculate how many trials for each condition (inhale/exhale/baseline) plus catch."""
        reps = self.config['repetitions']
        soa_count = len(self.config['soa_conditions_ms'])
        stimulus_types = LOOMING_STIMULI
        catch_percentage = self.config['catch_trial_percentage']
        
        trial_counts = {}
        for condition, include in self.config['included_conditions'].items():
            if include:
                # (# noise types) × (# SOA) × (reps)
                trial_counts[condition] = len(stimulus_types) * soa_count * reps
        
        regular_trials = sum(trial_counts.values())
        catch_trials = int(np.ceil(regular_trials * catch_percentage / 100))
        trial_counts['catch'] = catch_trials
        
        if self.config['debug_mode']:
            print("Trial count calculation:")
            for cond, count in trial_counts.items():
                print(f"  {cond}: {count} trials")
            print(f"  Total: {sum(trial_counts.values())} trials")
        
        return trial_counts
    
    def generate_breath_timestamps(self):
        """
        Generate timestamps for breath cycles at regular 8-second intervals.
        Each timestamp represents a point where a stimulus could be inserted.
        IMPORTANT: The breath phase (inhale/exhale) strictly alternates as this
        is a fixed physiological cycle that cannot be reordered.
        
        Returns:
            DataFrame with columns: timestamp, milliseconds, sample_index, breathphase
        """
        # Calculate parameters
        total_trials = sum(self.trial_counts.values())
        cycle_sec = self.config['box_breathing']['cycle_duration_sec']
        intro_sec = self.config['box_breathing']['intro_duration_sec']
        outro_sec = self.config['box_breathing']['outro_duration_sec']
        sample_rate = self.config['box_breathing']['sample_rate']
        
        # We need enough timestamps for all trials with intro/outro period
        total_duration_sec = intro_sec + (total_trials * cycle_sec) + outro_sec
        
        # Generate timestamps at regular intervals
        timestamps = []
        
        # Start with intro period
        current_time_sec = intro_sec  # First trial starts after intro
        
        # Always start with inhale (physiologically correct)
        # Strict alternation: inhale, exhale, inhale, exhale...
        for i in range(total_trials):
            # Convert time to minutes:seconds format
            minutes = int(current_time_sec / 60)
            seconds = current_time_sec % 60
            timestamp = f"{minutes:02}:{seconds:.1f}"
            
            # Calculate milliseconds and sample index
            milliseconds = current_time_sec * 1000
            sample_index = int(current_time_sec * sample_rate)
            
            # Strict alternation of breath phases - this cannot be randomized
            # In box breathing, phases must follow the natural breath cycle
            breathphase = 'inhale' if i % 2 == 0 else 'exhale'
            
            # Add this timestamp
            timestamps.append({
                'timestamp': timestamp,
                'milliseconds': milliseconds,
                'sample_index': sample_index,
                'breathphase': breathphase,
                'trial_index': i
            })
            
            # Advance to next cycle
            current_time_sec += cycle_sec
        
        # Convert to DataFrame
        timestamps_df = pd.DataFrame(timestamps)
        
        if self.config['debug_mode']:
            print(f"Generated {len(timestamps_df)} breath timestamps")
            print(f"First timestamp: {timestamps_df['timestamp'].iloc[0]}")
            print(f"Last timestamp: {timestamps_df['timestamp'].iloc[-1]}")
            print(f"Total audio duration: {total_duration_sec} seconds")
            
            # Check breath phase alternation
            phases = timestamps_df['breathphase'].tolist()
            print(f"Breath phases: {phases[:10]}... (alternating inhale-exhale pattern)")
        
        return timestamps_df
    
    def generate_counterbalanced_design(self, participant_id):
        """
        Generate a counterbalanced design for one participant.
        """
        # Set seed based on participant ID for reproducibility
        random.seed(self.base_seed + participant_id)
        np.random.seed(self.base_seed + participant_id)
        
        soa_conditions = self.config['soa_conditions_ms']
        jitter_options = self.config['jitter_options_ms']
        stimulus_types = LOOMING_STIMULI
        
        all_trials = []
        
        # Build the main conditions (inhale, exhale, baseline)
        for condition, count in self.trial_counts.items():
            if condition == 'catch' or count == 0:
                continue
            
            # Create all combos of (noise type × SOA)
            combo_list = []
            
            # Generate balanced combinations for each stimulus type and SOA
            for stim_type in stimulus_types:
                for soa in soa_conditions:
                    for _ in range(self.config['repetitions']):
                        combo_list.append((stim_type, soa))
            
            # Shuffle the combinations
            random.shuffle(combo_list)
            
            # Create trial records
            for stim_type, soa in combo_list:
                all_trials.append({
                    'participant_id': participant_id,
                    'trial_type': condition,  # inhalation, exhalation, baseline
                    'stimulus_type': stim_type,
                    'soa_value_ms': soa,
                    'jitter_ms': random.choice(jitter_options),
                    'is_tactile': True  # baseline & main trials have tactile
                })
        
        # Add catch trials (looming only, no tactile)
        catch_count = self.trial_counts.get('catch', 0)
        if catch_count > 0:
            catch_combos = []
            
            # Ensure each stimulus type and SOA appears equally in catch trials
            stim_counts = {stim: 0 for stim in stimulus_types}
            soa_counts = {soa: 0 for soa in soa_conditions}
            
            while len(catch_combos) < catch_count:
                # Find stimulus type with fewest occurrences
                min_stim = min(stim_counts, key=stim_counts.get)
                # Find SOA with fewest occurrences
                min_soa = min(soa_counts, key=soa_counts.get)
                
                catch_combos.append((min_stim, min_soa))
                stim_counts[min_stim] += 1
                soa_counts[min_soa] += 1
            
            # Create catch trial entries
            for stim_type, soa in catch_combos:
                all_trials.append({
                    'participant_id': participant_id,
                    'trial_type': 'catch',
                    'stimulus_type': stim_type,
                    'soa_value_ms': soa,
                    'jitter_ms': random.choice(jitter_options),
                    'is_tactile': False
                })
        
        # Convert to DataFrame and shuffle
        design_df = pd.DataFrame(all_trials)
        design_df = design_df.sample(frac=1, random_state=participant_id*100).reset_index(drop=True)
        
        # Add a trial_number
        design_df['trial_number'] = range(1, len(design_df) + 1)
        
        # Check distributions for proper counterbalancing
        if self.config['debug_mode']:
            print("\nTrial type distribution:")
            print(design_df['trial_type'].value_counts())
            
            print("\nStimulus type distribution by trial type:")
            for trial_type in design_df['trial_type'].unique():
                if trial_type == 'baseline':
                    continue  # baseline has no looming stimuli
                subset = design_df[design_df['trial_type'] == trial_type]
                print(f"  {trial_type}: {subset['stimulus_type'].value_counts().to_dict()}")
            
            print("\nSOA distribution by trial type:")
            for trial_type in design_df['trial_type'].unique():
                if trial_type == 'baseline':
                    continue  # baseline has no SOA relationship
                subset = design_df[design_df['trial_type'] == trial_type]
                print(f"  {trial_type}: {subset['soa_value_ms'].value_counts().to_dict()}")
        
        return design_df
    
    def assign_breath_holds(self, design_df):
        """
        Assign trial types to fixed breath phases (inhale/exhale cycles).
        
        IMPORTANT: Unlike the previous implementation, we don't choose which breath
        phase to assign a trial to - rather, we choose which trial type goes into
        each predefined breath phase slot. The breath phases are fixed in their
        alternating pattern.
        """
        # Generate timestamps with fixed alternating breath phases
        timestamps_df = self.generate_breath_timestamps()
        
        # Split timestamp dataframe by breath phase
        inhale_timestamps = timestamps_df[timestamps_df['breathphase'] == 'inhale'].copy()
        exhale_timestamps = timestamps_df[timestamps_df['breathphase'] == 'exhale'].copy()
        
        # Split trials by preferred breath phase
        inhalation_trials = design_df[design_df['trial_type'] == 'inhalation'].copy()
        exhalation_trials = design_df[design_df['trial_type'] == 'exhalation'].copy()
        other_trials = design_df[~design_df['trial_type'].isin(['inhalation', 'exhalation'])].copy()
        
        # Shuffle the trials to randomize their order
        inhalation_trials = inhalation_trials.sample(frac=1).reset_index(drop=True)
        exhalation_trials = exhalation_trials.sample(frac=1).reset_index(drop=True)
        other_trials = other_trials.sample(frac=1).reset_index(drop=True)
        
        # Check if we have enough timestamps of each type
        if len(inhale_timestamps) < len(inhalation_trials):
            print(f"Warning: Not enough inhale phases ({len(inhale_timestamps)}) for all inhalation trials ({len(inhalation_trials)})")
        
        if len(exhale_timestamps) < len(exhalation_trials):
            print(f"Warning: Not enough exhale phases ({len(exhale_timestamps)}) for all exhalation trials ({len(exhalation_trials)})")
        
        # Prepare result dataframe
        result_rows = []
        
        # First, assign inhalation trials to inhale phases (as many as possible)
        used_inhale_indices = []
        for i, trial in enumerate(inhalation_trials.itertuples()):
            if i < len(inhale_timestamps):
                timestamp = inhale_timestamps.iloc[i]
                trial_dict = trial._asdict()
                trial_dict.update({
                    'breathphase': 'inhale',
                    'milliseconds': timestamp['milliseconds'],
                    'timestamp_original': timestamp['timestamp'],
                    'sample_index': timestamp['sample_index'],
                    'breath_cycle_index': timestamp['trial_index']
                })
                result_rows.append(trial_dict)
                used_inhale_indices.append(i)
            else:
                break  # Out of inhale phases
        
        # Then, assign exhalation trials to exhale phases (as many as possible)
        used_exhale_indices = []
        for i, trial in enumerate(exhalation_trials.itertuples()):
            if i < len(exhale_timestamps):
                timestamp = exhale_timestamps.iloc[i]
                trial_dict = trial._asdict()
                trial_dict.update({
                    'breathphase': 'exhale',
                    'milliseconds': timestamp['milliseconds'],
                    'timestamp_original': timestamp['timestamp'],
                    'sample_index': timestamp['sample_index'],
                    'breath_cycle_index': timestamp['trial_index']
                })
                result_rows.append(trial_dict)
                used_exhale_indices.append(i)
            else:
                break  # Out of exhale phases
        
        # Collect remaining trials that couldn't get their preferred breath phase
        remaining_inhalation = [row for i, row in enumerate(inhalation_trials.itertuples()) 
                               if i >= len(inhale_timestamps)]
        remaining_exhalation = [row for i, row in enumerate(exhalation_trials.itertuples()) 
                                if i >= len(exhale_timestamps)]
        all_remaining = remaining_inhalation + remaining_exhalation + list(other_trials.itertuples())
        random.shuffle(all_remaining)  # Mix them up
        
        # Assign remaining trials to any available timestamps
        unused_inhale = [timestamp for i, timestamp in enumerate(inhale_timestamps.itertuples()) 
                         if i >= len(used_inhale_indices)]
        unused_exhale = [timestamp for i, timestamp in enumerate(exhale_timestamps.itertuples()) 
                        if i >= len(used_exhale_indices)]
        
        # Combine all unused timestamps in their natural order
        all_unused = []
        for timestamp in timestamps_df.itertuples():
            is_inhale = timestamp.breathphase == 'inhale'
            inhale_idx = timestamp.trial_index // 2  # Convert to index within inhale/exhale group
            exhale_idx = timestamp.trial_index // 2
            
            if (is_inhale and inhale_idx >= len(used_inhale_indices)) or \
               (not is_inhale and exhale_idx >= len(used_exhale_indices)):
                all_unused.append(timestamp)
        
        # Sort by trial_index to maintain the natural breath cycle order
        all_unused.sort(key=lambda x: x.trial_index)
        
        # Assign remaining trials to the available timestamps
        for i, trial in enumerate(all_remaining):
            if i < len(all_unused):
                timestamp = all_unused[i]
                trial_dict = trial._asdict()
                trial_dict.update({
                    'breathphase': timestamp.breathphase,
                    'milliseconds': timestamp.milliseconds,
                    'timestamp_original': timestamp.timestamp,
                    'sample_index': timestamp.sample_index,
                    'breath_cycle_index': timestamp.trial_index
                })
                result_rows.append(trial_dict)
            else:
                raise ValueError(f"Not enough breath cycles for all trials! Need {len(all_remaining) - len(all_unused)} more.")
        
        # Convert to DataFrame
        design_with_timestamps = pd.DataFrame(result_rows)
        
        # Sort by the breath cycle index to maintain the natural order of breaths
        design_with_timestamps = design_with_timestamps.sort_values(by='breath_cycle_index').reset_index(drop=True)
        
        # Calculate jittered_ms = base + jitter
        design_with_timestamps['jittered_ms'] = design_with_timestamps['milliseconds'] + design_with_timestamps['jitter_ms']
        
        # Format jittered timestamp
        def ms_to_timestamp(ms):
            if pd.isna(ms):
                return None
            m = int(ms // 60000)
            s = (ms % 60000) / 1000.0
            return f"{m:02}:{s:.1f}"
        
        design_with_timestamps['timestamp_after_jitter'] = design_with_timestamps['jittered_ms'].apply(ms_to_timestamp)
        
        # For non-catch trials, add tactile timestamp (original + jitter + SOA)
        design_with_timestamps['soa_ms'] = None
        design_with_timestamps['timestamp_with_soa'] = None
        
        non_catch_mask = design_with_timestamps['trial_type'] != 'catch'
        design_with_timestamps.loc[non_catch_mask, 'soa_ms'] = (
            design_with_timestamps.loc[non_catch_mask, 'jittered_ms'] + 
            design_with_timestamps.loc[non_catch_mask, 'soa_value_ms']
        )
        
        design_with_timestamps.loc[non_catch_mask, 'timestamp_with_soa'] = (
            design_with_timestamps.loc[non_catch_mask, 'soa_ms'].apply(ms_to_timestamp)
        )
        
        if self.config['debug_mode']:
            # Verify the breathphase alternation is maintained
            phases = design_with_timestamps.sort_values(by='breath_cycle_index')['breathphase'].tolist()
            alternating = True
            for i in range(1, len(phases)):
                if phases[i] == phases[i-1]:
                    alternating = False
                    break
            
            if alternating:
                print("✓ Breath phases maintain perfect alternation (inhale-exhale-inhale-...)")
            else:
                print("⚠ WARNING: Breath phases do not alternate correctly!")
            
            # Show trial type distribution across breath phases
            print("\nTrial Type Distribution in Breath Phases:")
            pivot = pd.crosstab(design_with_timestamps['trial_type'], design_with_timestamps['breathphase'])
            print(pivot)
        
        return design_with_timestamps
        # Assign remaining trials to the available timestamps
        for i, trial in enumerate(all_remaining):
            if i < len(all_unused):
                timestamp = all_unused[i]
                trial_dict = trial._asdict()
                trial_dict.update({
                    'breathphase': timestamp.breathphase,
                    'milliseconds': timestamp.milliseconds,
                    'timestamp_original': timestamp.timestamp,
                    'sample_index': timestamp.sample_index,
                    'breath_cycle_index': timestamp.trial_index
                })
                result_rows.append(trial_dict)
            else:
                raise ValueError(f"Not enough breath cycles for all trials! Need {len(all_remaining) - len(all_unused)} more.")
        
        # Convert to DataFrame
        design_with_timestamps = pd.DataFrame(result_rows)
        
        # Sort by the breath cycle index to maintain the natural order of breaths
        design_with_timestamps = design_with_timestamps.sort_values(by='breath_cycle_index').reset_index(drop=True)
        
        # Calculate jittered_ms = base + jitter
        design_with_timestamps['jittered_ms'] = design_with_timestamps['milliseconds'] + design_with_timestamps['jitter_ms']
        
        # Calculate jittered_ms = base + jitter
        design_with_timestamps['jittered_ms'] = design_with_timestamps['milliseconds'] + design_with_timestamps['jitter_ms']
        
        # Format jittered timestamp
        def ms_to_timestamp(ms):
            m = int(ms // 60000)
            s = (ms % 60000) / 1000.0
            return f"{m:02}:{s:.1f}"
        
        design_with_timestamps['timestamp_after_jitter'] = design_with_timestamps['jittered_ms'].apply(ms_to_timestamp)
        
        # For non-catch trials, add tactile timestamp (original + jitter + SOA)
        design_with_timestamps['soa_ms'] = None
        design_with_timestamps['timestamp_with_soa'] = None
        
        non_catch_mask = design_with_timestamps['trial_type'] != 'catch'
        design_with_timestamps.loc[non_catch_mask, 'soa_ms'] = (
            design_with_timestamps.loc[non_catch_mask, 'jittered_ms'] + 
            design_with_timestamps.loc[non_catch_mask, 'soa_value_ms']
        )
        
        design_with_timestamps.loc[non_catch_mask, 'timestamp_with_soa'] = (
            design_with_timestamps.loc[non_catch_mask, 'soa_ms'].apply(ms_to_timestamp)
        )
        
        return design_with_timestamps
    
    def finalize_design_csv(self, design_df):
        """
        Finalize the design CSV with proper column names and special case handling.
        
        Args:
            design_df: DataFrame with raw design data including timestamps
            
        Returns:
            DataFrame with finalized column names and special case handling
        """
        # Rename columns
        design_df.rename(columns={
            'stimulus_type': 'looming_stimulus_type',
            'timestamp_original': 'retentionphase_timestamp',
            'timestamp_after_jitter': 'looming_stimulus_timestamp',
            'timestamp_with_soa': 'tactile_stimulus_timestamp'
        }, inplace=True)
        
        # Handle baseline trials (no looming stimulus)
        baseline_mask = (design_df['trial_type'] == 'baseline')
        design_df.loc[baseline_mask, 'looming_stimulus_type'] = 'none'
        design_df.loc[baseline_mask, 'looming_stimulus_timestamp'] = None
        
        # Handle catch trials (no tactile stimulus)
        catch_mask = (design_df['trial_type'] == 'catch')
        design_df.loc[catch_mask, 'tactile_stimulus_timestamp'] = None
        
        # Add expected response flag (true for all except catch)
        design_df['expected_response'] = ~design_df['trial_type'].isin(['catch'])
        

            'soa_value_ms',              # time between looming and tactile
            'jitter_ms',                 # random jitter added to base timestamp
            'sample_index',              # sample index of retention phase
            'is_tactile',                # whether trial has tactile component
            'expected_response'          # whether response is expected
        ]
        
        # Keep only columns that exist in the DataFrame
        final_cols = [col for col in final_cols if col in design_df.columns]
        design_df = design_df[final_cols]
        
        # Validate final design
        self._validate_design(design_df)
        
        return design_df
    
    def _validate_design(self, design_df):
        """Check if the final design meets counterbalancing criteria."""
        if not self.config['debug_mode']:
            return
            
        print("\n=== DESIGN VALIDATION ===")
        
        # 1. Check trial type distribution
        print("Trial type counts:")
        trial_counts = design_df['trial_type'].value_counts()
        print(trial_counts)
        
        # 2. Check stimulus type distribution within each trial type
        print("\nStimulus distribution by trial type:")
        for trial_type in design_df['trial_type'].unique():
            if trial_type == 'baseline':
                continue  # No looming in baseline
                
            type_df = design_df[design_df['trial_type'] == trial_type]
            stim_counts = type_df['looming_stimulus_type'].value_counts()
            print(f"  {trial_type}: {stim_counts.to_dict()}")
        
        # 3. Check SOA distribution within each trial type
        print("\nSOA distribution by trial type:")
        for trial_type in design_df['trial_type'].unique():
            if trial_type == 'baseline':
                continue  # No SOA relationship in baseline
                
            type_df = design_df[design_df['trial_type'] == trial_type]
            soa_counts = type_df['soa_value_ms'].value_counts()
            print(f"  {trial_type}: {soa_counts.to_dict()}")
        
        # 4. Check breath phase distribution within each trial type
        print("\nBreath phase distribution by trial type:")
        for trial_type in design_df['trial_type'].unique():
            type_df = design_df[design_df['trial_type'] == trial_type]
            phase_counts = type_df['breathphase'].value_counts()
            print(f"  {trial_type}: {phase_counts.to_dict()}")
        
        # 5. Check sequence properties
        trial_sequence = design_df['trial_type'].tolist()
        
        # Count runs of consecutive trial types
        consecutive_counts = {}
        current_type = trial_sequence[0]
        current_count = 1
        
        for t in trial_sequence[1:]:
            if t == current_type:
                current_count += 1
            else:
                # Record the run we just finished
                length_key = f"{current_type}_{current_count}"
                consecutive_counts[length_key] = consecutive_counts.get(length_key, 0) + 1
                # Reset for new type
                current_type = t
                current_count = 1
        
        # Record the final run
        length_key = f"{current_type}_{current_count}"
        consecutive_counts[length_key] = consecutive_counts.get(length_key, 0) + 1
        
        print("\nRuns of consecutive trials:")
        for key, count in sorted(consecutive_counts.items()):
            print(f"  {key}: {count}")
        
        # 6. Calculate transition probabilities between trial types
        transitions = {}
        for i in range(len(trial_sequence) - 1):
            transition = (trial_sequence[i], trial_sequence[i+1])
            transitions[transition] = transitions.get(transition, 0) + 1
        
        print("\nTransition counts between trial types:")
        for transition, count in sorted(transitions.items()):
            print(f"  {transition[0]} → {transition[1]}: {count}")
    
    def process_participant(self, participant_id):
        """
        Generate the design, assign breath holds, finalize columns,
        and save as a CSV for one participant.
        """
        try:
            if self.config['debug_mode']:
                print(f"\n=== Processing participant {participant_id} ===")
            
            # 1) Generate base design (counterbalanced combos)
            design_df = self.generate_counterbalanced_design(participant_id)
            
            # 2) Assign each trial to a breath hold (inhale or exhale)
            design_df = self.assign_breath_holds(design_df)
            
            # 3) Finalize columns (rename, set baseline/catch to NA, etc.)
            design_df = self.finalize_design_csv(design_df)
            
            # 4) Save CSV
            out_path = os.path.join(self.experiment_log_dir,
                                    f"participant_{participant_id}_design.csv")
            design_df.to_csv(out_path, index=False)
            
            if self.config['debug_mode']:
                print(f"Saved participant {participant_id} design to {out_path}")
            
            return True, out_path
        except Exception as e:
            print(f"Error processing participant {participant_id}: {e}")
            traceback.print_exc()
            return False, None
    
    def run(self, participant_ids=None, num_participants=None):
        """
        Generate the design CSVs for multiple participants.
        
        Args:
            participant_ids: List of specific participant IDs to process
            num_participants: Number of participants to generate if participant_ids not provided
            
        Returns:
            Dictionary mapping participant IDs to generation results
        """
        if participant_ids is None and num_participants is None:
            num_participants = self.config.get('num_participants', 5)
        
        if participant_ids is None:
            participant_ids = list(range(num_participants))
        
        if self.config['debug_mode']:
            print("=== Starting PPSDesignGenerator ===")
            print(f"Repetitions: {self.config['repetitions']}")
            print(f"Participants: {participant_ids}")
            print(f"Box breathing settings: {self.config['box_breathing']}")
        
        results = {}
        for pid in participant_ids:
            success, path = self.process_participant(pid)
            results[pid] = {
                'success': success,
                'csv_path': path
            }
        
        return results

# ======================================================================
# AUDIO GENERATION FUNCTIONS
# ======================================================================

def generate_audio_from_design(design_csv_path, output_dir, base_audio_path=None):
    """
    Generate audio files from a design CSV.
    
    This is a placeholder for audio generation functionality.
    The actual implementation would:
    1. Read the design CSV
    2. Create or load the base box breathing audio
    3. Insert looming and tactile stimuli at the specified timestamps
    4. Save the resulting audio files
    
    Args:
        design_csv_path: Path to the design CSV
        output_dir: Directory to save audio files
        base_audio_path: Optional path to base box breathing audio
        
    Returns:
        Tuple of (looming_audio_path, tactile_audio_path)
    """
    # This is just a placeholder - implement actual audio generation here
    print(f"Audio generation from {design_csv_path} not implemented yet")
    return None, None

# ======================================================================
# MAIN FUNCTION
# ======================================================================

def main():
    """Main function to run the design generator."""
    # Create the design generator
    generator = PPSDesignGenerator(CONFIG)
    
    # Run for all participants defined in CONFIG
    results = generator.run()
    
    # Print summary
    print("\n=== Generation Summary ===")
    for pid, info in results.items():
        if info['success']:
            print(f"Participant {pid}: CSV generated at {info['csv_path']}")
        else:
            print(f"Participant {pid}: ERROR - no CSV generated.")
    
    print("\nTo generate audio files from these designs, implement the audio generation function.")

if __name__ == "__main__":
    main()

IndentationError: unexpected indent (630239049.py, line 529)

In [4]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
PPS Design Generator

This script generates experimental design files for a Peripersonal Space (PPS) experiment.
It creates CSV files with precise timing information for audio stimuli, using regular
8-second intervals for box breathing cycles.

Author: AI Assistant
"""

import os
import numpy as np
import pandas as pd
import random
import json
from itertools import product
from datetime import datetime
import traceback

# ======================================================================
# CONFIGURATION PARAMETERS
# ======================================================================

CONFIG = {
    'repetitions': 4,            # how many times each condition × SOA is repeated
    'num_participants': 5,       # default number of participants
    'included_conditions': {
        'inhalation': True,
        'exhalation': True,
        'baseline': True,
    },
    'catch_trial_percentage': 10,
    'soa_conditions_ms': [190, 400, 700, 1000, 1500],
    'jitter_options_ms': [100, 200, 300, 400, 500],
    'debug_mode': True,
    'paths': {
        'experiment_log_dir': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog",
    },
    'box_breathing': {
        'cycle_duration_sec': 8,      # Total duration of one breath cycle in seconds
        'intro_duration_sec': 60,     # Initial settling period before trials begin
        'outro_duration_sec': 30,     # Closing period after trials end
        'alternating_phases': True,   # Alternate between inhale and exhale phases
        'sample_rate': 48000,         # Audio sample rate
    }
}

# Noise types for looming (these can be any labels you want)
LOOMING_STIMULI = ['blue', 'brown', 'pink', 'white']

# ======================================================================
# CLASS: PPSDesignGenerator
# ======================================================================

class PPSDesignGenerator:
    """Generates CSV design files for Peripersonal Space (PPS) experiments."""
    
    def __init__(self, config=None):
        self.config = config or CONFIG
        
        # Prepare output directory
        self.experiment_log_dir = self.config['paths']['experiment_log_dir']
        os.makedirs(self.experiment_log_dir, exist_ok=True)
        
        # Calculate trial counts
        self.trial_counts = self._calculate_trial_counts()
        self.conditions = [cond for cond, inc in self.config['included_conditions'].items() if inc]
        
        # Set seed for reproducibility
        self.base_seed = 42
        
        if self.config['debug_mode']:
            print("Initialized PPSDesignGenerator with:")
            print(f"  - Repetitions: {self.config['repetitions']}")
            print(f"  - Conditions: {self.conditions}")
            print(f"  - SOA values: {self.config['soa_conditions_ms']}")
            print(f"  - Total trials per participant: {sum(self.trial_counts.values())}")
            print(f"  - Output directory: {self.experiment_log_dir}")
    
    def _calculate_trial_counts(self):
        """Calculate how many trials for each condition (inhale/exhale/baseline) plus catch."""
        reps = self.config['repetitions']
        soa_count = len(self.config['soa_conditions_ms'])
        stimulus_types = LOOMING_STIMULI
        catch_percentage = self.config['catch_trial_percentage']
        
        trial_counts = {}
        for condition, include in self.config['included_conditions'].items():
            if include:
                # (# noise types) × (# SOA) × (reps)
                trial_counts[condition] = len(stimulus_types) * soa_count * reps
        
        regular_trials = sum(trial_counts.values())
        catch_trials = int(np.ceil(regular_trials * catch_percentage / 100))
        trial_counts['catch'] = catch_trials
        
        if self.config['debug_mode']:
            print("Trial count calculation:")
            for cond, count in trial_counts.items():
                print(f"  {cond}: {count} trials")
            print(f"  Total: {sum(trial_counts.values())} trials")
        
        return trial_counts
    
    def generate_breath_timestamps(self):
        """
        Generate timestamps for breath cycles at regular 8-second intervals.
        Each timestamp represents a point where a stimulus could be inserted.
        IMPORTANT: The breath phase (inhale/exhale) strictly alternates as this
        is a fixed physiological cycle that cannot be reordered.
        
        Returns:
            DataFrame with columns: timestamp, milliseconds, sample_index, breathphase
        """
        # Calculate parameters
        total_trials = sum(self.trial_counts.values())
        cycle_sec = self.config['box_breathing']['cycle_duration_sec']
        intro_sec = self.config['box_breathing']['intro_duration_sec']
        outro_sec = self.config['box_breathing']['outro_duration_sec']
        sample_rate = self.config['box_breathing']['sample_rate']
        
        # We need enough timestamps for all trials with intro/outro period
        total_duration_sec = intro_sec + (total_trials * cycle_sec) + outro_sec
        
        # Generate timestamps at regular intervals
        timestamps = []
        
        # Start with intro period
        current_time_sec = intro_sec  # First trial starts after intro
        
        # Always start with inhale (physiologically correct)
        # Strict alternation: inhale, exhale, inhale, exhale...
        for i in range(total_trials):
            # Convert time to minutes:seconds format
            minutes = int(current_time_sec / 60)
            seconds = current_time_sec % 60
            timestamp = f"{minutes:02}:{seconds:.1f}"
            
            # Calculate milliseconds and sample index
            milliseconds = current_time_sec * 1000
            sample_index = int(current_time_sec * sample_rate)
            
            # Strict alternation of breath phases - this cannot be randomized
            # In box breathing, phases must follow the natural breath cycle
            breathphase = 'inhale' if i % 2 == 0 else 'exhale'
            
            # Add this timestamp
            timestamps.append({
                'timestamp': timestamp,
                'milliseconds': milliseconds,
                'sample_index': sample_index,
                'breathphase': breathphase,
                'trial_index': i
            })
            
            # Advance to next cycle
            current_time_sec += cycle_sec
        
        # Convert to DataFrame
        timestamps_df = pd.DataFrame(timestamps)
        
        if self.config['debug_mode']:
            print(f"Generated {len(timestamps_df)} breath timestamps")
            print(f"First timestamp: {timestamps_df['timestamp'].iloc[0]}")
            print(f"Last timestamp: {timestamps_df['timestamp'].iloc[-1]}")
            print(f"Total audio duration: {total_duration_sec} seconds")
            
            # Check breath phase alternation
            phases = timestamps_df['breathphase'].tolist()
            print(f"Breath phases: {phases[:10]}... (alternating inhale-exhale pattern)")
        
        return timestamps_df
    
    def generate_counterbalanced_design(self, participant_id):
        """
        Generate a counterbalanced design for one participant.
        """
        # Set seed based on participant ID for reproducibility
        random.seed(self.base_seed + participant_id)
        np.random.seed(self.base_seed + participant_id)
        
        soa_conditions = self.config['soa_conditions_ms']
        jitter_options = self.config['jitter_options_ms']
        stimulus_types = LOOMING_STIMULI
        
        all_trials = []
        
        # Build the main conditions (inhale, exhale, baseline)
        for condition, count in self.trial_counts.items():
            if condition == 'catch' or count == 0:
                continue
            
            # Create all combos of (noise type × SOA)
            combo_list = []
            
            # Generate balanced combinations for each stimulus type and SOA
            for stim_type in stimulus_types:
                for soa in soa_conditions:
                    for _ in range(self.config['repetitions']):
                        combo_list.append((stim_type, soa))
            
            # Shuffle the combinations
            random.shuffle(combo_list)
            
            # Create trial records
            for stim_type, soa in combo_list:
                all_trials.append({
                    'participant_id': participant_id,
                    'trial_type': condition,  # inhalation, exhalation, baseline
                    'stimulus_type': stim_type,
                    'soa_value_ms': soa,
                    'jitter_ms': random.choice(jitter_options),
                    'is_tactile': True  # baseline & main trials have tactile
                })
        
        # Add catch trials (looming only, no tactile)
        catch_count = self.trial_counts.get('catch', 0)
        if catch_count > 0:
            catch_combos = []
            
            # Ensure each stimulus type and SOA appears equally in catch trials
            stim_counts = {stim: 0 for stim in stimulus_types}
            soa_counts = {soa: 0 for soa in soa_conditions}
            
            while len(catch_combos) < catch_count:
                # Find stimulus type with fewest occurrences
                min_stim = min(stim_counts, key=stim_counts.get)
                # Find SOA with fewest occurrences
                min_soa = min(soa_counts, key=soa_counts.get)
                
                catch_combos.append((min_stim, min_soa))
                stim_counts[min_stim] += 1
                soa_counts[min_soa] += 1
            
            # Create catch trial entries
            for stim_type, soa in catch_combos:
                all_trials.append({
                    'participant_id': participant_id,
                    'trial_type': 'catch',
                    'stimulus_type': stim_type,
                    'soa_value_ms': soa,
                    'jitter_ms': random.choice(jitter_options),
                    'is_tactile': False
                })
        
        # Convert to DataFrame and shuffle
        design_df = pd.DataFrame(all_trials)
        design_df = design_df.sample(frac=1, random_state=participant_id*100).reset_index(drop=True)
        
        # Add a trial_number
        design_df['trial_number'] = range(1, len(design_df) + 1)
        
        # Check distributions for proper counterbalancing
        if self.config['debug_mode']:
            print("\nTrial type distribution:")
            print(design_df['trial_type'].value_counts())
            
            print("\nStimulus type distribution by trial type:")
            for trial_type in design_df['trial_type'].unique():
                if trial_type == 'baseline':
                    continue  # baseline has no looming stimuli
                subset = design_df[design_df['trial_type'] == trial_type]
                print(f"  {trial_type}: {subset['stimulus_type'].value_counts().to_dict()}")
            
            print("\nSOA distribution by trial type:")
            for trial_type in design_df['trial_type'].unique():
                if trial_type == 'baseline':
                    continue  # baseline has no SOA relationship
                subset = design_df[design_df['trial_type'] == trial_type]
                print(f"  {trial_type}: {subset['soa_value_ms'].value_counts().to_dict()}")
        
        return design_df
    
    def assign_breath_holds(self, design_df):
        """
        Assign trial types to fixed breath phases (inhale/exhale cycles).
        
        IMPORTANT: Unlike the previous implementation, we don't choose which breath
        phase to assign a trial to - rather, we choose which trial type goes into
        each predefined breath phase slot. The breath phases are fixed in their
        alternating pattern.
        """
        # Generate timestamps with fixed alternating breath phases
        timestamps_df = self.generate_breath_timestamps()
        
        # Split timestamp dataframe by breath phase
        inhale_timestamps = timestamps_df[timestamps_df['breathphase'] == 'inhale'].copy()
        exhale_timestamps = timestamps_df[timestamps_df['breathphase'] == 'exhale'].copy()
        
        # Split trials by preferred breath phase
        inhalation_trials = design_df[design_df['trial_type'] == 'inhalation'].copy()
        exhalation_trials = design_df[design_df['trial_type'] == 'exhalation'].copy()
        other_trials = design_df[~design_df['trial_type'].isin(['inhalation', 'exhalation'])].copy()
        
        # Shuffle the trials to randomize their order
        inhalation_trials = inhalation_trials.sample(frac=1).reset_index(drop=True)
        exhalation_trials = exhalation_trials.sample(frac=1).reset_index(drop=True)
        other_trials = other_trials.sample(frac=1).reset_index(drop=True)
        
        # Check if we have enough timestamps of each type
        if len(inhale_timestamps) < len(inhalation_trials):
            print(f"Warning: Not enough inhale phases ({len(inhale_timestamps)}) for all inhalation trials ({len(inhalation_trials)})")
        
        if len(exhale_timestamps) < len(exhalation_trials):
            print(f"Warning: Not enough exhale phases ({len(exhale_timestamps)}) for all exhalation trials ({len(exhalation_trials)})")
        
        # Prepare result dataframe
        result_rows = []
        
        # First, assign inhalation trials to inhale phases (as many as possible)
        used_inhale_indices = []
        for i, trial in enumerate(inhalation_trials.itertuples()):
            if i < len(inhale_timestamps):
                timestamp = inhale_timestamps.iloc[i]
                trial_dict = trial._asdict()
                trial_dict.update({
                    'breathphase': 'inhale',
                    'milliseconds': timestamp['milliseconds'],
                    'timestamp_original': timestamp['timestamp'],
                    'sample_index': timestamp['sample_index'],
                    'breath_cycle_index': timestamp['trial_index']
                })
                result_rows.append(trial_dict)
                used_inhale_indices.append(i)
            else:
                break  # Out of inhale phases
        
        # Then, assign exhalation trials to exhale phases (as many as possible)
        used_exhale_indices = []
        for i, trial in enumerate(exhalation_trials.itertuples()):
            if i < len(exhale_timestamps):
                timestamp = exhale_timestamps.iloc[i]
                trial_dict = trial._asdict()
                trial_dict.update({
                    'breathphase': 'exhale',
                    'milliseconds': timestamp['milliseconds'],
                    'timestamp_original': timestamp['timestamp'],
                    'sample_index': timestamp['sample_index'],
                    'breath_cycle_index': timestamp['trial_index']
                })
                result_rows.append(trial_dict)
                used_exhale_indices.append(i)
            else:
                break  # Out of exhale phases
        
        # Collect remaining trials that couldn't get their preferred breath phase
        remaining_inhalation = [row for i, row in enumerate(inhalation_trials.itertuples()) 
                               if i >= len(inhale_timestamps)]
        remaining_exhalation = [row for i, row in enumerate(exhalation_trials.itertuples()) 
                                if i >= len(exhale_timestamps)]
        all_remaining = remaining_inhalation + remaining_exhalation + list(other_trials.itertuples())
        random.shuffle(all_remaining)  # Mix them up
        
        # Assign remaining trials to any available timestamps
        unused_inhale = [timestamp for i, timestamp in enumerate(inhale_timestamps.itertuples()) 
                         if i >= len(used_inhale_indices)]
        unused_exhale = [timestamp for i, timestamp in enumerate(exhale_timestamps.itertuples()) 
                        if i >= len(used_exhale_indices)]
        
        # Combine all unused timestamps in their natural order
        all_unused = []
        for timestamp in timestamps_df.itertuples():
            is_inhale = timestamp.breathphase == 'inhale'
            inhale_idx = timestamp.trial_index // 2  # Convert to index within inhale/exhale group
            exhale_idx = timestamp.trial_index // 2
            
            if (is_inhale and inhale_idx >= len(used_inhale_indices)) or \
               (not is_inhale and exhale_idx >= len(used_exhale_indices)):
                all_unused.append(timestamp)
        
        # Sort by trial_index to maintain the natural breath cycle order
        all_unused.sort(key=lambda x: x.trial_index)
        
        # Assign remaining trials to the available timestamps
        for i, trial in enumerate(all_remaining):
            if i < len(all_unused):
                timestamp = all_unused[i]
                trial_dict = trial._asdict()
                trial_dict.update({
                    'breathphase': timestamp.breathphase,
                    'milliseconds': timestamp.milliseconds,
                    'timestamp_original': timestamp.timestamp,
                    'sample_index': timestamp.sample_index,
                    'breath_cycle_index': timestamp.trial_index
                })
                result_rows.append(trial_dict)
            else:
                raise ValueError(f"Not enough breath cycles for all trials! Need {len(all_remaining) - len(all_unused)} more.")
        
        # Convert to DataFrame
        design_with_timestamps = pd.DataFrame(result_rows)
        
        # Sort by the breath cycle index to maintain the natural order of breaths
        design_with_timestamps = design_with_timestamps.sort_values(by='breath_cycle_index').reset_index(drop=True)
        
        # Calculate jittered_ms = base + jitter
        design_with_timestamps['jittered_ms'] = design_with_timestamps['milliseconds'] + design_with_timestamps['jitter_ms']
        
        # Format jittered timestamp
        def ms_to_timestamp(ms):
            if pd.isna(ms):
                return None
            m = int(ms // 60000)
            s = (ms % 60000) / 1000.0
            return f"{m:02}:{s:.1f}"
        
        design_with_timestamps['timestamp_after_jitter'] = design_with_timestamps['jittered_ms'].apply(ms_to_timestamp)
        
        # For non-catch trials, add tactile timestamp (original + jitter + SOA)
        design_with_timestamps['soa_ms'] = None
        design_with_timestamps['timestamp_with_soa'] = None
        
        non_catch_mask = design_with_timestamps['trial_type'] != 'catch'
        design_with_timestamps.loc[non_catch_mask, 'soa_ms'] = (
            design_with_timestamps.loc[non_catch_mask, 'jittered_ms'] + 
            design_with_timestamps.loc[non_catch_mask, 'soa_value_ms']
        )
        
        design_with_timestamps.loc[non_catch_mask, 'timestamp_with_soa'] = (
            design_with_timestamps.loc[non_catch_mask, 'soa_ms'].apply(ms_to_timestamp)
        )
        
        if self.config['debug_mode']:
            # Verify the breathphase alternation is maintained
            phases = design_with_timestamps.sort_values(by='breath_cycle_index')['breathphase'].tolist()
            alternating = True
            for i in range(1, len(phases)):
                if phases[i] == phases[i-1]:
                    alternating = False
                    break
            
            if alternating:
                print("✓ Breath phases maintain perfect alternation (inhale-exhale-inhale-...)")
            else:
                print("⚠ WARNING: Breath phases do not alternate correctly!")
            
            # Show trial type distribution across breath phases
            print("\nTrial Type Distribution in Breath Phases:")
            pivot = pd.crosstab(design_with_timestamps['trial_type'], design_with_timestamps['breathphase'])
            print(pivot)
        
        return design_with_timestamps
        # Assign remaining trials to the available timestamps
        for i, trial in enumerate(all_remaining):
            if i < len(all_unused):
                timestamp = all_unused[i]
                trial_dict = trial._asdict()
                trial_dict.update({
                    'breathphase': timestamp.breathphase,
                    'milliseconds': timestamp.milliseconds,
                    'timestamp_original': timestamp.timestamp,
                    'sample_index': timestamp.sample_index,
                    'breath_cycle_index': timestamp.trial_index
                })
                result_rows.append(trial_dict)
            else:
                raise ValueError(f"Not enough breath cycles for all trials! Need {len(all_remaining) - len(all_unused)} more.")
        
        # Convert to DataFrame
        design_with_timestamps = pd.DataFrame(result_rows)
        
        # Sort by the breath cycle index to maintain the natural order of breaths
        design_with_timestamps = design_with_timestamps.sort_values(by='breath_cycle_index').reset_index(drop=True)
        
        # Calculate jittered_ms = base + jitter
        design_with_timestamps['jittered_ms'] = design_with_timestamps['milliseconds'] + design_with_timestamps['jitter_ms']
        
        # Calculate jittered_ms = base + jitter
        design_with_timestamps['jittered_ms'] = design_with_timestamps['milliseconds'] + design_with_timestamps['jitter_ms']
        
        # Format jittered timestamp
        def ms_to_timestamp(ms):
            m = int(ms // 60000)
            s = (ms % 60000) / 1000.0
            return f"{m:02}:{s:.1f}"
        
        design_with_timestamps['timestamp_after_jitter'] = design_with_timestamps['jittered_ms'].apply(ms_to_timestamp)
        
        # For non-catch trials, add tactile timestamp (original + jitter + SOA)
        design_with_timestamps['soa_ms'] = None
        design_with_timestamps['timestamp_with_soa'] = None
        
        non_catch_mask = design_with_timestamps['trial_type'] != 'catch'
        design_with_timestamps.loc[non_catch_mask, 'soa_ms'] = (
            design_with_timestamps.loc[non_catch_mask, 'jittered_ms'] + 
            design_with_timestamps.loc[non_catch_mask, 'soa_value_ms']
        )
        
        design_with_timestamps.loc[non_catch_mask, 'timestamp_with_soa'] = (
            design_with_timestamps.loc[non_catch_mask, 'soa_ms'].apply(ms_to_timestamp)
        )
        
        return design_with_timestamps
    
    def finalize_design_csv(self, design_df):
        """
        Finalize the design CSV with proper column names and special case handling.
        
        Args:
            design_df: DataFrame with raw design data including timestamps
            
        Returns:
            DataFrame with finalized column names and special case handling
        """
        # Rename columns
        design_df.rename(columns={
            'stimulus_type': 'looming_stimulus_type',
            'timestamp_original': 'retentionphase_timestamp',
            'timestamp_after_jitter': 'looming_stimulus_timestamp',
            'timestamp_with_soa': 'tactile_stimulus_timestamp'
        }, inplace=True)
        
        # Handle baseline trials (no looming stimulus)
        baseline_mask = (design_df['trial_type'] == 'baseline')
        design_df.loc[baseline_mask, 'looming_stimulus_type'] = 'none'
        design_df.loc[baseline_mask, 'looming_stimulus_timestamp'] = None
        
        # Handle catch trials (no tactile stimulus)
        catch_mask = (design_df['trial_type'] == 'catch')
        design_df.loc[catch_mask, 'tactile_stimulus_timestamp'] = None
        
        # Add expected response flag (true for all except catch)
        design_df['expected_response'] = ~design_df['trial_type'].isin(['catch'])
        

            'soa_value_ms',              # time between looming and tactile
            'jitter_ms',                 # random jitter added to base timestamp
            'sample_index',              # sample index of retention phase
            'is_tactile',                # whether trial has tactile component
            'expected_response'          # whether response is expected
        ]
        
        # Keep only columns that exist in the DataFrame
        final_cols = [col for col in final_cols if col in design_df.columns]
        design_df = design_df[final_cols]
        
        # Validate final design
        self._validate_design(design_df)
        
        return design_df
    
    def _validate_design(self, design_df):
        """Check if the final design meets counterbalancing criteria."""
        if not self.config['debug_mode']:
            return
            
        print("\n=== DESIGN VALIDATION ===")
        
        # 1. Check trial type distribution
        print("Trial type counts:")
        trial_counts = design_df['trial_type'].value_counts()
        print(trial_counts)
        
        # 2. Check stimulus type distribution within each trial type
        print("\nStimulus distribution by trial type:")
        for trial_type in design_df['trial_type'].unique():
            if trial_type == 'baseline':
                continue  # No looming in baseline
                
            type_df = design_df[design_df['trial_type'] == trial_type]
            stim_counts = type_df['looming_stimulus_type'].value_counts()
            print(f"  {trial_type}: {stim_counts.to_dict()}")
        
        # 3. Check SOA distribution within each trial type
        print("\nSOA distribution by trial type:")
        for trial_type in design_df['trial_type'].unique():
            if trial_type == 'baseline':
                continue  # No SOA relationship in baseline
                
            type_df = design_df[design_df['trial_type'] == trial_type]
            soa_counts = type_df['soa_value_ms'].value_counts()
            print(f"  {trial_type}: {soa_counts.to_dict()}")
        
        # 4. Check breath phase distribution within each trial type
        print("\nBreath phase distribution by trial type:")
        for trial_type in design_df['trial_type'].unique():
            type_df = design_df[design_df['trial_type'] == trial_type]
            phase_counts = type_df['breathphase'].value_counts()
            print(f"  {trial_type}: {phase_counts.to_dict()}")
        
        # 5. Check sequence properties
        trial_sequence = design_df['trial_type'].tolist()
        
        # Count runs of consecutive trial types
        consecutive_counts = {}
        current_type = trial_sequence[0]
        current_count = 1
        
        for t in trial_sequence[1:]:
            if t == current_type:
                current_count += 1
            else:
                # Record the run we just finished
                length_key = f"{current_type}_{current_count}"
                consecutive_counts[length_key] = consecutive_counts.get(length_key, 0) + 1
                # Reset for new type
                current_type = t
                current_count = 1
        
        # Record the final run
        length_key = f"{current_type}_{current_count}"
        consecutive_counts[length_key] = consecutive_counts.get(length_key, 0) + 1
        
        print("\nRuns of consecutive trials:")
        for key, count in sorted(consecutive_counts.items()):
            print(f"  {key}: {count}")
        
        # 6. Calculate transition probabilities between trial types
        transitions = {}
        for i in range(len(trial_sequence) - 1):
            transition = (trial_sequence[i], trial_sequence[i+1])
            transitions[transition] = transitions.get(transition, 0) + 1
        
        print("\nTransition counts between trial types:")
        for transition, count in sorted(transitions.items()):
            print(f"  {transition[0]} → {transition[1]}: {count}")
    
    def process_participant(self, participant_id):
        """
        Generate the design, assign breath holds, finalize columns,
        and save as a CSV for one participant.
        """
        try:
            if self.config['debug_mode']:
                print(f"\n=== Processing participant {participant_id} ===")
            
            # 1) Generate base design (counterbalanced combos)
            design_df = self.generate_counterbalanced_design(participant_id)
            
            # 2) Assign each trial to a breath hold (inhale or exhale)
            design_df = self.assign_breath_holds(design_df)
            
            # 3) Finalize columns (rename, set baseline/catch to NA, etc.)
            design_df = self.finalize_design_csv(design_df)
            
            # 4) Save CSV
            out_path = os.path.join(self.experiment_log_dir,
                                    f"participant_{participant_id}_design.csv")
            design_df.to_csv(out_path, index=False)
            
            if self.config['debug_mode']:
                print(f"Saved participant {participant_id} design to {out_path}")
            
            return True, out_path
        except Exception as e:
            print(f"Error processing participant {participant_id}: {e}")
            traceback.print_exc()
            return False, None
    
    def run(self, participant_ids=None, num_participants=None):
        """
        Generate the design CSVs for multiple participants.
        
        Args:
            participant_ids: List of specific participant IDs to process
            num_participants: Number of participants to generate if participant_ids not provided
            
        Returns:
            Dictionary mapping participant IDs to generation results
        """
        if participant_ids is None and num_participants is None:
            num_participants = self.config.get('num_participants', 5)
        
        if participant_ids is None:
            participant_ids = list(range(num_participants))
        
        if self.config['debug_mode']:
            print("=== Starting PPSDesignGenerator ===")
            print(f"Repetitions: {self.config['repetitions']}")
            print(f"Participants: {participant_ids}")
            print(f"Box breathing settings: {self.config['box_breathing']}")
        
        results = {}
        for pid in participant_ids:
            success, path = self.process_participant(pid)
            results[pid] = {
                'success': success,
                'csv_path': path
            }
        
        return results

# ======================================================================
# AUDIO GENERATION FUNCTIONS
# ======================================================================

def generate_audio_from_design(design_csv_path, output_dir, base_audio_path=None):
    """
    Generate audio files from a design CSV.
    
    This is a placeholder for audio generation functionality.
    The actual implementation would:
    1. Read the design CSV
    2. Create or load the base box breathing audio
    3. Insert looming and tactile stimuli at the specified timestamps
    4. Save the resulting audio files
    
    Args:
        design_csv_path: Path to the design CSV
        output_dir: Directory to save audio files
        base_audio_path: Optional path to base box breathing audio
        
    Returns:
        Tuple of (looming_audio_path, tactile_audio_path)
    """
    # This is just a placeholder - implement actual audio generation here
    print(f"Audio generation from {design_csv_path} not implemented yet")
    return None, None

# ======================================================================
# MAIN FUNCTION
# ======================================================================

def main():
    """Main function to run the design generator."""
    # Create the design generator
    generator = PPSDesignGenerator(CONFIG)
    
    # Run for all participants defined in CONFIG
    results = generator.run()
    
    # Print summary
    print("\n=== Generation Summary ===")
    for pid, info in results.items():
        if info['success']:
            print(f"Participant {pid}: CSV generated at {info['csv_path']}")
        else:
            print(f"Participant {pid}: ERROR - no CSV generated.")
    
    print("\nTo generate audio files from these designs, implement the audio generation function.")

if __name__ == "__main__":
    main()

IndentationError: unexpected indent (630239049.py, line 529)

In [5]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
PPS Design Generator

This script generates participant‐specific experimental design CSVs for a 
Peripersonal Space (PPS) experiment. Each design CSV is built upon a 
“skeleton” CSV that contains the retention slot (breathhold) timestamps 
from a box‐breathing audio file. If needed, additional retention slots are 
appended (using 8‑second increments) so that every trial can be assigned 
to a specific retention slot. New columns are added to include counterbalanced 
stimulus conditions (e.g., looming stimulus type, tactile stimulus timing).

Author: AI Assistant
"""

import os
import numpy as np
import pandas as pd
import random
import traceback

# ----------------------------------------------------------------------
# Helper function: Convert milliseconds to "MM:SS.s" format
# ----------------------------------------------------------------------
def convert_ms_to_timestamp(ms):
    m = int(ms // 60000)
    s = (ms % 60000) / 1000.0
    return f"{m:02}:{s:.1f}"

# ----------------------------------------------------------------------
# CONFIGURATION PARAMETERS
# ----------------------------------------------------------------------
CONFIG = {
    'repetitions': 4,            # How many times each SOA×stimulus combination repeats
    'num_participants': 5,       # Default number of participants
    'included_conditions': {
        'inhalation': True,
        'exhalation': True,
        'baseline': True,
    },
    'catch_trial_percentage': 10,
    'soa_conditions_ms': [190, 400, 700, 1000, 1500],
    'jitter_options_ms': [100, 200, 300, 400, 500],
    'debug_mode': True,
    'paths': {
        'experiment_log_dir': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog",
    },
    'skeleton_csv_path': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_timestamps.csv",
    'box_breathing': {
        'cycle_duration_sec': 8,      # Duration of one breath cycle in seconds
        'intro_duration_sec': 60,       # (Not used here; skeleton timestamps drive timing)
        'outro_duration_sec': 30,       # (Not used here)
        'sample_rate': 48000,           # Audio sample rate (samples per second)
    }
}

# Noise types for looming stimuli (can be any labels)
LOOMING_STIMULI = ['blue', 'brown', 'pink', 'white']

# ----------------------------------------------------------------------
# CLASS: PPSDesignGenerator
# ----------------------------------------------------------------------
class PPSDesignGenerator:
    """Generates participant-specific CSV design files for PPS experiments."""
    
    def __init__(self, config=None):
        self.config = config or CONFIG
        
        # Prepare output directory
        self.experiment_log_dir = self.config['paths']['experiment_log_dir']
        os.makedirs(self.experiment_log_dir, exist_ok=True)
        
        # Calculate trial counts
        self.trial_counts = self._calculate_trial_counts()
        self.conditions = [cond for cond, inc in self.config['included_conditions'].items() if inc]
        
        # Set seed for reproducibility
        self.base_seed = 42
        
        if self.config['debug_mode']:
            total_trials = sum(self.trial_counts.values())
            print("Initialized PPSDesignGenerator with:")
            print(f"  - Repetitions: {self.config['repetitions']}")
            print(f"  - Conditions: {self.conditions}")
            print(f"  - SOA values: {self.config['soa_conditions_ms']}")
            print(f"  - Total trials per participant: {total_trials}")
            print(f"  - Output directory: {self.experiment_log_dir}")
    
    def _calculate_trial_counts(self):
        """
        Calculate how many trials for each condition (inhalation, exhalation, baseline)
        and include catch trials.
        """
        reps = self.config['repetitions']
        soa_count = len(self.config['soa_conditions_ms'])
        stimulus_types = LOOMING_STIMULI
        catch_percentage = self.config['catch_trial_percentage']
        
        trial_counts = {}
        for condition, include in self.config['included_conditions'].items():
            if include:
                # (# noise types) × (# SOA) × (reps)
                trial_counts[condition] = len(stimulus_types) * soa_count * reps
        
        regular_trials = sum(trial_counts.values())
        catch_trials = int(np.ceil(regular_trials * catch_percentage / 100))
        trial_counts['catch'] = catch_trials
        
        if self.config['debug_mode']:
            print("Trial count calculation:")
            for cond, count in trial_counts.items():
                print(f"  {cond}: {count} trials")
            print(f"  Total: {sum(trial_counts.values())} trials")
        
        return trial_counts

    # ------------------------------------------------------------------
    # Load the skeleton CSV and extend it if needed
    # ------------------------------------------------------------------
    def load_retention_slots(self, required_slots):
        """
        Load the skeleton CSV of retention (breathhold) timestamps and extend it with 
        additional rows (using 8-second increments) if the number of rows is less than required_slots.
        
        Returns:
            DataFrame with columns: retentionslot_timestamp, milliseconds, sample_index, description, breathphase
        """
        skeleton_path = self.config['skeleton_csv_path']
        try:
            # Assuming the CSV is tab-delimited (adjust delimiter if needed)
            skeleton_df = pd.read_csv(skeleton_path, delimiter="\t")
        except Exception as e:
            raise ValueError(f"Error reading skeleton CSV from {skeleton_path}: {e}")
        
        current_slots = len(skeleton_df)
        required_slots = int(required_slots)
        sample_rate = self.config['box_breathing']['sample_rate']
        cycle_sec = self.config['box_breathing']['cycle_duration_sec']
        
        if self.config['debug_mode']:
            print(f"Loaded skeleton CSV with {current_slots} retention slots.")
        
        # If more slots are needed, add rows using fixed 8-second increments.
        if current_slots < required_slots:
            last_row = skeleton_df.iloc[-1]
            last_ms = float(last_row['milliseconds'])
            last_sample_index = int(last_row['sample_index'])
            # Determine the next hold number (based on description, if available)
            for i in range(required_slots - current_slots):
                new_ms = last_ms + (cycle_sec * 1000) * (i + 1)
                new_sample_index = last_sample_index + int(cycle_sec * sample_rate * (i + 1))
                new_timestamp = convert_ms_to_timestamp(new_ms)
                # Alternate breathphase based on overall order (even: inhale, odd: exhale)
                new_breathphase = 'inhale' if ((current_slots + i) % 2 == 0) else 'exhale'
                new_description = f"Additional hold {i+1}"
                new_row = {
                    'retentionslot_timestamp': new_timestamp,
                    'milliseconds': new_ms,
                    'sample_index': new_sample_index,
                    'description': new_description,
                    'breathphase': new_breathphase
                }
                skeleton_df = skeleton_df.append(new_row, ignore_index=True)
            if self.config['debug_mode']:
                print(f"Extended skeleton CSV to {len(skeleton_df)} retention slots.")
        
        # If there are more slots than required, take only the first required_slots rows.
        if len(skeleton_df) > required_slots:
            skeleton_df = skeleton_df.iloc[:required_slots]
        
        return skeleton_df

    # ------------------------------------------------------------------
    # Generate counterbalanced stimulus design (trial conditions)
    # ------------------------------------------------------------------
    def generate_counterbalanced_design(self, participant_id):
        """
        Generate a counterbalanced design (trial conditions) for one participant.
        Returns a DataFrame with columns: participant_id, trial_type, stimulus_type,
        soa_value_ms, jitter_ms, is_tactile, trial_number.
        """
        # Set seed based on participant ID for reproducibility
        random.seed(self.base_seed + participant_id)
        np.random.seed(self.base_seed + participant_id)
        
        soa_conditions = self.config['soa_conditions_ms']
        jitter_options = self.config['jitter_options_ms']
        stimulus_types = LOOMING_STIMULI
        
        all_trials = []
        
        # Build main conditions (inhalation, exhalation, baseline)
        for condition, count in self.trial_counts.items():
            if condition == 'catch' or count == 0:
                continue
            
            # Create all combos of (stimulus type × SOA)
            combo_list = []
            for stim_type in stimulus_types:
                for soa in soa_conditions:
                    for _ in range(self.config['repetitions']):
                        combo_list.append((stim_type, soa))
            random.shuffle(combo_list)
            
            for stim_type, soa in combo_list:
                all_trials.append({
                    'participant_id': participant_id,
                    'trial_type': condition,  # inhalation, exhalation, baseline
                    'stimulus_type': stim_type,
                    'soa_value_ms': soa,
                    'jitter_ms': random.choice(jitter_options),
                    'is_tactile': True  # main trials have a tactile component
                })
        
        # Add catch trials (looming only, no tactile)
        catch_count = self.trial_counts.get('catch', 0)
        if catch_count > 0:
            catch_combos = []
            stim_counts = {stim: 0 for stim in stimulus_types}
            soa_counts = {soa: 0 for soa in soa_conditions}
            while len(catch_combos) < catch_count:
                min_stim = min(stim_counts, key=stim_counts.get)
                min_soa = min(soa_counts, key=soa_counts.get)
                catch_combos.append((min_stim, min_soa))
                stim_counts[min_stim] += 1
                soa_counts[min_soa] += 1
            for stim_type, soa in catch_combos:
                all_trials.append({
                    'participant_id': participant_id,
                    'trial_type': 'catch',
                    'stimulus_type': stim_type,
                    'soa_value_ms': soa,
                    'jitter_ms': random.choice(jitter_options),
                    'is_tactile': False
                })
        
        design_df = pd.DataFrame(all_trials)
        design_df = design_df.sample(frac=1, random_state=participant_id*100).reset_index(drop=True)
        design_df['trial_number'] = range(1, len(design_df) + 1)
        
        if self.config['debug_mode']:
            print("\nTrial type distribution:")
            print(design_df['trial_type'].value_counts())
            print("\nStimulus type distribution by trial type:")
            for trial_type in design_df['trial_type'].unique():
                if trial_type == 'baseline':
                    continue
                subset = design_df[design_df['trial_type'] == trial_type]
                print(f"  {trial_type}: {subset['stimulus_type'].value_counts().to_dict()}")
            print("\nSOA distribution by trial type:")
            for trial_type in design_df['trial_type'].unique():
                if trial_type == 'baseline':
                    continue
                subset = design_df[design_df['trial_type'] == trial_type]
                print(f"  {trial_type}: {subset['soa_value_ms'].value_counts().to_dict()}")
        
        return design_df
    
    # ------------------------------------------------------------------
    # Assign retention slots (skeleton rows) to trials
    # ------------------------------------------------------------------
    def assign_breath_holds(self, design_df):
        """
        Assign each trial (from the counterbalanced design) to a retention slot 
        from the skeleton CSV. For inhalation and exhalation trials, slots matching 
        the appropriate breathphase are assigned first; all remaining trials are 
        allocated to leftover slots.
        """
        required_slots = len(design_df)
        retention_slots_df = self.load_retention_slots(required_slots)
        
        # Split retention slots by breathphase
        inhale_slots = retention_slots_df[retention_slots_df['breathphase'] == 'inhale'].copy()
        exhale_slots = retention_slots_df[retention_slots_df['breathphase'] == 'exhale'].copy()
        
        # Split design trials by trial type
        inhalation_trials = design_df[design_df['trial_type'] == 'inhalation'].copy()
        exhalation_trials = design_df[design_df['trial_type'] == 'exhalation'].copy()
        other_trials = design_df[~design_df['trial_type'].isin(['inhalation', 'exhalation'])].copy()
        
        result_rows = []
        # Assign inhalation trials to inhale slots in order
        for i, trial in enumerate(inhalation_trials.itertuples(index=False)):
            if i < len(inhale_slots):
                slot = inhale_slots.iloc[i]
                trial_dict = trial._asdict() if hasattr(trial, '_asdict') else trial.__dict__
                trial_dict.update({
                    'retentionslot_timestamp': slot['retentionslot_timestamp'],
                    'milliseconds': slot['milliseconds'],
                    'sample_index': slot['sample_index'],
                    'description': slot['description'],
                    'breathphase': slot['breathphase']
                })
                result_rows.append(trial_dict)
            else:
                break
        # Assign exhalation trials to exhale slots in order
        for i, trial in enumerate(exhalation_trials.itertuples(index=False)):
            if i < len(exhale_slots):
                slot = exhale_slots.iloc[i]
                trial_dict = trial._asdict() if hasattr(trial, '_asdict') else trial.__dict__
                trial_dict.update({
                    'retentionslot_timestamp': slot['retentionslot_timestamp'],
                    'milliseconds': slot['milliseconds'],
                    'sample_index': slot['sample_index'],
                    'description': slot['description'],
                    'breathphase': slot['breathphase']
                })
                result_rows.append(trial_dict)
            else:
                break
        # For remaining trials (baseline, catch), assign remaining slots in order
        assigned_count = len(inhalation_trials) + len(exhalation_trials)
        remaining_slots = retention_slots_df.iloc[assigned_count:]
        for i, trial in enumerate(other_trials.itertuples(index=False)):
            if i < len(remaining_slots):
                slot = remaining_slots.iloc[i]
                trial_dict = trial._asdict() if hasattr(trial, '_asdict') else trial.__dict__
                trial_dict.update({
                    'retentionslot_timestamp': slot['retentionslot_timestamp'],
                    'milliseconds': slot['milliseconds'],
                    'sample_index': slot['sample_index'],
                    'description': slot['description'],
                    'breathphase': slot['breathphase']
                })
                result_rows.append(trial_dict)
            else:
                raise ValueError("Not enough retention slots for all trials.")
        
        # Combine into one DataFrame and sort by the audio timeline (milliseconds)
        design_with_slots = pd.DataFrame(result_rows)
        design_with_slots = design_with_slots.sort_values(by='milliseconds').reset_index(drop=True)
        
        # Compute jittered onset for looming stimulus
        design_with_slots['jittered_ms'] = design_with_slots['milliseconds'] + design_with_slots['jitter_ms']
        design_with_slots['looming_stimulus_timestamp'] = design_with_slots['jittered_ms'].apply(convert_ms_to_timestamp)
        
        # For non-catch trials, compute tactile stimulus timestamp (looming + SOA)
        non_catch_mask = design_with_slots['trial_type'] != 'catch'
        design_with_slots.loc[non_catch_mask, 'soa_ms'] = (
            design_with_slots.loc[non_catch_mask, 'jittered_ms'] + 
            design_with_slots.loc[non_catch_mask, 'soa_value_ms']
        )
        design_with_slots.loc[non_catch_mask, 'tactile_stimulus_timestamp'] = (
            design_with_slots.loc[non_catch_mask, 'soa_ms'].apply(convert_ms_to_timestamp)
        )
        
        return design_with_slots
    
    # ------------------------------------------------------------------
    # Finalize the design CSV (rename columns and handle special cases)
    # ------------------------------------------------------------------
    def finalize_design_csv(self, design_df):
        """
        Finalize the design DataFrame by ensuring the final column order and
        handling special cases for baseline and catch trials.
        """
        # For baseline trials: no looming stimulus
        baseline_mask = (design_df['trial_type'] == 'baseline')
        design_df.loc[baseline_mask, 'stimulus_type'] = 'none'
        design_df.loc[baseline_mask, 'looming_stimulus_timestamp'] = None
        
        # For catch trials: no tactile stimulus
        catch_mask = (design_df['trial_type'] == 'catch')
        design_df.loc[catch_mask, 'tactile_stimulus_timestamp'] = None
        
        # Expected response: True for all except catch trials
        design_df['expected_response'] = ~design_df['trial_type'].isin(['catch'])
        
        # Define final column order
        final_cols = [
            'participant_id',
            'trial_number',
            'trial_type',
            'retentionslot_timestamp',
            'looming_stimulus_timestamp',
            'tactile_stimulus_timestamp',
            'stimulus_type',
            'soa_value_ms',
            'jitter_ms',
            'sample_index',
            'description',
            'breathphase',
            'expected_response'
        ]
        final_cols = [col for col in final_cols if col in design_df.columns]
        design_df = design_df[final_cols]
        
        self._validate_design(design_df)
        return design_df
    
    # ------------------------------------------------------------------
    # Validation: Check distributions and transitions
    # ------------------------------------------------------------------
    def _validate_design(self, design_df):
        if not self.config['debug_mode']:
            return
        print("\n=== DESIGN VALIDATION ===")
        print("Trial type counts:")
        print(design_df['trial_type'].value_counts())
        print("\nStimulus distribution by trial type:")
        for trial_type in design_df['trial_type'].unique():
            if trial_type == 'baseline':
                continue
            subset = design_df[design_df['trial_type'] == trial_type]
            print(f"  {trial_type}: {subset['stimulus_type'].value_counts().to_dict()}")
        print("\nSOA distribution by trial type:")
        for trial_type in design_df['trial_type'].unique():
            if trial_type == 'baseline':
                continue
            subset = design_df[design_df['trial_type'] == trial_type]
            print(f"  {trial_type}: {subset['soa_value_ms'].value_counts().to_dict()}")
        print("\nBreath phase distribution:")
        if 'breathphase' in design_df.columns:
            print(design_df['breathphase'].value_counts())
        trial_sequence = design_df['trial_type'].tolist()
        consecutive_counts = {}
        current_type = trial_sequence[0]
        current_count = 1
        for t in trial_sequence[1:]:
            if t == current_type:
                current_count += 1
            else:
                key = f"{current_type}_{current_count}"
                consecutive_counts[key] = consecutive_counts.get(key, 0) + 1
                current_type = t
                current_count = 1
        key = f"{current_type}_{current_count}"
        consecutive_counts[key] = consecutive_counts.get(key, 0) + 1
        print("\nRuns of consecutive trials:")
        for key, count in sorted(consecutive_counts.items()):
            print(f"  {key}: {count}")
    
    # ------------------------------------------------------------------
    # Process a single participant: generate design, assign slots, finalize, save CSV
    # ------------------------------------------------------------------
    def process_participant(self, participant_id):
        try:
            if self.config['debug_mode']:
                print(f"\n=== Processing participant {participant_id} ===")
            design_df = self.generate_counterbalanced_design(participant_id)
            design_df = self.assign_breath_holds(design_df)
            design_df = self.finalize_design_csv(design_df)
            out_path = os.path.join(self.experiment_log_dir, f"participant_{participant_id}_design.csv")
            design_df.to_csv(out_path, index=False)
            if self.config['debug_mode']:
                print(f"Saved participant {participant_id} design to {out_path}")
            return True, out_path
        except Exception as e:
            print(f"Error processing participant {participant_id}: {e}")
            traceback.print_exc()
            return False, None
    
    # ------------------------------------------------------------------
    # Run design generation for multiple participants
    # ------------------------------------------------------------------
    def run(self, participant_ids=None, num_participants=None):
        if participant_ids is None and num_participants is None:
            num_participants = self.config.get('num_participants', 5)
        if participant_ids is None:
            participant_ids = list(range(num_participants))
        if self.config['debug_mode']:
            print("=== Starting PPSDesignGenerator ===")
            print(f"Repetitions: {self.config['repetitions']}")
            print(f"Participants: {participant_ids}")
            print(f"Box breathing settings: {self.config['box_breathing']}")
        results = {}
        for pid in participant_ids:
            success, path = self.process_participant(pid)
            results[pid] = {'success': success, 'csv_path': path}
        return results

# ----------------------------------------------------------------------
# AUDIO GENERATION PLACEHOLDER
# ----------------------------------------------------------------------
def generate_audio_from_design(design_csv_path, output_dir, base_audio_path=None):
    """
    Generate audio files from a design CSV.
    (Placeholder function – implement audio generation as needed.)
    """
    print(f"Audio generation from {design_csv_path} not implemented yet")
    return None, None

# ----------------------------------------------------------------------
# MAIN FUNCTION
# ----------------------------------------------------------------------
def main():
    """Main function to run the design generator."""
    generator = PPSDesignGenerator(CONFIG)
    results = generator.run()
    print("\n=== Generation Summary ===")
    for pid, info in results.items():
        if info['success']:
            print(f"Participant {pid}: CSV generated at {info['csv_path']}")
        else:
            print(f"Participant {pid}: ERROR - no CSV generated.")
    print("\nTo generate audio files from these designs, implement the audio generation function.")

if __name__ == "__main__":
    main()


Trial count calculation:
  inhalation: 80 trials
  exhalation: 80 trials
  baseline: 80 trials
  catch: 24 trials
  Total: 264 trials
Initialized PPSDesignGenerator with:
  - Repetitions: 4
  - Conditions: ['inhalation', 'exhalation', 'baseline']
  - SOA values: [190, 400, 700, 1000, 1500]
  - Total trials per participant: 264
  - Output directory: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog
=== Starting PPSDesignGenerator ===
Repetitions: 4
Participants: [0, 1, 2, 3, 4]
Box breathing settings: {'cycle_duration_sec': 8, 'intro_duration_sec': 60, 'outro_duration_sec': 30, 'sample_rate': 48000}

=== Processing participant 0 ===

Trial type distribution:
trial_type
exhalation    80
inhalation    80
baseline      80
catch         24
Name: count, dtype: int64

Stimulus type distribution by trial type:
  exhalation: {'brown': 20, 'pink': 20, 'blue': 20, 'white': 20}
  catch: {'blue': 6, 'white': 6, 'brown': 6, 'pink': 6}
  inhalation: {'pink': 2

Traceback (most recent call last):
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\site-packages\pandas\core\indexes\base.py", line 3805, in get_loc
    return self._engine.get_loc(casted_key)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "index.pyx", line 167, in pandas._libs.index.IndexEngine.get_loc
  File "index.pyx", line 196, in pandas._libs.index.IndexEngine.get_loc
  File "pandas\\_libs\\hashtable_class_helper.pxi", line 7081, in pandas._libs.hashtable.PyObjectHashTable.get_item
  File "pandas\\_libs\\hashtable_class_helper.pxi", line 7089, in pandas._libs.hashtable.PyObjectHashTable.get_item
KeyError: 'milliseconds'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "C:\Users\cogpsy-vrlab\AppData\Local\Temp\ipykernel_7528\741025921.py", line 443, in process_participant
    design_df = self.assign_breath_holds(design_df)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\cogpsy-vrlab\AppData\Local\

In [7]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
PPS Design Generator

This script generates participant‐specific experimental design CSVs for a 
Peripersonal Space (PPS) experiment. Each design CSV is built upon a 
“scaffold” CSV that contains the retention slot (breathhold) timestamps 
from a box‐breathing audio file. If needed, additional retention slots are 
appended (using 8‑second increments) so that every trial can be assigned 
to a specific retention slot. New columns are added to include counterbalanced 
stimulus conditions (e.g., looming stimulus type, tactile stimulus timing).

Author: AI Assistant
"""

import os
import numpy as np
import pandas as pd
import random
import traceback

# ----------------------------------------------------------------------
# Helper function: Convert milliseconds to "MM:SS.s" format
# ----------------------------------------------------------------------
def convert_ms_to_timestamp(ms):
    m = int(ms // 60000)
    s = (ms % 60000) / 1000.0
    return f"{m:02}:{s:.1f}"

# ----------------------------------------------------------------------
# CONFIGURATION PARAMETERS
# ----------------------------------------------------------------------
CONFIG = {
    'repetitions': 4,            # How many times each SOA×stimulus combination repeats
    'num_participants': 5,       # Default number of participants
    'included_conditions': {
        'inhalation': True,
        'exhalation': True,
        'baseline': True,
    },
    'catch_trial_percentage': 10,
    'soa_conditions_ms': [190, 400, 700, 1000, 1500],
    'jitter_options_ms': [100, 200, 300, 400, 500],
    'debug_mode': True,
    'paths': {
        # Directory where participant-specific CSV files will be saved
        'experiment_log_dir': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog",
    },
    # Path to the original scaffold CSV containing retention (breathhold) timestamps
    'skeleton_csv_path': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_timestamps.csv",
    'box_breathing': {
        'cycle_duration_sec': 8,      # Duration of one breath cycle in seconds
        # Intro/outro are not used in this implementation because the scaffold drives timing.
        'intro_duration_sec': 60,
        'outro_duration_sec': 30,
        'sample_rate': 48000,           # Audio sample rate (samples per second)
    }
}

# Noise types for looming stimuli (can be any labels)
LOOMING_STIMULI = ['blue', 'brown', 'pink', 'white']

# ----------------------------------------------------------------------
# CLASS: PPSDesignGenerator
# ----------------------------------------------------------------------
class PPSDesignGenerator:
    """Generates participant-specific CSV design files for PPS experiments."""
    
    def __init__(self, config=None):
        self.config = config or CONFIG
        
        # Prepare output directory
        self.experiment_log_dir = self.config['paths']['experiment_log_dir']
        os.makedirs(self.experiment_log_dir, exist_ok=True)
        
        # Calculate trial counts based on included conditions and repetitions
        self.trial_counts = self._calculate_trial_counts()
        self.conditions = [cond for cond, inc in self.config['included_conditions'].items() if inc]
        
        # Set seed for reproducibility
        self.base_seed = 42
        
        if self.config['debug_mode']:
            total_trials = sum(self.trial_counts.values())
            print("Initialized PPSDesignGenerator with:")
            print(f"  - Repetitions: {self.config['repetitions']}")
            print(f"  - Conditions: {self.conditions}")
            print(f"  - SOA values: {self.config['soa_conditions_ms']}")
            print(f"  - Total trials per participant: {total_trials}")
            print(f"  - Output directory: {self.experiment_log_dir}")
    
    def _calculate_trial_counts(self):
        """
        Calculate the number of trials for each condition (inhalation, exhalation, baseline)
        and include catch trials.
        """
        reps = self.config['repetitions']
        soa_count = len(self.config['soa_conditions_ms'])
        stimulus_types = LOOMING_STIMULI
        catch_percentage = self.config['catch_trial_percentage']
        
        trial_counts = {}
        for condition, include in self.config['included_conditions'].items():
            if include:
                # (# noise types) × (# SOA) × (reps)
                trial_counts[condition] = len(stimulus_types) * soa_count * reps
        
        regular_trials = sum(trial_counts.values())
        catch_trials = int(np.ceil(regular_trials * catch_percentage / 100))
        trial_counts['catch'] = catch_trials
        
        if self.config['debug_mode']:
            print("Trial count calculation:")
            for cond, count in trial_counts.items():
                print(f"  {cond}: {count} trials")
            print(f"  Total: {sum(trial_counts.values())} trials")
        
        return trial_counts

    # ------------------------------------------------------------------
    # Load the scaffold CSV and extend it if needed
    # ------------------------------------------------------------------
    def load_retention_slots(self, required_slots):
        """
        Load the scaffold CSV of retention (breathhold) timestamps and extend it with 
        additional rows (using fixed 8‑second increments) if the number of rows is less than required_slots.
        
        Returns:
            DataFrame with columns: retentionslot_timestamp, milliseconds, sample_index, description, breathphase
        """
        skeleton_path = self.config['skeleton_csv_path']
        try:
            # Adjust delimiter if needed; here we assume tab-delimited.
            skeleton_df = pd.read_csv(skeleton_path, delimiter="\t")
            # Remove any extra whitespace from column names.
            skeleton_df.columns = skeleton_df.columns.str.strip()
        except Exception as e:
            raise ValueError(f"Error reading scaffold CSV from {skeleton_path}: {e}")
        
        current_slots = len(skeleton_df)
        required_slots = int(required_slots)
        sample_rate = self.config['box_breathing']['sample_rate']
        cycle_sec = self.config['box_breathing']['cycle_duration_sec']
        
        if self.config['debug_mode']:
            print(f"Loaded scaffold CSV with {current_slots} retention slots.")
        
        # If more slots are needed, add rows using fixed 8‑second increments.
        if current_slots < required_slots:
            last_row = skeleton_df.iloc[-1]
            last_ms = float(last_row['milliseconds'])
            last_sample_index = int(last_row['sample_index'])
            for i in range(required_slots - current_slots):
                new_ms = last_ms + (cycle_sec * 1000) * (i + 1)
                new_sample_index = last_sample_index + int(cycle_sec * sample_rate * (i + 1))
                new_timestamp = convert_ms_to_timestamp(new_ms)
                new_breathphase = 'inhale' if ((current_slots + i) % 2 == 0) else 'exhale'
                new_description = f"Additional hold {i+1}"
                new_row = {
                    'retentionslot_timestamp': new_timestamp,
                    'milliseconds': new_ms,
                    'sample_index': new_sample_index,
                    'description': new_description,
                    'breathphase': new_breathphase
                }
                skeleton_df = skeleton_df.append(new_row, ignore_index=True)
            if self.config['debug_mode']:
                print(f"Extended scaffold CSV to {len(skeleton_df)} retention slots.")
        
        # If there are more slots than required, take only the first required_slots rows.
        if len(skeleton_df) > required_slots:
            skeleton_df = skeleton_df.iloc[:required_slots]
        
        return skeleton_df

    # ------------------------------------------------------------------
    # Generate counterbalanced stimulus design (trial conditions)
    # ------------------------------------------------------------------
    def generate_counterbalanced_design(self, participant_id):
        """
        Generate a counterbalanced design (trial conditions) for one participant.
        
        Returns a DataFrame with columns: participant_id, trial_type, stimulus_type,
        soa_value_ms, jitter_ms, is_tactile, trial_number.
        """
        # Set seed based on participant ID for reproducibility.
        random.seed(self.base_seed + participant_id)
        np.random.seed(self.base_seed + participant_id)
        
        soa_conditions = self.config['soa_conditions_ms']
        jitter_options = self.config['jitter_options_ms']
        stimulus_types = LOOMING_STIMULI
        
        all_trials = []
        
        # Build main conditions (inhalation, exhalation, baseline)
        for condition, count in self.trial_counts.items():
            if condition == 'catch' or count == 0:
                continue
            
            # Create all combinations of (stimulus type × SOA)
            combo_list = []
            for stim_type in stimulus_types:
                for soa in soa_conditions:
                    for _ in range(self.config['repetitions']):
                        combo_list.append((stim_type, soa))
            random.shuffle(combo_list)
            
            for stim_type, soa in combo_list:
                all_trials.append({
                    'participant_id': participant_id,
                    'trial_type': condition,  # inhalation, exhalation, baseline
                    'stimulus_type': stim_type,
                    'soa_value_ms': soa,
                    'jitter_ms': random.choice(jitter_options),
                    'is_tactile': True  # main trials have tactile component
                })
        
        # Add catch trials (looming only, no tactile)
        catch_count = self.trial_counts.get('catch', 0)
        if catch_count > 0:
            catch_combos = []
            stim_counts = {stim: 0 for stim in stimulus_types}
            soa_counts = {soa: 0 for soa in soa_conditions}
            while len(catch_combos) < catch_count:
                min_stim = min(stim_counts, key=stim_counts.get)
                min_soa = min(soa_counts, key=soa_counts.get)
                catch_combos.append((min_stim, min_soa))
                stim_counts[min_stim] += 1
                soa_counts[min_soa] += 1
            for stim_type, soa in catch_combos:
                all_trials.append({
                    'participant_id': participant_id,
                    'trial_type': 'catch',
                    'stimulus_type': stim_type,
                    'soa_value_ms': soa,
                    'jitter_ms': random.choice(jitter_options),
                    'is_tactile': False
                })
        
        design_df = pd.DataFrame(all_trials)
        design_df = design_df.sample(frac=1, random_state=participant_id*100).reset_index(drop=True)
        design_df['trial_number'] = range(1, len(design_df) + 1)
        
        if self.config['debug_mode']:
            print("\nTrial type distribution:")
            print(design_df['trial_type'].value_counts())
            print("\nStimulus type distribution by trial type:")
            for trial_type in design_df['trial_type'].unique():
                if trial_type == 'baseline':
                    continue
                subset = design_df[design_df['trial_type'] == trial_type]
                print(f"  {trial_type}: {subset['stimulus_type'].value_counts().to_dict()}")
            print("\nSOA distribution by trial type:")
            for trial_type in design_df['trial_type'].unique():
                if trial_type == 'baseline':
                    continue
                subset = design_df[design_df['trial_type'] == trial_type]
                print(f"  {trial_type}: {subset['soa_value_ms'].value_counts().to_dict()}")
        
        return design_df

    # ------------------------------------------------------------------
    # Assign scaffold retention slots to trials
    # ------------------------------------------------------------------
    def assign_breath_holds(self, design_df):
        """
        Assign each trial (from the counterbalanced design) to a retention slot 
        from the scaffold CSV. For inhalation and exhalation trials, slots matching 
        the appropriate breathphase are assigned first; all remaining trials 
        receive leftover slots.
        """
        required_slots = len(design_df)
        retention_slots_df = self.load_retention_slots(required_slots)
        
        # Split retention slots by breathphase
        inhale_slots = retention_slots_df[retention_slots_df['breathphase'] == 'inhale'].copy()
        exhale_slots = retention_slots_df[retention_slots_df['breathphase'] == 'exhale'].copy()
        
        # Split design trials by trial type
        inhalation_trials = design_df[design_df['trial_type'] == 'inhalation'].copy()
        exhalation_trials = design_df[design_df['trial_type'] == 'exhalation'].copy()
        other_trials = design_df[~design_df['trial_type'].isin(['inhalation', 'exhalation'])].copy()
        
        result_rows = []
        # Assign inhalation trials to inhale slots (in order)
        for i, trial in enumerate(inhalation_trials.itertuples(index=False)):
            if i < len(inhale_slots):
                slot = inhale_slots.iloc[i]
                trial_dict = trial._asdict() if hasattr(trial, '_asdict') else trial.__dict__
                trial_dict.update({
                    'retentionslot_timestamp': slot['retentionslot_timestamp'],
                    'milliseconds': slot['milliseconds'],
                    'sample_index': slot['sample_index'],
                    'description': slot['description'],
                    'breathphase': slot['breathphase']
                })
                result_rows.append(trial_dict)
            else:
                break
        # Assign exhalation trials to exhale slots (in order)
        for i, trial in enumerate(exhalation_trials.itertuples(index=False)):
            if i < len(exhale_slots):
                slot = exhale_slots.iloc[i]
                trial_dict = trial._asdict() if hasattr(trial, '_asdict') else trial.__dict__
                trial_dict.update({
                    'retentionslot_timestamp': slot['retentionslot_timestamp'],
                    'milliseconds': slot['milliseconds'],
                    'sample_index': slot['sample_index'],
                    'description': slot['description'],
                    'breathphase': slot['breathphase']
                })
                result_rows.append(trial_dict)
            else:
                break
        # For remaining trials (baseline, catch), assign remaining slots in order.
        assigned_count = len(inhalation_trials) + len(exhalation_trials)
        remaining_slots = retention_slots_df.iloc[assigned_count:]
        for i, trial in enumerate(other_trials.itertuples(index=False)):
            if i < len(remaining_slots):
                slot = remaining_slots.iloc[i]
                trial_dict = trial._asdict() if hasattr(trial, '_asdict') else trial.__dict__
                trial_dict.update({
                    'retentionslot_timestamp': slot['retentionslot_timestamp'],
                    'milliseconds': slot['milliseconds'],
                    'sample_index': slot['sample_index'],
                    'description': slot['description'],
                    'breathphase': slot['breathphase']
                })
                result_rows.append(trial_dict)
            else:
                raise ValueError("Not enough retention slots for all trials.")
        
        # Combine into one DataFrame and sort by audio timeline (milliseconds)
        design_with_slots = pd.DataFrame(result_rows)
        design_with_slots = design_with_slots.sort_values(by='milliseconds').reset_index(drop=True)
        
        # Compute jittered onset for looming stimulus
        design_with_slots['jittered_ms'] = design_with_slots['milliseconds'] + design_with_slots['jitter_ms']
        design_with_slots['looming_stimulus_timestamp'] = design_with_slots['jittered_ms'].apply(convert_ms_to_timestamp)
        
        # For non-catch trials, compute tactile stimulus timestamp (looming + SOA)
        non_catch_mask = design_with_slots['trial_type'] != 'catch'
        design_with_slots.loc[non_catch_mask, 'soa_ms'] = (
            design_with_slots.loc[non_catch_mask, 'jittered_ms'] + 
            design_with_slots.loc[non_catch_mask, 'soa_value_ms']
        )
        design_with_slots.loc[non_catch_mask, 'tactile_stimulus_timestamp'] = (
            design_with_slots.loc[non_catch_mask, 'soa_ms'].apply(convert_ms_to_timestamp)
        )
        
        return design_with_slots

    # ------------------------------------------------------------------
    # Finalize the design CSV (rename columns, handle special cases)
    # ------------------------------------------------------------------
    def finalize_design_csv(self, design_df):
        """
        Finalize the design DataFrame by ensuring final column order and
        handling special cases for baseline and catch trials.
        """
        # For baseline trials: no looming stimulus
        baseline_mask = (design_df['trial_type'] == 'baseline')
        design_df.loc[baseline_mask, 'stimulus_type'] = 'none'
        design_df.loc[baseline_mask, 'looming_stimulus_timestamp'] = None
        
        # For catch trials: no tactile stimulus
        catch_mask = (design_df['trial_type'] == 'catch')
        design_df.loc[catch_mask, 'tactile_stimulus_timestamp'] = None
        
        # Expected response: True for all except catch trials
        design_df['expected_response'] = ~design_df['trial_type'].isin(['catch'])
        
        # Define final column order
        final_cols = [
            'participant_id',
            'trial_number',
            'trial_type',
            'retentionslot_timestamp',
            'looming_stimulus_timestamp',
            'tactile_stimulus_timestamp',
            'stimulus_type',
            'soa_value_ms',
            'jitter_ms',
            'sample_index',
            'description',
            'breathphase',
            'expected_response'
        ]
        final_cols = [col for col in final_cols if col in design_df.columns]
        design_df = design_df[final_cols]
        
        self._validate_design(design_df)
        return design_df
    
    # ------------------------------------------------------------------
    # Validation: Check distributions and transitions
    # ------------------------------------------------------------------
    def _validate_design(self, design_df):
        if not self.config['debug_mode']:
            return
        print("\n=== DESIGN VALIDATION ===")
        print("Trial type counts:")
        print(design_df['trial_type'].value_counts())
        print("\nStimulus distribution by trial type:")
        for trial_type in design_df['trial_type'].unique():
            if trial_type == 'baseline':
                continue
            subset = design_df[design_df['trial_type'] == trial_type]
            print(f"  {trial_type}: {subset['stimulus_type'].value_counts().to_dict()}")
        print("\nSOA distribution by trial type:")
        for trial_type in design_df['trial_type'].unique():
            if trial_type == 'baseline':
                continue
            subset = design_df[design_df['trial_type'] == trial_type]
            print(f"  {trial_type}: {subset['soa_value_ms'].value_counts().to_dict()}")
        print("\nBreath phase distribution:")
        if 'breathphase' in design_df.columns:
            print(design_df['breathphase'].value_counts())
        trial_sequence = design_df['trial_type'].tolist()
        consecutive_counts = {}
        current_type = trial_sequence[0]
        current_count = 1
        for t in trial_sequence[1:]:
            if t == current_type:
                current_count += 1
            else:
                key = f"{current_type}_{current_count}"
                consecutive_counts[key] = consecutive_counts.get(key, 0) + 1
                current_type = t
                current_count = 1
        key = f"{current_type}_{current_count}"
        consecutive_counts[key] = consecutive_counts.get(key, 0) + 1
        print("\nRuns of consecutive trials:")
        for key, count in sorted(consecutive_counts.items()):
            print(f"  {key}: {count}")
    
    # ------------------------------------------------------------------
    # Process a single participant: generate design, assign slots, finalize, save CSV
    # ------------------------------------------------------------------
    def process_participant(self, participant_id):
        try:
            if self.config['debug_mode']:
                print(f"\n=== Processing participant {participant_id} ===")
            design_df = self.generate_counterbalanced_design(participant_id)
            design_df = self.assign_breath_holds(design_df)
            design_df = self.finalize_design_csv(design_df)
            out_path = os.path.join(self.experiment_log_dir, f"participant_{participant_id}_design.csv")
            design_df.to_csv(out_path, index=False)
            if self.config['debug_mode']:
                print(f"Saved participant {participant_id} design to {out_path}")
            return True, out_path
        except Exception as e:
            print(f"Error processing participant {participant_id}: {e}")
            traceback.print_exc()
            return False, None
    
    # ------------------------------------------------------------------
    # Run design generation for multiple participants
    # ------------------------------------------------------------------
    def run(self, participant_ids=None, num_participants=None):
        if participant_ids is None and num_participants is None:
            num_participants = self.config.get('num_participants', 5)
        if participant_ids is None:
            participant_ids = list(range(num_participants))
        if self.config['debug_mode']:
            print("=== Starting PPSDesignGenerator ===")
            print(f"Repetitions: {self.config['repetitions']}")
            print(f"Participants: {participant_ids}")
            print(f"Box breathing settings: {self.config['box_breathing']}")
        results = {}
        for pid in participant_ids:
            success, path = self.process_participant(pid)
            results[pid] = {'success': success, 'csv_path': path}
        return results

# ----------------------------------------------------------------------
# AUDIO GENERATION PLACEHOLDER FUNCTION
# ----------------------------------------------------------------------
def generate_audio_from_design(design_csv_path, output_dir, base_audio_path=None):
    """
    Generate audio files from a design CSV.
    (Placeholder function – implement audio generation as needed.)
    """
    print(f"Audio generation from {design_csv_path} not implemented yet")
    return None, None

# ----------------------------------------------------------------------
# MAIN FUNCTION
# ----------------------------------------------------------------------
def main():
    """Main function to run the design generator."""
    generator = PPSDesignGenerator(CONFIG)
    results = generator.run()
    print("\n=== Generation Summary ===")
    for pid, info in results.items():
        if info['success']:
            print(f"Participant {pid}: CSV generated at {info['csv_path']}")
        else:
            print(f"Participant {pid}: ERROR - no CSV generated.")
    print("\nTo generate audio files from these designs, implement the audio generation function.")

if __name__ == "__main__":
    main()


Trial count calculation:
  inhalation: 80 trials
  exhalation: 80 trials
  baseline: 80 trials
  catch: 24 trials
  Total: 264 trials
Initialized PPSDesignGenerator with:
  - Repetitions: 4
  - Conditions: ['inhalation', 'exhalation', 'baseline']
  - SOA values: [190, 400, 700, 1000, 1500]
  - Total trials per participant: 264
  - Output directory: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog
=== Starting PPSDesignGenerator ===
Repetitions: 4
Participants: [0, 1, 2, 3, 4]
Box breathing settings: {'cycle_duration_sec': 8, 'intro_duration_sec': 60, 'outro_duration_sec': 30, 'sample_rate': 48000}

=== Processing participant 0 ===

Trial type distribution:
trial_type
exhalation    80
inhalation    80
baseline      80
catch         24
Name: count, dtype: int64

Stimulus type distribution by trial type:
  exhalation: {'brown': 20, 'pink': 20, 'blue': 20, 'white': 20}
  catch: {'blue': 6, 'white': 6, 'brown': 6, 'pink': 6}
  inhalation: {'pink': 2

Traceback (most recent call last):
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\site-packages\pandas\core\indexes\base.py", line 3805, in get_loc
    return self._engine.get_loc(casted_key)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "index.pyx", line 167, in pandas._libs.index.IndexEngine.get_loc
  File "index.pyx", line 196, in pandas._libs.index.IndexEngine.get_loc
  File "pandas\\_libs\\hashtable_class_helper.pxi", line 7081, in pandas._libs.hashtable.PyObjectHashTable.get_item
  File "pandas\\_libs\\hashtable_class_helper.pxi", line 7089, in pandas._libs.hashtable.PyObjectHashTable.get_item
KeyError: 'milliseconds'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "C:\Users\cogpsy-vrlab\AppData\Local\Temp\ipykernel_7528\3874838564.py", line 447, in process_participant
    design_df = self.assign_breath_holds(design_df)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\cogpsy-vrlab\AppData\Local

In [10]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
PPS Design Generator

This script generates participant‐specific experimental design CSVs for a 
Peripersonal Space (PPS) experiment. Each design CSV is built upon a 
“scaffold” CSV that contains the retention slot (breathhold) timestamps 
from a box‐breathing audio file. If needed, additional retention slots are 
appended (using fixed 8‑second increments) so that every trial can be assigned 
to a specific retention slot. Then, a counterbalanced set of trial conditions 
(e.g., trial type, looming stimulus type, SOA, jitter) is randomly allocated to 
each scaffold row while preserving the sequential order of timestamps.

Author: AI Assistant
"""

import os
import numpy as np
import pandas as pd
import random
import traceback

# ----------------------------------------------------------------------
# Helper function: Convert milliseconds to "MM:SS.s" format
# ----------------------------------------------------------------------
def convert_ms_to_timestamp(ms):
    m = int(ms // 60000)
    s = (ms % 60000) / 1000.0
    return f"{m:02}:{s:.1f}"

# ----------------------------------------------------------------------
# CONFIGURATION PARAMETERS
# ----------------------------------------------------------------------
CONFIG = {
    'repetitions': 4,            # How many times each SOA×stimulus combination repeats
    'num_participants': 5,       # Default number of participants
    'included_conditions': {
        'inhalation': True,
        'exhalation': True,
        'baseline': True,
    },
    'catch_trial_percentage': 10,
    'soa_conditions_ms': [190, 400, 700, 1000, 1500],
    'jitter_options_ms': [100, 200, 300, 400, 500],
    'debug_mode': True,
    'paths': {
        # Directory where participant-specific CSV files will be saved
        'experiment_log_dir': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog",
    },
    # Path to the original scaffold CSV containing retention (breathhold) timestamps.
    'skeleton_csv_path': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_timestamps.csv",
    'box_breathing': {
        'cycle_duration_sec': 8,      # Duration of one breath cycle in seconds
        # Intro/outro are not used here because the scaffold drives timing.
        'intro_duration_sec': 60,
        'outro_duration_sec': 30,
        'sample_rate': 48000,           # Audio sample rate (samples per second)
    }
}

# Noise types for looming stimuli (can be any labels)
LOOMING_STIMULI = ['blue', 'brown', 'pink', 'white']

# ----------------------------------------------------------------------
# CLASS: PPSDesignGenerator
# ----------------------------------------------------------------------
class PPSDesignGenerator:
    """Generates participant-specific CSV design files for PPS experiments."""
    
    def __init__(self, config=None):
        self.config = config or CONFIG
        
        # Prepare output directory
        self.experiment_log_dir = self.config['paths']['experiment_log_dir']
        os.makedirs(self.experiment_log_dir, exist_ok=True)
        
        # Calculate trial counts based on included conditions and repetitions
        self.trial_counts = self._calculate_trial_counts()
        self.conditions = [cond for cond, inc in self.config['included_conditions'].items() if inc]
        
        # Set seed for reproducibility
        self.base_seed = 42
        
        if self.config['debug_mode']:
            total_trials = sum(self.trial_counts.values())
            print("Initialized PPSDesignGenerator with:")
            print(f"  - Repetitions: {self.config['repetitions']}")
            print(f"  - Conditions: {self.conditions}")
            print(f"  - SOA values: {self.config['soa_conditions_ms']}")
            print(f"  - Total trials per participant: {total_trials}")
            print(f"  - Output directory: {self.experiment_log_dir}")
    
    def _calculate_trial_counts(self):
        """
        Calculate the number of trials for each condition (inhalation, exhalation, baseline)
        and include catch trials.
        """
        reps = self.config['repetitions']
        soa_count = len(self.config['soa_conditions_ms'])
        stimulus_types = LOOMING_STIMULI
        catch_percentage = self.config['catch_trial_percentage']
        
        trial_counts = {}
        for condition, include in self.config['included_conditions'].items():
            if include:
                # (# noise types) × (# SOA) × (reps)
                trial_counts[condition] = len(stimulus_types) * soa_count * reps
        
        regular_trials = sum(trial_counts.values())
        catch_trials = int(np.ceil(regular_trials * catch_percentage / 100))
        trial_counts['catch'] = catch_trials
        
        if self.config['debug_mode']:
            print("Trial count calculation:")
            for cond, count in trial_counts.items():
                print(f"  {cond}: {count} trials")
            print(f"  Total: {sum(trial_counts.values())} trials")
        
        return trial_counts

    # ------------------------------------------------------------------
    # Load the scaffold CSV and extend it if needed
    # ------------------------------------------------------------------
    def load_retention_slots(self, required_slots):
        """
        Load the scaffold CSV of retention (breathhold) timestamps and extend it with 
        additional rows (using fixed 8‑second increments) if the number of rows is less than required_slots.
        
        Returns:
            DataFrame with columns: retentionslot_timestamp, milliseconds, sample_index, description, breathphase
        """
        skeleton_path = self.config['skeleton_csv_path']
        try:
            # Attempt to auto-detect the delimiter using the Python engine.
            skeleton_df = pd.read_csv(skeleton_path, sep=None, engine='python')
            # If only one column is found, try using tab as delimiter.
            if len(skeleton_df.columns) == 1:
                skeleton_df = pd.read_csv(skeleton_path, delimiter="\t")
            # Standardize column names: strip whitespace and convert to lowercase.
            skeleton_df.columns = [col.strip().lower() for col in skeleton_df.columns]
            if self.config['debug_mode']:
                print("Scaffold CSV columns:", skeleton_df.columns.tolist())
        except Exception as e:
            raise ValueError(f"Error reading scaffold CSV from {skeleton_path}: {e}")
        
        current_slots = len(skeleton_df)
        required_slots = int(required_slots)
        sample_rate = self.config['box_breathing']['sample_rate']
        cycle_sec = self.config['box_breathing']['cycle_duration_sec']
        
        if self.config['debug_mode']:
            print(f"Loaded scaffold CSV with {current_slots} retention slots.")
        
        # Extend the scaffold if more slots are needed.
        if current_slots < required_slots:
            last_row = skeleton_df.iloc[-1]
            last_ms = float(last_row['milliseconds'])
            last_sample_index = int(last_row['sample_index'])
            new_rows = []
            for i in range(required_slots - current_slots):
                new_ms = last_ms + (cycle_sec * 1000) * (i + 1)
                new_sample_index = last_sample_index + int(cycle_sec * sample_rate * (i + 1))
                new_timestamp = convert_ms_to_timestamp(new_ms)
                new_breathphase = 'inhale' if ((current_slots + i) % 2 == 0) else 'exhale'
                new_description = f"Additional hold {i+1}"
                new_rows.append({
                    'retentionslot_timestamp': new_timestamp,
                    'milliseconds': new_ms,
                    'sample_index': new_sample_index,
                    'description': new_description,
                    'breathphase': new_breathphase
                })
            if new_rows:
                skeleton_df = pd.concat([skeleton_df, pd.DataFrame(new_rows)], ignore_index=True)
            if self.config['debug_mode']:
                print(f"Extended scaffold CSV to {len(skeleton_df)} retention slots.")
        
        # If there are more slots than required, take only the first required_slots rows.
        if len(skeleton_df) > required_slots:
            skeleton_df = skeleton_df.iloc[:required_slots]
        
        return skeleton_df

    # ------------------------------------------------------------------
    # Generate counterbalanced stimulus design (trial conditions)
    # ------------------------------------------------------------------
    def generate_counterbalanced_design(self, participant_id):
        """
        Generate a counterbalanced design (trial conditions) for one participant.
        
        Returns a DataFrame with columns: participant_id, trial_type, stimulus_type,
        soa_value_ms, jitter_ms, is_tactile, trial_number.
        """
        # Set seed based on participant ID for reproducibility.
        random.seed(self.base_seed + participant_id)
        np.random.seed(self.base_seed + participant_id)
        
        soa_conditions = self.config['soa_conditions_ms']
        jitter_options = self.config['jitter_options_ms']
        stimulus_types = LOOMING_STIMULI
        
        all_trials = []
        
        # Build main conditions (inhalation, exhalation, baseline)
        for condition, count in self.trial_counts.items():
            if condition == 'catch' or count == 0:
                continue
            
            # Create all combinations of (stimulus type × SOA)
            combo_list = []
            for stim_type in stimulus_types:
                for soa in soa_conditions:
                    for _ in range(self.config['repetitions']):
                        combo_list.append((stim_type, soa))
            random.shuffle(combo_list)
            
            for stim_type, soa in combo_list:
                all_trials.append({
                    'participant_id': participant_id,
                    'trial_type': condition,  # inhalation, exhalation, baseline
                    'stimulus_type': stim_type,
                    'soa_value_ms': soa,
                    'jitter_ms': random.choice(jitter_options),
                    'is_tactile': True  # main trials have tactile component
                })
        
        # Add catch trials (looming only, no tactile)
        catch_count = self.trial_counts.get('catch', 0)
        if catch_count > 0:
            catch_combos = []
            stim_counts = {stim: 0 for stim in stimulus_types}
            soa_counts = {soa: 0 for soa in soa_conditions}
            while len(catch_combos) < catch_count:
                min_stim = min(stim_counts, key=stim_counts.get)
                min_soa = min(soa_counts, key=soa_counts.get)
                catch_combos.append((min_stim, min_soa))
                stim_counts[min_stim] += 1
                soa_counts[min_soa] += 1
            for stim_type, soa in catch_combos:
                all_trials.append({
                    'participant_id': participant_id,
                    'trial_type': 'catch',
                    'stimulus_type': stim_type,
                    'soa_value_ms': soa,
                    'jitter_ms': random.choice(jitter_options),
                    'is_tactile': False
                })
        
        design_df = pd.DataFrame(all_trials)
        # Shuffle the design conditions (the assignment pool)
        design_df = design_df.sample(frac=1, random_state=participant_id*100).reset_index(drop=True)
        # (We will assign new trial numbers after merging.)
        design_df['trial_number'] = range(1, len(design_df) + 1)
        
        if self.config['debug_mode']:
            print("\nTrial type distribution:")
            print(design_df['trial_type'].value_counts())
            print("\nStimulus type distribution by trial type:")
            for trial_type in design_df['trial_type'].unique():
                if trial_type == 'baseline':
                    continue
                subset = design_df[design_df['trial_type'] == trial_type]
                print(f"  {trial_type}: {subset['stimulus_type'].value_counts().to_dict()}")
            print("\nSOA distribution by trial type:")
            for trial_type in design_df['trial_type'].unique():
                if trial_type == 'baseline':
                    continue
                subset = design_df[design_df['trial_type'] == trial_type]
                print(f"  {trial_type}: {subset['soa_value_ms'].value_counts().to_dict()}")
        
        return design_df

    # ------------------------------------------------------------------
    # Assign scaffold retention slots to trials while preserving scaffold order
    # ------------------------------------------------------------------
    def assign_breath_holds(self, design_df):
        """
        Assign each trial (from the counterbalanced design) to a retention slot 
        from the scaffold CSV. The scaffold order (and thus timestamps and trial numbers) 
        remains sequential. For each scaffold row (in order), we choose a design condition 
        from the remaining pool, preferring one whose trial_type matches the scaffold’s breathphase.
        """
        required_slots = len(design_df)
        scaffold_df = self.load_retention_slots(required_slots)
        
        # Convert design_df to a list (pool) of condition dictionaries.
        design_pool = design_df.to_dict('records')
        assigned_conditions = []
        
        # Iterate over the scaffold rows in order.
        for idx, row in scaffold_df.iterrows():
            bp = row['breathphase'].lower()
            # Try to select a candidate whose trial_type matches the breathphase.
            if bp == 'inhale':
                candidates = [cond for cond in design_pool if cond['trial_type'] == 'inhalation']
            elif bp == 'exhale':
                candidates = [cond for cond in design_pool if cond['trial_type'] == 'exhalation']
            else:
                candidates = []
            # If no matching candidates are available, use any remaining condition.
            if not candidates:
                candidates = design_pool
            # Randomly select one candidate.
            chosen = random.choice(candidates)
            design_pool.remove(chosen)
            # Merge the scaffold row values (which are sequential) into the chosen condition.
            chosen.update({
                'retentionslot_timestamp': row['retentionslot_timestamp'],
                'milliseconds': row['milliseconds'],
                'sample_index': row['sample_index'],
                'description': row['description'],
                'breathphase': row['breathphase']
            })
            assigned_conditions.append(chosen)
        
        # Create the final DataFrame (which follows the scaffold order).
        final_df = pd.DataFrame(assigned_conditions)
        # Reassign trial numbers sequentially.
        final_df['trial_number'] = range(1, len(final_df) + 1)
        
        # Compute additional timing variables.
        final_df['jittered_ms'] = final_df['milliseconds'] + final_df['jitter_ms']
        final_df['looming_stimulus_timestamp'] = final_df['jittered_ms'].apply(convert_ms_to_timestamp)
        non_catch_mask = final_df['trial_type'] != 'catch'
        final_df.loc[non_catch_mask, 'soa_ms'] = final_df.loc[non_catch_mask, 'jittered_ms'] + final_df.loc[non_catch_mask, 'soa_value_ms']
        final_df.loc[non_catch_mask, 'tactile_stimulus_timestamp'] = final_df.loc[non_catch_mask, 'soa_ms'].apply(convert_ms_to_timestamp)
        
        return final_df

    # ------------------------------------------------------------------
    # Finalize the design CSV (rename columns, handle special cases)
    # ------------------------------------------------------------------
    def finalize_design_csv(self, design_df):
        """
        Finalize the design DataFrame by ensuring final column order and
        handling special cases for baseline and catch trials.
        """
        # For baseline trials: no looming stimulus.
        baseline_mask = (design_df['trial_type'] == 'baseline')
        design_df.loc[baseline_mask, 'stimulus_type'] = 'none'
        design_df.loc[baseline_mask, 'looming_stimulus_timestamp'] = None
        
        # For catch trials: no tactile stimulus.
        catch_mask = (design_df['trial_type'] == 'catch')
        design_df.loc[catch_mask, 'tactile_stimulus_timestamp'] = None
        
        # Expected response: True for all except catch trials.
        design_df['expected_response'] = ~design_df['trial_type'].isin(['catch'])
        
        # Define final column order.
        final_cols = [
            'participant_id',
            'trial_number',
            'trial_type',
            'retentionslot_timestamp',
            'looming_stimulus_timestamp',
            'tactile_stimulus_timestamp',
            'stimulus_type',
            'soa_value_ms',
            'jitter_ms',
            'sample_index',
            'description',
            'breathphase',
            'expected_response'
        ]
        final_cols = [col for col in final_cols if col in design_df.columns]
        design_df = design_df[final_cols]
        
        self._validate_design(design_df)
        return design_df
    
    # ------------------------------------------------------------------
    # Validation: Check distributions and transitions
    # ------------------------------------------------------------------
    def _validate_design(self, design_df):
        if not self.config['debug_mode']:
            return
        print("\n=== DESIGN VALIDATION ===")
        print("Trial type counts:")
        print(design_df['trial_type'].value_counts())
        print("\nStimulus distribution by trial type:")
        for trial_type in design_df['trial_type'].unique():
            if trial_type == 'baseline':
                continue
            subset = design_df[design_df['trial_type'] == trial_type]
            print(f"  {trial_type}: {subset['stimulus_type'].value_counts().to_dict()}")
        print("\nSOA distribution by trial type:")
        for trial_type in design_df['trial_type'].unique():
            if trial_type == 'baseline':
                continue
            subset = design_df[design_df['trial_type'] == trial_type]
            print(f"  {trial_type}: {subset['soa_value_ms'].value_counts().to_dict()}")
        print("\nBreath phase distribution:")
        if 'breathphase' in design_df.columns:
            print(design_df['breathphase'].value_counts())
        trial_sequence = design_df['trial_type'].tolist()
        consecutive_counts = {}
        current_type = trial_sequence[0]
        current_count = 1
        for t in trial_sequence[1:]:
            if t == current_type:
                current_count += 1
            else:
                key = f"{current_type}_{current_count}"
                consecutive_counts[key] = consecutive_counts.get(key, 0) + 1
                current_type = t
                current_count = 1
        key = f"{current_type}_{current_count}"
        consecutive_counts[key] = consecutive_counts.get(key, 0) + 1
        print("\nRuns of consecutive trials:")
        for key, count in sorted(consecutive_counts.items()):
            print(f"  {key}: {count}")
    
    # ------------------------------------------------------------------
    # Process a single participant: generate design, assign slots, finalize, save CSV
    # ------------------------------------------------------------------
    def process_participant(self, participant_id):
        try:
            if self.config['debug_mode']:
                print(f"\n=== Processing participant {participant_id} ===")
            design_df = self.generate_counterbalanced_design(participant_id)
            design_df = self.assign_breath_holds(design_df)
            design_df = self.finalize_design_csv(design_df)
            out_path = os.path.join(self.experiment_log_dir, f"participant_{participant_id}_design.csv")
            design_df.to_csv(out_path, index=False)
            if self.config['debug_mode']:
                print(f"Saved participant {participant_id} design to {out_path}")
            return True, out_path
        except Exception as e:
            print(f"Error processing participant {participant_id}: {e}")
            traceback.print_exc()
            return False, None
    
    # ------------------------------------------------------------------
    # Run design generation for multiple participants
    # ------------------------------------------------------------------
    def run(self, participant_ids=None, num_participants=None):
        if participant_ids is None and num_participants is None:
            num_participants = self.config.get('num_participants', 5)
        if participant_ids is None:
            participant_ids = list(range(num_participants))
        if self.config['debug_mode']:
            print("=== Starting PPSDesignGenerator ===")
            print(f"Repetitions: {self.config['repetitions']}")
            print(f"Participants: {participant_ids}")
            print(f"Box breathing settings: {self.config['box_breathing']}")
        results = {}
        for pid in participant_ids:
            success, path = self.process_participant(pid)
            results[pid] = {'success': success, 'csv_path': path}
        return results

# ----------------------------------------------------------------------
# AUDIO GENERATION PLACEHOLDER FUNCTION
# ----------------------------------------------------------------------
def generate_audio_from_design(design_csv_path, output_dir, base_audio_path=None):
    """
    Generate audio files from a design CSV.
    (Placeholder function – implement audio generation as needed.)
    """
    print(f"Audio generation from {design_csv_path} not implemented yet")
    return None, None

# ----------------------------------------------------------------------
# MAIN FUNCTION
# ----------------------------------------------------------------------
def main():
    """Main function to run the design generator."""
    generator = PPSDesignGenerator(CONFIG)
    results = generator.run()
    print("\n=== Generation Summary ===")
    for pid, info in results.items():
        if info['success']:
            print(f"Participant {pid}: CSV generated at {info['csv_path']}")
        else:
            print(f"Participant {pid}: ERROR - no CSV generated.")
    print("\nTo generate audio files from these designs, implement the audio generation function.")

if __name__ == "__main__":
    main()


Trial count calculation:
  inhalation: 80 trials
  exhalation: 80 trials
  baseline: 80 trials
  catch: 24 trials
  Total: 264 trials
Initialized PPSDesignGenerator with:
  - Repetitions: 4
  - Conditions: ['inhalation', 'exhalation', 'baseline']
  - SOA values: [190, 400, 700, 1000, 1500]
  - Total trials per participant: 264
  - Output directory: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog
=== Starting PPSDesignGenerator ===
Repetitions: 4
Participants: [0, 1, 2, 3, 4]
Box breathing settings: {'cycle_duration_sec': 8, 'intro_duration_sec': 60, 'outro_duration_sec': 30, 'sample_rate': 48000}

=== Processing participant 0 ===

Trial type distribution:
trial_type
exhalation    80
inhalation    80
baseline      80
catch         24
Name: count, dtype: int64

Stimulus type distribution by trial type:
  exhalation: {'brown': 20, 'pink': 20, 'blue': 20, 'white': 20}
  catch: {'blue': 6, 'white': 6, 'brown': 6, 'pink': 6}
  inhalation: {'pink': 2

Traceback (most recent call last):
  File "C:\Users\cogpsy-vrlab\AppData\Local\Temp\ipykernel_7528\3164721178.py", line 426, in process_participant
    design_df.to_csv(out_path, index=False)
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\site-packages\pandas\util\_decorators.py", line 333, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\site-packages\pandas\core\generic.py", line 3967, in to_csv
    return DataFrameRenderer(formatter).to_csv(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\site-packages\pandas\io\formats\format.py", line 1014, in to_csv
    csv_formatter.save()
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\site-packages\pandas\io\formats\csvs.py", line 251, in save
    with get_handle(
         ^^^^^^^^^^^
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\site-packages\pandas\io\common.py", line 873, in get_handle
    handle = open(
             ^^^^^
PermissionEr

In [12]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
PPS Design Generator

This script generates participant‐specific experimental design CSVs for a 
Peripersonal Space (PPS) experiment. Each design CSV is built upon a 
“scaffold” CSV that contains the retention slot (breathhold) timestamps 
from a box‐breathing audio file. If needed, additional retention slots are 
appended (using fixed 8‑second increments) so that every trial can be assigned 
to a specific retention slot. Then, a counterbalanced set of trial conditions 
(e.g., trial type, looming stimulus type, SOA, jitter) is randomly allocated 
to each scaffold row while preserving the sequential order of the scaffold. 
The final output retains the original column order from the scaffold, with 
new trial-specific variables appended as new columns.

Author: AI Assistant
"""

import os
import numpy as np
import pandas as pd
import random
import traceback

# ----------------------------------------------------------------------
# Helper function: Convert milliseconds to "MM:SS.s" format
# ----------------------------------------------------------------------
def convert_ms_to_timestamp(ms):
    m = int(ms // 60000)
    s = (ms % 60000) / 1000.0
    return f"{m:02}:{s:.1f}"

# ----------------------------------------------------------------------
# CONFIGURATION PARAMETERS
# ----------------------------------------------------------------------
CONFIG = {
    'repetitions': 4,            # How many times each SOA×stimulus combination repeats
    'num_participants': 5,       # Default number of participants
    'included_conditions': {
        'inhalation': True,
        'exhalation': True,
        'baseline': True,
    },
    'catch_trial_percentage': 10,
    'soa_conditions_ms': [190, 400, 700, 1000, 1500],
    'jitter_options_ms': [100, 200, 300, 400, 500],
    'debug_mode': True,
    'paths': {
        # Directory where participant-specific CSV files will be saved
        'experiment_log_dir': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog",
    },
    # Path to the original scaffold CSV containing retention (breathhold) timestamps.
    'skeleton_csv_path': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_timestamps.csv",
    'box_breathing': {
        'cycle_duration_sec': 8,      # Duration of one breath cycle in seconds
        # Intro/outro are not used here because the scaffold drives timing.
        'intro_duration_sec': 60,
        'outro_duration_sec': 30,
        'sample_rate': 48000,           # Audio sample rate (samples per second)
    }
}

# Noise types for looming stimuli (can be any labels)
LOOMING_STIMULI = ['blue', 'brown', 'pink', 'white']

# ----------------------------------------------------------------------
# CLASS: PPSDesignGenerator
# ----------------------------------------------------------------------
class PPSDesignGenerator:
    """Generates participant-specific CSV design files for PPS experiments."""
    
    def __init__(self, config=None):
        self.config = config or CONFIG
        
        # Prepare output directory
        self.experiment_log_dir = self.config['paths']['experiment_log_dir']
        os.makedirs(self.experiment_log_dir, exist_ok=True)
        
        # Calculate trial counts based on included conditions and repetitions
        self.trial_counts = self._calculate_trial_counts()
        self.conditions = [cond for cond, inc in self.config['included_conditions'].items() if inc]
        
        # Set seed for reproducibility
        self.base_seed = 42
        
        if self.config['debug_mode']:
            total_trials = sum(self.trial_counts.values())
            print("Initialized PPSDesignGenerator with:")
            print(f"  - Repetitions: {self.config['repetitions']}")
            print(f"  - Conditions: {self.conditions}")
            print(f"  - SOA values: {self.config['soa_conditions_ms']}")
            print(f"  - Total trials per participant: {total_trials}")
            print(f"  - Output directory: {self.experiment_log_dir}")
    
    def _calculate_trial_counts(self):
        """
        Calculate the number of trials for each condition (inhalation, exhalation, baseline)
        and include catch trials.
        """
        reps = self.config['repetitions']
        soa_count = len(self.config['soa_conditions_ms'])
        stimulus_types = LOOMING_STIMULI
        catch_percentage = self.config['catch_trial_percentage']
        
        trial_counts = {}
        for condition, include in self.config['included_conditions'].items():
            if include:
                # (# noise types) × (# SOA) × (reps)
                trial_counts[condition] = len(stimulus_types) * soa_count * reps
        
        regular_trials = sum(trial_counts.values())
        catch_trials = int(np.ceil(regular_trials * catch_percentage / 100))
        trial_counts['catch'] = catch_trials
        
        if self.config['debug_mode']:
            print("Trial count calculation:")
            for cond, count in trial_counts.items():
                print(f"  {cond}: {count} trials")
            print(f"  Total: {sum(trial_counts.values())} trials")
        
        return trial_counts

    # ------------------------------------------------------------------
    # Load the scaffold CSV and extend it if needed
    # ------------------------------------------------------------------
    def load_retention_slots(self, required_slots):
        """
        Load the scaffold CSV of retention (breathhold) timestamps and extend it with 
        additional rows (using fixed 8‑second increments) if the number of rows is less than required_slots.
        
        Returns:
            DataFrame with columns: retentionslot_timestamp, milliseconds, sample_index, description, breathphase
        """
        skeleton_path = self.config['skeleton_csv_path']
        try:
            # Attempt to auto-detect the delimiter using the Python engine.
            skeleton_df = pd.read_csv(skeleton_path, sep=None, engine='python')
            if len(skeleton_df.columns) == 1:
                skeleton_df = pd.read_csv(skeleton_path, delimiter="\t")
            # Standardize column names: strip whitespace and convert to lowercase.
            skeleton_df.columns = [col.strip().lower() for col in skeleton_df.columns]
            # Save the scaffold column order.
            self.scaffold_order = skeleton_df.columns.tolist()
            if self.config['debug_mode']:
                print("Scaffold CSV columns:", self.scaffold_order)
        except Exception as e:
            raise ValueError(f"Error reading scaffold CSV from {skeleton_path}: {e}")
        
        current_slots = len(skeleton_df)
        required_slots = int(required_slots)
        sample_rate = self.config['box_breathing']['sample_rate']
        cycle_sec = self.config['box_breathing']['cycle_duration_sec']
        
        if self.config['debug_mode']:
            print(f"Loaded scaffold CSV with {current_slots} retention slots.")
        
        # Extend the scaffold if more slots are needed.
        if current_slots < required_slots:
            last_row = skeleton_df.iloc[-1]
            last_ms = float(last_row['milliseconds'])
            last_sample_index = int(last_row['sample_index'])
            new_rows = []
            for i in range(required_slots - current_slots):
                new_ms = last_ms + (cycle_sec * 1000) * (i + 1)
                new_sample_index = last_sample_index + int(cycle_sec * sample_rate * (i + 1))
                new_timestamp = convert_ms_to_timestamp(new_ms)
                new_breathphase = 'inhale' if ((current_slots + i) % 2 == 0) else 'exhale'
                new_description = f"Additional hold {i+1}"
                new_rows.append({
                    'retentionslot_timestamp': new_timestamp,
                    'milliseconds': new_ms,
                    'sample_index': new_sample_index,
                    'description': new_description,
                    'breathphase': new_breathphase
                })
            if new_rows:
                skeleton_df = pd.concat([skeleton_df, pd.DataFrame(new_rows)], ignore_index=True)
            if self.config['debug_mode']:
                print(f"Extended scaffold CSV to {len(skeleton_df)} retention slots.")
        
        # If there are more slots than required, take only the first required_slots rows.
        if len(skeleton_df) > required_slots:
            skeleton_df = skeleton_df.iloc[:required_slots]
        
        return skeleton_df

    # ------------------------------------------------------------------
    # Generate counterbalanced stimulus design (trial conditions)
    # ------------------------------------------------------------------
    def generate_counterbalanced_design(self, participant_id):
        """
        Generate a counterbalanced design (trial conditions) for one participant.
        
        Returns a DataFrame with columns: participant_id, trial_type, stimulus_type,
        soa_value_ms, jitter_ms, is_tactile, trial_number.
        """
        random.seed(self.base_seed + participant_id)
        np.random.seed(self.base_seed + participant_id)
        
        soa_conditions = self.config['soa_conditions_ms']
        jitter_options = self.config['jitter_options_ms']
        stimulus_types = LOOMING_STIMULI
        
        all_trials = []
        
        # Build main conditions (inhalation, exhalation, baseline)
        for condition, count in self.trial_counts.items():
            if condition == 'catch' or count == 0:
                continue
            
            combo_list = []
            for stim_type in stimulus_types:
                for soa in soa_conditions:
                    for _ in range(self.config['repetitions']):
                        combo_list.append((stim_type, soa))
            random.shuffle(combo_list)
            
            for stim_type, soa in combo_list:
                all_trials.append({
                    'participant_id': participant_id,
                    'trial_type': condition,  # inhalation, exhalation, baseline
                    'stimulus_type': stim_type,
                    'soa_value_ms': soa,
                    'jitter_ms': random.choice(jitter_options),
                    'is_tactile': True  # main trials have tactile component
                })
        
        # Add catch trials (looming only, no tactile)
        catch_count = self.trial_counts.get('catch', 0)
        if catch_count > 0:
            catch_combos = []
            stim_counts = {stim: 0 for stim in stimulus_types}
            soa_counts = {soa: 0 for soa in soa_conditions}
            while len(catch_combos) < catch_count:
                min_stim = min(stim_counts, key=stim_counts.get)
                min_soa = min(soa_counts, key=soa_counts.get)
                catch_combos.append((min_stim, min_soa))
                stim_counts[min_stim] += 1
                soa_counts[min_soa] += 1
            for stim_type, soa in catch_combos:
                all_trials.append({
                    'participant_id': participant_id,
                    'trial_type': 'catch',
                    'stimulus_type': stim_type,
                    'soa_value_ms': soa,
                    'jitter_ms': random.choice(jitter_options),
                    'is_tactile': False
                })
        
        design_df = pd.DataFrame(all_trials)
        design_df = design_df.sample(frac=1, random_state=participant_id*100).reset_index(drop=True)
        design_df['trial_number'] = range(1, len(design_df) + 1)
        
        if self.config['debug_mode']:
            print("\nTrial type distribution:")
            print(design_df['trial_type'].value_counts())
            print("\nStimulus type distribution by trial type:")
            for trial_type in design_df['trial_type'].unique():
                if trial_type == 'baseline':
                    continue
                subset = design_df[design_df['trial_type'] == trial_type]
                print(f"  {trial_type}: {subset['stimulus_type'].value_counts().to_dict()}")
            print("\nSOA distribution by trial type:")
            for trial_type in design_df['trial_type'].unique():
                if trial_type == 'baseline':
                    continue
                subset = design_df[design_df['trial_type'] == trial_type]
                print(f"  {trial_type}: {subset['soa_value_ms'].value_counts().to_dict()}")
        
        return design_df

    # ------------------------------------------------------------------
    # Assign scaffold retention slots to trials while preserving scaffold order
    # ------------------------------------------------------------------
    def assign_breath_holds(self, design_df):
        """
        Assign each trial (from the counterbalanced design) to a retention slot 
        from the scaffold CSV. The scaffold order (and thus timestamps and trial numbers) 
        remains sequential. For each scaffold row (in order), a design condition is selected 
        (preferentially matching the scaffold's breathphase), then merged with the scaffold row.
        """
        required_slots = len(design_df)
        scaffold_df = self.load_retention_slots(required_slots)
        
        design_pool = design_df.to_dict('records')
        assigned_conditions = []
        
        # Iterate over scaffold rows in sequential order.
        for idx, row in scaffold_df.iterrows():
            bp = row['breathphase'].lower()
            if bp == 'inhale':
                candidates = [cond for cond in design_pool if cond['trial_type'] == 'inhalation']
            elif bp == 'exhale':
                candidates = [cond for cond in design_pool if cond['trial_type'] == 'exhalation']
            else:
                candidates = []
            if not candidates:
                candidates = design_pool
            chosen = random.choice(candidates)
            design_pool.remove(chosen)
            # Merge scaffold row into chosen condition.
            chosen.update({
                'retentionslot_timestamp': row['retentionslot_timestamp'],
                'milliseconds': row['milliseconds'],
                'sample_index': row['sample_index'],
                'description': row['description'],
                'breathphase': row['breathphase']
            })
            assigned_conditions.append(chosen)
        
        final_df = pd.DataFrame(assigned_conditions)
        final_df['trial_number'] = range(1, len(final_df) + 1)
        
        final_df['jittered_ms'] = final_df['milliseconds'] + final_df['jitter_ms']
        final_df['looming_stimulus_timestamp'] = final_df['jittered_ms'].apply(convert_ms_to_timestamp)
        non_catch_mask = final_df['trial_type'] != 'catch'
        final_df.loc[non_catch_mask, 'soa_ms'] = final_df.loc[non_catch_mask, 'jittered_ms'] + final_df.loc[non_catch_mask, 'soa_value_ms']
        final_df.loc[non_catch_mask, 'tactile_stimulus_timestamp'] = final_df.loc[non_catch_mask, 'soa_ms'].apply(convert_ms_to_timestamp)
        
        return final_df

    # ------------------------------------------------------------------
    # Finalize the design CSV (retain scaffold column order, add new columns)
    # ------------------------------------------------------------------
    def finalize_design_csv(self, design_df):
        """
        Finalize the design DataFrame by retaining the scaffold column order from the 
        original file and appending new variables as additional columns.
        """
        # Special cases:
        baseline_mask = (design_df['trial_type'] == 'baseline')
        design_df.loc[baseline_mask, 'stimulus_type'] = 'none'
        design_df.loc[baseline_mask, 'looming_stimulus_timestamp'] = None
        
        catch_mask = (design_df['trial_type'] == 'catch')
        design_df.loc[catch_mask, 'tactile_stimulus_timestamp'] = None
        
        design_df['expected_response'] = ~design_df['trial_type'].isin(['catch'])
        
        # Retain scaffold column order.
        scaffold_cols = self.scaffold_order if hasattr(self, 'scaffold_order') else ['retentionslot_timestamp', 'milliseconds', 'sample_index', 'description', 'breathphase']
        # Define new columns in a desired order.
        new_cols = ['participant_id', 'trial_number', 'trial_type', 'stimulus_type', 'soa_value_ms', 'jitter_ms', 'is_tactile', 'looming_stimulus_timestamp', 'tactile_stimulus_timestamp', 'soa_ms', 'jittered_ms', 'expected_response']
        # Final order: scaffold columns first, then any new columns not already in scaffold.
        final_order = scaffold_cols + [col for col in new_cols if col not in scaffold_cols and col in design_df.columns]
        # Include any remaining columns.
        remaining = [col for col in design_df.columns if col not in final_order]
        final_order.extend(remaining)
        design_df = design_df[final_order]
        
        self._validate_design(design_df)
        return design_df

    # ------------------------------------------------------------------
    # Validation: Check distributions and transitions
    # ------------------------------------------------------------------
    def _validate_design(self, design_df):
        if not self.config['debug_mode']:
            return
        print("\n=== DESIGN VALIDATION ===")
        print("Trial type counts:")
        print(design_df['trial_type'].value_counts())
        print("\nStimulus distribution by trial type:")
        for trial_type in design_df['trial_type'].unique():
            if trial_type == 'baseline':
                continue
            subset = design_df[design_df['trial_type'] == trial_type]
            print(f"  {trial_type}: {subset['stimulus_type'].value_counts().to_dict()}")
        print("\nSOA distribution by trial type:")
        for trial_type in design_df['trial_type'].unique():
            if trial_type == 'baseline':
                continue
            subset = design_df[design_df['trial_type'] == trial_type]
            print(f"  {trial_type}: {subset['soa_value_ms'].value_counts().to_dict()}")
        print("\nBreath phase distribution:")
        if 'breathphase' in design_df.columns:
            print(design_df['breathphase'].value_counts())
        trial_sequence = design_df['trial_type'].tolist()
        consecutive_counts = {}
        current_type = trial_sequence[0]
        current_count = 1
        for t in trial_sequence[1:]:
            if t == current_type:
                current_count += 1
            else:
                key = f"{current_type}_{current_count}"
                consecutive_counts[key] = consecutive_counts.get(key, 0) + 1
                current_type = t
                current_count = 1
        key = f"{current_type}_{current_count}"
        consecutive_counts[key] = consecutive_counts.get(key, 0) + 1
        print("\nRuns of consecutive trials:")
        for key, count in sorted(consecutive_counts.items()):
            print(f"  {key}: {count}")
    
    # ------------------------------------------------------------------
    # Process a single participant: generate design, assign slots, finalize, save CSV
    # ------------------------------------------------------------------
    def process_participant(self, participant_id):
        try:
            if self.config['debug_mode']:
                print(f"\n=== Processing participant {participant_id} ===")
            design_df = self.generate_counterbalanced_design(participant_id)
            design_df = self.assign_breath_holds(design_df)
            design_df = self.finalize_design_csv(design_df)
            out_path = os.path.join(self.experiment_log_dir, f"participant_{participant_id}_design.csv")
            design_df.to_csv(out_path, index=False)
            if self.config['debug_mode']:
                print(f"Saved participant {participant_id} design to {out_path}")
            return True, out_path
        except Exception as e:
            print(f"Error processing participant {participant_id}: {e}")
            traceback.print_exc()
            return False, None
    
    # ------------------------------------------------------------------
    # Run design generation for multiple participants
    # ------------------------------------------------------------------
    def run(self, participant_ids=None, num_participants=None):
        if participant_ids is None and num_participants is None:
            num_participants = self.config.get('num_participants', 5)
        if participant_ids is None:
            participant_ids = list(range(num_participants))
        if self.config['debug_mode']:
            print("=== Starting PPSDesignGenerator ===")
            print(f"Repetitions: {self.config['repetitions']}")
            print(f"Participants: {participant_ids}")
            print(f"Box breathing settings: {self.config['box_breathing']}")
        results = {}
        for pid in participant_ids:
            success, path = self.process_participant(pid)
            results[pid] = {'success': success, 'csv_path': path}
        return results

# ----------------------------------------------------------------------
# AUDIO GENERATION PLACEHOLDER FUNCTION
# ----------------------------------------------------------------------
def generate_audio_from_design(design_csv_path, output_dir, base_audio_path=None):
    """
    Generate audio files from a design CSV.
    (Placeholder function – implement audio generation as needed.)
    """
    print(f"Audio generation from {design_csv_path} not implemented yet")
    return None, None

# ----------------------------------------------------------------------
# MAIN FUNCTION
# ----------------------------------------------------------------------
def main():
    """Main function to run the design generator."""
    generator = PPSDesignGenerator(CONFIG)
    results = generator.run()
    print("\n=== Generation Summary ===")
    for pid, info in results.items():
        if info['success']:
            print(f"Participant {pid}: CSV generated at {info['csv_path']}")
        else:
            print(f"Participant {pid}: ERROR - no CSV generated.")
    print("\nTo generate audio files from these designs, implement the audio generation function.")

if __name__ == "__main__":
    main()


Trial count calculation:
  inhalation: 80 trials
  exhalation: 80 trials
  baseline: 80 trials
  catch: 24 trials
  Total: 264 trials
Initialized PPSDesignGenerator with:
  - Repetitions: 4
  - Conditions: ['inhalation', 'exhalation', 'baseline']
  - SOA values: [190, 400, 700, 1000, 1500]
  - Total trials per participant: 264
  - Output directory: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog
=== Starting PPSDesignGenerator ===
Repetitions: 4
Participants: [0, 1, 2, 3, 4]
Box breathing settings: {'cycle_duration_sec': 8, 'intro_duration_sec': 60, 'outro_duration_sec': 30, 'sample_rate': 48000}

=== Processing participant 0 ===

Trial type distribution:
trial_type
exhalation    80
inhalation    80
baseline      80
catch         24
Name: count, dtype: int64

Stimulus type distribution by trial type:
  exhalation: {'brown': 20, 'pink': 20, 'blue': 20, 'white': 20}
  catch: {'blue': 6, 'white': 6, 'brown': 6, 'pink': 6}
  inhalation: {'pink': 2

In [14]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
PPS Design Generator

This script generates participant‐specific experimental design CSVs for a 
Peripersonal Space (PPS) experiment. Each design CSV is built upon a 
“scaffold” CSV that contains the retention slot (breathhold) timestamps 
from a box‐breathing audio file. If needed, additional retention slots are 
appended (using fixed 8‑second increments) so that every trial can be assigned 
to a specific retention slot. Then, a counterbalanced set of trial conditions 
(e.g., trial type, looming stimulus type, SOA, jitter) is randomly allocated 
to each scaffold row while preserving the sequential order of the scaffold. 
The final output retains the original column order from the scaffold, with 
new trial-specific variables appended as new columns.

Trial Types:
  - Inhalation: looming stimulus cue and tactile stimulus.
  - Exhalation: looming stimulus cue and tactile stimulus.
  - Baseline: tactile stimulus only (looming stimulus is NA).
  - Catch: looming stimulus cue only (no tactile stimulus).

Author: AI Assistant
"""

import os
import numpy as np
import pandas as pd
import random
import traceback

# ----------------------------------------------------------------------
# Helper function: Convert milliseconds to "MM:SS.s" format
# ----------------------------------------------------------------------
def convert_ms_to_timestamp(ms):
    m = int(ms // 60000)
    s = (ms % 60000) / 1000.0
    return f"{m:02}:{s:.1f}"

# ----------------------------------------------------------------------
# CONFIGURATION PARAMETERS
# ----------------------------------------------------------------------
CONFIG = {
    'repetitions': 4,            # How many times each SOA×stimulus combination repeats
    'num_participants': 5,       # Default number of participants
    'included_conditions': {
        'inhalation': True,
        'exhalation': True,
        'baseline': True,
    },
    'catch_trial_percentage': 10,
    'soa_conditions_ms': [190, 400, 700, 1000, 1500],
    'jitter_options_ms': [100, 200, 300, 400, 500],
    'debug_mode': True,
    'paths': {
        # Directory where participant-specific CSV files will be saved
        'experiment_log_dir': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog",
    },
    # Path to the original scaffold CSV containing retention (breathhold) timestamps.
    'skeleton_csv_path': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_timestamps.csv",
    'box_breathing': {
        'cycle_duration_sec': 8,      # Duration of one breath cycle in seconds
        # Intro/outro are not used here because the scaffold drives timing.
        'intro_duration_sec': 60,
        'outro_duration_sec': 30,
        'sample_rate': 48000,           # Audio sample rate (samples per second)
    }
}

# Noise types for looming stimuli (can be any labels)
LOOMING_STIMULI = ['blue', 'brown', 'pink', 'white']

# ----------------------------------------------------------------------
# CLASS: PPSDesignGenerator
# ----------------------------------------------------------------------
class PPSDesignGenerator:
    """Generates participant-specific CSV design files for PPS experiments."""
    
    def __init__(self, config=None):
        self.config = config or CONFIG
        
        # Prepare output directory
        self.experiment_log_dir = self.config['paths']['experiment_log_dir']
        os.makedirs(self.experiment_log_dir, exist_ok=True)
        
        # Calculate trial counts based on included conditions and repetitions.
        # For each condition (inhalation, exhalation, baseline) the count is:
        # (# noise types) × (# SOA) × (repetitions). Then, 10% catch trials are added.
        self.trial_counts = self._calculate_trial_counts()
        self.conditions = [cond for cond, inc in self.config['included_conditions'].items() if inc]
        
        # Set seed for reproducibility.
        self.base_seed = 42
        
        if self.config['debug_mode']:
            total_trials = sum(self.trial_counts.values())
            print("Initialized PPSDesignGenerator with:")
            print(f"  - Repetitions: {self.config['repetitions']}")
            print(f"  - Conditions: {self.conditions}")
            print(f"  - SOA values: {self.config['soa_conditions_ms']}")
            print(f"  - Total trials per participant: {total_trials}")
            print(f"  - Output directory: {self.experiment_log_dir}")
    
    def _calculate_trial_counts(self):
        reps = self.config['repetitions']
        soa_count = len(self.config['soa_conditions_ms'])
        stimulus_types = LOOMING_STIMULI
        catch_percentage = self.config['catch_trial_percentage']
        
        trial_counts = {}
        for condition, include in self.config['included_conditions'].items():
            if include:
                # For inhalation, exhalation, baseline:
                trial_counts[condition] = len(stimulus_types) * soa_count * reps
        regular_trials = sum(trial_counts.values())
        catch_trials = int(np.ceil(regular_trials * catch_percentage / 100))
        trial_counts['catch'] = catch_trials
        
        if self.config['debug_mode']:
            print("Trial count calculation:")
            for cond, count in trial_counts.items():
                print(f"  {cond}: {count} trials")
            print(f"  Total: {sum(trial_counts.values())} trials")
        
        return trial_counts

    # ------------------------------------------------------------------
    # Load the scaffold CSV and extend it if needed.
    # Retain the original column order.
    # ------------------------------------------------------------------
    def load_retention_slots(self, required_slots):
        skeleton_path = self.config['skeleton_csv_path']
        try:
            skeleton_df = pd.read_csv(skeleton_path, sep=None, engine='python')
            if len(skeleton_df.columns) == 1:
                skeleton_df = pd.read_csv(skeleton_path, delimiter="\t")
            skeleton_df.columns = [col.strip().lower() for col in skeleton_df.columns]
            # Save the scaffold column order.
            self.scaffold_order = skeleton_df.columns.tolist()
            if self.config['debug_mode']:
                print("Scaffold CSV columns:", self.scaffold_order)
        except Exception as e:
            raise ValueError(f"Error reading scaffold CSV from {skeleton_path}: {e}")
        
        current_slots = len(skeleton_df)
        required_slots = int(required_slots)
        sample_rate = self.config['box_breathing']['sample_rate']
        cycle_sec = self.config['box_breathing']['cycle_duration_sec']
        
        if self.config['debug_mode']:
            print(f"Loaded scaffold CSV with {current_slots} retention slots.")
        
        if current_slots < required_slots:
            last_row = skeleton_df.iloc[-1]
            last_ms = float(last_row['milliseconds'])
            last_sample_index = int(last_row['sample_index'])
            new_rows = []
            for i in range(required_slots - current_slots):
                new_ms = last_ms + (cycle_sec * 1000) * (i + 1)
                new_sample_index = last_sample_index + int(cycle_sec * sample_rate * (i + 1))
                new_timestamp = convert_ms_to_timestamp(new_ms)
                new_breathphase = 'inhale' if ((current_slots + i) % 2 == 0) else 'exhale'
                new_description = f"Additional hold {i+1}"
                new_rows.append({
                    'retentionslot_timestamp': new_timestamp,
                    'milliseconds': new_ms,
                    'sample_index': new_sample_index,
                    'description': new_description,
                    'breathphase': new_breathphase
                })
            if new_rows:
                skeleton_df = pd.concat([skeleton_df, pd.DataFrame(new_rows)], ignore_index=True)
            if self.config['debug_mode']:
                print(f"Extended scaffold CSV to {len(skeleton_df)} retention slots.")
        
        if len(skeleton_df) > required_slots:
            skeleton_df = skeleton_df.iloc[:required_slots]
        
        return skeleton_df

    # ------------------------------------------------------------------
    # Generate counterbalanced stimulus design (trial conditions)
    # ------------------------------------------------------------------
    def generate_counterbalanced_design(self, participant_id):
        random.seed(self.base_seed + participant_id)
        np.random.seed(self.base_seed + participant_id)
        
        soa_conditions = self.config['soa_conditions_ms']
        jitter_options = self.config['jitter_options_ms']
        stimulus_types = LOOMING_STIMULI
        
        all_trials = []
        
        # For each main condition (inhalation, exhalation, baseline), create combos.
        for condition, count in self.trial_counts.items():
            if condition == 'catch' or count == 0:
                continue
            combo_list = []
            for stim_type in stimulus_types:
                for soa in soa_conditions:
                    for _ in range(self.config['repetitions']):
                        combo_list.append((stim_type, soa))
            random.shuffle(combo_list)
            for stim_type, soa in combo_list:
                all_trials.append({
                    'participant_id': participant_id,
                    'trial_type': condition,  # inhalation, exhalation, or baseline
                    'stimulus_type': stim_type,
                    'soa_value_ms': soa,
                    'jitter_ms': random.choice(jitter_options),
                    'is_tactile': True  # for main trials tactile stimulus is present
                })
        
        # Add catch trials (looming stimulus only, no tactile stimulus).
        catch_count = self.trial_counts.get('catch', 0)
        if catch_count > 0:
            catch_combos = []
            stim_counts = {stim: 0 for stim in stimulus_types}
            soa_counts = {soa: 0 for soa in soa_conditions}
            while len(catch_combos) < catch_count:
                min_stim = min(stim_counts, key=stim_counts.get)
                min_soa = min(soa_counts, key=soa_counts.get)
                catch_combos.append((min_stim, min_soa))
                stim_counts[min_stim] += 1
                soa_counts[min_soa] += 1
            for stim_type, soa in catch_combos:
                all_trials.append({
                    'participant_id': participant_id,
                    'trial_type': 'catch',
                    'stimulus_type': stim_type,
                    'soa_value_ms': soa,
                    'jitter_ms': random.choice(jitter_options),
                    'is_tactile': False  # catch trials have no tactile stimulus
                })
        
        design_df = pd.DataFrame(all_trials)
        design_df = design_df.sample(frac=1, random_state=participant_id*100).reset_index(drop=True)
        design_df['trial_number'] = range(1, len(design_df) + 1)
        
        if self.config['debug_mode']:
            print("\nTrial type distribution:")
            print(design_df['trial_type'].value_counts())
            print("\nStimulus type distribution by trial type:")
            for trial_type in design_df['trial_type'].unique():
                if trial_type == 'baseline':
                    continue
                subset = design_df[design_df['trial_type'] == trial_type]
                print(f"  {trial_type}: {subset['stimulus_type'].value_counts().to_dict()}")
            print("\nSOA distribution by trial type:")
            for trial_type in design_df['trial_type'].unique():
                if trial_type == 'baseline':
                    continue
                subset = design_df[design_df['trial_type'] == trial_type]
                print(f"  {trial_type}: {subset['soa_value_ms'].value_counts().to_dict()}")
        
        return design_df

    # ------------------------------------------------------------------
    # Assign scaffold retention slots to trials by randomly mapping the design pool in order.
    # This preserves the scaffold’s sequential order (timestamps, trial numbers).
    # ------------------------------------------------------------------
    def assign_breath_holds(self, design_df):
        required_slots = len(design_df)
        scaffold_df = self.load_retention_slots(required_slots)
        
        design_pool = design_df.to_dict('records')
        random.shuffle(design_pool)  # Randomize the assignment
        assigned_conditions = []
        
        for idx, row in scaffold_df.iterrows():
            chosen = design_pool[idx]
            # Merge scaffold row into chosen condition.
            chosen.update({
                'retentionslot_timestamp': row['retentionslot_timestamp'],
                'milliseconds': row['milliseconds'],
                'sample_index': row['sample_index'],
                'description': row['description'],
                'breathphase': row['breathphase']
            })
            assigned_conditions.append(chosen)
        
        final_df = pd.DataFrame(assigned_conditions)
        final_df['trial_number'] = range(1, len(final_df) + 1)
        
        final_df['jittered_ms'] = final_df['milliseconds'] + final_df['jitter_ms']
        final_df['looming_stimulus_timestamp'] = final_df['jittered_ms'].apply(convert_ms_to_timestamp)
        non_catch_mask = final_df['trial_type'] != 'catch'
        final_df.loc[non_catch_mask, 'soa_ms'] = final_df.loc[non_catch_mask, 'jittered_ms'] + final_df.loc[non_catch_mask, 'soa_value_ms']
        final_df.loc[non_catch_mask, 'tactile_stimulus_timestamp'] = final_df.loc[non_catch_mask, 'soa_ms'].apply(convert_ms_to_timestamp)
        
        return final_df

    # ------------------------------------------------------------------
    # Finalize the design CSV by retaining the scaffold column order and appending new columns.
    # Also, for baseline trials, remove looming stimulus info; for catch trials, remove tactile stimulus info.
    # ------------------------------------------------------------------
    def finalize_design_csv(self, design_df):
        # For baseline trials: no looming stimulus.
        baseline_mask = (design_df['trial_type'] == 'baseline')
        design_df.loc[baseline_mask, 'stimulus_type'] = None
        design_df.loc[baseline_mask, 'looming_stimulus_timestamp'] = None
        
        # For catch trials: no tactile stimulus.
        catch_mask = (design_df['trial_type'] == 'catch')
        design_df.loc[catch_mask, 'tactile_stimulus_timestamp'] = None
        
        design_df['expected_response'] = ~design_df['trial_type'].isin(['catch'])
        
        # Retain scaffold column order as first columns.
        scaffold_cols = self.scaffold_order if hasattr(self, 'scaffold_order') else ['retentionslot_timestamp', 'milliseconds', 'sample_index', 'description', 'breathphase']
        # Define new columns in a desired order.
        new_cols = ['participant_id', 'trial_number', 'trial_type', 'stimulus_type', 'soa_value_ms', 'jitter_ms', 'is_tactile', 'looming_stimulus_timestamp', 'tactile_stimulus_timestamp', 'soa_ms', 'jittered_ms', 'expected_response']
        # Final order: scaffold columns first, then new columns that are not in scaffold.
        final_order = scaffold_cols + [col for col in new_cols if col not in scaffold_cols and col in design_df.columns]
        # Add any remaining columns.
        remaining = [col for col in design_df.columns if col not in final_order]
        final_order.extend(remaining)
        design_df = design_df[final_order]
        
        self._validate_design(design_df)
        return design_df

    # ------------------------------------------------------------------
    # Validation: Print distributions and sequence properties.
    # ------------------------------------------------------------------
    def _validate_design(self, design_df):
        if not self.config['debug_mode']:
            return
        print("\n=== DESIGN VALIDATION ===")
        print("Trial type counts:")
        print(design_df['trial_type'].value_counts())
        print("\nStimulus distribution by trial type:")
        for trial_type in design_df['trial_type'].unique():
            if trial_type == 'baseline':
                continue
            subset = design_df[design_df['trial_type'] == trial_type]
            print(f"  {trial_type}: {subset['stimulus_type'].value_counts().to_dict()}")
        print("\nSOA distribution by trial type:")
        for trial_type in design_df['trial_type'].unique():
            if trial_type == 'baseline':
                continue
            subset = design_df[design_df['trial_type'] == trial_type]
            print(f"  {trial_type}: {subset['soa_value_ms'].value_counts().to_dict()}")
        print("\nBreath phase distribution:")
        if 'breathphase' in design_df.columns:
            print(design_df['breathphase'].value_counts())
        trial_sequence = design_df['trial_type'].tolist()
        consecutive_counts = {}
        current_type = trial_sequence[0]
        current_count = 1
        for t in trial_sequence[1:]:
            if t == current_type:
                current_count += 1
            else:
                key = f"{current_type}_{current_count}"
                consecutive_counts[key] = consecutive_counts.get(key, 0) + 1
                current_type = t
                current_count = 1
        key = f"{current_type}_{current_count}"
        consecutive_counts[key] = consecutive_counts.get(key, 0) + 1
        print("\nRuns of consecutive trials:")
        for key, count in sorted(consecutive_counts.items()):
            print(f"  {key}: {count}")
    
    # ------------------------------------------------------------------
    # Process a single participant: generate design, assign slots, finalize, and save CSV.
    # ------------------------------------------------------------------
    def process_participant(self, participant_id):
        try:
            if self.config['debug_mode']:
                print(f"\n=== Processing participant {participant_id} ===")
            design_df = self.generate_counterbalanced_design(participant_id)
            design_df = self.assign_breath_holds(design_df)
            design_df = self.finalize_design_csv(design_df)
            out_path = os.path.join(self.experiment_log_dir, f"participant_{participant_id}_design.csv")
            design_df.to_csv(out_path, index=False)
            if self.config['debug_mode']:
                print(f"Saved participant {participant_id} design to {out_path}")
            return True, out_path
        except Exception as e:
            print(f"Error processing participant {participant_id}: {e}")
            traceback.print_exc()
            return False, None
    
    # ------------------------------------------------------------------
    # Run design generation for multiple participants.
    # ------------------------------------------------------------------
    def run(self, participant_ids=None, num_participants=None):
        if participant_ids is None and num_participants is None:
            num_participants = self.config.get('num_participants', 5)
        if participant_ids is None:
            participant_ids = list(range(num_participants))
        if self.config['debug_mode']:
            print("=== Starting PPSDesignGenerator ===")
            print(f"Repetitions: {self.config['repetitions']}")
            print(f"Participants: {participant_ids}")
            print(f"Box breathing settings: {self.config['box_breathing']}")
        results = {}
        for pid in participant_ids:
            success, path = self.process_participant(pid)
            results[pid] = {'success': success, 'csv_path': path}
        return results

# ----------------------------------------------------------------------
# AUDIO GENERATION PLACEHOLDER FUNCTION
# ----------------------------------------------------------------------
def generate_audio_from_design(design_csv_path, output_dir, base_audio_path=None):
    """
    Generate audio files from a design CSV.
    (Placeholder function – implement audio generation as needed.)
    """
    print(f"Audio generation from {design_csv_path} not implemented yet")
    return None, None

# ----------------------------------------------------------------------
# MAIN FUNCTION
# ----------------------------------------------------------------------
def main():
    """Main function to run the design generator."""
    generator = PPSDesignGenerator(CONFIG)
    results = generator.run()
    print("\n=== Generation Summary ===")
    for pid, info in results.items():
        if info['success']:
            print(f"Participant {pid}: CSV generated at {info['csv_path']}")
        else:
            print(f"Participant {pid}: ERROR - no CSV generated.")
    print("\nTo generate audio files from these designs, implement the audio generation function.")

if __name__ == "__main__":
    main()


Trial count calculation:
  inhalation: 80 trials
  exhalation: 80 trials
  baseline: 80 trials
  catch: 24 trials
  Total: 264 trials
Initialized PPSDesignGenerator with:
  - Repetitions: 4
  - Conditions: ['inhalation', 'exhalation', 'baseline']
  - SOA values: [190, 400, 700, 1000, 1500]
  - Total trials per participant: 264
  - Output directory: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog
=== Starting PPSDesignGenerator ===
Repetitions: 4
Participants: [0, 1, 2, 3, 4]
Box breathing settings: {'cycle_duration_sec': 8, 'intro_duration_sec': 60, 'outro_duration_sec': 30, 'sample_rate': 48000}

=== Processing participant 0 ===

Trial type distribution:
trial_type
exhalation    80
inhalation    80
baseline      80
catch         24
Name: count, dtype: int64

Stimulus type distribution by trial type:
  exhalation: {'brown': 20, 'pink': 20, 'blue': 20, 'white': 20}
  catch: {'blue': 6, 'white': 6, 'brown': 6, 'pink': 6}
  inhalation: {'pink': 2

In [16]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
PPS Design Generator

This script generates participant‐specific experimental design CSVs for a 
Peripersonal Space (PPS) experiment. Each design CSV is built upon a 
“scaffold” CSV that contains the retention slot (breathhold) timestamps 
from a box‐breathing audio file. If needed, additional retention slots are 
appended (using fixed 8‑second increments) so that every trial can be assigned 
to a specific retention slot. Then, a counterbalanced set of trial conditions 
(e.g., trial type, looming stimulus type, SOA, jitter) is allocated to each 
scaffold row while preserving the sequential order of the scaffold. The final 
output retains the original column order from the scaffold, with new trial‐specific 
variables appended as new columns.

Trial Types:
  - Inhalation: looming stimulus cue and tactile stimulus.
  - Exhalation: looming stimulus cue and tactile stimulus.
  - Baseline: tactile stimulus only (looming stimulus is NA).
  - Catch: looming stimulus cue only (no tactile stimulus).

Rationale for assignment:
  - The design pool is created so that 1/3 of trials are inhalation, 1/3 exhalation, 
    and 1/3 baseline (plus an additional 10% catch trials).
  - During assignment, scaffold rows are processed in order. For each row, if its 
    breathphase is “inhale” the algorithm first attempts to assign an inhalation trial. 
    If none remain, it assigns a baseline or catch trial, balancing the number of baseline 
    (and catch) trials assigned to inhale versus exhale rows. The same logic applies for 
    exhale rows.
    
Author: AI Assistant
"""

import os
import numpy as np
import pandas as pd
import random
import traceback

# ----------------------------------------------------------------------
# Helper function: Convert milliseconds to "MM:SS.s" format
# ----------------------------------------------------------------------
def convert_ms_to_timestamp(ms):
    m = int(ms // 60000)
    s = (ms % 60000) / 1000.0
    return f"{m:02}:{s:.1f}"

# ----------------------------------------------------------------------
# CONFIGURATION PARAMETERS
# ----------------------------------------------------------------------
CONFIG = {
    'repetitions': 4,            # How many times each SOA×stimulus combination repeats
    'num_participants': 5,       # Default number of participants
    'included_conditions': {
        'inhalation': True,
        'exhalation': True,
        'baseline': True,
    },
    'catch_trial_percentage': 10,
    'soa_conditions_ms': [190, 400, 700, 1000, 1500],
    'jitter_options_ms': [100, 200, 300, 400, 500],
    'debug_mode': True,
    'paths': {
        # Directory where participant-specific CSV files will be saved
        'experiment_log_dir': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog",
    },
    # Path to the original scaffold CSV containing retention (breathhold) timestamps.
    'skeleton_csv_path': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_timestamps.csv",
    'box_breathing': {
        'cycle_duration_sec': 8,      # Duration of one breath cycle in seconds
        # Intro/outro are not used here because the scaffold drives timing.
        'intro_duration_sec': 60,
        'outro_duration_sec': 30,
        'sample_rate': 48000,           # Audio sample rate (samples per second)
    }
}

# Noise types for looming stimuli (can be any labels)
LOOMING_STIMULI = ['blue', 'brown', 'pink', 'white']

# ----------------------------------------------------------------------
# CLASS: PPSDesignGenerator
# ----------------------------------------------------------------------
class PPSDesignGenerator:
    """Generates participant-specific CSV design files for PPS experiments."""
    
    def __init__(self, config=None):
        self.config = config or CONFIG
        
        # Prepare output directory.
        self.experiment_log_dir = self.config['paths']['experiment_log_dir']
        os.makedirs(self.experiment_log_dir, exist_ok=True)
        
        # Calculate trial counts.
        # For each condition (inhalation, exhalation, baseline): count = (# noise types) × (# SOA) × (repetitions).
        # Then, add 10% catch trials.
        self.trial_counts = self._calculate_trial_counts()
        self.conditions = [cond for cond, inc in self.config['included_conditions'].items() if inc]
        
        # Set seed for reproducibility.
        self.base_seed = 42
        
        if self.config['debug_mode']:
            total_trials = sum(self.trial_counts.values())
            print("Initialized PPSDesignGenerator with:")
            print(f"  - Repetitions: {self.config['repetitions']}")
            print(f"  - Conditions: {self.conditions}")
            print(f"  - SOA values: {self.config['soa_conditions_ms']}")
            print(f"  - Total trials per participant: {total_trials}")
            print(f"  - Output directory: {self.experiment_log_dir}")
    
    def _calculate_trial_counts(self):
        reps = self.config['repetitions']
        soa_count = len(self.config['soa_conditions_ms'])
        stimulus_types = LOOMING_STIMULI
        catch_percentage = self.config['catch_trial_percentage']
        
        trial_counts = {}
        for condition, include in self.config['included_conditions'].items():
            if include:
                trial_counts[condition] = len(stimulus_types) * soa_count * reps
        regular_trials = sum(trial_counts.values())
        catch_trials = int(np.ceil(regular_trials * catch_percentage / 100))
        trial_counts['catch'] = catch_trials
        
        if self.config['debug_mode']:
            print("Trial count calculation:")
            for cond, count in trial_counts.items():
                print(f"  {cond}: {count} trials")
            print(f"  Total: {sum(trial_counts.values())} trials")
        
        return trial_counts

    # ------------------------------------------------------------------
    # Load the scaffold CSV and extend it if needed. Retain original column order.
    # ------------------------------------------------------------------
    def load_retention_slots(self, required_slots):
        skeleton_path = self.config['skeleton_csv_path']
        try:
            skeleton_df = pd.read_csv(skeleton_path, sep=None, engine='python')
            if len(skeleton_df.columns) == 1:
                skeleton_df = pd.read_csv(skeleton_path, delimiter="\t")
            skeleton_df.columns = [col.strip().lower() for col in skeleton_df.columns]
            # Save original scaffold column order.
            self.scaffold_order = skeleton_df.columns.tolist()
            if self.config['debug_mode']:
                print("Scaffold CSV columns:", self.scaffold_order)
        except Exception as e:
            raise ValueError(f"Error reading scaffold CSV from {skeleton_path}: {e}")
        
        current_slots = len(skeleton_df)
        required_slots = int(required_slots)
        sample_rate = self.config['box_breathing']['sample_rate']
        cycle_sec = self.config['box_breathing']['cycle_duration_sec']
        
        if self.config['debug_mode']:
            print(f"Loaded scaffold CSV with {current_slots} retention slots.")
        
        if current_slots < required_slots:
            last_row = skeleton_df.iloc[-1]
            last_ms = float(last_row['milliseconds'])
            last_sample_index = int(last_row['sample_index'])
            new_rows = []
            for i in range(required_slots - current_slots):
                new_ms = last_ms + (cycle_sec * 1000) * (i + 1)
                new_sample_index = last_sample_index + int(cycle_sec * sample_rate * (i + 1))
                new_timestamp = convert_ms_to_timestamp(new_ms)
                new_breathphase = 'inhale' if ((current_slots + i) % 2 == 0) else 'exhale'
                new_description = f"Additional hold {i+1}"
                new_rows.append({
                    'retentionslot_timestamp': new_timestamp,
                    'milliseconds': new_ms,
                    'sample_index': new_sample_index,
                    'description': new_description,
                    'breathphase': new_breathphase
                })
            if new_rows:
                skeleton_df = pd.concat([skeleton_df, pd.DataFrame(new_rows)], ignore_index=True)
            if self.config['debug_mode']:
                print(f"Extended scaffold CSV to {len(skeleton_df)} retention slots.")
        
        if len(skeleton_df) > required_slots:
            skeleton_df = skeleton_df.iloc[:required_slots]
        
        return skeleton_df

    # ------------------------------------------------------------------
    # Generate counterbalanced stimulus design (trial conditions)
    # ------------------------------------------------------------------
    def generate_counterbalanced_design(self, participant_id):
        random.seed(self.base_seed + participant_id)
        np.random.seed(self.base_seed + participant_id)
        
        soa_conditions = self.config['soa_conditions_ms']
        jitter_options = self.config['jitter_options_ms']
        stimulus_types = LOOMING_STIMULI
        
        all_trials = []
        
        # Generate for main conditions: inhalation, exhalation, and baseline.
        for condition, count in self.trial_counts.items():
            if condition == 'catch' or count == 0:
                continue
            combo_list = []
            for stim_type in stimulus_types:
                for soa in soa_conditions:
                    for _ in range(self.config['repetitions']):
                        combo_list.append((stim_type, soa))
            random.shuffle(combo_list)
            for stim_type, soa in combo_list:
                all_trials.append({
                    'participant_id': participant_id,
                    'trial_type': condition,  # inhalation, exhalation, or baseline
                    'stimulus_type': stim_type,
                    'soa_value_ms': soa,
                    'jitter_ms': random.choice(jitter_options),
                    'is_tactile': True  # For these main trials tactile is present.
                })
        
        # Add catch trials (looming only, no tactile stimulus).
        catch_count = self.trial_counts.get('catch', 0)
        if catch_count > 0:
            catch_combos = []
            stim_counts = {stim: 0 for stim in stimulus_types}
            soa_counts = {soa: 0 for soa in soa_conditions}
            while len(catch_combos) < catch_count:
                min_stim = min(stim_counts, key=stim_counts.get)
                min_soa = min(soa_counts, key=soa_counts.get)
                catch_combos.append((min_stim, min_soa))
                stim_counts[min_stim] += 1
                soa_counts[min_soa] += 1
            for stim_type, soa in catch_combos:
                all_trials.append({
                    'participant_id': participant_id,
                    'trial_type': 'catch',
                    'stimulus_type': stim_type,
                    'soa_value_ms': soa,
                    'jitter_ms': random.choice(jitter_options),
                    'is_tactile': False
                })
        
        design_df = pd.DataFrame(all_trials)
        design_df = design_df.sample(frac=1, random_state=participant_id*100).reset_index(drop=True)
        design_df['trial_number'] = range(1, len(design_df) + 1)
        
        if self.config['debug_mode']:
            print("\nTrial type distribution:")
            print(design_df['trial_type'].value_counts())
            print("\nStimulus type distribution by trial type:")
            for trial_type in design_df['trial_type'].unique():
                if trial_type == 'baseline':
                    continue
                subset = design_df[design_df['trial_type'] == trial_type]
                print(f"  {trial_type}: {subset['stimulus_type'].value_counts().to_dict()}")
            print("\nSOA distribution by trial type:")
            for trial_type in design_df['trial_type'].unique():
                if trial_type == 'baseline':
                    continue
                subset = design_df[design_df['trial_type'] == trial_type]
                print(f"  {trial_type}: {subset['soa_value_ms'].value_counts().to_dict()}")
        
        return design_df

    # ------------------------------------------------------------------
    # Assign scaffold retention slots to trials with balanced baseline (and catch) assignment.
    # The scaffold order is preserved.
    # ------------------------------------------------------------------
    def assign_breath_holds(self, design_df):
        required_slots = len(design_df)
        scaffold_df = self.load_retention_slots(required_slots)
        
        # Partition design pool by trial type.
        design_inhalation = [x for x in design_df.to_dict('records') if x['trial_type'] == 'inhalation']
        design_exhalation = [x for x in design_df.to_dict('records') if x['trial_type'] == 'exhalation']
        design_baseline = [x for x in design_df.to_dict('records') if x['trial_type'] == 'baseline']
        design_catch = [x for x in design_df.to_dict('records') if x['trial_type'] == 'catch']
        
        # For baseline and catch, we'll try to balance assignment between inhale and exhale.
        baseline_inhale = []
        baseline_exhale = []
        catch_inhale = []
        catch_exhale = []
        
        # For simplicity, shuffle each pool.
        random.shuffle(design_inhalation)
        random.shuffle(design_exhalation)
        random.shuffle(design_baseline)
        random.shuffle(design_catch)
        
        # Determine half counts.
        half_baseline = len(design_baseline) // 2
        half_catch = len(design_catch) // 2
        
        # Split baseline and catch pools.
        baseline_inhale = design_baseline[:half_baseline]
        baseline_exhale = design_baseline[half_baseline:]
        catch_inhale = design_catch[:half_catch]
        catch_exhale = design_catch[half_catch:]
        
        # Now, iterate over scaffold rows in order.
        assigned_conditions = []
        for idx, row in scaffold_df.iterrows():
            bp = row['breathphase'].lower()
            chosen = None
            if bp == 'inhale':
                if design_inhalation:
                    chosen = design_inhalation.pop(0)
                elif baseline_inhale:
                    chosen = baseline_inhale.pop(0)
                elif catch_inhale:
                    chosen = catch_inhale.pop(0)
                else:
                    # If nothing left in inhale-specific pools, pull from remaining overall.
                    remaining = design_inhalation + design_exhalation + baseline_exhale + catch_exhale
                    if remaining:
                        chosen = remaining.pop(0)
            elif bp == 'exhale':
                if design_exhalation:
                    chosen = design_exhalation.pop(0)
                elif baseline_exhale:
                    chosen = baseline_exhale.pop(0)
                elif catch_exhale:
                    chosen = catch_exhale.pop(0)
                else:
                    remaining = design_inhalation + design_exhalation + baseline_inhale + catch_inhale
                    if remaining:
                        chosen = remaining.pop(0)
            else:
                # Fallback: assign any.
                remaining = design_inhalation + design_exhalation + baseline_inhale + baseline_exhale + catch_inhale + catch_exhale
                if remaining:
                    chosen = remaining.pop(0)
            if not chosen:
                continue
            chosen.update({
                'retentionslot_timestamp': row['retentionslot_timestamp'],
                'milliseconds': row['milliseconds'],
                'sample_index': row['sample_index'],
                'description': row['description'],
                'breathphase': row['breathphase']
            })
            assigned_conditions.append(chosen)
        
        final_df = pd.DataFrame(assigned_conditions)
        final_df['trial_number'] = range(1, len(final_df) + 1)
        
        final_df['jittered_ms'] = final_df['milliseconds'] + final_df['jitter_ms']
        final_df['looming_stimulus_timestamp'] = final_df['jittered_ms'].apply(convert_ms_to_timestamp)
        non_catch_mask = final_df['trial_type'] != 'catch'
        final_df.loc[non_catch_mask, 'soa_ms'] = final_df.loc[non_catch_mask, 'jittered_ms'] + final_df.loc[non_catch_mask, 'soa_value_ms']
        final_df.loc[non_catch_mask, 'tactile_stimulus_timestamp'] = final_df.loc[non_catch_mask, 'soa_ms'].apply(convert_ms_to_timestamp)
        
        return final_df

    # ------------------------------------------------------------------
    # Finalize the design CSV: retain original scaffold column order; append new columns.
    # Also, set looming stimulus to NA for baseline trials and tactile stimulus to NA for catch trials.
    # ------------------------------------------------------------------
    def finalize_design_csv(self, design_df):
        # For baseline trials: remove looming stimulus.
        baseline_mask = (design_df['trial_type'] == 'baseline')
        design_df.loc[baseline_mask, 'stimulus_type'] = None
        design_df.loc[baseline_mask, 'looming_stimulus_timestamp'] = None
        
        # For catch trials: remove tactile stimulus.
        catch_mask = (design_df['trial_type'] == 'catch')
        design_df.loc[catch_mask, 'tactile_stimulus_timestamp'] = None
        
        design_df['expected_response'] = ~design_df['trial_type'].isin(['catch'])
        
        # Retain scaffold columns first in original order.
        scaffold_cols = self.scaffold_order if hasattr(self, 'scaffold_order') else ['retentionslot_timestamp', 'milliseconds', 'sample_index', 'description', 'breathphase']
        new_cols = ['participant_id', 'trial_number', 'trial_type', 'stimulus_type', 'soa_value_ms', 'jitter_ms', 'is_tactile', 'looming_stimulus_timestamp', 'tactile_stimulus_timestamp', 'soa_ms', 'jittered_ms', 'expected_response']
        final_order = scaffold_cols + [col for col in new_cols if col not in scaffold_cols and col in design_df.columns]
        remaining = [col for col in design_df.columns if col not in final_order]
        final_order.extend(remaining)
        design_df = design_df[final_order]
        
        self._validate_design(design_df)
        return design_df

    # ------------------------------------------------------------------
    # Validation: Print distributions and sequence properties.
    # ------------------------------------------------------------------
    def _validate_design(self, design_df):
        if not self.config['debug_mode']:
            return
        print("\n=== DESIGN VALIDATION ===")
        print("Trial type counts:")
        print(design_df['trial_type'].value_counts())
        print("\nStimulus distribution by trial type:")
        for trial_type in design_df['trial_type'].unique():
            if trial_type == 'baseline':
                continue
            subset = design_df[design_df['trial_type'] == trial_type]
            print(f"  {trial_type}: {subset['stimulus_type'].value_counts().to_dict()}")
        print("\nSOA distribution by trial type:")
        for trial_type in design_df['trial_type'].unique():
            if trial_type == 'baseline':
                continue
            subset = design_df[design_df['trial_type'] == trial_type]
            print(f"  {trial_type}: {subset['soa_value_ms'].value_counts().to_dict()}")
        print("\nBreath phase distribution:")
        if 'breathphase' in design_df.columns:
            print(design_df['breathphase'].value_counts())
        trial_sequence = design_df['trial_type'].tolist()
        consecutive_counts = {}
        current_type = trial_sequence[0]
        current_count = 1
        for t in trial_sequence[1:]:
            if t == current_type:
                current_count += 1
            else:
                key = f"{current_type}_{current_count}"
                consecutive_counts[key] = consecutive_counts.get(key, 0) + 1
                current_type = t
                current_count = 1
        key = f"{current_type}_{current_count}"
        consecutive_counts[key] = consecutive_counts.get(key, 0) + 1
        print("\nRuns of consecutive trials:")
        for key, count in sorted(consecutive_counts.items()):
            print(f"  {key}: {count}")
    
    # ------------------------------------------------------------------
    # Process a single participant: generate design, assign slots, finalize, and save CSV.
    # ------------------------------------------------------------------
    def process_participant(self, participant_id):
        try:
            if self.config['debug_mode']:
                print(f"\n=== Processing participant {participant_id} ===")
            design_df = self.generate_counterbalanced_design(participant_id)
            design_df = self.assign_breath_holds(design_df)
            design_df = self.finalize_design_csv(design_df)
            out_path = os.path.join(self.experiment_log_dir, f"participant_{participant_id}_design.csv")
            design_df.to_csv(out_path, index=False)
            if self.config['debug_mode']:
                print(f"Saved participant {participant_id} design to {out_path}")
            return True, out_path
        except Exception as e:
            print(f"Error processing participant {participant_id}: {e}")
            traceback.print_exc()
            return False, None
    
    # ------------------------------------------------------------------
    # Run design generation for multiple participants.
    # ------------------------------------------------------------------
    def run(self, participant_ids=None, num_participants=None):
        if participant_ids is None and num_participants is None:
            num_participants = self.config.get('num_participants', 5)
        if participant_ids is None:
            participant_ids = list(range(num_participants))
        if self.config['debug_mode']:
            print("=== Starting PPSDesignGenerator ===")
            print(f"Repetitions: {self.config['repetitions']}")
            print(f"Participants: {participant_ids}")
            print(f"Box breathing settings: {self.config['box_breathing']}")
        results = {}
        for pid in participant_ids:
            success, path = self.process_participant(pid)
            results[pid] = {'success': success, 'csv_path': path}
        return results

# ----------------------------------------------------------------------
# AUDIO GENERATION PLACEHOLDER FUNCTION
# ----------------------------------------------------------------------
def generate_audio_from_design(design_csv_path, output_dir, base_audio_path=None):
    """
    Generate audio files from a design CSV.
    (Placeholder function – implement audio generation as needed.)
    """
    print(f"Audio generation from {design_csv_path} not implemented yet")
    return None, None

# ----------------------------------------------------------------------
# MAIN FUNCTION
# ----------------------------------------------------------------------
def main():
    """Main function to run the design generator."""
    generator = PPSDesignGenerator(CONFIG)
    results = generator.run()
    print("\n=== Generation Summary ===")
    for pid, info in results.items():
        if info['success']:
            print(f"Participant {pid}: CSV generated at {info['csv_path']}")
        else:
            print(f"Participant {pid}: ERROR - no CSV generated.")
    print("\nTo generate audio files from these designs, implement the audio generation function.")

if __name__ == "__main__":
    main()


Trial count calculation:
  inhalation: 80 trials
  exhalation: 80 trials
  baseline: 80 trials
  catch: 24 trials
  Total: 264 trials
Initialized PPSDesignGenerator with:
  - Repetitions: 4
  - Conditions: ['inhalation', 'exhalation', 'baseline']
  - SOA values: [190, 400, 700, 1000, 1500]
  - Total trials per participant: 264
  - Output directory: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog
=== Starting PPSDesignGenerator ===
Repetitions: 4
Participants: [0, 1, 2, 3, 4]
Box breathing settings: {'cycle_duration_sec': 8, 'intro_duration_sec': 60, 'outro_duration_sec': 30, 'sample_rate': 48000}

=== Processing participant 0 ===

Trial type distribution:
trial_type
exhalation    80
inhalation    80
baseline      80
catch         24
Name: count, dtype: int64

Stimulus type distribution by trial type:
  exhalation: {'brown': 20, 'pink': 20, 'blue': 20, 'white': 20}
  catch: {'blue': 6, 'white': 6, 'brown': 6, 'pink': 6}
  inhalation: {'pink': 2

In [18]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
PPS Design Generator

This script generates participant‐specific experimental design CSVs for a 
Peripersonal Space (PPS) experiment. Each design CSV is built upon a 
“scaffold” CSV that contains the retention slot (breathhold) timestamps 
from a box‐breathing audio file. If needed, additional retention slots are 
appended (using fixed 8‑second increments) so that every trial can be assigned 
to a specific retention slot. Then, a counterbalanced set of trial conditions 
(e.g., trial type, looming stimulus type, SOA, jitter) is allocated to each 
scaffold row while preserving the sequential order of the scaffold. The final 
output retains the original column order from the scaffold, with new trial‐specific 
variables appended as new columns.

Trial Types:
  - Inhalation: looming stimulus cue and tactile stimulus.
  - Exhalation: looming stimulus cue and tactile stimulus.
  - Baseline: tactile stimulus only (looming stimulus is NA).
  - Catch: looming stimulus cue only (no tactile stimulus).

Rationale for assignment:
  - The design pool is built so that 1/3 of main trials are inhalation, 
    1/3 exhalation, and 1/3 baseline (plus an additional 10% catch trials).
  - In assignment, scaffold rows are processed in order. For each scaffold row,
    if its breathphase is "inhale" the algorithm first uses an inhalation trial.
    Once those are exhausted, it uses baseline trials split evenly between inhale and exhale,
    and similarly for catch trials. This ensures baseline and catch trials are balanced 
    between the two breathphases.

Author: AI Assistant
"""

import os
import numpy as np
import pandas as pd
import random
import traceback

# ----------------------------------------------------------------------
# Helper function: Convert milliseconds to "MM:SS.s" format
# ----------------------------------------------------------------------
def convert_ms_to_timestamp(ms):
    m = int(ms // 60000)
    s = (ms % 60000) / 1000.0
    return f"{m:02}:{s:.1f}"

# ----------------------------------------------------------------------
# CONFIGURATION PARAMETERS
# ----------------------------------------------------------------------
CONFIG = {
    'repetitions': 4,            # How many times each SOA×stimulus combination repeats
    'num_participants': 5,       # Default number of participants
    'included_conditions': {
        'inhalation': True,
        'exhalation': True,
        'baseline': True,
    },
    'catch_trial_percentage': 10,
    'soa_conditions_ms': [190, 400, 700, 1000, 1500],
    'jitter_options_ms': [100, 200, 300, 400, 500],
    'debug_mode': True,
    'paths': {
        # Directory where participant-specific CSV files will be saved
        'experiment_log_dir': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog",
    },
    # Path to the original scaffold CSV containing retention (breathhold) timestamps.
    'skeleton_csv_path': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_timestamps.csv",
    'box_breathing': {
        'cycle_duration_sec': 8,      # Duration of one breath cycle in seconds
        # Intro/outro are not used here because the scaffold drives timing.
        'intro_duration_sec': 60,
        'outro_duration_sec': 30,
        'sample_rate': 48000,           # Audio sample rate (samples per second)
    }
}

# Noise types for looming stimuli (can be any labels)
LOOMING_STIMULI = ['blue', 'brown', 'pink', 'white']

# ----------------------------------------------------------------------
# CLASS: PPSDesignGenerator
# ----------------------------------------------------------------------
class PPSDesignGenerator:
    """Generates participant-specific CSV design files for PPS experiments."""
    
    def __init__(self, config=None):
        self.config = config or CONFIG
        
        # Prepare output directory.
        self.experiment_log_dir = self.config['paths']['experiment_log_dir']
        os.makedirs(self.experiment_log_dir, exist_ok=True)
        
        # Calculate trial counts.
        # For each condition (inhalation, exhalation, baseline): count = (# noise types) × (# SOA) × (repetitions).
        # Then, add 10% catch trials.
        self.trial_counts = self._calculate_trial_counts()
        self.conditions = [cond for cond, inc in self.config['included_conditions'].items() if inc]
        
        # Set seed for reproducibility.
        self.base_seed = 42
        
        if self.config['debug_mode']:
            total_trials = sum(self.trial_counts.values())
            print("Initialized PPSDesignGenerator with:")
            print(f"  - Repetitions: {self.config['repetitions']}")
            print(f"  - Conditions: {self.conditions}")
            print(f"  - SOA values: {self.config['soa_conditions_ms']}")
            print(f"  - Total trials per participant: {total_trials}")
            print(f"  - Output directory: {self.experiment_log_dir}")
    
    def _calculate_trial_counts(self):
        reps = self.config['repetitions']
        soa_count = len(self.config['soa_conditions_ms'])
        stimulus_types = LOOMING_STIMULI
        catch_percentage = self.config['catch_trial_percentage']
        
        trial_counts = {}
        for condition, include in self.config['included_conditions'].items():
            if include:
                trial_counts[condition] = len(stimulus_types) * soa_count * reps
        regular_trials = sum(trial_counts.values())
        catch_trials = int(np.ceil(regular_trials * catch_percentage / 100))
        trial_counts['catch'] = catch_trials
        
        if self.config['debug_mode']:
            print("Trial count calculation:")
            for cond, count in trial_counts.items():
                print(f"  {cond}: {count} trials")
            print(f"  Total: {sum(trial_counts.values())} trials")
        
        return trial_counts

    # ------------------------------------------------------------------
    # Load the scaffold CSV and extend it if needed.
    # Retain the original column order.
    # ------------------------------------------------------------------
    def load_retention_slots(self, required_slots):
        skeleton_path = self.config['skeleton_csv_path']
        try:
            skeleton_df = pd.read_csv(skeleton_path, sep=None, engine='python')
            if len(skeleton_df.columns) == 1:
                skeleton_df = pd.read_csv(skeleton_path, delimiter="\t")
            skeleton_df.columns = [col.strip().lower() for col in skeleton_df.columns]
            self.scaffold_order = skeleton_df.columns.tolist()
            if self.config['debug_mode']:
                print("Scaffold CSV columns:", self.scaffold_order)
        except Exception as e:
            raise ValueError(f"Error reading scaffold CSV from {skeleton_path}: {e}")
        
        current_slots = len(skeleton_df)
        required_slots = int(required_slots)
        sample_rate = self.config['box_breathing']['sample_rate']
        cycle_sec = self.config['box_breathing']['cycle_duration_sec']
        
        if self.config['debug_mode']:
            print(f"Loaded scaffold CSV with {current_slots} retention slots.")
        
        if current_slots < required_slots:
            last_row = skeleton_df.iloc[-1]
            last_ms = float(last_row['milliseconds'])
            last_sample_index = int(last_row['sample_index'])
            new_rows = []
            for i in range(required_slots - current_slots):
                new_ms = last_ms + (cycle_sec * 1000) * (i + 1)
                new_sample_index = last_sample_index + int(cycle_sec * sample_rate * (i + 1))
                new_timestamp = convert_ms_to_timestamp(new_ms)
                new_breathphase = 'inhale' if ((current_slots + i) % 2 == 0) else 'exhale'
                new_description = f"Additional hold {i+1}"
                new_rows.append({
                    'retentionslot_timestamp': new_timestamp,
                    'milliseconds': new_ms,
                    'sample_index': new_sample_index,
                    'description': new_description,
                    'breathphase': new_breathphase
                })
            if new_rows:
                skeleton_df = pd.concat([skeleton_df, pd.DataFrame(new_rows)], ignore_index=True)
            if self.config['debug_mode']:
                print(f"Extended scaffold CSV to {len(skeleton_df)} retention slots.")
        
        if len(skeleton_df) > required_slots:
            skeleton_df = skeleton_df.iloc[:required_slots]
        
        return skeleton_df

    # ------------------------------------------------------------------
    # Generate counterbalanced stimulus design (trial conditions)
    # ------------------------------------------------------------------
    def generate_counterbalanced_design(self, participant_id):
        random.seed(self.base_seed + participant_id)
        np.random.seed(self.base_seed + participant_id)
        
        soa_conditions = self.config['soa_conditions_ms']
        jitter_options = self.config['jitter_options_ms']
        stimulus_types = LOOMING_STIMULI
        
        all_trials = []
        
        # Generate main conditions: inhalation, exhalation, baseline.
        for condition, count in self.trial_counts.items():
            if condition == 'catch' or count == 0:
                continue
            combo_list = []
            for stim_type in stimulus_types:
                for soa in soa_conditions:
                    for _ in range(self.config['repetitions']):
                        combo_list.append((stim_type, soa))
            random.shuffle(combo_list)
            for stim_type, soa in combo_list:
                all_trials.append({
                    'participant_id': participant_id,
                    'trial_type': condition,  # inhalation, exhalation, or baseline
                    'stimulus_type': stim_type,
                    'soa_value_ms': soa,
                    'jitter_ms': random.choice(jitter_options),
                    'is_tactile': True  # For main trials, tactile is present.
                })
        
        # Add catch trials (looming only, no tactile).
        catch_count = self.trial_counts.get('catch', 0)
        if catch_count > 0:
            catch_combos = []
            stim_counts = {stim: 0 for stim in stimulus_types}
            soa_counts = {soa: 0 for soa in soa_conditions}
            while len(catch_combos) < catch_count:
                min_stim = min(stim_counts, key=stim_counts.get)
                min_soa = min(soa_counts, key=soa_counts.get)
                catch_combos.append((min_stim, min_soa))
                stim_counts[min_stim] += 1
                soa_counts[min_soa] += 1
            for stim_type, soa in catch_combos:
                all_trials.append({
                    'participant_id': participant_id,
                    'trial_type': 'catch',
                    'stimulus_type': stim_type,
                    'soa_value_ms': soa,
                    'jitter_ms': random.choice(jitter_options),
                    'is_tactile': False  # No tactile in catch trials.
                })
        
        design_df = pd.DataFrame(all_trials)
        design_df = design_df.sample(frac=1, random_state=participant_id*100).reset_index(drop=True)
        design_df['trial_number'] = range(1, len(design_df) + 1)
        
        if self.config['debug_mode']:
            print("\nTrial type distribution:")
            print(design_df['trial_type'].value_counts())
            print("\nStimulus type distribution by trial type:")
            for trial_type in design_df['trial_type'].unique():
                if trial_type == 'baseline':
                    continue
                subset = design_df[design_df['trial_type'] == trial_type]
                print(f"  {trial_type}: {subset['stimulus_type'].value_counts().to_dict()}")
            print("\nSOA distribution by trial type:")
            for trial_type in design_df['trial_type'].unique():
                if trial_type == 'baseline':
                    continue
                subset = design_df[design_df['trial_type'] == trial_type]
                print(f"  {trial_type}: {subset['soa_value_ms'].value_counts().to_dict()}")
        
        return design_df

    # ------------------------------------------------------------------
    # Assign scaffold retention slots to trials with balanced baseline and catch assignment.
    # The scaffold order is preserved.
    # ------------------------------------------------------------------
    def assign_breath_holds(self, design_df):
        required_slots = len(design_df)
        scaffold_df = self.load_retention_slots(required_slots)
        
        # Partition design pool by trial type.
        design_inhalation = [x for x in design_df.to_dict('records') if x['trial_type'] == 'inhalation']
        design_exhalation = [x for x in design_df.to_dict('records') if x['trial_type'] == 'exhalation']
        design_baseline = [x for x in design_df.to_dict('records') if x['trial_type'] == 'baseline']
        design_catch = [x for x in design_df.to_dict('records') if x['trial_type'] == 'catch']
        
        # For baseline and catch, split each evenly for inhale and exhale.
        random.shuffle(design_inhalation)
        random.shuffle(design_exhalation)
        random.shuffle(design_baseline)
        random.shuffle(design_catch)
        
        half_baseline = len(design_baseline) // 2
        half_catch = len(design_catch) // 2
        
        baseline_inhale = design_baseline[:half_baseline]
        baseline_exhale = design_baseline[half_baseline:]
        catch_inhale = design_catch[:half_catch]
        catch_exhale = design_catch[half_catch:]
        
        assigned_conditions = []
        # Iterate over scaffold rows (which are sequential).
        for idx, row in scaffold_df.iterrows():
            bp = row['breathphase'].lower()
            chosen = None
            if bp == 'inhale':
                if design_inhalation:
                    chosen = design_inhalation.pop(0)
                elif baseline_inhale:
                    chosen = baseline_inhale.pop(0)
                elif catch_inhale:
                    chosen = catch_inhale.pop(0)
                else:
                    remaining = design_inhalation + design_exhalation + baseline_exhale + catch_exhale
                    if remaining:
                        chosen = remaining.pop(0)
            elif bp == 'exhale':
                if design_exhalation:
                    chosen = design_exhalation.pop(0)
                elif baseline_exhale:
                    chosen = baseline_exhale.pop(0)
                elif catch_exhale:
                    chosen = catch_exhale.pop(0)
                else:
                    remaining = design_inhalation + design_exhalation + baseline_inhale + catch_inhale
                    if remaining:
                        chosen = remaining.pop(0)
            else:
                remaining = design_inhalation + design_exhalation + baseline_inhale + baseline_exhale + catch_inhale + catch_exhale
                if remaining:
                    chosen = remaining.pop(0)
            if not chosen:
                continue
            chosen.update({
                'retentionslot_timestamp': row['retentionslot_timestamp'],
                'milliseconds': row['milliseconds'],
                'sample_index': row['sample_index'],
                'description': row['description'],
                'breathphase': row['breathphase']
            })
            assigned_conditions.append(chosen)
        
        final_df = pd.DataFrame(assigned_conditions)
        final_df['trial_number'] = range(1, len(final_df) + 1)
        
        final_df['jittered_ms'] = final_df['milliseconds'] + final_df['jitter_ms']
        final_df['looming_stimulus_timestamp'] = final_df['jittered_ms'].apply(convert_ms_to_timestamp)
        non_catch_mask = final_df['trial_type'] != 'catch'
        final_df.loc[non_catch_mask, 'soa_ms'] = final_df.loc[non_catch_mask, 'jittered_ms'] + final_df.loc[non_catch_mask, 'soa_value_ms']
        final_df.loc[non_catch_mask, 'tactile_stimulus_timestamp'] = final_df.loc[non_catch_mask, 'soa_ms'].apply(convert_ms_to_timestamp)
        
        return final_df

    # ------------------------------------------------------------------
    # Finalize the design CSV: retain original scaffold column order and append new columns.
    # Also, for baseline trials, remove looming stimulus info; for catch trials, remove tactile stimulus info.
    # ------------------------------------------------------------------
    def finalize_design_csv(self, design_df):
        baseline_mask = (design_df['trial_type'] == 'baseline')
        design_df.loc[baseline_mask, 'stimulus_type'] = None
        design_df.loc[baseline_mask, 'looming_stimulus_timestamp'] = None
        
        catch_mask = (design_df['trial_type'] == 'catch')
        design_df.loc[catch_mask, 'tactile_stimulus_timestamp'] = None
        
        design_df['expected_response'] = ~design_df['trial_type'].isin(['catch'])
        
        scaffold_cols = self.scaffold_order if hasattr(self, 'scaffold_order') else ['retentionslot_timestamp', 'milliseconds', 'sample_index', 'description', 'breathphase']
        new_cols = ['participant_id', 'trial_number', 'trial_type', 'stimulus_type', 'soa_value_ms', 'jitter_ms', 'is_tactile', 'looming_stimulus_timestamp', 'tactile_stimulus_timestamp', 'soa_ms', 'jittered_ms', 'expected_response']
        final_order = scaffold_cols + [col for col in new_cols if col not in scaffold_cols and col in design_df.columns]
        remaining = [col for col in design_df.columns if col not in final_order]
        final_order.extend(remaining)
        design_df = design_df[final_order]
        
        self._validate_design(design_df)
        return design_df

    # ------------------------------------------------------------------
    # Validation: Print distributions and sequence properties.
    # ------------------------------------------------------------------
    def _validate_design(self, design_df):
        if not self.config['debug_mode']:
            return
        print("\n=== DESIGN VALIDATION ===")
        print("Trial type counts:")
        print(design_df['trial_type'].value_counts())
        print("\nStimulus distribution by trial type:")
        for trial_type in design_df['trial_type'].unique():
            if trial_type == 'baseline':
                continue
            subset = design_df[design_df['trial_type'] == trial_type]
            print(f"  {trial_type}: {subset['stimulus_type'].value_counts().to_dict()}")
        print("\nSOA distribution by trial type:")
        for trial_type in design_df['trial_type'].unique():
            if trial_type == 'baseline':
                continue
            subset = design_df[design_df['trial_type'] == trial_type]
            print(f"  {trial_type}: {subset['soa_value_ms'].value_counts().to_dict()}")
        print("\nBreath phase distribution:")
        if 'breathphase' in design_df.columns:
            print(design_df['breathphase'].value_counts())
        trial_sequence = design_df['trial_type'].tolist()
        consecutive_counts = {}
        current_type = trial_sequence[0]
        current_count = 1
        for t in trial_sequence[1:]:
            if t == current_type:
                current_count += 1
            else:
                key = f"{current_type}_{current_count}"
                consecutive_counts[key] = consecutive_counts.get(key, 0) + 1
                current_type = t
                current_count = 1
        key = f"{current_type}_{current_count}"
        consecutive_counts[key] = consecutive_counts.get(key, 0) + 1
        print("\nRuns of consecutive trials:")
        for key, count in sorted(consecutive_counts.items()):
            print(f"  {key}: {count}")
    
    # ------------------------------------------------------------------
    # Process a single participant: generate design, assign slots, finalize, and save CSV.
    # ------------------------------------------------------------------
    def process_participant(self, participant_id):
        try:
            if self.config['debug_mode']:
                print(f"\n=== Processing participant {participant_id} ===")
            design_df = self.generate_counterbalanced_design(participant_id)
            design_df = self.assign_breath_holds(design_df)
            design_df = self.finalize_design_csv(design_df)
            out_path = os.path.join(self.experiment_log_dir, f"participant_{participant_id}_design.csv")
            design_df.to_csv(out_path, index=False)
            if self.config['debug_mode']:
                print(f"Saved participant {participant_id} design to {out_path}")
            return True, out_path
        except Exception as e:
            print(f"Error processing participant {participant_id}: {e}")
            traceback.print_exc()
            return False, None
    
    # ------------------------------------------------------------------
    # Run design generation for multiple participants.
    # ------------------------------------------------------------------
    def run(self, participant_ids=None, num_participants=None):
        if participant_ids is None and num_participants is None:
            num_participants = self.config.get('num_participants', 5)
        if participant_ids is None:
            participant_ids = list(range(num_participants))
        if self.config['debug_mode']:
            print("=== Starting PPSDesignGenerator ===")
            print(f"Repetitions: {self.config['repetitions']}")
            print(f"Participants: {participant_ids}")
            print(f"Box breathing settings: {self.config['box_breathing']}")
        results = {}
        for pid in participant_ids:
            success, path = self.process_participant(pid)
            results[pid] = {'success': success, 'csv_path': path}
        return results

# ----------------------------------------------------------------------
# AUDIO GENERATION PLACEHOLDER FUNCTION
# ----------------------------------------------------------------------
def generate_audio_from_design(design_csv_path, output_dir, base_audio_path=None):
    """
    Generate audio files from a design CSV.
    (Placeholder function – implement audio generation as needed.)
    """
    print(f"Audio generation from {design_csv_path} not implemented yet")
    return None, None

# ----------------------------------------------------------------------
# MAIN FUNCTION
# ----------------------------------------------------------------------
def main():
    """Main function to run the design generator."""
    generator = PPSDesignGenerator(CONFIG)
    results = generator.run()
    print("\n=== Generation Summary ===")
    for pid, info in results.items():
        if info['success']:
            print(f"Participant {pid}: CSV generated at {info['csv_path']}")
        else:
            print(f"Participant {pid}: ERROR - no CSV generated.")
    print("\nTo generate audio files from these designs, implement the audio generation function.")

if __name__ == "__main__":
    main()


Trial count calculation:
  inhalation: 80 trials
  exhalation: 80 trials
  baseline: 80 trials
  catch: 24 trials
  Total: 264 trials
Initialized PPSDesignGenerator with:
  - Repetitions: 4
  - Conditions: ['inhalation', 'exhalation', 'baseline']
  - SOA values: [190, 400, 700, 1000, 1500]
  - Total trials per participant: 264
  - Output directory: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog
=== Starting PPSDesignGenerator ===
Repetitions: 4
Participants: [0, 1, 2, 3, 4]
Box breathing settings: {'cycle_duration_sec': 8, 'intro_duration_sec': 60, 'outro_duration_sec': 30, 'sample_rate': 48000}

=== Processing participant 0 ===

Trial type distribution:
trial_type
exhalation    80
inhalation    80
baseline      80
catch         24
Name: count, dtype: int64

Stimulus type distribution by trial type:
  exhalation: {'brown': 20, 'pink': 20, 'blue': 20, 'white': 20}
  catch: {'blue': 6, 'white': 6, 'brown': 6, 'pink': 6}
  inhalation: {'pink': 2

In [21]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
PPS Design Generator

This script generates participant‐specific experimental design CSVs for a 
Peripersonal Space (PPS) experiment. Each design CSV is built upon a 
“scaffold” CSV that contains the retention slot (breathhold) timestamps 
from a box‐breathing audio file. If needed, additional retention slots are 
appended (using fixed 8‑second increments) so that every trial can be assigned 
to a specific retention slot. Then, a counterbalanced set of trial conditions 
(e.g., trial type, looming stimulus type, SOA, jitter) is allocated to each 
scaffold row according to the following rules:
  
  - Inhalation trials (looming cue + tactile) are only assigned to scaffold rows 
    whose breathphase is “inhale.”
  - Exhalation trials (looming cue + tactile) are only assigned to scaffold rows 
    whose breathphase is “exhale.”
  - Baseline trials (tactile only; looming = NA) and catch trials (looming cue only; no tactile)
    are randomly interspersed and balanced across inhale and exhale rows.
    
The final output retains the original column order from the scaffold, with new 
trial‐specific variables appended as new columns.

Author: AI Assistant
"""

import os
import numpy as np
import pandas as pd
import random
import traceback

# ----------------------------------------------------------------------
# Helper function: Convert milliseconds to "MM:SS.s" format
# ----------------------------------------------------------------------
def convert_ms_to_timestamp(ms):
    m = int(ms // 60000)
    s = (ms % 60000) / 1000.0
    return f"{m:02}:{s:.1f}"

# ----------------------------------------------------------------------
# CONFIGURATION PARAMETERS
# ----------------------------------------------------------------------
CONFIG = {
    'repetitions': 12,            # How many times each SOA×stimulus combination repeats
    'num_participants': 5,       # Default number of participants
    'included_conditions': {
        'inhalation': True,
        'exhalation': True,
        'baseline': True,
    },
    'catch_trial_percentage': 10,
    'soa_conditions_ms': [190, 400, 700, 1000, 1500],
    'jitter_options_ms': [100, 200, 300, 400, 500],
    'debug_mode': True,
    'paths': {
        # Directory where participant-specific CSV files will be saved
        'experiment_log_dir': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog",
    },
    # Path to the original scaffold CSV containing retention (breathhold) timestamps.
    'skeleton_csv_path': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_timestamps.csv",
    'box_breathing': {
        'cycle_duration_sec': 8,      # Duration of one breath cycle in seconds
        # Intro/outro are not used here because the scaffold drives timing.
        'intro_duration_sec': 60,
        'outro_duration_sec': 30,
        'sample_rate': 48000,           # Audio sample rate (samples per second)
    }
}

# Noise types for looming stimuli (can be any labels)
LOOMING_STIMULI = ['blue', 'brown', 'pink', 'white']

# ----------------------------------------------------------------------
# CLASS: PPSDesignGenerator
# ----------------------------------------------------------------------
class PPSDesignGenerator:
    """Generates participant-specific CSV design files for PPS experiments."""
    
    def __init__(self, config=None):
        self.config = config or CONFIG
        
        # Prepare output directory.
        self.experiment_log_dir = self.config['paths']['experiment_log_dir']
        os.makedirs(self.experiment_log_dir, exist_ok=True)
        
        # Calculate trial counts.
        # For each condition (inhalation, exhalation, baseline): count = (# noise types) × (# SOA) × (repetitions).
        # Then, add 10% catch trials.
        self.trial_counts = self._calculate_trial_counts()
        self.conditions = [cond for cond, inc in self.config['included_conditions'].items() if inc]
        
        # Set seed for reproducibility.
        self.base_seed = 42
        
        if self.config['debug_mode']:
            total_trials = sum(self.trial_counts.values())
            print("Initialized PPSDesignGenerator with:")
            print(f"  - Repetitions: {self.config['repetitions']}")
            print(f"  - Conditions: {self.conditions}")
            print(f"  - SOA values: {self.config['soa_conditions_ms']}")
            print(f"  - Total trials per participant: {total_trials}")
            print(f"  - Output directory: {self.experiment_log_dir}")
    
    def _calculate_trial_counts(self):
        reps = self.config['repetitions']
        soa_count = len(self.config['soa_conditions_ms'])
        stimulus_types = LOOMING_STIMULI
        catch_percentage = self.config['catch_trial_percentage']
        
        trial_counts = {}
        for condition, include in self.config['included_conditions'].items():
            if include:
                trial_counts[condition] = len(stimulus_types) * soa_count * reps
        regular_trials = sum(trial_counts.values())
        catch_trials = int(np.ceil(regular_trials * catch_percentage / 100))
        trial_counts['catch'] = catch_trials
        
        if self.config['debug_mode']:
            print("Trial count calculation:")
            for cond, count in trial_counts.items():
                print(f"  {cond}: {count} trials")
            print(f"  Total: {sum(trial_counts.values())} trials")
        
        return trial_counts

    # ------------------------------------------------------------------
    # Load the scaffold CSV and extend it if needed.
    # Retain the original column order.
    # ------------------------------------------------------------------
    def load_retention_slots(self, required_slots):
        skeleton_path = self.config['skeleton_csv_path']
        try:
            skeleton_df = pd.read_csv(skeleton_path, sep=None, engine='python')
            if len(skeleton_df.columns) == 1:
                skeleton_df = pd.read_csv(skeleton_path, delimiter="\t")
            skeleton_df.columns = [col.strip().lower() for col in skeleton_df.columns]
            self.scaffold_order = skeleton_df.columns.tolist()
            if self.config['debug_mode']:
                print("Scaffold CSV columns:", self.scaffold_order)
        except Exception as e:
            raise ValueError(f"Error reading scaffold CSV from {skeleton_path}: {e}")
        
        current_slots = len(skeleton_df)
        required_slots = int(required_slots)
        sample_rate = self.config['box_breathing']['sample_rate']
        cycle_sec = self.config['box_breathing']['cycle_duration_sec']
        
        if self.config['debug_mode']:
            print(f"Loaded scaffold CSV with {current_slots} retention slots.")
        
        if current_slots < required_slots:
            last_row = skeleton_df.iloc[-1]
            last_ms = float(last_row['milliseconds'])
            last_sample_index = int(last_row['sample_index'])
            new_rows = []
            for i in range(required_slots - current_slots):
                new_ms = last_ms + (cycle_sec * 1000) * (i + 1)
                new_sample_index = last_sample_index + int(cycle_sec * sample_rate * (i + 1))
                new_timestamp = convert_ms_to_timestamp(new_ms)
                new_breathphase = 'inhale' if ((current_slots + i) % 2 == 0) else 'exhale'
                new_description = f"Additional hold {i+1}"
                new_rows.append({
                    'retentionslot_timestamp': new_timestamp,
                    'milliseconds': new_ms,
                    'sample_index': new_sample_index,
                    'description': new_description,
                    'breathphase': new_breathphase
                })
            if new_rows:
                skeleton_df = pd.concat([skeleton_df, pd.DataFrame(new_rows)], ignore_index=True)
            if self.config['debug_mode']:
                print(f"Extended scaffold CSV to {len(skeleton_df)} retention slots.")
        
        if len(skeleton_df) > required_slots:
            skeleton_df = skeleton_df.iloc[:required_slots]
        
        return skeleton_df

    # ------------------------------------------------------------------
    # Generate counterbalanced stimulus design (trial conditions)
    # ------------------------------------------------------------------
    def generate_counterbalanced_design(self, participant_id):
        random.seed(self.base_seed + participant_id)
        np.random.seed(self.base_seed + participant_id)
        
        soa_conditions = self.config['soa_conditions_ms']
        jitter_options = self.config['jitter_options_ms']
        stimulus_types = LOOMING_STIMULI
        
        all_trials = []
        
        # Generate main conditions: inhalation, exhalation, and baseline.
        for condition, count in self.trial_counts.items():
            if condition == 'catch' or count == 0:
                continue
            combo_list = []
            for stim_type in stimulus_types:
                for soa in soa_conditions:
                    for _ in range(self.config['repetitions']):
                        combo_list.append((stim_type, soa))
            random.shuffle(combo_list)
            for stim_type, soa in combo_list:
                all_trials.append({
                    'participant_id': participant_id,
                    'trial_type': condition,  # inhalation, exhalation, or baseline
                    'stimulus_type': stim_type,
                    'soa_value_ms': soa,
                    'jitter_ms': random.choice(jitter_options),
                    'is_tactile': True  # For these main trials, tactile is present.
                })
        
        # Add catch trials (looming only, no tactile).
        catch_count = self.trial_counts.get('catch', 0)
        if catch_count > 0:
            catch_combos = []
            stim_counts = {stim: 0 for stim in stimulus_types}
            soa_counts = {soa: 0 for soa in soa_conditions}
            while len(catch_combos) < catch_count:
                min_stim = min(stim_counts, key=stim_counts.get)
                min_soa = min(soa_counts, key=soa_counts.get)
                catch_combos.append((min_stim, min_soa))
                stim_counts[min_stim] += 1
                soa_counts[min_soa] += 1
            for stim_type, soa in catch_combos:
                all_trials.append({
                    'participant_id': participant_id,
                    'trial_type': 'catch',
                    'stimulus_type': stim_type,
                    'soa_value_ms': soa,
                    'jitter_ms': random.choice(jitter_options),
                    'is_tactile': False  # No tactile in catch trials.
                })
        
        design_df = pd.DataFrame(all_trials)
        design_df = design_df.sample(frac=1, random_state=participant_id*100).reset_index(drop=True)
        design_df['trial_number'] = range(1, len(design_df) + 1)
        
        if self.config['debug_mode']:
            print("\nTrial type distribution:")
            print(design_df['trial_type'].value_counts())
            print("\nStimulus type distribution by trial type:")
            for trial_type in design_df['trial_type'].unique():
                if trial_type == 'baseline':
                    continue
                subset = design_df[design_df['trial_type'] == trial_type]
                print(f"  {trial_type}: {subset['stimulus_type'].value_counts().to_dict()}")
            print("\nSOA distribution by trial type:")
            for trial_type in design_df['trial_type'].unique():
                if trial_type == 'baseline':
                    continue
                subset = design_df[design_df['trial_type'] == trial_type]
                print(f"  {trial_type}: {subset['soa_value_ms'].value_counts().to_dict()}")
        
        return design_df

    # ------------------------------------------------------------------
    # Assign scaffold retention slots to trials ensuring that:
    # - Inhalation trials are only assigned to scaffold rows with breathphase "inhale".
    # - Exhalation trials are only assigned to scaffold rows with breathphase "exhale".
    # - Baseline and catch trials are randomly distributed across inhale and exhale rows.
    # The scaffold order (timestamps, trial numbers) is preserved.
    # ------------------------------------------------------------------
    def assign_breath_holds(self, design_df):
        required_slots = len(design_df)
        scaffold_df = self.load_retention_slots(required_slots)
        
        # Partition the design pool into groups.
        pool_inhale = [d for d in design_df.to_dict('records') if d['trial_type'] == 'inhalation']
        pool_exhale = [d for d in design_df.to_dict('records') if d['trial_type'] == 'exhalation']
        pool_baseline = [d for d in design_df.to_dict('records') if d['trial_type'] == 'baseline']
        pool_catch = [d for d in design_df.to_dict('records') if d['trial_type'] == 'catch']
        
        # Count available scaffold rows for each breathphase.
        scaffold_inhale = scaffold_df[scaffold_df['breathphase'].str.lower() == 'inhale'].to_dict('records')
        scaffold_exhale = scaffold_df[scaffold_df['breathphase'].str.lower() == 'exhale'].to_dict('records')
        
        # Ideally, inhalation trials go to inhale rows and exhalation to exhale rows.
        assigned = []
        # Assign inhalation pool to inhale rows.
        for i, row in enumerate(scaffold_inhale):
            if i < len(pool_inhale):
                chosen = pool_inhale[i]
            else:
                chosen = None
            assigned.append((row, chosen, 'inhale'))
        # Assign exhalation pool to exhale rows.
        for i, row in enumerate(scaffold_exhale):
            if i < len(pool_exhale):
                chosen = pool_exhale[i]
            else:
                chosen = None
            assigned.append((row, chosen, 'exhale'))
        
        # Now, determine how many remaining slots in inhale and exhale need to be filled
        # with baseline and catch trials.
        remaining_inhale = [row for row, cond, bp in assigned if cond is None and bp == 'inhale']
        remaining_exhale = [row for row, cond, bp in assigned if cond is None and bp == 'exhale']
        
        # Combine baseline and catch pools, and shuffle them.
        pool_bc = pool_baseline + pool_catch
        random.shuffle(pool_bc)
        
        # Split the combined pool into two groups matching the number of remaining slots.
        num_rem_inhale = len(remaining_inhale)
        num_rem_exhale = len(remaining_exhale)
        # If there are more baseline+catch trials than needed, split evenly.
        bc_inhale = pool_bc[:num_rem_inhale]
        bc_exhale = pool_bc[num_rem_inhale:num_rem_inhale+num_rem_exhale]
        
        # Now, for assigned list, fill in those that were None.
        new_assigned = []
        for row, cond, bp in assigned:
            if cond is None:
                if bp == 'inhale':
                    cond = bc_inhale.pop(0) if bc_inhale else {}
                elif bp == 'exhale':
                    cond = bc_exhale.pop(0) if bc_exhale else {}
            # Merge scaffold row into condition.
            cond.update({
                'retentionslot_timestamp': row['retentionslot_timestamp'],
                'milliseconds': row['milliseconds'],
                'sample_index': row['sample_index'],
                'description': row['description'],
                'breathphase': row['breathphase']
            })
            new_assigned.append(cond)
        
        final_df = pd.DataFrame(new_assigned)
        final_df['trial_number'] = range(1, len(final_df) + 1)
        final_df['jittered_ms'] = final_df['milliseconds'] + final_df['jitter_ms']
        final_df['looming_stimulus_timestamp'] = final_df['jittered_ms'].apply(convert_ms_to_timestamp)
        non_catch_mask = final_df['trial_type'] != 'catch'
        final_df.loc[non_catch_mask, 'soa_ms'] = final_df.loc[non_catch_mask, 'jittered_ms'] + final_df.loc[non_catch_mask, 'soa_value_ms']
        final_df.loc[non_catch_mask, 'tactile_stimulus_timestamp'] = final_df.loc[non_catch_mask, 'soa_ms'].apply(convert_ms_to_timestamp)
        
        return final_df

    # ------------------------------------------------------------------
    # Finalize the design CSV: Retain the scaffold column order and append new columns.
    # Also, for baseline trials, remove looming stimulus info; for catch trials, remove tactile stimulus info.
    # ------------------------------------------------------------------
    def finalize_design_csv(self, design_df):
        baseline_mask = (design_df['trial_type'] == 'baseline')
        design_df.loc[baseline_mask, 'stimulus_type'] = None
        design_df.loc[baseline_mask, 'looming_stimulus_timestamp'] = None
        
        catch_mask = (design_df['trial_type'] == 'catch')
        design_df.loc[catch_mask, 'tactile_stimulus_timestamp'] = None
        
        design_df['expected_response'] = ~design_df['trial_type'].isin(['catch'])
        
        scaffold_cols = self.scaffold_order if hasattr(self, 'scaffold_order') else ['retentionslot_timestamp', 'milliseconds', 'sample_index', 'description', 'breathphase']
        new_cols = ['participant_id', 'trial_number', 'trial_type', 'stimulus_type', 'soa_value_ms', 'jitter_ms', 'is_tactile', 'looming_stimulus_timestamp', 'tactile_stimulus_timestamp', 'soa_ms', 'jittered_ms', 'expected_response']
        final_order = scaffold_cols + [col for col in new_cols if col not in scaffold_cols and col in design_df.columns]
        remaining = [col for col in design_df.columns if col not in final_order]
        final_order.extend(remaining)
        design_df = design_df[final_order]
        
        self._validate_design(design_df)
        return design_df

    # ------------------------------------------------------------------
    # Validation: Print distributions and sequence properties.
    # ------------------------------------------------------------------
    def _validate_design(self, design_df):
        if not self.config['debug_mode']:
            return
        print("\n=== DESIGN VALIDATION ===")
        print("Trial type counts:")
        print(design_df['trial_type'].value_counts())
        print("\nStimulus distribution by trial type:")
        for trial_type in design_df['trial_type'].unique():
            if trial_type == 'baseline':
                continue
            subset = design_df[design_df['trial_type'] == trial_type]
            print(f"  {trial_type}: {subset['stimulus_type'].value_counts().to_dict()}")
        print("\nSOA distribution by trial type:")
        for trial_type in design_df['trial_type'].unique():
            if trial_type == 'baseline':
                continue
            subset = design_df[design_df['trial_type'] == trial_type]
            print(f"  {trial_type}: {subset['soa_value_ms'].value_counts().to_dict()}")
        print("\nBreath phase distribution:")
        if 'breathphase' in design_df.columns:
            print(design_df['breathphase'].value_counts())
        trial_sequence = design_df['trial_type'].tolist()
        consecutive_counts = {}
        current_type = trial_sequence[0]
        current_count = 1
        for t in trial_sequence[1:]:
            if t == current_type:
                current_count += 1
            else:
                key = f"{current_type}_{current_count}"
                consecutive_counts[key] = consecutive_counts.get(key, 0) + 1
                current_type = t
                current_count = 1
        key = f"{current_type}_{current_count}"
        consecutive_counts[key] = consecutive_counts.get(key, 0) + 1
        print("\nRuns of consecutive trials:")
        for key, count in sorted(consecutive_counts.items()):
            print(f"  {key}: {count}")
    
    # ------------------------------------------------------------------
    # Process a single participant: generate design, assign slots, finalize, and save CSV.
    # ------------------------------------------------------------------
    def process_participant(self, participant_id):
        try:
            if self.config['debug_mode']:
                print(f"\n=== Processing participant {participant_id} ===")
            design_df = self.generate_counterbalanced_design(participant_id)
            design_df = self.assign_breath_holds(design_df)
            design_df = self.finalize_design_csv(design_df)
            out_path = os.path.join(self.experiment_log_dir, f"participant_{participant_id}_design.csv")
            design_df.to_csv(out_path, index=False)
            if self.config['debug_mode']:
                print(f"Saved participant {participant_id} design to {out_path}")
            return True, out_path
        except Exception as e:
            print(f"Error processing participant {participant_id}: {e}")
            traceback.print_exc()
            return False, None
    
    # ------------------------------------------------------------------
    # Run design generation for multiple participants.
    # ------------------------------------------------------------------
    def run(self, participant_ids=None, num_participants=None):
        if participant_ids is None and num_participants is None:
            num_participants = self.config.get('num_participants', 5)
        if participant_ids is None:
            participant_ids = list(range(num_participants))
        if self.config['debug_mode']:
            print("=== Starting PPSDesignGenerator ===")
            print(f"Repetitions: {self.config['repetitions']}")
            print(f"Participants: {participant_ids}")
            print(f"Box breathing settings: {self.config['box_breathing']}")
        results = {}
        for pid in participant_ids:
            success, path = self.process_participant(pid)
            results[pid] = {'success': success, 'csv_path': path}
        return results

# ----------------------------------------------------------------------
# AUDIO GENERATION PLACEHOLDER FUNCTION
# ----------------------------------------------------------------------
def generate_audio_from_design(design_csv_path, output_dir, base_audio_path=None):
    """
    Generate audio files from a design CSV.
    (Placeholder function – implement audio generation as needed.)
    """
    print(f"Audio generation from {design_csv_path} not implemented yet")
    return None, None

# ----------------------------------------------------------------------
# MAIN FUNCTION
# ----------------------------------------------------------------------
def main():
    """Main function to run the design generator."""
    generator = PPSDesignGenerator(CONFIG)
    results = generator.run()
    print("\n=== Generation Summary ===")
    for pid, info in results.items():
        if info['success']:
            print(f"Participant {pid}: CSV generated at {info['csv_path']}")
        else:
            print(f"Participant {pid}: ERROR - no CSV generated.")
    print("\nTo generate audio files from these designs, implement the audio generation function.")

if __name__ == "__main__":
    main()


Trial count calculation:
  inhalation: 240 trials
  exhalation: 240 trials
  baseline: 240 trials
  catch: 72 trials
  Total: 792 trials
Initialized PPSDesignGenerator with:
  - Repetitions: 12
  - Conditions: ['inhalation', 'exhalation', 'baseline']
  - SOA values: [190, 400, 700, 1000, 1500]
  - Total trials per participant: 792
  - Output directory: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog
=== Starting PPSDesignGenerator ===
Repetitions: 12
Participants: [0, 1, 2, 3, 4]
Box breathing settings: {'cycle_duration_sec': 8, 'intro_duration_sec': 60, 'outro_duration_sec': 30, 'sample_rate': 48000}

=== Processing participant 0 ===

Trial type distribution:
trial_type
baseline      240
inhalation    240
exhalation    240
catch          72
Name: count, dtype: int64

Stimulus type distribution by trial type:
  inhalation: {'white': 60, 'blue': 60, 'brown': 60, 'pink': 60}
  exhalation: {'pink': 60, 'white': 60, 'blue': 60, 'brown': 60}
  catc

In [24]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
PPS Design Generator

This script generates participant‐specific experimental design CSVs for a 
Peripersonal Space (PPS) experiment. Each design CSV is built upon a 
"scaffold" CSV that contains the retention slot (breathhold) timestamps 
from a box‐breathing audio file. If needed, additional retention slots are 
appended (using fixed 8‑second increments) so that every trial can be assigned 
to a specific retention slot. Then, a counterbalanced set of trial conditions 
(e.g., trial type, looming stimulus type, SOA, jitter) is allocated to each 
scaffold row according to the following rules:
  
  - Inhalation trials (looming cue + tactile) are only assigned to scaffold rows 
    whose breathphase is "inhale."
  - Exhalation trials (looming cue + tactile) are only assigned to scaffold rows 
    whose breathphase is "exhale."
  - Baseline (tactile only; looming = NA) and catch (looming only; no tactile) trials 
    are allocated from the remaining pool in a random, counterbalanced manner 
    between the inhale and exhale rows.
    
The final output retains the original column order from the scaffold, with new 
trial‐specific variables appended as new columns.

Author: AI Assistant
"""

import os
import numpy as np
import pandas as pd
import random
import traceback

# ----------------------------------------------------------------------
# Helper function: Convert milliseconds to "MM:SS.s" format
# ----------------------------------------------------------------------
def convert_ms_to_timestamp(ms):
    m = int(ms // 60000)
    s = (ms % 60000) / 1000.0
    return f"{m:02}:{s:.1f}"

# ----------------------------------------------------------------------
# CONFIGURATION PARAMETERS
# ----------------------------------------------------------------------
CONFIG = {
    'repetitions': 4,            # How many times each SOA×stimulus combination repeats
    'num_participants': 5,       # Default number of participants
    'included_conditions': {
        'inhalation': True,
        'exhalation': True,
        'baseline': True,
    },
    'catch_trial_percentage': 10,
    'soa_conditions_ms': [190, 400, 700, 1000, 1500],
    'jitter_options_ms': [100, 200, 300, 400, 500],
    'debug_mode': True,
    'paths': {
        # Directory where participant-specific CSV files will be saved
        'experiment_log_dir': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog",
    },
    # Path to the original scaffold CSV containing retention (breathhold) timestamps.
    'skeleton_csv_path': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_timestamps.csv",
    'box_breathing': {
        'cycle_duration_sec': 8,      # Duration of one breath cycle in seconds
        # Intro/outro are not used here because the scaffold drives timing.
        'intro_duration_sec': 60,
        'outro_duration_sec': 30,
        'sample_rate': 48000,           # Audio sample rate (samples per second)
    }
}

# Noise types for looming stimuli (can be any labels)
LOOMING_STIMULI = ['blue', 'brown', 'pink', 'white']

# ----------------------------------------------------------------------
# CLASS: PPSDesignGenerator
# ----------------------------------------------------------------------
class PPSDesignGenerator:
    """Generates participant-specific CSV design files for PPS experiments."""
    
    def __init__(self, config=None):
        self.config = config or CONFIG
        
        # Prepare output directory.
        self.experiment_log_dir = self.config['paths']['experiment_log_dir']
        os.makedirs(self.experiment_log_dir, exist_ok=True)
        
        # Calculate trial counts.
        # For each condition (inhalation, exhalation, baseline): count = (# noise types) × (# SOA) × (repetitions).
        # Then, add 10% catch trials.
        self.trial_counts = self._calculate_trial_counts()
        self.conditions = [cond for cond, inc in self.config['included_conditions'].items() if inc]
        
        # Set seed for reproducibility.
        self.base_seed = 42
        
        if self.config['debug_mode']:
            total_trials = sum(self.trial_counts.values())
            print("Initialized PPSDesignGenerator with:")
            print(f"  - Repetitions: {self.config['repetitions']}")
            print(f"  - Conditions: {self.conditions}")
            print(f"  - SOA values: {self.config['soa_conditions_ms']}")
            print(f"  - Total trials per participant: {total_trials}")
            print(f"  - Output directory: {self.experiment_log_dir}")
    
    def _calculate_trial_counts(self):
        reps = self.config['repetitions']
        soa_count = len(self.config['soa_conditions_ms'])
        stimulus_types = LOOMING_STIMULI
        catch_percentage = self.config['catch_trial_percentage']
        
        trial_counts = {}
        for condition, include in self.config['included_conditions'].items():
            if include:
                trial_counts[condition] = len(stimulus_types) * soa_count * reps
        regular_trials = sum(trial_counts.values())
        catch_trials = int(np.ceil(regular_trials * catch_percentage / 100))
        trial_counts['catch'] = catch_trials
        
        if self.config['debug_mode']:
            print("Trial count calculation:")
            for cond, count in trial_counts.items():
                print(f"  {cond}: {count} trials")
            print(f"  Total: {sum(trial_counts.values())} trials")
        
        return trial_counts

    # ------------------------------------------------------------------
    # Load the scaffold CSV and extend it if needed.
    # Retain the original column order.
    # ------------------------------------------------------------------
    def load_retention_slots(self, required_slots):
        skeleton_path = self.config['skeleton_csv_path']
        try:
            skeleton_df = pd.read_csv(skeleton_path, sep=None, engine='python')
            if len(skeleton_df.columns) == 1:
                skeleton_df = pd.read_csv(skeleton_path, delimiter="\t")
            skeleton_df.columns = [col.strip().lower() for col in skeleton_df.columns]
            self.scaffold_order = skeleton_df.columns.tolist()
            if self.config['debug_mode']:
                print("Scaffold CSV columns:", self.scaffold_order)
        except Exception as e:
            raise ValueError(f"Error reading scaffold CSV from {skeleton_path}: {e}")
        
        current_slots = len(skeleton_df)
        required_slots = int(required_slots)
        sample_rate = self.config['box_breathing']['sample_rate']
        cycle_sec = self.config['box_breathing']['cycle_duration_sec']
        
        if self.config['debug_mode']:
            print(f"Loaded scaffold CSV with {current_slots} retention slots.")
        
        if current_slots < required_slots:
            last_row = skeleton_df.iloc[-1]
            last_ms = float(last_row['milliseconds'])
            last_sample_index = int(last_row['sample_index'])
            new_rows = []
            for i in range(required_slots - current_slots):
                new_ms = last_ms + (cycle_sec * 1000) * (i + 1)
                new_sample_index = last_sample_index + int(cycle_sec * sample_rate * (i + 1))
                new_timestamp = convert_ms_to_timestamp(new_ms)
                new_breathphase = 'inhale' if ((current_slots + i) % 2 == 0) else 'exhale'
                new_description = f"Additional hold {i+1}"
                new_rows.append({
                    'timestamp': new_timestamp,
                    'milliseconds': new_ms,
                    'sample_index': new_sample_index,
                    'description': new_description,
                    'breathphase': new_breathphase
                })
            if new_rows:
                skeleton_df = pd.concat([skeleton_df, pd.DataFrame(new_rows)], ignore_index=True)
            if self.config['debug_mode']:
                print(f"Extended scaffold CSV to {len(skeleton_df)} retention slots.")
        
        if len(skeleton_df) > required_slots:
            skeleton_df = skeleton_df.iloc[:required_slots]
        
        return skeleton_df

    # ------------------------------------------------------------------
    # Generate counterbalanced stimulus design (trial conditions)
    # ------------------------------------------------------------------
    def generate_counterbalanced_design(self, participant_id):
        random.seed(self.base_seed + participant_id)
        np.random.seed(self.base_seed + participant_id)
        
        soa_conditions = self.config['soa_conditions_ms']
        jitter_options = self.config['jitter_options_ms']
        stimulus_types = LOOMING_STIMULI
        
        all_trials = []
        
        # Generate main conditions: inhalation, exhalation, and baseline.
        for condition, count in self.trial_counts.items():
            if condition == 'catch' or count == 0:
                continue
            combo_list = []
            for stim_type in stimulus_types:
                for soa in soa_conditions:
                    for _ in range(self.config['repetitions']):
                        combo_list.append((stim_type, soa))
            random.shuffle(combo_list)
            for stim_type, soa in combo_list:
                all_trials.append({
                    'participant_id': participant_id,
                    'trial_type': condition,  # "inhalation", "exhalation", or "baseline"
                    'stimulus_type': stim_type,
                    'soa_value_ms': soa,
                    'jitter_ms': random.choice(jitter_options),
                    'is_tactile': True  # For these main trials, tactile is present.
                })
        
        # Add catch trials (looming only, no tactile).
        catch_count = self.trial_counts.get('catch', 0)
        if catch_count > 0:
            catch_combos = []
            stim_counts = {stim: 0 for stim in stimulus_types}
            soa_counts = {soa: 0 for soa in soa_conditions}
            while len(catch_combos) < catch_count:
                min_stim = min(stim_counts, key=stim_counts.get)
                min_soa = min(soa_counts, key=soa_counts.get)
                catch_combos.append((min_stim, min_soa))
                stim_counts[min_stim] += 1
                soa_counts[min_soa] += 1
            for stim_type, soa in catch_combos:
                all_trials.append({
                    'participant_id': participant_id,
                    'trial_type': 'catch',
                    'stimulus_type': stim_type,
                    'soa_value_ms': soa,
                    'jitter_ms': random.choice(jitter_options),
                    'is_tactile': False
                })
        
        design_df = pd.DataFrame(all_trials)
        design_df = design_df.sample(frac=1, random_state=participant_id*100).reset_index(drop=True)
        design_df['trial_number'] = range(1, len(design_df) + 1)
        
        if self.config['debug_mode']:
            print("\nTrial type distribution:")
            print(design_df['trial_type'].value_counts())
            print("\nStimulus type distribution by trial type:")
            for trial_type in design_df['trial_type'].unique():
                if trial_type == 'baseline':
                    continue
                subset = design_df[design_df['trial_type'] == trial_type]
                print(f"  {trial_type}: {subset['stimulus_type'].value_counts().to_dict()}")
            print("\nSOA distribution by trial type:")
            for trial_type in design_df['trial_type'].unique():
                if trial_type == 'baseline':
                    continue
                subset = design_df[design_df['trial_type'] == trial_type]
                print(f"  {trial_type}: {subset['soa_value_ms'].value_counts().to_dict()}")
        
        return design_df

    # ------------------------------------------------------------------
    # Assign scaffold retention slots to trials while strictly maintaining the breathphase cycle.
    # ------------------------------------------------------------------
    def assign_breath_holds(self, design_df):
        """
        Assign scaffold retention slots to trials while strictly maintaining the breathphase cycle.
        
        Key requirements:
        1. Breathphase cycle (inhale-exhale-inhale-exhale) must remain intact
        2. Inhalation trials must go to inhale breathphase slots
        3. Exhalation trials must go to exhale breathphase slots
        4. Baseline and catch trials can go to either breathphase type
        5. The order of baseline and catch trials should be randomized
        """
        required_slots = len(design_df)
        scaffold_df = self.load_retention_slots(required_slots)
        
        # Ensure scaffold is properly sorted by timestamp/milliseconds
        scaffold_df = scaffold_df.sort_values(by='milliseconds').reset_index(drop=True)
        
        # Partition design trials by trial type
        pool_inh = [d for d in design_df.to_dict('records') if d['trial_type'] == 'inhalation']
        pool_exh = [d for d in design_df.to_dict('records') if d['trial_type'] == 'exhalation']
        pool_baseline = [d for d in design_df.to_dict('records') if d['trial_type'] == 'baseline']
        pool_catch = [d for d in design_df.to_dict('records') if d['trial_type'] == 'catch']
        
        # Count scaffold slots by breathphase
        inhale_slots = scaffold_df[scaffold_df['breathphase'].str.lower() == 'inhale']
        exhale_slots = scaffold_df[scaffold_df['breathphase'].str.lower() == 'exhale']
        n_inhale_slots = len(inhale_slots)
        n_exhale_slots = len(exhale_slots)
        
        if self.config['debug_mode']:
            print(f"Scaffold: {n_inhale_slots} inhale slots, {n_exhale_slots} exhale slots")
            print(f"Trials: {len(pool_inh)} inhalation, {len(pool_exh)} exhalation, " 
                  f"{len(pool_baseline)} baseline, {len(pool_catch)} catch")
        
        # Verify we have enough slots for mandatory trials
        if len(pool_inh) > n_inhale_slots:
            raise ValueError(f"Not enough inhale slots ({n_inhale_slots}) for inhalation trials ({len(pool_inh)})")
        if len(pool_exh) > n_exhale_slots:
            raise ValueError(f"Not enough exhale slots ({n_exhale_slots}) for exhalation trials ({len(pool_exh)})")
        
        # Calculate remaining slots per breathphase
        remaining_inhale = n_inhale_slots - len(pool_inh)
        remaining_exhale = n_exhale_slots - len(pool_exh)
        
        # Total baseline+catch trials
        total_flex_trials = len(pool_baseline) + len(pool_catch)
        
        # Verify we have enough slots for all trials
        if remaining_inhale + remaining_exhale != total_flex_trials:
            raise ValueError(f"Mismatch between available slots ({remaining_inhale + remaining_exhale}) "
                            f"and flexible trials ({total_flex_trials})")
        
        # Decide how to distribute baseline and catch trials across breathphases
        # Combine them first, then distribute proportionally
        flexible_trials = pool_baseline + pool_catch
        random.shuffle(flexible_trials)  # Randomize order
        
        # Split flexible trials between inhale and exhale slots proportionally
        inhale_flexible = flexible_trials[:remaining_inhale]
        exhale_flexible = flexible_trials[remaining_inhale:]
        
        if self.config['debug_mode']:
            print(f"Distributing {len(inhale_flexible)} flexible trials to inhale slots")
            print(f"Distributing {len(exhale_flexible)} flexible trials to exhale slots")
        
        # Create final allocation pools
        inhale_pool = pool_inh + inhale_flexible
        exhale_pool = pool_exh + exhale_flexible
        
        # Shuffle within each pool to randomize order
        random.shuffle(inhale_pool)
        random.shuffle(exhale_pool)
        
        # Create a dictionary mapping breathphase to trial pool
        phase_to_pool = {
            'inhale': inhale_pool,
            'exhale': exhale_pool
        }
        
        # Initialize counters for each breathphase
        pool_counters = {
            'inhale': 0,
            'exhale': 0
        }
        
        # Final list to store assigned trials in scaffold order
        final_assignments = []
        
        # Assign trials to scaffold rows, maintaining the original order
        for _, row in scaffold_df.iterrows():
            breathphase = row['breathphase'].lower()
            
            # Get the next trial from the appropriate pool
            if pool_counters[breathphase] < len(phase_to_pool[breathphase]):
                trial = phase_to_pool[breathphase][pool_counters[breathphase]]
                pool_counters[breathphase] += 1
                
                # Combine scaffold information with trial information
                combined = trial.copy()
                for col in row.index:
                    combined[col] = row[col]
                
                final_assignments.append(combined)
        
        # Create final dataframe and sort by scaffold order
        final_df = pd.DataFrame(final_assignments).sort_values(by='milliseconds').reset_index(drop=True)
        final_df['trial_number'] = range(1, len(final_df) + 1)
        
        # Compute additional timing information
        final_df['jittered_ms'] = final_df['milliseconds'] + final_df['jitter_ms']
        final_df['looming_stimulus_timestamp'] = final_df['jittered_ms'].apply(convert_ms_to_timestamp)
        
        # Add SOA timing for non-catch trials
        non_catch_mask = final_df['trial_type'] != 'catch'
        final_df.loc[non_catch_mask, 'soa_ms'] = final_df.loc[non_catch_mask, 'jittered_ms'] + final_df.loc[non_catch_mask, 'soa_value_ms']
        final_df.loc[non_catch_mask, 'tactile_stimulus_timestamp'] = final_df.loc[non_catch_mask, 'soa_ms'].apply(convert_ms_to_timestamp)
        
        # Verify the assignment respects the breathphase constraints
        self._validate_breathphase_assignment(final_df)
        
        return final_df

    def _validate_breathphase_assignment(self, final_df):
        """Verify that the trial assignment respects breathphase constraints."""
        if not self.config['debug_mode']:
            return
            
        print("\n=== BREATHPHASE ASSIGNMENT VALIDATION ===")
        
        # Check for inhalation trials assigned to inhale
        inh_trials = final_df[final_df['trial_type'] == 'inhalation']
        inh_correct = (inh_trials['breathphase'].str.lower() == 'inhale').all()
        print(f"Inhalation trials on inhale breathphase: {inh_correct}")
        
        # Check for exhalation trials assigned to exhale
        exh_trials = final_df[final_df['trial_type'] == 'exhalation']
        exh_correct = (exh_trials['breathphase'].str.lower() == 'exhale').all()
        print(f"Exhalation trials on exhale breathphase: {exh_correct}")
        
        # Count distribution of baseline and catch trials across breathphases
        baseline_inhale = ((final_df['trial_type'] == 'baseline') & 
                         (final_df['breathphase'].str.lower() == 'inhale')).sum()
        baseline_exhale = ((final_df['trial_type'] == 'baseline') & 
                          (final_df['breathphase'].str.lower() == 'exhale')).sum()
        catch_inhale = ((final_df['trial_type'] == 'catch') & 
                       (final_df['breathphase'].str.lower() == 'inhale')).sum()
        catch_exhale = ((final_df['trial_type'] == 'catch') & 
                       (final_df['breathphase'].str.lower() == 'exhale')).sum()
        
        print(f"Baseline trials: {baseline_inhale} on inhale, {baseline_exhale} on exhale")
        print(f"Catch trials: {catch_inhale} on inhale, {catch_exhale} on exhale")
        
        # Verify breathphase cycle is maintained
        prev_phase = None
        cycle_maintained = True
        expected_cycle = ['inhale', 'exhale'] * (len(final_df) // 2 + 1)
        
        for i, phase in enumerate(final_df['breathphase'].str.lower()):
            if i < len(expected_cycle) and phase != expected_cycle[i]:
                cycle_maintained = False
                print(f"Breathphase cycle disrupted at index {i}: expected {expected_cycle[i]}, got {phase}")
                break
        
        print(f"Breathphase cycle maintained: {cycle_maintained}")

    # ------------------------------------------------------------------
    # Finalize the design CSV: Retain the scaffold column order and append new columns.
    # Also, for baseline trials remove looming stimulus info; for catch trials remove tactile stimulus info.
    # ------------------------------------------------------------------
    def finalize_design_csv(self, design_df):
        baseline_mask = (design_df['trial_type'] == 'baseline')
        design_df.loc[baseline_mask, 'stimulus_type'] = None
        design_df.loc[baseline_mask, 'looming_stimulus_timestamp'] = None
        
        catch_mask = (design_df['trial_type'] == 'catch')
        design_df.loc[catch_mask, 'tactile_stimulus_timestamp'] = None
        
        design_df['expected_response'] = ~design_df['trial_type'].isin(['catch'])
        
        scaffold_cols = self.scaffold_order if hasattr(self, 'scaffold_order') else ['timestamp', 'milliseconds', 'sample_index', 'description', 'breathphase']
        new_cols = ['participant_id', 'trial_number', 'trial_type', 'stimulus_type', 'soa_value_ms', 'jitter_ms', 'is_tactile', 'looming_stimulus_timestamp', 'tactile_stimulus_timestamp', 'soa_ms', 'jittered_ms', 'expected_response']
        final_order = scaffold_cols + [col for col in new_cols if col not in scaffold_cols and col in design_df.columns]
        remaining = [col for col in design_df.columns if col not in final_order]
        final_order.extend(remaining)
        design_df = design_df[final_order]
        
        self._validate_design(design_df)
        return design_df

    # ------------------------------------------------------------------
    # Validation: Print distributions and sequence properties.
    # ------------------------------------------------------------------
    def _validate_design(self, design_df):
        if not self.config['debug_mode']:
            return
        print("\n=== DESIGN VALIDATION ===")
        print("Trial type counts:")
        print(design_df['trial_type'].value_counts())
        print("\nStimulus distribution by trial type:")
        for trial_type in design_df['trial_type'].unique():
            if trial_type == 'baseline':
                continue
            subset = design_df[design_df['trial_type'] == trial_type]
            print(f"  {trial_type}: {subset['stimulus_type'].value_counts().to_dict()}")
        print("\nSOA distribution by trial type:")
        for trial_type in design_df['trial_type'].unique():
            if trial_type == 'baseline':
                continue
            subset = design_df[design_df['trial_type'] == trial_type]
            print(f"  {trial_type}: {subset['soa_value_ms'].value_counts().to_dict()}")
        print("\nBreath phase distribution:")
        if 'breathphase' in design_df.columns:
            print(design_df['breathphase'].value_counts())
        trial_sequence = design_df['trial_type'].tolist()
        consecutive_counts = {}
        current_type = trial_sequence[0]
        current_count = 1
        for t in trial_sequence[1:]:
            if t == current_type:
                current_count += 1
            else:
                key = f"{current_type}_{current_count}"
                consecutive_counts[key] = consecutive_counts.get(key, 0) + 1
                current_type = t
                current_count = 1
        key = f"{current_type}_{current_count}"
        consecutive_counts[key] = consecutive_counts.get(key, 0) + 1
        print("\nRuns of consecutive trials:")
        for key, count in sorted(consecutive_counts.items()):
            print(f"  {key}: {count}")

        # Additional validation: Check for balanced distribution across participants
        # This will be populated when running multiple participants
        print("\nBreathphase-by-trial-type balance:")
        breathphase_balance = design_df.groupby(['breathphase', 'trial_type']).size()
        print(breathphase_balance)
    
    # ------------------------------------------------------------------
    # Process a single participant: generate design, assign slots, finalize, and save CSV.
    # ------------------------------------------------------------------
    def process_participant(self, participant_id):
        try:
            if self.config['debug_mode']:
                print(f"\n=== Processing participant {participant_id} ===")
            design_df = self.generate_counterbalanced_design(participant_id)
            design_df = self.assign_breath_holds(design_df)
            design_df = self.finalize_design_csv(design_df)
            out_path = os.path.join(self.experiment_log_dir, f"participant_{participant_id}_design.csv")
            design_df.to_csv(out_path, index=False)
            if self.config['debug_mode']:
                print(f"Saved participant {participant_id} design to {out_path}")
            return True, out_path
        except Exception as e:
            print(f"Error processing participant {participant_id}: {e}")
            traceback.print_exc()
            return False, None
    
    # ------------------------------------------------------------------
    # Run design generation for multiple participants.
    # ------------------------------------------------------------------
    def run(self, participant_ids=None, num_participants=None):
        if participant_ids is None and num_participants is None:
            num_participants = self.config.get('num_participants', 5)
        if participant_ids is None:
            participant_ids = list(range(num_participants))
        if self.config['debug_mode']:
            print("=== Starting PPSDesignGenerator ===")
            print(f"Repetitions: {self.config['repetitions']}")
            print(f"Participants: {participant_ids}")
            print(f"Box breathing settings: {self.config['box_breathing']}")
        results = {}
        for pid in participant_ids:
            success, path = self.process_participant(pid)
            results[pid] = {'success': success, 'csv_path': path}
        
        # Summary of all participants
        if self.config['debug_mode'] and len(results) > 1:
            print("\n=== CROSS-PARTICIPANT VALIDATION ===")
            all_success = all(info['success'] for info in results.values())
            print(f"All participants generated successfully: {all_success}")
        
        return results

# ----------------------------------------------------------------------
# AUDIO GENERATION PLACEHOLDER FUNCTION
# ----------------------------------------------------------------------
def generate_audio_from_design(design_csv_path, output_dir, base_audio_path=None):
    """
    Generate audio files from a design CSV.
    (Placeholder function – implement audio generation as needed.)
    """
    print(f"Audio generation from {design_csv_path} not implemented yet")
    return None, None

# ----------------------------------------------------------------------
# MAIN FUNCTION
# ----------------------------------------------------------------------
def main():
    """Main function to run the design generator."""
    generator = PPSDesignGenerator(CONFIG)
    results = generator.run()
    print("\n=== Generation Summary ===")
    for pid, info in results.items():
        if info['success']:
            print(f"Participant {pid}: CSV generated at {info['csv_path']}")
        else:
            print(f"Participant {pid}: ERROR - no CSV generated.")
    print("\nTo generate audio files from these designs, implement the audio generation function.")

if __name__ == "__main__":
    main()

Trial count calculation:
  inhalation: 80 trials
  exhalation: 80 trials
  baseline: 80 trials
  catch: 24 trials
  Total: 264 trials
Initialized PPSDesignGenerator with:
  - Repetitions: 4
  - Conditions: ['inhalation', 'exhalation', 'baseline']
  - SOA values: [190, 400, 700, 1000, 1500]
  - Total trials per participant: 264
  - Output directory: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog
=== Starting PPSDesignGenerator ===
Repetitions: 4
Participants: [0, 1, 2, 3, 4]
Box breathing settings: {'cycle_duration_sec': 8, 'intro_duration_sec': 60, 'outro_duration_sec': 30, 'sample_rate': 48000}

=== Processing participant 0 ===

Trial type distribution:
trial_type
exhalation    80
inhalation    80
baseline      80
catch         24
Name: count, dtype: int64

Stimulus type distribution by trial type:
  exhalation: {'brown': 20, 'pink': 20, 'blue': 20, 'white': 20}
  catch: {'blue': 6, 'white': 6, 'brown': 6, 'pink': 6}
  inhalation: {'pink': 2

In [26]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
PPS Design Generator

This script generates participant‐specific experimental design CSVs for a 
Peripersonal Space (PPS) experiment. Each design CSV is built upon a 
"scaffold" CSV that contains the retention slot (breathhold) timestamps 
from a box‐breathing audio file. If needed, additional retention slots are 
appended (using fixed 8‑second increments) so that every trial can be assigned 
to a specific retention slot. Then, a counterbalanced set of trial conditions 
(e.g., trial type, looming stimulus type, SOA, jitter) is allocated to each 
scaffold row according to the following rules:
  
  - Inhalation trials (looming cue + tactile) are only assigned to scaffold rows 
    whose breathphase is "inhale."
  - Exhalation trials (looming cue + tactile) are only assigned to scaffold rows 
    whose breathphase is "exhale."
  - Baseline (tactile only; looming = NA) and catch (looming only; no tactile) trials 
    are allocated from the remaining pool in a random, counterbalanced manner 
    between the inhale and exhale rows.
    
The final output retains the original column order from the scaffold, with new 
trial‐specific variables appended as new columns.

Author: AI Assistant
"""

import os
import numpy as np
import pandas as pd
import random
import traceback

# ----------------------------------------------------------------------
# Helper function: Convert milliseconds to "MM:SS.s" format
# ----------------------------------------------------------------------
def convert_ms_to_timestamp(ms):
    m = int(ms // 60000)
    s = (ms % 60000) / 1000.0
    return f"{m:02}:{s:.1f}"

# ----------------------------------------------------------------------
# CONFIGURATION PARAMETERS
# ----------------------------------------------------------------------
CONFIG = {
    'repetitions': 4,            # How many times each SOA×stimulus combination repeats
    'num_participants': 5,       # Default number of participants
    'included_conditions': {
        'inhalation': True,
        'exhalation': True,
        'baseline': True,
    },
    'catch_trial_percentage': 10,
    'soa_conditions_ms': [190, 400, 700, 1000, 1500],
    'jitter_options_ms': [100, 200, 300, 400, 500],
    'debug_mode': True,
    'paths': {
        # Directory where participant-specific CSV files will be saved
        'experiment_log_dir': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog",
    },
    # Path to the original scaffold CSV containing retention (breathhold) timestamps.
    'skeleton_csv_path': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_timestamps.csv",
    'box_breathing': {
        'cycle_duration_sec': 8,      # Duration of one breath cycle in seconds
        # Intro/outro are not used here because the scaffold drives timing.
        'intro_duration_sec': 60,
        'outro_duration_sec': 30,
        'sample_rate': 48000,           # Audio sample rate (samples per second)
    }
}

# Noise types for looming stimuli (can be any labels)
LOOMING_STIMULI = ['blue', 'brown', 'pink', 'white']

# ----------------------------------------------------------------------
# CLASS: PPSDesignGenerator
# ----------------------------------------------------------------------
class PPSDesignGenerator:
    """Generates participant-specific CSV design files for PPS experiments."""
    
    def __init__(self, config=None):
        self.config = config or CONFIG
        
        # Prepare output directory.
        self.experiment_log_dir = self.config['paths']['experiment_log_dir']
        os.makedirs(self.experiment_log_dir, exist_ok=True)
        
        # Calculate trial counts.
        # For each condition (inhalation, exhalation, baseline): count = (# noise types) × (# SOA) × (repetitions).
        # Then, add 10% catch trials.
        self.trial_counts = self._calculate_trial_counts()
        self.conditions = [cond for cond, inc in self.config['included_conditions'].items() if inc]
        
        # Set seed for reproducibility.
        self.base_seed = 42
        
        if self.config['debug_mode']:
            total_trials = sum(self.trial_counts.values())
            print("Initialized PPSDesignGenerator with:")
            print(f"  - Repetitions: {self.config['repetitions']}")
            print(f"  - Conditions: {self.conditions}")
            print(f"  - SOA values: {self.config['soa_conditions_ms']}")
            print(f"  - Total trials per participant: {total_trials}")
            print(f"  - Output directory: {self.experiment_log_dir}")
    
    def _calculate_trial_counts(self):
        reps = self.config['repetitions']
        soa_count = len(self.config['soa_conditions_ms'])
        stimulus_types = LOOMING_STIMULI
        catch_percentage = self.config['catch_trial_percentage']
        
        trial_counts = {}
        for condition, include in self.config['included_conditions'].items():
            if include:
                trial_counts[condition] = len(stimulus_types) * soa_count * reps
        regular_trials = sum(trial_counts.values())
        catch_trials = int(np.ceil(regular_trials * catch_percentage / 100))
        trial_counts['catch'] = catch_trials
        
        if self.config['debug_mode']:
            print("Trial count calculation:")
            for cond, count in trial_counts.items():
                print(f"  {cond}: {count} trials")
            print(f"  Total: {sum(trial_counts.values())} trials")
        
        return trial_counts

    # ------------------------------------------------------------------
    # Load the scaffold CSV and extend it if needed.
    # Retain the original column order.
    # ------------------------------------------------------------------
    def load_retention_slots(self, required_slots):
        skeleton_path = self.config['skeleton_csv_path']
        try:
            skeleton_df = pd.read_csv(skeleton_path, sep=None, engine='python')
            if len(skeleton_df.columns) == 1:
                skeleton_df = pd.read_csv(skeleton_path, delimiter="\t")
            skeleton_df.columns = [col.strip().lower() for col in skeleton_df.columns]
            self.scaffold_order = skeleton_df.columns.tolist()
            if self.config['debug_mode']:
                print("Scaffold CSV columns:", self.scaffold_order)
        except Exception as e:
            raise ValueError(f"Error reading scaffold CSV from {skeleton_path}: {e}")
        
        current_slots = len(skeleton_df)
        required_slots = int(required_slots)
        sample_rate = self.config['box_breathing']['sample_rate']
        cycle_sec = self.config['box_breathing']['cycle_duration_sec']
        
        if self.config['debug_mode']:
            print(f"Loaded scaffold CSV with {current_slots} retention slots.")
            # Display the last few existing timestamps
            print(f"Last timestamp in skeleton CSV: {skeleton_df.iloc[-1]['timestamp']} ({skeleton_df.iloc[-1]['milliseconds']} ms)")
        
        # Create a renaming mapping to standardize column names (if 'retentionslot_timestamp' exists, use that)
        if 'timestamp' in skeleton_df.columns and 'retentionslot_timestamp' not in skeleton_df.columns:
            skeleton_df = skeleton_df.rename(columns={'timestamp': 'retentionslot_timestamp'})
            
        if current_slots < required_slots:
            last_row = skeleton_df.iloc[-1]
            last_ms = float(last_row['milliseconds'])
            last_sample_index = int(last_row['sample_index'])
            new_rows = []
            
            for i in range(required_slots - current_slots):
                # Increment by 8 seconds (8000 ms) for each new row
                new_ms = last_ms + (cycle_sec * 1000) * (i + 1)
                new_sample_index = last_sample_index + int(cycle_sec * sample_rate * (i + 1))
                new_timestamp = convert_ms_to_timestamp(new_ms)
                
                # Alternate breathphase: if last was 'exhale', next is 'inhale' and vice versa
                last_phase = last_row['breathphase'].lower()
                if i == 0:  # First new row
                    new_breathphase = 'inhale' if last_phase == 'exhale' else 'exhale'
                else:  # Subsequent rows
                    prev_phase = new_rows[i-1]['breathphase'].lower()
                    new_breathphase = 'inhale' if prev_phase == 'exhale' else 'exhale'
                
                new_description = f"Additional hold {i+1}"
                
                # Create the new row with the correct timestamp column name
                new_row = {
                    'retentionslot_timestamp': new_timestamp,
                    'milliseconds': new_ms,
                    'sample_index': new_sample_index,
                    'description': new_description,
                    'breathphase': new_breathphase
                }
                new_rows.append(new_row)
                
                if self.config['debug_mode'] and i < 5:  # Show first few new rows
                    print(f"Adding new row: {new_timestamp} ({new_ms} ms) - {new_breathphase}")
            
            if new_rows:
                new_rows_df = pd.DataFrame(new_rows)
                # Make sure the new DataFrame has the same columns as the original
                for col in skeleton_df.columns:
                    if col not in new_rows_df.columns:
                        new_rows_df[col] = None
                
                skeleton_df = pd.concat([skeleton_df, new_rows_df[skeleton_df.columns]], ignore_index=True)
                
            if self.config['debug_mode']:
                print(f"Extended scaffold CSV to {len(skeleton_df)} retention slots.")
                print(f"New timestamp range: {skeleton_df['retentionslot_timestamp'].iloc[0]} to {skeleton_df['retentionslot_timestamp'].iloc[-1]}")
        
        if len(skeleton_df) > required_slots:
            skeleton_df = skeleton_df.iloc[:required_slots]
        
        return skeleton_df

    # ------------------------------------------------------------------
    # Generate counterbalanced stimulus design (trial conditions)
    # ------------------------------------------------------------------
    def generate_counterbalanced_design(self, participant_id):
        random.seed(self.base_seed + participant_id)
        np.random.seed(self.base_seed + participant_id)
        
        soa_conditions = self.config['soa_conditions_ms']
        jitter_options = self.config['jitter_options_ms']
        stimulus_types = LOOMING_STIMULI
        
        all_trials = []
        
        # Generate main conditions: inhalation, exhalation, and baseline.
        for condition, count in self.trial_counts.items():
            if condition == 'catch' or count == 0:
                continue
            combo_list = []
            for stim_type in stimulus_types:
                for soa in soa_conditions:
                    for _ in range(self.config['repetitions']):
                        combo_list.append((stim_type, soa))
            random.shuffle(combo_list)
            for stim_type, soa in combo_list:
                all_trials.append({
                    'participant_id': participant_id,
                    'trial_type': condition,  # "inhalation", "exhalation", or "baseline"
                    'stimulus_type': stim_type,
                    'soa_value_ms': soa,
                    'jitter_ms': random.choice(jitter_options),
                    'is_tactile': True  # For these main trials, tactile is present.
                })
        
        # Add catch trials (looming only, no tactile).
        catch_count = self.trial_counts.get('catch', 0)
        if catch_count > 0:
            catch_combos = []
            stim_counts = {stim: 0 for stim in stimulus_types}
            soa_counts = {soa: 0 for soa in soa_conditions}
            while len(catch_combos) < catch_count:
                min_stim = min(stim_counts, key=stim_counts.get)
                min_soa = min(soa_counts, key=soa_counts.get)
                catch_combos.append((min_stim, min_soa))
                stim_counts[min_stim] += 1
                soa_counts[min_soa] += 1
            for stim_type, soa in catch_combos:
                all_trials.append({
                    'participant_id': participant_id,
                    'trial_type': 'catch',
                    'stimulus_type': stim_type,
                    'soa_value_ms': soa,
                    'jitter_ms': random.choice(jitter_options),
                    'is_tactile': False
                })
        
        design_df = pd.DataFrame(all_trials)
        design_df = design_df.sample(frac=1, random_state=participant_id*100).reset_index(drop=True)
        design_df['trial_number'] = range(1, len(design_df) + 1)
        
        if self.config['debug_mode']:
            print("\nTrial type distribution:")
            print(design_df['trial_type'].value_counts())
            print("\nStimulus type distribution by trial type:")
            for trial_type in design_df['trial_type'].unique():
                if trial_type == 'baseline':
                    continue
                subset = design_df[design_df['trial_type'] == trial_type]
                print(f"  {trial_type}: {subset['stimulus_type'].value_counts().to_dict()}")
            print("\nSOA distribution by trial type:")
            for trial_type in design_df['trial_type'].unique():
                if trial_type == 'baseline':
                    continue
                subset = design_df[design_df['trial_type'] == trial_type]
                print(f"  {trial_type}: {subset['soa_value_ms'].value_counts().to_dict()}")
        
        return design_df

    # ------------------------------------------------------------------
    # Assign scaffold retention slots to trials while strictly maintaining the breathphase cycle.
    # ------------------------------------------------------------------
    def assign_breath_holds(self, design_df):
        """
        Assign scaffold retention slots to trials while strictly maintaining the breathphase cycle.
        
        Key requirements:
        1. Breathphase cycle (inhale-exhale-inhale-exhale) must remain intact
        2. Inhalation trials must go to inhale breathphase slots
        3. Exhalation trials must go to exhale breathphase slots
        4. Baseline and catch trials can go to either breathphase type
        5. The order of baseline and catch trials should be randomized
        """
        required_slots = len(design_df)
        scaffold_df = self.load_retention_slots(required_slots)
        
        # Ensure scaffold is properly sorted by timestamp/milliseconds
        scaffold_df = scaffold_df.sort_values(by='milliseconds').reset_index(drop=True)
        
        # Partition design trials by trial type
        pool_inh = [d for d in design_df.to_dict('records') if d['trial_type'] == 'inhalation']
        pool_exh = [d for d in design_df.to_dict('records') if d['trial_type'] == 'exhalation']
        pool_baseline = [d for d in design_df.to_dict('records') if d['trial_type'] == 'baseline']
        pool_catch = [d for d in design_df.to_dict('records') if d['trial_type'] == 'catch']
        
        # Count scaffold slots by breathphase
        inhale_slots = scaffold_df[scaffold_df['breathphase'].str.lower() == 'inhale']
        exhale_slots = scaffold_df[scaffold_df['breathphase'].str.lower() == 'exhale']
        n_inhale_slots = len(inhale_slots)
        n_exhale_slots = len(exhale_slots)
        
        if self.config['debug_mode']:
            print(f"Scaffold: {n_inhale_slots} inhale slots, {n_exhale_slots} exhale slots")
            print(f"Trials: {len(pool_inh)} inhalation, {len(pool_exh)} exhalation, " 
                  f"{len(pool_baseline)} baseline, {len(pool_catch)} catch")
        
        # Verify we have enough slots for mandatory trials
        if len(pool_inh) > n_inhale_slots:
            raise ValueError(f"Not enough inhale slots ({n_inhale_slots}) for inhalation trials ({len(pool_inh)})")
        if len(pool_exh) > n_exhale_slots:
            raise ValueError(f"Not enough exhale slots ({n_exhale_slots}) for exhalation trials ({len(pool_exh)})")
        
        # Calculate remaining slots per breathphase
        remaining_inhale = n_inhale_slots - len(pool_inh)
        remaining_exhale = n_exhale_slots - len(pool_exh)
        
        # Total baseline+catch trials
        total_flex_trials = len(pool_baseline) + len(pool_catch)
        
        # Verify we have enough slots for all trials
        if remaining_inhale + remaining_exhale != total_flex_trials:
            raise ValueError(f"Mismatch between available slots ({remaining_inhale + remaining_exhale}) "
                            f"and flexible trials ({total_flex_trials})")
        
        # Decide how to distribute baseline and catch trials across breathphases
        # Combine them first, then distribute proportionally
        flexible_trials = pool_baseline + pool_catch
        random.shuffle(flexible_trials)  # Randomize order
        
        # Split flexible trials between inhale and exhale slots proportionally
        inhale_flexible = flexible_trials[:remaining_inhale]
        exhale_flexible = flexible_trials[remaining_inhale:]
        
        if self.config['debug_mode']:
            print(f"Distributing {len(inhale_flexible)} flexible trials to inhale slots")
            print(f"Distributing {len(exhale_flexible)} flexible trials to exhale slots")
        
        # Create final allocation pools
        inhale_pool = pool_inh + inhale_flexible
        exhale_pool = pool_exh + exhale_flexible
        
        # Shuffle within each pool to randomize order
        random.shuffle(inhale_pool)
        random.shuffle(exhale_pool)
        
        # Create a dictionary mapping breathphase to trial pool
        phase_to_pool = {
            'inhale': inhale_pool,
            'exhale': exhale_pool
        }
        
        # Initialize counters for each breathphase
        pool_counters = {
            'inhale': 0,
            'exhale': 0
        }
        
        # Final list to store assigned trials in scaffold order
        final_assignments = []
        
        # Assign trials to scaffold rows, maintaining the original order
        for _, row in scaffold_df.iterrows():
            breathphase = row['breathphase'].lower()
            
            # Get the next trial from the appropriate pool
            if pool_counters[breathphase] < len(phase_to_pool[breathphase]):
                trial = phase_to_pool[breathphase][pool_counters[breathphase]]
                pool_counters[breathphase] += 1
                
                # Combine scaffold information with trial information
                combined = trial.copy()
                for col in row.index:
                    combined[col] = row[col]
                
                final_assignments.append(combined)
        
        # Create final dataframe and sort by scaffold order
        final_df = pd.DataFrame(final_assignments).sort_values(by='milliseconds').reset_index(drop=True)
        final_df['trial_number'] = range(1, len(final_df) + 1)
        
        # Compute additional timing information
        final_df['jittered_ms'] = final_df['milliseconds'] + final_df['jitter_ms']
        final_df['looming_stimulus_timestamp'] = final_df['jittered_ms'].apply(convert_ms_to_timestamp)
        
        # Add SOA timing for non-catch trials
        non_catch_mask = final_df['trial_type'] != 'catch'
        final_df.loc[non_catch_mask, 'soa_ms'] = final_df.loc[non_catch_mask, 'jittered_ms'] + final_df.loc[non_catch_mask, 'soa_value_ms']
        final_df.loc[non_catch_mask, 'tactile_stimulus_timestamp'] = final_df.loc[non_catch_mask, 'soa_ms'].apply(convert_ms_to_timestamp)
        
        # Verify the assignment respects the breathphase constraints
        self._validate_breathphase_assignment(final_df)
        
        return final_df

    def _validate_breathphase_assignment(self, final_df):
        """Verify that the trial assignment respects breathphase constraints."""
        if not self.config['debug_mode']:
            return
            
        print("\n=== BREATHPHASE ASSIGNMENT VALIDATION ===")
        
        # Check for inhalation trials assigned to inhale
        inh_trials = final_df[final_df['trial_type'] == 'inhalation']
        inh_correct = (inh_trials['breathphase'].str.lower() == 'inhale').all()
        print(f"Inhalation trials on inhale breathphase: {inh_correct}")
        
        # Check for exhalation trials assigned to exhale
        exh_trials = final_df[final_df['trial_type'] == 'exhalation']
        exh_correct = (exh_trials['breathphase'].str.lower() == 'exhale').all()
        print(f"Exhalation trials on exhale breathphase: {exh_correct}")
        
        # Count distribution of baseline and catch trials across breathphases
        baseline_inhale = ((final_df['trial_type'] == 'baseline') & 
                         (final_df['breathphase'].str.lower() == 'inhale')).sum()
        baseline_exhale = ((final_df['trial_type'] == 'baseline') & 
                          (final_df['breathphase'].str.lower() == 'exhale')).sum()
        catch_inhale = ((final_df['trial_type'] == 'catch') & 
                       (final_df['breathphase'].str.lower() == 'inhale')).sum()
        catch_exhale = ((final_df['trial_type'] == 'catch') & 
                       (final_df['breathphase'].str.lower() == 'exhale')).sum()
        
        print(f"Baseline trials: {baseline_inhale} on inhale, {baseline_exhale} on exhale")
        print(f"Catch trials: {catch_inhale} on inhale, {catch_exhale} on exhale")
        
        # Verify breathphase cycle is maintained
        prev_phase = None
        cycle_maintained = True
        expected_cycle = ['inhale', 'exhale'] * (len(final_df) // 2 + 1)
        
        for i, phase in enumerate(final_df['breathphase'].str.lower()):
            if i < len(expected_cycle) and phase != expected_cycle[i]:
                cycle_maintained = False
                print(f"Breathphase cycle disrupted at index {i}: expected {expected_cycle[i]}, got {phase}")
                break
        
        print(f"Breathphase cycle maintained: {cycle_maintained}")

    # ------------------------------------------------------------------
    # Finalize the design CSV: Retain the scaffold column order and append new columns.
    # Also, for baseline trials remove looming stimulus info; for catch trials remove tactile stimulus info.
    # ------------------------------------------------------------------
    def finalize_design_csv(self, design_df):
        baseline_mask = (design_df['trial_type'] == 'baseline')
        design_df.loc[baseline_mask, 'stimulus_type'] = None
        design_df.loc[baseline_mask, 'looming_stimulus_timestamp'] = None
        
        catch_mask = (design_df['trial_type'] == 'catch')
        design_df.loc[catch_mask, 'tactile_stimulus_timestamp'] = None
        
        design_df['expected_response'] = ~design_df['trial_type'].isin(['catch'])
        
        scaffold_cols = self.scaffold_order if hasattr(self, 'scaffold_order') else ['timestamp', 'milliseconds', 'sample_index', 'description', 'breathphase']
        new_cols = ['participant_id', 'trial_number', 'trial_type', 'stimulus_type', 'soa_value_ms', 'jitter_ms', 'is_tactile', 'looming_stimulus_timestamp', 'tactile_stimulus_timestamp', 'soa_ms', 'jittered_ms', 'expected_response']
        final_order = scaffold_cols + [col for col in new_cols if col not in scaffold_cols and col in design_df.columns]
        remaining = [col for col in design_df.columns if col not in final_order]
        final_order.extend(remaining)
        design_df = design_df[final_order]
        
        self._validate_design(design_df)
        return design_df

    # ------------------------------------------------------------------
    # Validation: Print distributions and sequence properties.
    # ------------------------------------------------------------------
    def _validate_design(self, design_df):
        """Perform comprehensive validation and counterbalancing checks on the design."""
        if not self.config['debug_mode']:
            return
            
        print("\n=== DESIGN VALIDATION ===")
        
        # 1. Overall trial count summary
        total_trials = len(design_df)
        trial_counts = design_df['trial_type'].value_counts().to_dict()
        
        print(f"TOTAL TRIALS: {total_trials}")
        print("Trial type distribution:")
        for trial_type, count in sorted(trial_counts.items()):
            percent = (count / total_trials) * 100
            print(f"  {trial_type.upper()}: {count} trials ({percent:.1f}%)")
        
        # 2. Detailed breakdown by trial type
        print("\nStimulus distribution by trial type:")
        for trial_type in sorted(design_df['trial_type'].unique()):
            if trial_type == 'baseline':
                continue
            subset = design_df[design_df['trial_type'] == trial_type]
            stim_counts = subset['stimulus_type'].value_counts().to_dict()
            print(f"  {trial_type.upper()}:")
            for stim, count in sorted(stim_counts.items()):
                percent = (count / len(subset)) * 100
                print(f"    {stim}: {count} trials ({percent:.1f}%)")
        
        # 3. SOA distribution checks
        print("\nSOA distribution by trial type:")
        for trial_type in sorted(design_df['trial_type'].unique()):
            if trial_type == 'baseline':
                continue
            subset = design_df[design_df['trial_type'] == trial_type]
            soa_counts = subset['soa_value_ms'].value_counts().to_dict()
            print(f"  {trial_type.upper()}:")
            for soa, count in sorted(soa_counts.items()):
                percent = (count / len(subset)) * 100
                print(f"    {soa} ms: {count} trials ({percent:.1f}%)")
        
        # 4. Jitter distribution
        print("\nJitter distribution:")
        jitter_counts = design_df['jitter_ms'].value_counts().to_dict()
        for jitter, count in sorted(jitter_counts.items()):
            percent = (count / total_trials) * 100
            print(f"  {jitter} ms: {count} trials ({percent:.1f}%)")
        
        # 5. Breath phase checks
        print("\nBreath phase distribution:")
        if 'breathphase' in design_df.columns:
            phase_counts = design_df['breathphase'].value_counts().to_dict()
            for phase, count in sorted(phase_counts.items()):
                percent = (count / total_trials) * 100
                print(f"  {phase}: {count} trials ({percent:.1f}%)")
        
        # 6. Cross-tabulation of trial types by breath phase
        print("\nTrial types by breath phase (COUNTERBALANCE CHECK):")
        if 'breathphase' in design_df.columns:
            trial_by_phase = pd.crosstab(
                design_df['trial_type'], 
                design_df['breathphase']
            )
            print(trial_by_phase)
            
            # Calculate chi-square statistic for independence (excluding mandatory assignments)
            try:
                # Create subset without inhalation/exhalation (which are deterministically assigned)
                flex_df = design_df[design_df['trial_type'].isin(['baseline', 'catch'])]
                if len(flex_df) > 0:
                    flex_table = pd.crosstab(flex_df['trial_type'], flex_df['breathphase'])
                    chi2 = 0
                    row_sums = flex_table.sum(axis=1)
                    col_sums = flex_table.sum(axis=0) 
                    total = flex_table.sum().sum()
                    
                    for i in flex_table.index:
                        for j in flex_table.columns:
                            observed = flex_table.loc[i, j]
                            expected = row_sums[i] * col_sums[j] / total
                            if expected > 0:
                                chi2 += ((observed - expected) ** 2) / expected
                    
                    print(f"\nChi-square for baseline/catch distribution: {chi2:.2f}")
                    print(f"Lower values indicate better balance between breathphases.")
            except Exception as e:
                print(f"Error calculating chi-square: {e}")
        
        # 7. Consecutive trials analysis
        trial_sequence = design_df['trial_type'].tolist()
        consecutive_counts = {}
        current_type = trial_sequence[0]
        current_count = 1
        for t in trial_sequence[1:]:
            if t == current_type:
                current_count += 1
            else:
                key = f"{current_type}_{current_count}"
                consecutive_counts[key] = consecutive_counts.get(key, 0) + 1
                current_type = t
                current_count = 1
        key = f"{current_type}_{current_count}"
        consecutive_counts[key] = consecutive_counts.get(key, 0) + 1
        
        print("\nRuns of consecutive trial types:")
        for key, count in sorted(consecutive_counts.items()):
            print(f"  {key}: {count}")
        
        # 8. SOA and stimulus type correlation check
        print("\nSOA-by-stimulus-type COUNTERBALANCE CHECK:")
        non_baseline = design_df[design_df['trial_type'] != 'baseline']
        if len(non_baseline) > 0:
            soa_by_stim = pd.crosstab(
                non_baseline['soa_value_ms'], 
                non_baseline['stimulus_type']
            )
            print(soa_by_stim)
            
            # Calculate coefficient of variation as a balance metric
            cv_values = []
            for _, row in soa_by_stim.iterrows():
                if row.sum() > 0:
                    cv = row.std() / row.mean() if row.mean() > 0 else 0
                    cv_values.append(cv)
            
            mean_cv = sum(cv_values) / len(cv_values) if cv_values else 0
            print(f"\nCoefficient of variation across SOA-stimulus combinations: {mean_cv:.4f}")
            print(f"Lower values indicate better counterbalancing.")
        
        # 9. Breathphase alternation check
        if 'breathphase' in design_df.columns:
            phases = design_df.sort_values('milliseconds')['breathphase'].str.lower().tolist()
            alternation_violations = 0
            for i in range(1, len(phases)):
                if phases[i] == phases[i-1]:
                    alternation_violations += 1
            
            print(f"\nBreathphase alternation violations: {alternation_violations}")
            print(f"This should be 0 - the inhale/exhale cycle must be maintained.")
        
        # 10. Timing analysis
        if 'jittered_ms' in design_df.columns:
            time_diffs = []
            sorted_df = design_df.sort_values('jittered_ms')
            for i in range(1, len(sorted_df)):
                diff = sorted_df.iloc[i]['jittered_ms'] - sorted_df.iloc[i-1]['jittered_ms']
                time_diffs.append(diff)
            
            if time_diffs:
                min_diff = min(time_diffs)
                max_diff = max(time_diffs)
                avg_diff = sum(time_diffs) / len(time_diffs)
                print(f"\nStimulus timing intervals:")
                print(f"  Minimum interval: {min_diff:.1f} ms")
                print(f"  Maximum interval: {max_diff:.1f} ms")
                print(f"  Average interval: {avg_diff:.1f} ms")
    
    # ------------------------------------------------------------------
    # Process a single participant: generate design, assign slots, finalize, and save CSV.
    # ------------------------------------------------------------------
    def process_participant(self, participant_id):
        try:
            if self.config['debug_mode']:
                print(f"\n=== Processing participant {participant_id} ===")
            design_df = self.generate_counterbalanced_design(participant_id)
            design_df = self.assign_breath_holds(design_df)
            design_df = self.finalize_design_csv(design_df)
            out_path = os.path.join(self.experiment_log_dir, f"participant_{participant_id}_design.csv")
            design_df.to_csv(out_path, index=False)
            if self.config['debug_mode']:
                print(f"Saved participant {participant_id} design to {out_path}")
            return True, out_path
        except Exception as e:
            print(f"Error processing participant {participant_id}: {e}")
            traceback.print_exc()
            return False, None
    
    # ------------------------------------------------------------------
    # Run design generation for multiple participants.
    # ------------------------------------------------------------------
    def run(self, participant_ids=None, num_participants=None):
        if participant_ids is None and num_participants is None:
            num_participants = self.config.get('num_participants', 5)
        if participant_ids is None:
            participant_ids = list(range(num_participants))
        if self.config['debug_mode']:
            print("=== Starting PPSDesignGenerator ===")
            print(f"Repetitions: {self.config['repetitions']}")
            print(f"Participants: {participant_ids}")
            print(f"Box breathing settings: {self.config['box_breathing']}")
        results = {}
        for pid in participant_ids:
            success, path = self.process_participant(pid)
            results[pid] = {'success': success, 'csv_path': path}
        
        # Summary of all participants
        if self.config['debug_mode'] and len(results) > 1:
            print("\n=== CROSS-PARTICIPANT VALIDATION ===")
            all_success = all(info['success'] for info in results.values())
            print(f"All participants generated successfully: {all_success}")
        
        return results

# ----------------------------------------------------------------------
# AUDIO GENERATION PLACEHOLDER FUNCTION
# ----------------------------------------------------------------------
def generate_audio_from_design(design_csv_path, output_dir, base_audio_path=None):
    """
    Generate audio files from a design CSV.
    (Placeholder function – implement audio generation as needed.)
    """
    print(f"Audio generation from {design_csv_path} not implemented yet")
    return None, None

# ----------------------------------------------------------------------
# MAIN FUNCTION
# ----------------------------------------------------------------------
def main():
    """Main function to run the design generator."""
    generator = PPSDesignGenerator(CONFIG)
    results = generator.run()
    print("\n=== Generation Summary ===")
    for pid, info in results.items():
        if info['success']:
            print(f"Participant {pid}: CSV generated at {info['csv_path']}")
        else:
            print(f"Participant {pid}: ERROR - no CSV generated.")
    print("\nTo generate audio files from these designs, implement the audio generation function.")

if __name__ == "__main__":
    main()

Trial count calculation:
  inhalation: 80 trials
  exhalation: 80 trials
  baseline: 80 trials
  catch: 24 trials
  Total: 264 trials
Initialized PPSDesignGenerator with:
  - Repetitions: 4
  - Conditions: ['inhalation', 'exhalation', 'baseline']
  - SOA values: [190, 400, 700, 1000, 1500]
  - Total trials per participant: 264
  - Output directory: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog
=== Starting PPSDesignGenerator ===
Repetitions: 4
Participants: [0, 1, 2, 3, 4]
Box breathing settings: {'cycle_duration_sec': 8, 'intro_duration_sec': 60, 'outro_duration_sec': 30, 'sample_rate': 48000}

=== Processing participant 0 ===

Trial type distribution:
trial_type
exhalation    80
inhalation    80
baseline      80
catch         24
Name: count, dtype: int64

Stimulus type distribution by trial type:
  exhalation: {'brown': 20, 'pink': 20, 'blue': 20, 'white': 20}
  catch: {'blue': 6, 'white': 6, 'brown': 6, 'pink': 6}
  inhalation: {'pink': 2

Traceback (most recent call last):
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\site-packages\pandas\core\indexes\base.py", line 3805, in get_loc
    return self._engine.get_loc(casted_key)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "index.pyx", line 167, in pandas._libs.index.IndexEngine.get_loc
  File "index.pyx", line 196, in pandas._libs.index.IndexEngine.get_loc
  File "pandas\\_libs\\hashtable_class_helper.pxi", line 7081, in pandas._libs.hashtable.PyObjectHashTable.get_item
  File "pandas\\_libs\\hashtable_class_helper.pxi", line 7089, in pandas._libs.hashtable.PyObjectHashTable.get_item
KeyError: 'timestamp'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "C:\Users\cogpsy-vrlab\AppData\Local\Temp\ipykernel_7528\3392054419.py", line 650, in process_participant
    design_df = self.assign_breath_holds(design_df)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\cogpsy-vrlab\AppData\Local\Te

In [27]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
PPS Design Generator

This script generates participant‐specific experimental design CSVs for a 
Peripersonal Space (PPS) experiment. Each design CSV is built upon a 
"scaffold" CSV that contains the retention slot (breathhold) timestamps 
from a box‐breathing audio file. If needed, additional retention slots are 
appended (using fixed 8‑second increments) so that every trial can be assigned 
to a specific retention slot. Then, a counterbalanced set of trial conditions 
(e.g., trial type, looming stimulus type, SOA, jitter) is allocated to each 
scaffold row according to the following rules:
  
  - Inhalation trials (looming cue + tactile) are only assigned to scaffold rows 
    whose breathphase is "inhale."
  - Exhalation trials (looming cue + tactile) are only assigned to scaffold rows 
    whose breathphase is "exhale."
  - Baseline (tactile only; looming = NA) and catch (looming only; no tactile) trials 
    are allocated from the remaining pool in a random, counterbalanced manner 
    between the inhale and exhale rows.
    
The final output retains the original column order from the scaffold, with new 
trial‐specific variables appended as new columns.

Author: AI Assistant
"""

import os
import numpy as np
import pandas as pd
import random
import traceback

# ----------------------------------------------------------------------
# Helper function: Convert milliseconds to "MM:SS.s" format
# ----------------------------------------------------------------------
def convert_ms_to_timestamp(ms):
    m = int(ms // 60000)
    s = (ms % 60000) / 1000.0
    return f"{m:02}:{s:.1f}"

# ----------------------------------------------------------------------
# CONFIGURATION PARAMETERS
# ----------------------------------------------------------------------
CONFIG = {
    'repetitions': 4,            # How many times each SOA×stimulus combination repeats
    'num_participants': 5,       # Default number of participants
    'included_conditions': {
        'inhalation': True,
        'exhalation': True,
        'baseline': True,
    },
    'catch_trial_percentage': 10,
    'soa_conditions_ms': [190, 400, 700, 1000, 1500],
    'jitter_options_ms': [100, 200, 300, 400, 500],
    'debug_mode': True,
    'paths': {
        # Directory where participant-specific CSV files will be saved
        'experiment_log_dir': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog",
    },
    # Path to the original scaffold CSV containing retention (breathhold) timestamps.
    'skeleton_csv_path': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_timestamps.csv",
    'box_breathing': {
        'cycle_duration_sec': 8,      # Duration of one breath cycle in seconds
        # Intro/outro are not used here because the scaffold drives timing.
        'intro_duration_sec': 60,
        'outro_duration_sec': 30,
        'sample_rate': 48000,           # Audio sample rate (samples per second)
    }
}

# Noise types for looming stimuli (can be any labels)
LOOMING_STIMULI = ['blue', 'brown', 'pink', 'white']

# ----------------------------------------------------------------------
# CLASS: PPSDesignGenerator
# ----------------------------------------------------------------------
class PPSDesignGenerator:
    """Generates participant-specific CSV design files for PPS experiments."""
    
    def __init__(self, config=None):
        self.config = config or CONFIG
        
        # Prepare output directory.
        self.experiment_log_dir = self.config['paths']['experiment_log_dir']
        os.makedirs(self.experiment_log_dir, exist_ok=True)
        
        # Calculate trial counts.
        # For each condition (inhalation, exhalation, baseline): count = (# noise types) × (# SOA) × (repetitions).
        # Then, add 10% catch trials.
        self.trial_counts = self._calculate_trial_counts()
        self.conditions = [cond for cond, inc in self.config['included_conditions'].items() if inc]
        
        # Set seed for reproducibility.
        self.base_seed = 42
        
        if self.config['debug_mode']:
            total_trials = sum(self.trial_counts.values())
            print("Initialized PPSDesignGenerator with:")
            print(f"  - Repetitions: {self.config['repetitions']}")
            print(f"  - Conditions: {self.conditions}")
            print(f"  - SOA values: {self.config['soa_conditions_ms']}")
            print(f"  - Total trials per participant: {total_trials}")
            print(f"  - Output directory: {self.experiment_log_dir}")
    
    def _calculate_trial_counts(self):
        """
        Calculate the number of trials for each condition.
        
        This method sets up exactly 12 trials per SOA condition (190, 400, etc.)
        for each trial type (inhalation, exhalation, baseline).
        """
        soa_conditions = self.config['soa_conditions_ms']
        trials_per_soa = 12  # Fixed at 12 trials per SOA per condition
        catch_percentage = self.config['catch_trial_percentage']
        
        trial_counts = {}
        for condition, include in self.config['included_conditions'].items():
            if include:
                trial_counts[condition] = len(soa_conditions) * trials_per_soa
        
        regular_trials = sum(trial_counts.values())
        catch_trials = int(np.ceil(regular_trials * catch_percentage / 100))
        trial_counts['catch'] = catch_trials
        
        if self.config['debug_mode']:
            print("Trial count calculation:")
            for cond, count in trial_counts.items():
                print(f"  {cond}: {count} trials")
            print(f"  Total: {sum(trial_counts.values())} trials")
        
        return trial_counts

    # ------------------------------------------------------------------
    # Load the scaffold CSV and extend it if needed.
    # Retain the original column order.
    # ------------------------------------------------------------------
    def load_retention_slots(self, required_slots):
        """
        Load the scaffold CSV containing retention slots and extend it if needed.
        Handles different column naming variations by standardizing them.
        """
        skeleton_path = self.config['skeleton_csv_path']
        try:
            # Try to read the CSV, handling different separators
            skeleton_df = pd.read_csv(skeleton_path, sep=None, engine='python')
            if len(skeleton_df.columns) == 1:
                skeleton_df = pd.read_csv(skeleton_path, delimiter="\t")
            
            # Normalize column names (strip whitespace and convert to lowercase)
            skeleton_df.columns = [col.strip().lower() for col in skeleton_df.columns]
            self.scaffold_order = skeleton_df.columns.tolist()
            
            if self.config['debug_mode']:
                print("Scaffold CSV columns:", self.scaffold_order)
                
            # Determine timestamp column name - could be 'timestamp' or 'retentionslot_timestamp'
            timestamp_col = None
            if 'timestamp' in skeleton_df.columns:
                timestamp_col = 'timestamp'
            elif 'retentionslot_timestamp' in skeleton_df.columns:
                timestamp_col = 'retentionslot_timestamp'
            
            # If neither standard name exists, try to find a column that might contain timestamp info
            if timestamp_col is None:
                for col in skeleton_df.columns:
                    if 'time' in col.lower() and not col.lower() == 'milliseconds':
                        timestamp_col = col
                        break
            
            if timestamp_col is None:
                raise ValueError("Could not find timestamp column in scaffold CSV.")
                
            # Standardize the timestamp column name to 'retentionslot_timestamp'
            if timestamp_col != 'retentionslot_timestamp':
                skeleton_df = skeleton_df.rename(columns={timestamp_col: 'retentionslot_timestamp'})
                if timestamp_col in self.scaffold_order:
                    self.scaffold_order[self.scaffold_order.index(timestamp_col)] = 'retentionslot_timestamp'
                
        except Exception as e:
            raise ValueError(f"Error reading scaffold CSV from {skeleton_path}: {e}")
        
        current_slots = len(skeleton_df)
        required_slots = int(required_slots)
        sample_rate = self.config['box_breathing']['sample_rate']
        cycle_sec = self.config['box_breathing']['cycle_duration_sec']
        
        if self.config['debug_mode']:
            print(f"Loaded scaffold CSV with {current_slots} retention slots.")
            # Display the last few existing timestamps
            try:
                print(f"Last timestamp in skeleton CSV: {skeleton_df.iloc[-1]['retentionslot_timestamp']} ({skeleton_df.iloc[-1]['milliseconds']} ms)")
            except Exception as e:
                print(f"Could not display last timestamp: {e}")
        
        if current_slots < required_slots:
            last_row = skeleton_df.iloc[-1]
            last_ms = float(last_row['milliseconds'])
            last_sample_index = int(last_row['sample_index'])
            new_rows = []
            
            # Get the last breathphase to ensure proper alternation
            last_breathphase = last_row['breathphase'].lower()
            
            for i in range(required_slots - current_slots):
                # Increment by 8 seconds (8000 ms) for each new row
                new_ms = last_ms + (cycle_sec * 1000) * (i + 1)
                new_sample_index = last_sample_index + int(cycle_sec * sample_rate * (i + 1))
                new_timestamp = convert_ms_to_timestamp(new_ms)
                
                # Alternate breathphase: if last was 'exhale', next is 'inhale' and vice versa
                if i == 0:  # First new row
                    new_breathphase = 'inhale' if last_breathphase == 'exhale' else 'exhale'
                else:  # Subsequent rows
                    prev_phase = new_rows[i-1]['breathphase'].lower()
                    new_breathphase = 'inhale' if prev_phase == 'exhale' else 'exhale'
                
                new_description = f"Additional hold {i+1}"
                
                # Create the new row with the correct timestamp column name
                new_row = {
                    'retentionslot_timestamp': new_timestamp,
                    'milliseconds': new_ms,
                    'sample_index': new_sample_index,
                    'description': new_description,
                    'breathphase': new_breathphase
                }
                new_rows.append(new_row)
                
                if self.config['debug_mode'] and i < 5:  # Show first few new rows
                    print(f"Adding new row: {new_timestamp} ({new_ms} ms) - {new_breathphase}")
            
            if new_rows:
                new_rows_df = pd.DataFrame(new_rows)
                # Ensure new_rows_df has all columns from skeleton_df
                for col in skeleton_df.columns:
                    if col not in new_rows_df.columns:
                        new_rows_df[col] = None
                
                # Concatenate with only the columns from the original DataFrame
                skeleton_df = pd.concat([skeleton_df, new_rows_df[skeleton_df.columns]], ignore_index=True)
            
            if self.config['debug_mode']:
                print(f"Extended scaffold CSV to {len(skeleton_df)} retention slots.")
                print(f"New timestamp range: {skeleton_df['retentionslot_timestamp'].iloc[0]} to {skeleton_df['retentionslot_timestamp'].iloc[-1]}")
        
        if len(skeleton_df) > required_slots:
            skeleton_df = skeleton_df.iloc[:required_slots]
        
        return skeleton_df

    # ------------------------------------------------------------------
    # Generate counterbalanced stimulus design (trial conditions)
    # ------------------------------------------------------------------
    def generate_counterbalanced_design(self, participant_id):
        random.seed(self.base_seed + participant_id)
        np.random.seed(self.base_seed + participant_id)
        
        soa_conditions = self.config['soa_conditions_ms']
        jitter_options = self.config['jitter_options_ms']
        stimulus_types = LOOMING_STIMULI
        
        all_trials = []
        
        # Generate main conditions: inhalation, exhalation, and baseline.
        for condition, count in self.trial_counts.items():
            if condition == 'catch' or count == 0:
                continue
            combo_list = []
            for stim_type in stimulus_types:
                for soa in soa_conditions:
                    for _ in range(self.config['repetitions']):
                        combo_list.append((stim_type, soa))
            random.shuffle(combo_list)
            for stim_type, soa in combo_list:
                all_trials.append({
                    'participant_id': participant_id,
                    'trial_type': condition,  # "inhalation", "exhalation", or "baseline"
                    'stimulus_type': stim_type,
                    'soa_value_ms': soa,
                    'jitter_ms': random.choice(jitter_options),
                    'is_tactile': True  # For these main trials, tactile is present.
                })
        
        # Add catch trials (looming only, no tactile).
        catch_count = self.trial_counts.get('catch', 0)
        if catch_count > 0:
            catch_combos = []
            stim_counts = {stim: 0 for stim in stimulus_types}
            soa_counts = {soa: 0 for soa in soa_conditions}
            while len(catch_combos) < catch_count:
                min_stim = min(stim_counts, key=stim_counts.get)
                min_soa = min(soa_counts, key=soa_counts.get)
                catch_combos.append((min_stim, min_soa))
                stim_counts[min_stim] += 1
                soa_counts[min_soa] += 1
            for stim_type, soa in catch_combos:
                all_trials.append({
                    'participant_id': participant_id,
                    'trial_type': 'catch',
                    'stimulus_type': stim_type,
                    'soa_value_ms': soa,
                    'jitter_ms': random.choice(jitter_options),
                    'is_tactile': False
                })
        
        design_df = pd.DataFrame(all_trials)
        design_df = design_df.sample(frac=1, random_state=participant_id*100).reset_index(drop=True)
        design_df['trial_number'] = range(1, len(design_df) + 1)
        
        if self.config['debug_mode']:
            print("\nTrial type distribution:")
            print(design_df['trial_type'].value_counts())
            print("\nStimulus type distribution by trial type:")
            for trial_type in design_df['trial_type'].unique():
                if trial_type == 'baseline':
                    continue
                subset = design_df[design_df['trial_type'] == trial_type]
                print(f"  {trial_type}: {subset['stimulus_type'].value_counts().to_dict()}")
            print("\nSOA distribution by trial type:")
            for trial_type in design_df['trial_type'].unique():
                if trial_type == 'baseline':
                    continue
                subset = design_df[design_df['trial_type'] == trial_type]
                print(f"  {trial_type}: {subset['soa_value_ms'].value_counts().to_dict()}")
        
        return design_df

    # ------------------------------------------------------------------
    # Assign scaffold retention slots to trials while strictly maintaining the breathphase cycle.
    # ------------------------------------------------------------------
    def assign_breath_holds(self, design_df):
        """
        Assign scaffold retention slots to trials while strictly maintaining the breathphase cycle.
        
        Key requirements:
        1. Breathphase cycle (inhale-exhale-inhale-exhale) must remain intact
        2. Inhalation trials must go to inhale breathphase slots
        3. Exhalation trials must go to exhale breathphase slots
        4. Baseline and catch trials can go to either breathphase type
        5. The order of baseline and catch trials should be randomized
        """
        required_slots = len(design_df)
        scaffold_df = self.load_retention_slots(required_slots)
        
        # Ensure scaffold is properly sorted by timestamp/milliseconds
        scaffold_df = scaffold_df.sort_values(by='milliseconds').reset_index(drop=True)
        
        # Partition design trials by trial type
        pool_inh = [d for d in design_df.to_dict('records') if d['trial_type'] == 'inhalation']
        pool_exh = [d for d in design_df.to_dict('records') if d['trial_type'] == 'exhalation']
        pool_baseline = [d for d in design_df.to_dict('records') if d['trial_type'] == 'baseline']
        pool_catch = [d for d in design_df.to_dict('records') if d['trial_type'] == 'catch']
        
        # Count scaffold slots by breathphase
        inhale_slots = scaffold_df[scaffold_df['breathphase'].str.lower() == 'inhale']
        exhale_slots = scaffold_df[scaffold_df['breathphase'].str.lower() == 'exhale']
        n_inhale_slots = len(inhale_slots)
        n_exhale_slots = len(exhale_slots)
        
        if self.config['debug_mode']:
            print(f"Scaffold: {n_inhale_slots} inhale slots, {n_exhale_slots} exhale slots")
            print(f"Trials: {len(pool_inh)} inhalation, {len(pool_exh)} exhalation, " 
                  f"{len(pool_baseline)} baseline, {len(pool_catch)} catch")
        
        # Verify we have enough slots for mandatory trials
        if len(pool_inh) > n_inhale_slots:
            raise ValueError(f"Not enough inhale slots ({n_inhale_slots}) for inhalation trials ({len(pool_inh)})")
        if len(pool_exh) > n_exhale_slots:
            raise ValueError(f"Not enough exhale slots ({n_exhale_slots}) for exhalation trials ({len(pool_exh)})")
        
        # Calculate remaining slots per breathphase
        remaining_inhale = n_inhale_slots - len(pool_inh)
        remaining_exhale = n_exhale_slots - len(pool_exh)
        
        # Total baseline+catch trials
        total_flex_trials = len(pool_baseline) + len(pool_catch)
        
        # Verify we have enough slots for all trials
        if remaining_inhale + remaining_exhale != total_flex_trials:
            raise ValueError(f"Mismatch between available slots ({remaining_inhale + remaining_exhale}) "
                            f"and flexible trials ({total_flex_trials})")
        
        # Decide how to distribute baseline and catch trials across breathphases
        # Combine them first, then distribute proportionally
        flexible_trials = pool_baseline + pool_catch
        random.shuffle(flexible_trials)  # Randomize order
        
        # Split flexible trials between inhale and exhale slots proportionally
        inhale_flexible = flexible_trials[:remaining_inhale]
        exhale_flexible = flexible_trials[remaining_inhale:]
        
        if self.config['debug_mode']:
            print(f"Distributing {len(inhale_flexible)} flexible trials to inhale slots")
            print(f"Distributing {len(exhale_flexible)} flexible trials to exhale slots")
        
        # Create final allocation pools
        inhale_pool = pool_inh + inhale_flexible
        exhale_pool = pool_exh + exhale_flexible
        
        # Shuffle within each pool to randomize order
        random.shuffle(inhale_pool)
        random.shuffle(exhale_pool)
        
        # Create a dictionary mapping breathphase to trial pool
        phase_to_pool = {
            'inhale': inhale_pool,
            'exhale': exhale_pool
        }
        
        # Initialize counters for each breathphase
        pool_counters = {
            'inhale': 0,
            'exhale': 0
        }
        
        # Final list to store assigned trials in scaffold order
        final_assignments = []
        
        # Assign trials to scaffold rows, maintaining the original order
        for _, row in scaffold_df.iterrows():
            breathphase = row['breathphase'].lower()
            
            # Get the next trial from the appropriate pool
            if pool_counters[breathphase] < len(phase_to_pool[breathphase]):
                trial = phase_to_pool[breathphase][pool_counters[breathphase]]
                pool_counters[breathphase] += 1
                
                # Combine scaffold information with trial information
                combined = trial.copy()
                for col in row.index:
                    combined[col] = row[col]
                
                final_assignments.append(combined)
        
        # Create final dataframe and sort by scaffold order
        final_df = pd.DataFrame(final_assignments).sort_values(by='milliseconds').reset_index(drop=True)
        final_df['trial_number'] = range(1, len(final_df) + 1)
        
        # Compute additional timing information
        final_df['jittered_ms'] = final_df['milliseconds'] + final_df['jitter_ms']
        final_df['looming_stimulus_timestamp'] = final_df['jittered_ms'].apply(convert_ms_to_timestamp)
        
        # Add SOA timing for non-catch trials
        non_catch_mask = final_df['trial_type'] != 'catch'
        final_df.loc[non_catch_mask, 'soa_ms'] = final_df.loc[non_catch_mask, 'jittered_ms'] + final_df.loc[non_catch_mask, 'soa_value_ms']
        final_df.loc[non_catch_mask, 'tactile_stimulus_timestamp'] = final_df.loc[non_catch_mask, 'soa_ms'].apply(convert_ms_to_timestamp)
        
        # Verify the assignment respects the breathphase constraints
        self._validate_breathphase_assignment(final_df)
        
        return final_df

    def _validate_breathphase_assignment(self, final_df):
        """Verify that the trial assignment respects breathphase constraints."""
        if not self.config['debug_mode']:
            return
            
        print("\n=== BREATHPHASE ASSIGNMENT VALIDATION ===")
        
        # Check for inhalation trials assigned to inhale
        inh_trials = final_df[final_df['trial_type'] == 'inhalation']
        inh_correct = (inh_trials['breathphase'].str.lower() == 'inhale').all()
        print(f"Inhalation trials on inhale breathphase: {inh_correct}")
        
        # Check for exhalation trials assigned to exhale
        exh_trials = final_df[final_df['trial_type'] == 'exhalation']
        exh_correct = (exh_trials['breathphase'].str.lower() == 'exhale').all()
        print(f"Exhalation trials on exhale breathphase: {exh_correct}")
        
        # Count distribution of baseline and catch trials across breathphases
        baseline_inhale = ((final_df['trial_type'] == 'baseline') & 
                         (final_df['breathphase'].str.lower() == 'inhale')).sum()
        baseline_exhale = ((final_df['trial_type'] == 'baseline') & 
                          (final_df['breathphase'].str.lower() == 'exhale')).sum()
        catch_inhale = ((final_df['trial_type'] == 'catch') & 
                       (final_df['breathphase'].str.lower() == 'inhale')).sum()
        catch_exhale = ((final_df['trial_type'] == 'catch') & 
                       (final_df['breathphase'].str.lower() == 'exhale')).sum()
        
        print(f"Baseline trials: {baseline_inhale} on inhale, {baseline_exhale} on exhale")
        print(f"Catch trials: {catch_inhale} on inhale, {catch_exhale} on exhale")
        
        # Verify breathphase cycle is maintained
        prev_phase = None
        cycle_maintained = True
        expected_cycle = ['inhale', 'exhale'] * (len(final_df) // 2 + 1)
        
        for i, phase in enumerate(final_df['breathphase'].str.lower()):
            if i < len(expected_cycle) and phase != expected_cycle[i]:
                cycle_maintained = False
                print(f"Breathphase cycle disrupted at index {i}: expected {expected_cycle[i]}, got {phase}")
                break
        
        print(f"Breathphase cycle maintained: {cycle_maintained}")

    # ------------------------------------------------------------------
    # Finalize the design CSV: Retain the scaffold column order and append new columns.
    # Also, for baseline trials remove looming stimulus info; for catch trials remove tactile stimulus info.
    # ------------------------------------------------------------------
    def finalize_design_csv(self, design_df):
        baseline_mask = (design_df['trial_type'] == 'baseline')
        design_df.loc[baseline_mask, 'stimulus_type'] = None
        design_df.loc[baseline_mask, 'looming_stimulus_timestamp'] = None
        
        catch_mask = (design_df['trial_type'] == 'catch')
        design_df.loc[catch_mask, 'tactile_stimulus_timestamp'] = None
        
        design_df['expected_response'] = ~design_df['trial_type'].isin(['catch'])
        
        scaffold_cols = self.scaffold_order if hasattr(self, 'scaffold_order') else ['timestamp', 'milliseconds', 'sample_index', 'description', 'breathphase']
        new_cols = ['participant_id', 'trial_number', 'trial_type', 'stimulus_type', 'soa_value_ms', 'jitter_ms', 'is_tactile', 'looming_stimulus_timestamp', 'tactile_stimulus_timestamp', 'soa_ms', 'jittered_ms', 'expected_response']
        final_order = scaffold_cols + [col for col in new_cols if col not in scaffold_cols and col in design_df.columns]
        remaining = [col for col in design_df.columns if col not in final_order]
        final_order.extend(remaining)
        design_df = design_df[final_order]
        
        self._validate_design(design_df)
        return design_df

    # ------------------------------------------------------------------
    # Validation: Print distributions and sequence properties.
    # ------------------------------------------------------------------
    def _validate_design(self, design_df):
        """Perform comprehensive validation and counterbalancing checks on the design."""
        if not self.config['debug_mode']:
            return
            
        print("\n=== DESIGN VALIDATION ===")
        
        # 1. Overall trial count summary
        total_trials = len(design_df)
        trial_counts = design_df['trial_type'].value_counts().to_dict()
        
        print(f"TOTAL TRIALS: {total_trials}")
        print("Trial type distribution:")
        for trial_type, count in sorted(trial_counts.items()):
            percent = (count / total_trials) * 100
            print(f"  {trial_type.upper()}: {count} trials ({percent:.1f}%)")
        
        # 2. Detailed breakdown by trial type
        print("\nStimulus distribution by trial type:")
        for trial_type in sorted(design_df['trial_type'].unique()):
            if trial_type == 'baseline':
                continue
            subset = design_df[design_df['trial_type'] == trial_type]
            stim_counts = subset['stimulus_type'].value_counts().to_dict()
            print(f"  {trial_type.upper()}:")
            for stim, count in sorted(stim_counts.items()):
                percent = (count / len(subset)) * 100
                print(f"    {stim}: {count} trials ({percent:.1f}%)")
        
        # 3. SOA distribution checks
        print("\nSOA distribution by trial type:")
        for trial_type in sorted(design_df['trial_type'].unique()):
            if trial_type == 'baseline':
                continue
            subset = design_df[design_df['trial_type'] == trial_type]
            soa_counts = subset['soa_value_ms'].value_counts().to_dict()
            print(f"  {trial_type.upper()}:")
            for soa, count in sorted(soa_counts.items()):
                percent = (count / len(subset)) * 100
                print(f"    {soa} ms: {count} trials ({percent:.1f}%)")
        
        # 4. Jitter distribution
        print("\nJitter distribution:")
        jitter_counts = design_df['jitter_ms'].value_counts().to_dict()
        for jitter, count in sorted(jitter_counts.items()):
            percent = (count / total_trials) * 100
            print(f"  {jitter} ms: {count} trials ({percent:.1f}%)")
        
        # 5. Breath phase checks
        print("\nBreath phase distribution:")
        if 'breathphase' in design_df.columns:
            phase_counts = design_df['breathphase'].value_counts().to_dict()
            for phase, count in sorted(phase_counts.items()):
                percent = (count / total_trials) * 100
                print(f"  {phase}: {count} trials ({percent:.1f}%)")
        
        # 6. Cross-tabulation of trial types by breath phase
        print("\nTrial types by breath phase (COUNTERBALANCE CHECK):")
        if 'breathphase' in design_df.columns:
            trial_by_phase = pd.crosstab(
                design_df['trial_type'], 
                design_df['breathphase']
            )
            print(trial_by_phase)
            
            # Calculate chi-square statistic for independence (excluding mandatory assignments)
            try:
                # Create subset without inhalation/exhalation (which are deterministically assigned)
                flex_df = design_df[design_df['trial_type'].isin(['baseline', 'catch'])]
                if len(flex_df) > 0:
                    flex_table = pd.crosstab(flex_df['trial_type'], flex_df['breathphase'])
                    chi2 = 0
                    row_sums = flex_table.sum(axis=1)
                    col_sums = flex_table.sum(axis=0) 
                    total = flex_table.sum().sum()
                    
                    for i in flex_table.index:
                        for j in flex_table.columns:
                            observed = flex_table.loc[i, j]
                            expected = row_sums[i] * col_sums[j] / total
                            if expected > 0:
                                chi2 += ((observed - expected) ** 2) / expected
                    
                    print(f"\nChi-square for baseline/catch distribution: {chi2:.2f}")
                    print(f"Lower values indicate better balance between breathphases.")
            except Exception as e:
                print(f"Error calculating chi-square: {e}")
        
        # 7. Consecutive trials analysis
        trial_sequence = design_df['trial_type'].tolist()
        consecutive_counts = {}
        current_type = trial_sequence[0]
        current_count = 1
        for t in trial_sequence[1:]:
            if t == current_type:
                current_count += 1
            else:
                key = f"{current_type}_{current_count}"
                consecutive_counts[key] = consecutive_counts.get(key, 0) + 1
                current_type = t
                current_count = 1
        key = f"{current_type}_{current_count}"
        consecutive_counts[key] = consecutive_counts.get(key, 0) + 1
        
        print("\nRuns of consecutive trial types:")
        for key, count in sorted(consecutive_counts.items()):
            print(f"  {key}: {count}")
        
        # 8. SOA and stimulus type correlation check
        print("\nSOA-by-stimulus-type COUNTERBALANCE CHECK:")
        non_baseline = design_df[design_df['trial_type'] != 'baseline']
        if len(non_baseline) > 0:
            soa_by_stim = pd.crosstab(
                non_baseline['soa_value_ms'], 
                non_baseline['stimulus_type']
            )
            print(soa_by_stim)
            
            # Calculate coefficient of variation as a balance metric
            cv_values = []
            for _, row in soa_by_stim.iterrows():
                if row.sum() > 0:
                    cv = row.std() / row.mean() if row.mean() > 0 else 0
                    cv_values.append(cv)
            
            mean_cv = sum(cv_values) / len(cv_values) if cv_values else 0
            print(f"\nCoefficient of variation across SOA-stimulus combinations: {mean_cv:.4f}")
            print(f"Lower values indicate better counterbalancing.")
        
        # 9. Breathphase alternation check
        if 'breathphase' in design_df.columns:
            phases = design_df.sort_values('milliseconds')['breathphase'].str.lower().tolist()
            alternation_violations = 0
            for i in range(1, len(phases)):
                if phases[i] == phases[i-1]:
                    alternation_violations += 1
            
            print(f"\nBreathphase alternation violations: {alternation_violations}")
            print(f"This should be 0 - the inhale/exhale cycle must be maintained.")
        
        # 10. Timing analysis
        if 'jittered_ms' in design_df.columns:
            time_diffs = []
            sorted_df = design_df.sort_values('jittered_ms')
            for i in range(1, len(sorted_df)):
                diff = sorted_df.iloc[i]['jittered_ms'] - sorted_df.iloc[i-1]['jittered_ms']
                time_diffs.append(diff)
            
            if time_diffs:
                min_diff = min(time_diffs)
                max_diff = max(time_diffs)
                avg_diff = sum(time_diffs) / len(time_diffs)
                print(f"\nStimulus timing intervals:")
                print(f"  Minimum interval: {min_diff:.1f} ms")
                print(f"  Maximum interval: {max_diff:.1f} ms")
                print(f"  Average interval: {avg_diff:.1f} ms")
    
    # ------------------------------------------------------------------
    # Process a single participant: generate design, assign slots, finalize, and save CSV.
    # ------------------------------------------------------------------
    def process_participant(self, participant_id):
        try:
            if self.config['debug_mode']:
                print(f"\n=== Processing participant {participant_id} ===")
            design_df = self.generate_counterbalanced_design(participant_id)
            design_df = self.assign_breath_holds(design_df)
            design_df = self.finalize_design_csv(design_df)
            out_path = os.path.join(self.experiment_log_dir, f"participant_{participant_id}_design.csv")
            design_df.to_csv(out_path, index=False)
            if self.config['debug_mode']:
                print(f"Saved participant {participant_id} design to {out_path}")
            return True, out_path
        except Exception as e:
            print(f"Error processing participant {participant_id}: {e}")
            traceback.print_exc()
            return False, None
    
    # ------------------------------------------------------------------
    # Run design generation for multiple participants.
    # ------------------------------------------------------------------
    def run(self, participant_ids=None, num_participants=None):
        if participant_ids is None and num_participants is None:
            num_participants = self.config.get('num_participants', 5)
        if participant_ids is None:
            participant_ids = list(range(num_participants))
        if self.config['debug_mode']:
            print("=== Starting PPSDesignGenerator ===")
            print(f"Repetitions: {self.config['repetitions']}")
            print(f"Participants: {participant_ids}")
            print(f"Box breathing settings: {self.config['box_breathing']}")
        results = {}
        for pid in participant_ids:
            success, path = self.process_participant(pid)
            results[pid] = {'success': success, 'csv_path': path}
        
        # Summary of all participants
        if self.config['debug_mode'] and len(results) > 1:
            print("\n=== CROSS-PARTICIPANT VALIDATION ===")
            all_success = all(info['success'] for info in results.values())
            print(f"All participants generated successfully: {all_success}")
        
        return results

# ----------------------------------------------------------------------
# AUDIO GENERATION PLACEHOLDER FUNCTION
# ----------------------------------------------------------------------
def generate_audio_from_design(design_csv_path, output_dir, base_audio_path=None):
    """
    Generate audio files from a design CSV.
    (Placeholder function – implement audio generation as needed.)
    """
    print(f"Audio generation from {design_csv_path} not implemented yet")
    return None, None

# ----------------------------------------------------------------------
# MAIN FUNCTION
# ----------------------------------------------------------------------
def main():
    """Main function to run the design generator."""
    # Print configuration summary
    print("\n===== PPS EXPERIMENT DESIGN GENERATOR =====")
    print(f"Repetitions: {CONFIG['repetitions']}")
    print(f"SOA Conditions: {CONFIG['soa_conditions_ms']}")
    print(f"Jitter Options: {CONFIG['jitter_options_ms']}")
    print(f"Looming Stimuli: {LOOMING_STIMULI}")
    print(f"Number of Participants: {CONFIG['num_participants']}")
    
    # Calculate expected trial counts
    reps = CONFIG['repetitions']
    soa_count = len(CONFIG['soa_conditions_ms'])
    stim_count = len(LOOMING_STIMULI)
    
    base_trials_per_condition = stim_count * soa_count * reps
    enabled_conditions = [cond for cond, enabled in CONFIG['included_conditions'].items() if enabled]
    total_base_trials = base_trials_per_condition * len(enabled_conditions)
    catch_trials = int(np.ceil(total_base_trials * CONFIG['catch_trial_percentage'] / 100))
    
    print("\n===== EXPECTED TRIAL COUNTS =====")
    print(f"Base trials per condition: {stim_count} stimuli × {soa_count} SOAs × {reps} repetitions = {base_trials_per_condition}")
    for cond in enabled_conditions:
        print(f"  {cond.upper()}: {base_trials_per_condition} trials")
    print(f"Catch trials ({CONFIG['catch_trial_percentage']}%): {catch_trials}")
    print(f"TOTAL TRIALS PER PARTICIPANT: {total_base_trials + catch_trials}")
    
    # Run the generator
    generator = PPSDesignGenerator(CONFIG)
    results = generator.run()
    
    # Summarize results
    print("\n===== GENERATION SUMMARY =====")
    success_count = sum(1 for info in results.values() if info['success'])
    print(f"Successfully generated designs for {success_count}/{len(results)} participants.")
    
    for pid, info in results.items():
        if info['success']:
            print(f"Participant {pid}: CSV generated at {info['csv_path']}")
        else:
            print(f"Participant {pid}: ERROR - no CSV generated.")
    
    print("\nTo generate audio files from these designs, implement the audio generation function.")

if __name__ == "__main__":
    main()


===== PPS EXPERIMENT DESIGN GENERATOR =====
Repetitions: 4
SOA Conditions: [190, 400, 700, 1000, 1500]
Jitter Options: [100, 200, 300, 400, 500]
Looming Stimuli: ['blue', 'brown', 'pink', 'white']
Number of Participants: 5

===== EXPECTED TRIAL COUNTS =====
Base trials per condition: 4 stimuli × 5 SOAs × 4 repetitions = 80
  INHALATION: 80 trials
  EXHALATION: 80 trials
  BASELINE: 80 trials
Catch trials (10%): 24
TOTAL TRIALS PER PARTICIPANT: 264
Trial count calculation:
  inhalation: 60 trials
  exhalation: 60 trials
  baseline: 60 trials
  catch: 18 trials
  Total: 198 trials
Initialized PPSDesignGenerator with:
  - Repetitions: 4
  - Conditions: ['inhalation', 'exhalation', 'baseline']
  - SOA values: [190, 400, 700, 1000, 1500]
  - Total trials per participant: 198
  - Output directory: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog
=== Starting PPSDesignGenerator ===
Repetitions: 4
Participants: [0, 1, 2, 3, 4]
Box breathing settings: 

Traceback (most recent call last):
  File "C:\Users\cogpsy-vrlab\AppData\Local\Temp\ipykernel_7528\764555180.py", line 693, in process_participant
    design_df.to_csv(out_path, index=False)
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\site-packages\pandas\util\_decorators.py", line 333, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\site-packages\pandas\core\generic.py", line 3967, in to_csv
    return DataFrameRenderer(formatter).to_csv(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\site-packages\pandas\io\formats\format.py", line 1014, in to_csv
    csv_formatter.save()
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\site-packages\pandas\io\formats\csvs.py", line 251, in save
    with get_handle(
         ^^^^^^^^^^^
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\site-packages\pandas\io\common.py", line 873, in get_handle
    handle = open(
             ^^^^^
PermissionErr


Stimulus timing intervals:
  Minimum interval: 7600.0 ms
  Maximum interval: 8400.0 ms
  Average interval: 8001.6 ms
Saved participant 2 design to C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog\participant_2_design.csv

=== Processing participant 3 ===

Trial type distribution:
trial_type
inhalation    80
baseline      80
exhalation    80
catch         18
Name: count, dtype: int64

Stimulus type distribution by trial type:
  inhalation: {'white': 20, 'pink': 20, 'brown': 20, 'blue': 20}
  exhalation: {'brown': 20, 'pink': 20, 'blue': 20, 'white': 20}
  catch: {'blue': 5, 'brown': 5, 'pink': 4, 'white': 4}

SOA distribution by trial type:
  inhalation: {190: 16, 700: 16, 1000: 16, 400: 16, 1500: 16}
  exhalation: {1000: 16, 1500: 16, 400: 16, 700: 16, 190: 16}
  catch: {190: 4, 400: 4, 700: 4, 1500: 3, 1000: 3}
Scaffold CSV columns: ['retentionslot_timestamp', 'milliseconds', 'sample_index', 'description', 'breathphase']
Loaded scaffold CSV w

In [28]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
PPS Design Generator

This script generates participant‐specific experimental design CSVs for a 
Peripersonal Space (PPS) experiment. Each design CSV is built upon a 
"scaffold" CSV that contains the retention slot (breathhold) timestamps 
from a box‐breathing audio file. If needed, additional retention slots are 
appended (using fixed 8‑second increments) so that every trial can be assigned 
to a specific retention slot. Then, a counterbalanced set of trial conditions 
(e.g., trial type, looming stimulus type, SOA, jitter) is allocated to each 
scaffold row according to the following rules:
  
  - Inhalation trials (looming cue + tactile) are only assigned to scaffold rows 
    whose breathphase is "inhale."
  - Exhalation trials (looming cue + tactile) are only assigned to scaffold rows 
    whose breathphase is "exhale."
  - Baseline (tactile only; looming = NA) and catch (looming only; no tactile) trials 
    are allocated from the remaining pool in a random, counterbalanced manner 
    between the inhale and exhale rows.
    
The final output retains the original column order from the scaffold, with new 
trial‐specific variables appended as new columns.

Author: AI Assistant
"""

import os
import numpy as np
import pandas as pd
import random
import traceback

# ----------------------------------------------------------------------
# Helper function: Convert milliseconds to "MM:SS.s" format
# ----------------------------------------------------------------------
def convert_ms_to_timestamp(ms):
    m = int(ms // 60000)
    s = (ms % 60000) / 1000.0
    return f"{m:02}:{s:.1f}"

# ----------------------------------------------------------------------
# CONFIGURATION PARAMETERS
# ----------------------------------------------------------------------
CONFIG = {
    'repetitions': 4,            # How many times each SOA×stimulus combination repeats
    'num_participants': 5,       # Default number of participants
    'included_conditions': {
        'inhalation': True,
        'exhalation': True,
        'baseline': True,
    },
    'catch_trial_percentage': 10,
    'soa_conditions_ms': [190, 400, 700, 1000, 1500],
    'jitter_options_ms': [100, 200, 300, 400, 500],
    'debug_mode': True,
    'paths': {
        # Directory where participant-specific CSV files will be saved
        'experiment_log_dir': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog",
    },
    # Path to the original scaffold CSV containing retention (breathhold) timestamps.
    'skeleton_csv_path': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_timestamps.csv",
    'box_breathing': {
        'cycle_duration_sec': 8,      # Duration of one breath cycle in seconds
        # Intro/outro are not used here because the scaffold drives timing.
        'intro_duration_sec': 60,
        'outro_duration_sec': 30,
        'sample_rate': 48000,           # Audio sample rate (samples per second)
    }
}

# Noise types for looming stimuli (can be any labels)
LOOMING_STIMULI = ['blue', 'brown', 'pink', 'white']

# ----------------------------------------------------------------------
# CLASS: PPSDesignGenerator
# ----------------------------------------------------------------------
class PPSDesignGenerator:
    """Generates participant-specific CSV design files for PPS experiments."""
    
    def __init__(self, config=None):
        self.config = config or CONFIG
        
        # Prepare output directory.
        self.experiment_log_dir = self.config['paths']['experiment_log_dir']
        os.makedirs(self.experiment_log_dir, exist_ok=True)
        
        # Calculate trial counts.
        # For each condition (inhalation, exhalation, baseline): count = (# noise types) × (# SOA) × (repetitions).
        # Then, add 10% catch trials.
        self.trial_counts = self._calculate_trial_counts()
        self.conditions = [cond for cond, inc in self.config['included_conditions'].items() if inc]
        
        # Set seed for reproducibility.
        self.base_seed = 42
        
        if self.config['debug_mode']:
            total_trials = sum(self.trial_counts.values())
            print("Initialized PPSDesignGenerator with:")
            print(f"  - Repetitions: {self.config['repetitions']}")
            print(f"  - Conditions: {self.conditions}")
            print(f"  - SOA values: {self.config['soa_conditions_ms']}")
            print(f"  - Total trials per participant: {total_trials}")
            print(f"  - Output directory: {self.experiment_log_dir}")
    
    def _calculate_trial_counts(self):
        """
        Calculate the number of trials for each condition.
        
        This method sets up exactly 12 trials per SOA condition (190, 400, etc.)
        for each trial type (inhalation, exhalation, baseline).
        """
        soa_conditions = self.config['soa_conditions_ms']
        trials_per_soa = 12  # Fixed at 12 trials per SOA per condition
        catch_percentage = self.config['catch_trial_percentage']
        
        trial_counts = {}
        for condition, include in self.config['included_conditions'].items():
            if include:
                trial_counts[condition] = len(soa_conditions) * trials_per_soa
        
        regular_trials = sum(trial_counts.values())
        catch_trials = int(np.ceil(regular_trials * catch_percentage / 100))
        trial_counts['catch'] = catch_trials
        
        if self.config['debug_mode']:
            print("Trial count calculation:")
            for cond, count in trial_counts.items():
                print(f"  {cond}: {count} trials")
            print(f"  Total: {sum(trial_counts.values())} trials")
        
        return trial_counts

    # ------------------------------------------------------------------
    # Load the scaffold CSV and extend it if needed.
    # Retain the original column order.
    # ------------------------------------------------------------------
    def load_retention_slots(self, required_slots):
        """
        Load the scaffold CSV containing retention slots and extend it if needed.
        Handles different column naming variations by standardizing them.
        """
        skeleton_path = self.config['skeleton_csv_path']
        try:
            # Try to read the CSV, handling different separators
            skeleton_df = pd.read_csv(skeleton_path, sep=None, engine='python')
            if len(skeleton_df.columns) == 1:
                skeleton_df = pd.read_csv(skeleton_path, delimiter="\t")
            
            # Normalize column names (strip whitespace and convert to lowercase)
            skeleton_df.columns = [col.strip().lower() for col in skeleton_df.columns]
            self.scaffold_order = skeleton_df.columns.tolist()
            
            if self.config['debug_mode']:
                print("Scaffold CSV columns:", self.scaffold_order)
                
            # Determine timestamp column name - could be 'timestamp' or 'retentionslot_timestamp'
            timestamp_col = None
            if 'timestamp' in skeleton_df.columns:
                timestamp_col = 'timestamp'
            elif 'retentionslot_timestamp' in skeleton_df.columns:
                timestamp_col = 'retentionslot_timestamp'
            
            # If neither standard name exists, try to find a column that might contain timestamp info
            if timestamp_col is None:
                for col in skeleton_df.columns:
                    if 'time' in col.lower() and not col.lower() == 'milliseconds':
                        timestamp_col = col
                        break
            
            if timestamp_col is None:
                raise ValueError("Could not find timestamp column in scaffold CSV.")
                
            # Standardize the timestamp column name to 'retentionslot_timestamp'
            if timestamp_col != 'retentionslot_timestamp':
                skeleton_df = skeleton_df.rename(columns={timestamp_col: 'retentionslot_timestamp'})
                if timestamp_col in self.scaffold_order:
                    self.scaffold_order[self.scaffold_order.index(timestamp_col)] = 'retentionslot_timestamp'
                
        except Exception as e:
            raise ValueError(f"Error reading scaffold CSV from {skeleton_path}: {e}")
        
        current_slots = len(skeleton_df)
        required_slots = int(required_slots)
        sample_rate = self.config['box_breathing']['sample_rate']
        cycle_sec = self.config['box_breathing']['cycle_duration_sec']
        
        if self.config['debug_mode']:
            print(f"Loaded scaffold CSV with {current_slots} retention slots.")
            # Display the last few existing timestamps
            try:
                print(f"Last timestamp in skeleton CSV: {skeleton_df.iloc[-1]['retentionslot_timestamp']} ({skeleton_df.iloc[-1]['milliseconds']} ms)")
            except Exception as e:
                print(f"Could not display last timestamp: {e}")
        
        if current_slots < required_slots:
            last_row = skeleton_df.iloc[-1]
            last_ms = float(last_row['milliseconds'])
            last_sample_index = int(last_row['sample_index'])
            new_rows = []
            
            # Get the last breathphase to ensure proper alternation
            last_breathphase = last_row['breathphase'].lower()
            
            for i in range(required_slots - current_slots):
                # Increment by 8 seconds (8000 ms) for each new row
                new_ms = last_ms + (cycle_sec * 1000) * (i + 1)
                new_sample_index = last_sample_index + int(cycle_sec * sample_rate * (i + 1))
                new_timestamp = convert_ms_to_timestamp(new_ms)
                
                # Alternate breathphase: if last was 'exhale', next is 'inhale' and vice versa
                if i == 0:  # First new row
                    new_breathphase = 'inhale' if last_breathphase == 'exhale' else 'exhale'
                else:  # Subsequent rows
                    prev_phase = new_rows[i-1]['breathphase'].lower()
                    new_breathphase = 'inhale' if prev_phase == 'exhale' else 'exhale'
                
                new_description = f"Additional hold {i+1}"
                
                # Create the new row with the correct timestamp column name
                new_row = {
                    'retentionslot_timestamp': new_timestamp,
                    'milliseconds': new_ms,
                    'sample_index': new_sample_index,
                    'description': new_description,
                    'breathphase': new_breathphase
                }
                new_rows.append(new_row)
                
                if self.config['debug_mode'] and i < 5:  # Show first few new rows
                    print(f"Adding new row: {new_timestamp} ({new_ms} ms) - {new_breathphase}")
            
            if new_rows:
                new_rows_df = pd.DataFrame(new_rows)
                # Ensure new_rows_df has all columns from skeleton_df
                for col in skeleton_df.columns:
                    if col not in new_rows_df.columns:
                        new_rows_df[col] = None
                
                # Concatenate with only the columns from the original DataFrame
                skeleton_df = pd.concat([skeleton_df, new_rows_df[skeleton_df.columns]], ignore_index=True)
            
            if self.config['debug_mode']:
                print(f"Extended scaffold CSV to {len(skeleton_df)} retention slots.")
                print(f"New timestamp range: {skeleton_df['retentionslot_timestamp'].iloc[0]} to {skeleton_df['retentionslot_timestamp'].iloc[-1]}")
        
        if len(skeleton_df) > required_slots:
            skeleton_df = skeleton_df.iloc[:required_slots]
        
        return skeleton_df

    # ------------------------------------------------------------------
    # Generate counterbalanced stimulus design (trial conditions)
    # ------------------------------------------------------------------
    def generate_counterbalanced_design(self, participant_id):
        """
        Generate a counterbalanced design for a participant with exactly 
        12 trials per SOA per condition.
        
        Catch trials don't have SOA values assigned since they don't have tactile stimuli.
        """
        random.seed(self.base_seed + participant_id)
        np.random.seed(self.base_seed + participant_id)
        
        soa_conditions = self.config['soa_conditions_ms']
        jitter_options = self.config['jitter_options_ms']
        stimulus_types = LOOMING_STIMULI
        
        all_trials = []
        
        # Generate main conditions: inhalation, exhalation, and baseline with fixed 12 trials per SOA
        for condition, count in self.trial_counts.items():
            if condition == 'catch' or count == 0:
                continue
            
            # For each SOA, generate exactly 12 trials
            for soa in soa_conditions:
                # Calculate how many trials per stimulus type (as evenly as possible)
                trials_per_soa = 12
                trials_per_stim = trials_per_soa // len(stimulus_types)  # Integer division
                remainder = trials_per_soa % len(stimulus_types)
                
                # Distribute trials across stimulus types
                stim_trials = []
                for i, stim_type in enumerate(stimulus_types):
                    # Add extra trial for remainder distribution
                    extra = 1 if i < remainder else 0
                    for _ in range(trials_per_stim + extra):
                        stim_trials.append(stim_type)
                
                # Shuffle the stimulus types
                random.shuffle(stim_trials)
                
                # Create trials for this SOA
                for stim_type in stim_trials:
                    all_trials.append({
                        'participant_id': participant_id,
                        'trial_type': condition,
                        'stimulus_type': stim_type,
                        'soa_value_ms': soa,
                        'jitter_ms': random.choice(jitter_options),
                        'is_tactile': True  # For these main trials, tactile is present.
                    })
        
        # Add catch trials (looming only, no tactile) - no SOA values needed
        catch_count = self.trial_counts.get('catch', 0)
        if catch_count > 0:
            # Distribute catch trials evenly across stimulus types only
            catch_trials = []
            stim_counts = {stim: 0 for stim in stimulus_types}
            
            for _ in range(catch_count):
                # Choose the stimulus type with the lowest count
                min_stim = min(stim_counts, key=stim_counts.get)
                
                catch_trials.append({
                    'participant_id': participant_id,
                    'trial_type': 'catch',
                    'stimulus_type': min_stim,
                    'soa_value_ms': None,  # No SOA for catch trials
                    'jitter_ms': random.choice(jitter_options),
                    'is_tactile': False
                })
                
                stim_counts[min_stim] += 1
            
            # Shuffle catch trials and add to all trials
            random.shuffle(catch_trials)
            all_trials.extend(catch_trials)
        
        # Shuffle all trials and assign trial numbers
        all_trials_df = pd.DataFrame(all_trials)
        all_trials_df = all_trials_df.sample(frac=1, random_state=participant_id*100).reset_index(drop=True)
        all_trials_df['trial_number'] = range(1, len(all_trials_df) + 1)
        
        if self.config['debug_mode']:
            total_trials = len(all_trials_df)
            print("\nTrial distribution:")
            print(f"Total trials: {total_trials}")
            
            # Count by trial type
            for trial_type in sorted(all_trials_df['trial_type'].unique()):
                type_df = all_trials_df[all_trials_df['trial_type'] == trial_type]
                print(f"Total {trial_type}: {len(type_df)}")
                
                # Count by SOA for non-baseline and non-catch trial types
                if trial_type not in ['baseline', 'catch']:
                    for soa in sorted(soa_conditions):
                        soa_count = len(type_df[type_df['soa_value_ms'] == soa])
                        print(f"  {trial_type}_{soa}: {soa_count}")
                elif trial_type == 'baseline':
                    # For baseline, show the total
                    baseline_count = len(type_df)
                    print(f"  baseline_trials: {baseline_count}")
                elif trial_type == 'catch':
                    # For catch trials, show stimulus distribution
                    catch_count = len(type_df)
                    print(f"  catch_trials: {catch_count} (no tactile stimulus, no SOA)")
            
            # Show stimulus type distribution
            print("\nStimulus type distribution by trial type:")
            for trial_type in sorted(all_trials_df['trial_type'].unique()):
                if trial_type == 'baseline':
                    continue
                subset = all_trials_df[all_trials_df['trial_type'] == trial_type]
                print(f"  {trial_type}: {subset['stimulus_type'].value_counts().to_dict()}")
        
        return all_trials_df

    # ------------------------------------------------------------------
    # Assign scaffold retention slots to trials while strictly maintaining the breathphase cycle.
    # ------------------------------------------------------------------
    def assign_breath_holds(self, design_df):
        """
        Assign scaffold retention slots to trials while strictly maintaining the breathphase cycle.
        
        Key requirements:
        1. Breathphase cycle (inhale-exhale-inhale-exhale) must remain intact
        2. Inhalation trials must go to inhale breathphase slots
        3. Exhalation trials must go to exhale breathphase slots
        4. Baseline and catch trials can go to either breathphase type
        5. The order of baseline and catch trials should be randomized
        """
        required_slots = len(design_df)
        scaffold_df = self.load_retention_slots(required_slots)
        
        # Ensure scaffold is properly sorted by timestamp/milliseconds
        scaffold_df = scaffold_df.sort_values(by='milliseconds').reset_index(drop=True)
        
        # Partition design trials by trial type
        pool_inh = [d for d in design_df.to_dict('records') if d['trial_type'] == 'inhalation']
        pool_exh = [d for d in design_df.to_dict('records') if d['trial_type'] == 'exhalation']
        pool_baseline = [d for d in design_df.to_dict('records') if d['trial_type'] == 'baseline']
        pool_catch = [d for d in design_df.to_dict('records') if d['trial_type'] == 'catch']
        
        # Count scaffold slots by breathphase
        inhale_slots = scaffold_df[scaffold_df['breathphase'].str.lower() == 'inhale']
        exhale_slots = scaffold_df[scaffold_df['breathphase'].str.lower() == 'exhale']
        n_inhale_slots = len(inhale_slots)
        n_exhale_slots = len(exhale_slots)
        
        if self.config['debug_mode']:
            print(f"Scaffold: {n_inhale_slots} inhale slots, {n_exhale_slots} exhale slots")
            print(f"Trials: {len(pool_inh)} inhalation, {len(pool_exh)} exhalation, " 
                  f"{len(pool_baseline)} baseline, {len(pool_catch)} catch")
        
        # Verify we have enough slots for mandatory trials
        if len(pool_inh) > n_inhale_slots:
            raise ValueError(f"Not enough inhale slots ({n_inhale_slots}) for inhalation trials ({len(pool_inh)})")
        if len(pool_exh) > n_exhale_slots:
            raise ValueError(f"Not enough exhale slots ({n_exhale_slots}) for exhalation trials ({len(pool_exh)})")
        
        # Calculate remaining slots per breathphase
        remaining_inhale = n_inhale_slots - len(pool_inh)
        remaining_exhale = n_exhale_slots - len(pool_exh)
        
        # Total baseline+catch trials
        total_flex_trials = len(pool_baseline) + len(pool_catch)
        
        # Verify we have enough slots for all trials
        if remaining_inhale + remaining_exhale != total_flex_trials:
            raise ValueError(f"Mismatch between available slots ({remaining_inhale + remaining_exhale}) "
                            f"and flexible trials ({total_flex_trials})")
        
        # Decide how to distribute baseline and catch trials across breathphases
        # Combine them first, then distribute proportionally
        flexible_trials = pool_baseline + pool_catch
        random.shuffle(flexible_trials)  # Randomize order
        
        # Split flexible trials between inhale and exhale slots proportionally
        inhale_flexible = flexible_trials[:remaining_inhale]
        exhale_flexible = flexible_trials[remaining_inhale:]
        
        if self.config['debug_mode']:
            print(f"Distributing {len(inhale_flexible)} flexible trials to inhale slots")
            print(f"Distributing {len(exhale_flexible)} flexible trials to exhale slots")
        
        # Create final allocation pools
        inhale_pool = pool_inh + inhale_flexible
        exhale_pool = pool_exh + exhale_flexible
        
        # Shuffle within each pool to randomize order
        random.shuffle(inhale_pool)
        random.shuffle(exhale_pool)
        
        # Create a dictionary mapping breathphase to trial pool
        phase_to_pool = {
            'inhale': inhale_pool,
            'exhale': exhale_pool
        }
        
        # Initialize counters for each breathphase
        pool_counters = {
            'inhale': 0,
            'exhale': 0
        }
        
        # Final list to store assigned trials in scaffold order
        final_assignments = []
        
        # Assign trials to scaffold rows, maintaining the original order
        for _, row in scaffold_df.iterrows():
            breathphase = row['breathphase'].lower()
            
            # Get the next trial from the appropriate pool
            if pool_counters[breathphase] < len(phase_to_pool[breathphase]):
                trial = phase_to_pool[breathphase][pool_counters[breathphase]]
                pool_counters[breathphase] += 1
                
                # Combine scaffold information with trial information
                combined = trial.copy()
                for col in row.index:
                    combined[col] = row[col]
                
                final_assignments.append(combined)
        
        # Create final dataframe and sort by scaffold order
        final_df = pd.DataFrame(final_assignments).sort_values(by='milliseconds').reset_index(drop=True)
        final_df['trial_number'] = range(1, len(final_df) + 1)
        
        # Compute additional timing information
        final_df['jittered_ms'] = final_df['milliseconds'] + final_df['jitter_ms']
        final_df['looming_stimulus_timestamp'] = final_df['jittered_ms'].apply(convert_ms_to_timestamp)
        
        # Add SOA timing for non-catch trials
        non_catch_mask = final_df['trial_type'] != 'catch'
        final_df.loc[non_catch_mask, 'soa_ms'] = final_df.loc[non_catch_mask, 'jittered_ms'] + final_df.loc[non_catch_mask, 'soa_value_ms']
        final_df.loc[non_catch_mask, 'tactile_stimulus_timestamp'] = final_df.loc[non_catch_mask, 'soa_ms'].apply(convert_ms_to_timestamp)
        
        # Verify the assignment respects the breathphase constraints
        self._validate_breathphase_assignment(final_df)
        
        return final_df

    def _validate_breathphase_assignment(self, final_df):
        """Verify that the trial assignment respects breathphase constraints."""
        if not self.config['debug_mode']:
            return
            
        print("\n=== BREATHPHASE ASSIGNMENT VALIDATION ===")
        
        # Check for inhalation trials assigned to inhale
        inh_trials = final_df[final_df['trial_type'] == 'inhalation']
        inh_correct = (inh_trials['breathphase'].str.lower() == 'inhale').all()
        print(f"Inhalation trials on inhale breathphase: {inh_correct}")
        
        # Check for exhalation trials assigned to exhale
        exh_trials = final_df[final_df['trial_type'] == 'exhalation']
        exh_correct = (exh_trials['breathphase'].str.lower() == 'exhale').all()
        print(f"Exhalation trials on exhale breathphase: {exh_correct}")
        
        # Count distribution of baseline and catch trials across breathphases
        baseline_inhale = ((final_df['trial_type'] == 'baseline') & 
                         (final_df['breathphase'].str.lower() == 'inhale')).sum()
        baseline_exhale = ((final_df['trial_type'] == 'baseline') & 
                          (final_df['breathphase'].str.lower() == 'exhale')).sum()
        catch_inhale = ((final_df['trial_type'] == 'catch') & 
                       (final_df['breathphase'].str.lower() == 'inhale')).sum()
        catch_exhale = ((final_df['trial_type'] == 'catch') & 
                       (final_df['breathphase'].str.lower() == 'exhale')).sum()
        
        print(f"Baseline trials: {baseline_inhale} on inhale, {baseline_exhale} on exhale")
        print(f"Catch trials: {catch_inhale} on inhale, {catch_exhale} on exhale")
        
        # Verify breathphase cycle is maintained
        prev_phase = None
        cycle_maintained = True
        expected_cycle = ['inhale', 'exhale'] * (len(final_df) // 2 + 1)
        
        for i, phase in enumerate(final_df['breathphase'].str.lower()):
            if i < len(expected_cycle) and phase != expected_cycle[i]:
                cycle_maintained = False
                print(f"Breathphase cycle disrupted at index {i}: expected {expected_cycle[i]}, got {phase}")
                break
        
        print(f"Breathphase cycle maintained: {cycle_maintained}")

    # ------------------------------------------------------------------
    # Finalize the design CSV: Retain the scaffold column order and append new columns.
    # Also, for baseline trials remove looming stimulus info; for catch trials remove tactile stimulus info.
    # ------------------------------------------------------------------
    def finalize_design_csv(self, design_df):
        baseline_mask = (design_df['trial_type'] == 'baseline')
        design_df.loc[baseline_mask, 'stimulus_type'] = None
        design_df.loc[baseline_mask, 'looming_stimulus_timestamp'] = None
        
        catch_mask = (design_df['trial_type'] == 'catch')
        design_df.loc[catch_mask, 'tactile_stimulus_timestamp'] = None
        
        design_df['expected_response'] = ~design_df['trial_type'].isin(['catch'])
        
        scaffold_cols = self.scaffold_order if hasattr(self, 'scaffold_order') else ['timestamp', 'milliseconds', 'sample_index', 'description', 'breathphase']
        new_cols = ['participant_id', 'trial_number', 'trial_type', 'stimulus_type', 'soa_value_ms', 'jitter_ms', 'is_tactile', 'looming_stimulus_timestamp', 'tactile_stimulus_timestamp', 'soa_ms', 'jittered_ms', 'expected_response']
        final_order = scaffold_cols + [col for col in new_cols if col not in scaffold_cols and col in design_df.columns]
        remaining = [col for col in design_df.columns if col not in final_order]
        final_order.extend(remaining)
        design_df = design_df[final_order]
        
        self._validate_design(design_df)
        return design_df

    # ------------------------------------------------------------------
    # Validation: Print distributions and sequence properties.
    # ------------------------------------------------------------------
    def _validate_design(self, design_df):
        """Perform comprehensive validation and counterbalancing checks on the design."""
        if not self.config['debug_mode']:
            return
            
        print("\n=== DESIGN VALIDATION ===")
        
        # 1. Overall trial count summary
        total_trials = len(design_df)
        trial_counts = design_df['trial_type'].value_counts().to_dict()
        
        print(f"TOTAL TRIALS: {total_trials}")
        print("Trial type distribution:")
        for trial_type, count in sorted(trial_counts.items()):
            percent = (count / total_trials) * 100
            print(f"  {trial_type.upper()}: {count} trials ({percent:.1f}%)")
        
        # 2. Detailed breakdown by trial type
        print("\nStimulus distribution by trial type:")
        for trial_type in sorted(design_df['trial_type'].unique()):
            if trial_type == 'baseline':
                continue
            subset = design_df[design_df['trial_type'] == trial_type]
            stim_counts = subset['stimulus_type'].value_counts().to_dict()
            print(f"  {trial_type.upper()}:")
            for stim, count in sorted(stim_counts.items()):
                percent = (count / len(subset)) * 100
                print(f"    {stim}: {count} trials ({percent:.1f}%)")
        
        # 3. SOA distribution checks
        print("\nSOA distribution by trial type:")
        for trial_type in sorted(design_df['trial_type'].unique()):
            if trial_type == 'baseline':
                continue
            subset = design_df[design_df['trial_type'] == trial_type]
            soa_counts = subset['soa_value_ms'].value_counts().to_dict()
            print(f"  {trial_type.upper()}:")
            for soa, count in sorted(soa_counts.items()):
                percent = (count / len(subset)) * 100
                print(f"    {soa} ms: {count} trials ({percent:.1f}%)")
        
        # 4. Jitter distribution
        print("\nJitter distribution:")
        jitter_counts = design_df['jitter_ms'].value_counts().to_dict()
        for jitter, count in sorted(jitter_counts.items()):
            percent = (count / total_trials) * 100
            print(f"  {jitter} ms: {count} trials ({percent:.1f}%)")
        
        # 5. Breath phase checks
        print("\nBreath phase distribution:")
        if 'breathphase' in design_df.columns:
            phase_counts = design_df['breathphase'].value_counts().to_dict()
            for phase, count in sorted(phase_counts.items()):
                percent = (count / total_trials) * 100
                print(f"  {phase}: {count} trials ({percent:.1f}%)")
        
        # 6. Cross-tabulation of trial types by breath phase
        print("\nTrial types by breath phase (COUNTERBALANCE CHECK):")
        if 'breathphase' in design_df.columns:
            trial_by_phase = pd.crosstab(
                design_df['trial_type'], 
                design_df['breathphase']
            )
            print(trial_by_phase)
            
            # Calculate chi-square statistic for independence (excluding mandatory assignments)
            try:
                # Create subset without inhalation/exhalation (which are deterministically assigned)
                flex_df = design_df[design_df['trial_type'].isin(['baseline', 'catch'])]
                if len(flex_df) > 0:
                    flex_table = pd.crosstab(flex_df['trial_type'], flex_df['breathphase'])
                    chi2 = 0
                    row_sums = flex_table.sum(axis=1)
                    col_sums = flex_table.sum(axis=0) 
                    total = flex_table.sum().sum()
                    
                    for i in flex_table.index:
                        for j in flex_table.columns:
                            observed = flex_table.loc[i, j]
                            expected = row_sums[i] * col_sums[j] / total
                            if expected > 0:
                                chi2 += ((observed - expected) ** 2) / expected
                    
                    print(f"\nChi-square for baseline/catch distribution: {chi2:.2f}")
                    print(f"Lower values indicate better balance between breathphases.")
            except Exception as e:
                print(f"Error calculating chi-square: {e}")
        
        # 7. Consecutive trials analysis
        trial_sequence = design_df['trial_type'].tolist()
        consecutive_counts = {}
        current_type = trial_sequence[0]
        current_count = 1
        for t in trial_sequence[1:]:
            if t == current_type:
                current_count += 1
            else:
                key = f"{current_type}_{current_count}"
                consecutive_counts[key] = consecutive_counts.get(key, 0) + 1
                current_type = t
                current_count = 1
        key = f"{current_type}_{current_count}"
        consecutive_counts[key] = consecutive_counts.get(key, 0) + 1
        
        print("\nRuns of consecutive trial types:")
        for key, count in sorted(consecutive_counts.items()):
            print(f"  {key}: {count}")
        
        # 8. SOA and stimulus type correlation check
        print("\nSOA-by-stimulus-type COUNTERBALANCE CHECK:")
        non_baseline = design_df[design_df['trial_type'] != 'baseline']
        if len(non_baseline) > 0:
            soa_by_stim = pd.crosstab(
                non_baseline['soa_value_ms'], 
                non_baseline['stimulus_type']
            )
            print(soa_by_stim)
            
            # Calculate coefficient of variation as a balance metric
            cv_values = []
            for _, row in soa_by_stim.iterrows():
                if row.sum() > 0:
                    cv = row.std() / row.mean() if row.mean() > 0 else 0
                    cv_values.append(cv)
            
            mean_cv = sum(cv_values) / len(cv_values) if cv_values else 0
            print(f"\nCoefficient of variation across SOA-stimulus combinations: {mean_cv:.4f}")
            print(f"Lower values indicate better counterbalancing.")
        
        # 9. Breathphase alternation check
        if 'breathphase' in design_df.columns:
            phases = design_df.sort_values('milliseconds')['breathphase'].str.lower().tolist()
            alternation_violations = 0
            for i in range(1, len(phases)):
                if phases[i] == phases[i-1]:
                    alternation_violations += 1
            
            print(f"\nBreathphase alternation violations: {alternation_violations}")
            print(f"This should be 0 - the inhale/exhale cycle must be maintained.")
        
        # 10. Timing analysis
        if 'jittered_ms' in design_df.columns:
            time_diffs = []
            sorted_df = design_df.sort_values('jittered_ms')
            for i in range(1, len(sorted_df)):
                diff = sorted_df.iloc[i]['jittered_ms'] - sorted_df.iloc[i-1]['jittered_ms']
                time_diffs.append(diff)
            
            if time_diffs:
                min_diff = min(time_diffs)
                max_diff = max(time_diffs)
                avg_diff = sum(time_diffs) / len(time_diffs)
                print(f"\nStimulus timing intervals:")
                print(f"  Minimum interval: {min_diff:.1f} ms")
                print(f"  Maximum interval: {max_diff:.1f} ms")
                print(f"  Average interval: {avg_diff:.1f} ms")
    
    # ------------------------------------------------------------------
    # Process a single participant: generate design, assign slots, finalize, and save CSV.
    # ------------------------------------------------------------------
    def process_participant(self, participant_id):
        try:
            if self.config['debug_mode']:
                print(f"\n=== Processing participant {participant_id} ===")
            design_df = self.generate_counterbalanced_design(participant_id)
            design_df = self.assign_breath_holds(design_df)
            design_df = self.finalize_design_csv(design_df)
            out_path = os.path.join(self.experiment_log_dir, f"participant_{participant_id}_design.csv")
            design_df.to_csv(out_path, index=False)
            if self.config['debug_mode']:
                print(f"Saved participant {participant_id} design to {out_path}")
            return True, out_path
        except Exception as e:
            print(f"Error processing participant {participant_id}: {e}")
            traceback.print_exc()
            return False, None
    
    # ------------------------------------------------------------------
    # Run design generation for multiple participants.
    # ------------------------------------------------------------------
    def run(self, participant_ids=None, num_participants=None):
        if participant_ids is None and num_participants is None:
            num_participants = self.config.get('num_participants', 5)
        if participant_ids is None:
            participant_ids = list(range(num_participants))
        if self.config['debug_mode']:
            print("=== Starting PPSDesignGenerator ===")
            print(f"Repetitions: {self.config['repetitions']}")
            print(f"Participants: {participant_ids}")
            print(f"Box breathing settings: {self.config['box_breathing']}")
        results = {}
        for pid in participant_ids:
            success, path = self.process_participant(pid)
            results[pid] = {'success': success, 'csv_path': path}
        
        # Summary of all participants
        if self.config['debug_mode'] and len(results) > 1:
            print("\n=== CROSS-PARTICIPANT VALIDATION ===")
            all_success = all(info['success'] for info in results.values())
            print(f"All participants generated successfully: {all_success}")
        
        return results

# ----------------------------------------------------------------------
# AUDIO GENERATION PLACEHOLDER FUNCTION
# ----------------------------------------------------------------------
def generate_audio_from_design(design_csv_path, output_dir, base_audio_path=None):
    """
    Generate audio files from a design CSV.
    (Placeholder function – implement audio generation as needed.)
    """
    print(f"Audio generation from {design_csv_path} not implemented yet")
    return None, None

# ----------------------------------------------------------------------
# MAIN FUNCTION
# ----------------------------------------------------------------------
def main():
    """Main function to run the design generator."""
    # Print configuration summary
    print("\n===== PPS EXPERIMENT DESIGN GENERATOR =====")
    print(f"Repetitions: {CONFIG['repetitions']}")
    print(f"SOA Conditions: {CONFIG['soa_conditions_ms']}")
    print(f"Jitter Options: {CONFIG['jitter_options_ms']}")
    print(f"Looming Stimuli: {LOOMING_STIMULI}")
    print(f"Number of Participants: {CONFIG['num_participants']}")
    
    # Calculate expected trial counts
    reps = CONFIG['repetitions']
    soa_count = len(CONFIG['soa_conditions_ms'])
    stim_count = len(LOOMING_STIMULI)
    
    base_trials_per_condition = stim_count * soa_count * reps
    enabled_conditions = [cond for cond, enabled in CONFIG['included_conditions'].items() if enabled]
    total_base_trials = base_trials_per_condition * len(enabled_conditions)
    catch_trials = int(np.ceil(total_base_trials * CONFIG['catch_trial_percentage'] / 100))
    
    print("\n===== EXPECTED TRIAL COUNTS =====")
    print(f"Base trials per condition: {stim_count} stimuli × {soa_count} SOAs × {reps} repetitions = {base_trials_per_condition}")
    for cond in enabled_conditions:
        print(f"  {cond.upper()}: {base_trials_per_condition} trials")
    print(f"Catch trials ({CONFIG['catch_trial_percentage']}%): {catch_trials}")
    print(f"TOTAL TRIALS PER PARTICIPANT: {total_base_trials + catch_trials}")
    
    # Run the generator
    generator = PPSDesignGenerator(CONFIG)
    results = generator.run()
    
    # Summarize results
    print("\n===== GENERATION SUMMARY =====")
    success_count = sum(1 for info in results.values() if info['success'])
    print(f"Successfully generated designs for {success_count}/{len(results)} participants.")
    
    for pid, info in results.items():
        if info['success']:
            print(f"Participant {pid}: CSV generated at {info['csv_path']}")
        else:
            print(f"Participant {pid}: ERROR - no CSV generated.")
    
    print("\nTo generate audio files from these designs, implement the audio generation function.")

if __name__ == "__main__":
    main()


===== PPS EXPERIMENT DESIGN GENERATOR =====
Repetitions: 4
SOA Conditions: [190, 400, 700, 1000, 1500]
Jitter Options: [100, 200, 300, 400, 500]
Looming Stimuli: ['blue', 'brown', 'pink', 'white']
Number of Participants: 5

===== EXPECTED TRIAL COUNTS =====
Base trials per condition: 4 stimuli × 5 SOAs × 4 repetitions = 80
  INHALATION: 80 trials
  EXHALATION: 80 trials
  BASELINE: 80 trials
Catch trials (10%): 24
TOTAL TRIALS PER PARTICIPANT: 264
Trial count calculation:
  inhalation: 60 trials
  exhalation: 60 trials
  baseline: 60 trials
  catch: 18 trials
  Total: 198 trials
Initialized PPSDesignGenerator with:
  - Repetitions: 4
  - Conditions: ['inhalation', 'exhalation', 'baseline']
  - SOA values: [190, 400, 700, 1000, 1500]
  - Total trials per participant: 198
  - Output directory: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog
=== Starting PPSDesignGenerator ===
Repetitions: 4
Participants: [0, 1, 2, 3, 4]
Box breathing settings: 

In [1]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
PPS Design Generator

This script generates participant‐specific experimental design CSVs for a 
Peripersonal Space (PPS) experiment. Each design CSV is built upon a 
"scaffold" CSV that contains the retention slot (breathhold) timestamps 
from a box‐breathing audio file. If needed, additional retention slots are 
appended (using fixed 8‑second increments) so that every trial can be assigned 
to a specific retention slot. Then, a counterbalanced set of trial conditions 
(e.g., trial type, looming stimulus type, SOA, jitter) is allocated to each 
scaffold row according to the following rules:
  
  - Inhalation trials (looming cue + tactile) are only assigned to scaffold rows 
    whose breathphase is "inhale."
  - Exhalation trials (looming cue + tactile) are only assigned to scaffold rows 
    whose breathphase is "exhale."
  - Baseline (tactile only; looming = NA) and catch (looming only; no tactile) trials 
    are allocated from the remaining pool in a random, counterbalanced manner 
    between the inhale and exhale rows.
    
The final output retains the original column order from the scaffold, with new 
trial‐specific variables appended as new columns.

Author: AI Assistant
"""

import os
import numpy as np
import pandas as pd
import random
import traceback

# ----------------------------------------------------------------------
# Helper function: Convert milliseconds to "MM:SS.s" format
# ----------------------------------------------------------------------
def convert_ms_to_timestamp(ms):
    m = int(ms // 60000)
    s = (ms % 60000) / 1000.0
    return f"{m:02}:{s:.1f}"

# ----------------------------------------------------------------------
# CONFIGURATION PARAMETERS
# ----------------------------------------------------------------------
CONFIG = {
    'repetitions': 12,           # Number of trials per SOA condition
    'num_participants': 100,       # Default number of participants
    'included_conditions': {
        'inhalation': True,
        'exhalation': True,
        'baseline': True,
    },
    'catch_trial_percentage': 10,
    'soa_conditions_ms': [190, 400, 700, 1000, 1500],
    'jitter_options_ms': [100, 200, 300, 400, 500],
    'debug_mode': True,
    'paths': {
        # Directory where participant-specific CSV files will be saved
        'experiment_log_dir': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog",
    },
    # Path to the original scaffold CSV containing retention (breathhold) timestamps.
    'skeleton_csv_path': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_timestamps.csv",
    'box_breathing': {
        'cycle_duration_sec': 8,      # Duration of one breath cycle in seconds
        # Intro/outro are not used here because the scaffold drives timing.
        'intro_duration_sec': 60,
        'outro_duration_sec': 30,
        'sample_rate': 48000,           # Audio sample rate (samples per second)
    }
}

# Noise types for looming stimuli (can be any labels)
LOOMING_STIMULI = ['blue', 'brown', 'pink', 'white']

# ----------------------------------------------------------------------
# CLASS: PPSDesignGenerator
# ----------------------------------------------------------------------
class PPSDesignGenerator:
    """Generates participant-specific CSV design files for PPS experiments."""
    
    def __init__(self, config=None):
        self.config = config or CONFIG
        
        # Prepare output directory.
        self.experiment_log_dir = self.config['paths']['experiment_log_dir']
        os.makedirs(self.experiment_log_dir, exist_ok=True)
        
        # Calculate trial counts.
        # For each condition (inhalation, exhalation, baseline): count = (# noise types) × (# SOA) × (repetitions).
        # Then, add 10% catch trials.
        self.trial_counts = self._calculate_trial_counts()
        self.conditions = [cond for cond, inc in self.config['included_conditions'].items() if inc]
        
        # Set seed for reproducibility.
        self.base_seed = 42
        
        if self.config['debug_mode']:
            total_trials = sum(self.trial_counts.values())
            print("Initialized PPSDesignGenerator with:")
            print(f"  - Repetitions: {self.config['repetitions']}")
            print(f"  - Conditions: {self.conditions}")
            print(f"  - SOA values: {self.config['soa_conditions_ms']}")
            print(f"  - Total trials per participant: {total_trials}")
            print(f"  - Output directory: {self.experiment_log_dir}")
    
    def _calculate_trial_counts(self):
        """
        Calculate the number of trials for each condition, using the repetitions parameter
        to determine how many trials per SOA per condition.
        """
        soa_conditions = self.config['soa_conditions_ms']
        repetitions = self.config['repetitions']  # This now directly controls trials per SOA
        catch_percentage = self.config['catch_trial_percentage']
        
        trial_counts = {}
        for condition, include in self.config['included_conditions'].items():
            if include:
                # Each condition gets repetitions × number of SOAs trials
                trial_counts[condition] = len(soa_conditions) * repetitions
        
        regular_trials = sum(trial_counts.values())
        catch_trials = int(np.ceil(regular_trials * catch_percentage / 100))
        trial_counts['catch'] = catch_trials
        
        if self.config['debug_mode']:
            print("Trial count calculation:")
            for cond, count in trial_counts.items():
                print(f"  {cond}: {count} trials")
            print(f"  Total: {sum(trial_counts.values())} trials")
        
        return trial_counts

    # ------------------------------------------------------------------
    # Load the scaffold CSV and extend it if needed.
    # Retain the original column order.
    # ------------------------------------------------------------------
    def load_retention_slots(self, required_slots):
        """
        Load the scaffold CSV containing retention slots and extend it if needed.
        Handles different column naming variations by standardizing them.
        """
        skeleton_path = self.config['skeleton_csv_path']
        try:
            # Try to read the CSV, handling different separators
            skeleton_df = pd.read_csv(skeleton_path, sep=None, engine='python')
            if len(skeleton_df.columns) == 1:
                skeleton_df = pd.read_csv(skeleton_path, delimiter="\t")
            
            # Normalize column names (strip whitespace and convert to lowercase)
            skeleton_df.columns = [col.strip().lower() for col in skeleton_df.columns]
            self.scaffold_order = skeleton_df.columns.tolist()
            
            if self.config['debug_mode']:
                print("Scaffold CSV columns:", self.scaffold_order)
                
            # Determine timestamp column name - could be 'timestamp' or 'retentionslot_timestamp'
            timestamp_col = None
            if 'timestamp' in skeleton_df.columns:
                timestamp_col = 'timestamp'
            elif 'retentionslot_timestamp' in skeleton_df.columns:
                timestamp_col = 'retentionslot_timestamp'
            
            # If neither standard name exists, try to find a column that might contain timestamp info
            if timestamp_col is None:
                for col in skeleton_df.columns:
                    if 'time' in col.lower() and not col.lower() == 'milliseconds':
                        timestamp_col = col
                        break
            
            if timestamp_col is None:
                raise ValueError("Could not find timestamp column in scaffold CSV.")
                
            # Standardize the timestamp column name to 'retentionslot_timestamp'
            if timestamp_col != 'retentionslot_timestamp':
                skeleton_df = skeleton_df.rename(columns={timestamp_col: 'retentionslot_timestamp'})
                if timestamp_col in self.scaffold_order:
                    self.scaffold_order[self.scaffold_order.index(timestamp_col)] = 'retentionslot_timestamp'
                
        except Exception as e:
            raise ValueError(f"Error reading scaffold CSV from {skeleton_path}: {e}")
        
        current_slots = len(skeleton_df)
        required_slots = int(required_slots)
        sample_rate = self.config['box_breathing']['sample_rate']
        cycle_sec = self.config['box_breathing']['cycle_duration_sec']
        
        if self.config['debug_mode']:
            print(f"Loaded scaffold CSV with {current_slots} retention slots.")
            # Display the last few existing timestamps
            try:
                print(f"Last timestamp in skeleton CSV: {skeleton_df.iloc[-1]['retentionslot_timestamp']} ({skeleton_df.iloc[-1]['milliseconds']} ms)")
            except Exception as e:
                print(f"Could not display last timestamp: {e}")
        
        if current_slots < required_slots:
            last_row = skeleton_df.iloc[-1]
            last_ms = float(last_row['milliseconds'])
            last_sample_index = int(last_row['sample_index'])
            new_rows = []
            
            # Get the last breathphase to ensure proper alternation
            last_breathphase = last_row['breathphase'].lower()
            
            for i in range(required_slots - current_slots):
                # Increment by 8 seconds (8000 ms) for each new row
                new_ms = last_ms + (cycle_sec * 1000) * (i + 1)
                new_sample_index = last_sample_index + int(cycle_sec * sample_rate * (i + 1))
                new_timestamp = convert_ms_to_timestamp(new_ms)
                
                # Alternate breathphase: if last was 'exhale', next is 'inhale' and vice versa
                if i == 0:  # First new row
                    new_breathphase = 'inhale' if last_breathphase == 'exhale' else 'exhale'
                else:  # Subsequent rows
                    prev_phase = new_rows[i-1]['breathphase'].lower()
                    new_breathphase = 'inhale' if prev_phase == 'exhale' else 'exhale'
                
                new_description = f"Additional hold {i+1}"
                
                # Create the new row with the correct timestamp column name
                new_row = {
                    'retentionslot_timestamp': new_timestamp,
                    'milliseconds': new_ms,
                    'sample_index': new_sample_index,
                    'description': new_description,
                    'breathphase': new_breathphase
                }
                new_rows.append(new_row)
                
                if self.config['debug_mode'] and i < 5:  # Show first few new rows
                    print(f"Adding new row: {new_timestamp} ({new_ms} ms) - {new_breathphase}")
            
            if new_rows:
                new_rows_df = pd.DataFrame(new_rows)
                # Ensure new_rows_df has all columns from skeleton_df
                for col in skeleton_df.columns:
                    if col not in new_rows_df.columns:
                        new_rows_df[col] = None
                
                # Concatenate with only the columns from the original DataFrame
                skeleton_df = pd.concat([skeleton_df, new_rows_df[skeleton_df.columns]], ignore_index=True)
            
            if self.config['debug_mode']:
                print(f"Extended scaffold CSV to {len(skeleton_df)} retention slots.")
                print(f"New timestamp range: {skeleton_df['retentionslot_timestamp'].iloc[0]} to {skeleton_df['retentionslot_timestamp'].iloc[-1]}")
        
        if len(skeleton_df) > required_slots:
            skeleton_df = skeleton_df.iloc[:required_slots]
        
        return skeleton_df

    # ------------------------------------------------------------------
    # Generate counterbalanced stimulus design (trial conditions)
    # ------------------------------------------------------------------
    def generate_counterbalanced_design(self, participant_id):
        """
        Generate a counterbalanced design for a participant with trials per SOA
        determined by the repetitions parameter in the config.
        
        Catch trials don't have SOA values assigned since they don't have tactile stimuli.
        """
        random.seed(self.base_seed + participant_id)
        np.random.seed(self.base_seed + participant_id)
        
        soa_conditions = self.config['soa_conditions_ms']
        jitter_options = self.config['jitter_options_ms']
        stimulus_types = LOOMING_STIMULI
        repetitions = self.config['repetitions']  # Trials per SOA per condition
        
        all_trials = []
        
        # Generate main conditions: inhalation, exhalation, and baseline
        for condition, count in self.trial_counts.items():
            if condition == 'catch' or count == 0:
                continue
            
            # For each SOA, generate exactly 'repetitions' trials
            for soa in soa_conditions:
                # Calculate how many trials per stimulus type (as evenly as possible)
                trials_per_soa = repetitions
                trials_per_stim = trials_per_soa // len(stimulus_types)  # Integer division
                remainder = trials_per_soa % len(stimulus_types)
                
                # Distribute trials across stimulus types
                stim_trials = []
                for i, stim_type in enumerate(stimulus_types):
                    # Add extra trial for remainder distribution
                    extra = 1 if i < remainder else 0
                    for _ in range(trials_per_stim + extra):
                        stim_trials.append(stim_type)
                
                # Shuffle the stimulus types
                random.shuffle(stim_trials)
                
                # Create trials for this SOA
                for stim_type in stim_trials:
                    all_trials.append({
                        'participant_id': participant_id,
                        'trial_type': condition,
                        'stimulus_type': stim_type,
                        'soa_value_ms': soa,
                        'jitter_ms': random.choice(jitter_options),
                        'is_tactile': True  # For these main trials, tactile is present.
                    })
        
        # Add catch trials (looming only, no tactile) - no SOA values needed
        catch_count = self.trial_counts.get('catch', 0)
        if catch_count > 0:
            # Distribute catch trials evenly across stimulus types only
            catch_trials = []
            stim_counts = {stim: 0 for stim in stimulus_types}
            
            for _ in range(catch_count):
                # Choose the stimulus type with the lowest count
                min_stim = min(stim_counts, key=stim_counts.get)
                
                catch_trials.append({
                    'participant_id': participant_id,
                    'trial_type': 'catch',
                    'stimulus_type': min_stim,
                    'soa_value_ms': None,  # No SOA for catch trials
                    'jitter_ms': random.choice(jitter_options),
                    'is_tactile': False
                })
                
                stim_counts[min_stim] += 1
            
            # Shuffle catch trials and add to all trials
            random.shuffle(catch_trials)
            all_trials.extend(catch_trials)
        
        # Shuffle all trials and assign trial numbers
        all_trials_df = pd.DataFrame(all_trials)
        all_trials_df = all_trials_df.sample(frac=1, random_state=participant_id*100).reset_index(drop=True)
        all_trials_df['trial_number'] = range(1, len(all_trials_df) + 1)
        
        if self.config['debug_mode']:
            total_trials = len(all_trials_df)
            print("\nTrial distribution:")
            print(f"Total trials: {total_trials}")
            
            # Count by trial type
            for trial_type in sorted(all_trials_df['trial_type'].unique()):
                type_df = all_trials_df[all_trials_df['trial_type'] == trial_type]
                print(f"Total {trial_type}: {len(type_df)}")
                
                # Count by SOA for non-baseline and non-catch trial types
                if trial_type not in ['baseline', 'catch']:
                    for soa in sorted(soa_conditions):
                        soa_count = len(type_df[type_df['soa_value_ms'] == soa])
                        print(f"  {trial_type}_{soa}: {soa_count}")
                elif trial_type == 'baseline':
                    # For baseline, show SOA distribution (though not used for looming)
                    for soa in sorted(soa_conditions):
                        soa_count = len(type_df[type_df['soa_value_ms'] == soa])
                        print(f"  baseline_{soa}: {soa_count}")
                elif trial_type == 'catch':
                    # For catch trials, show stimulus distribution
                    catch_count = len(type_df)
                    print(f"  catch_trials: {catch_count} (no tactile stimulus, no SOA)")
            
            # Show stimulus type distribution
            print("\nStimulus type distribution by trial type:")
            for trial_type in sorted(all_trials_df['trial_type'].unique()):
                if trial_type == 'baseline':
                    continue
                subset = all_trials_df[all_trials_df['trial_type'] == trial_type]
                print(f"  {trial_type}: {subset['stimulus_type'].value_counts().to_dict()}")
        
        return all_trials_df

    # ------------------------------------------------------------------
    # Assign scaffold retention slots to trials while strictly maintaining the breathphase cycle.
    # ------------------------------------------------------------------
    def assign_breath_holds(self, design_df):
        """
        Assign scaffold retention slots to trials while strictly maintaining the breathphase cycle.
        
        Key requirements:
        1. Breathphase cycle (inhale-exhale-inhale-exhale) must remain intact
        2. Inhalation trials must go to inhale breathphase slots
        3. Exhalation trials must go to exhale breathphase slots
        4. Baseline and catch trials can go to either breathphase type
        5. The order of baseline and catch trials should be randomized
        """
        required_slots = len(design_df)
        scaffold_df = self.load_retention_slots(required_slots)
        
        # Ensure scaffold is properly sorted by timestamp/milliseconds
        scaffold_df = scaffold_df.sort_values(by='milliseconds').reset_index(drop=True)
        
        # Partition design trials by trial type
        pool_inh = [d for d in design_df.to_dict('records') if d['trial_type'] == 'inhalation']
        pool_exh = [d for d in design_df.to_dict('records') if d['trial_type'] == 'exhalation']
        pool_baseline = [d for d in design_df.to_dict('records') if d['trial_type'] == 'baseline']
        pool_catch = [d for d in design_df.to_dict('records') if d['trial_type'] == 'catch']
        
        # Count scaffold slots by breathphase
        inhale_slots = scaffold_df[scaffold_df['breathphase'].str.lower() == 'inhale']
        exhale_slots = scaffold_df[scaffold_df['breathphase'].str.lower() == 'exhale']
        n_inhale_slots = len(inhale_slots)
        n_exhale_slots = len(exhale_slots)
        
        if self.config['debug_mode']:
            print(f"Scaffold: {n_inhale_slots} inhale slots, {n_exhale_slots} exhale slots")
            print(f"Trials: {len(pool_inh)} inhalation, {len(pool_exh)} exhalation, " 
                  f"{len(pool_baseline)} baseline, {len(pool_catch)} catch")
        
        # Verify we have enough slots for mandatory trials
        if len(pool_inh) > n_inhale_slots:
            raise ValueError(f"Not enough inhale slots ({n_inhale_slots}) for inhalation trials ({len(pool_inh)})")
        if len(pool_exh) > n_exhale_slots:
            raise ValueError(f"Not enough exhale slots ({n_exhale_slots}) for exhalation trials ({len(pool_exh)})")
        
        # Calculate remaining slots per breathphase
        remaining_inhale = n_inhale_slots - len(pool_inh)
        remaining_exhale = n_exhale_slots - len(pool_exh)
        
        # Total baseline+catch trials
        total_flex_trials = len(pool_baseline) + len(pool_catch)
        
        # Verify we have enough slots for all trials
        if remaining_inhale + remaining_exhale != total_flex_trials:
            raise ValueError(f"Mismatch between available slots ({remaining_inhale + remaining_exhale}) "
                            f"and flexible trials ({total_flex_trials})")
        
        # Decide how to distribute baseline and catch trials across breathphases
        # Combine them first, then distribute proportionally
        flexible_trials = pool_baseline + pool_catch
        random.shuffle(flexible_trials)  # Randomize order
        
        # Split flexible trials between inhale and exhale slots proportionally
        inhale_flexible = flexible_trials[:remaining_inhale]
        exhale_flexible = flexible_trials[remaining_inhale:]
        
        if self.config['debug_mode']:
            print(f"Distributing {len(inhale_flexible)} flexible trials to inhale slots")
            print(f"Distributing {len(exhale_flexible)} flexible trials to exhale slots")
        
        # Create final allocation pools
        inhale_pool = pool_inh + inhale_flexible
        exhale_pool = pool_exh + exhale_flexible
        
        # Shuffle within each pool to randomize order
        random.shuffle(inhale_pool)
        random.shuffle(exhale_pool)
        
        # Create a dictionary mapping breathphase to trial pool
        phase_to_pool = {
            'inhale': inhale_pool,
            'exhale': exhale_pool
        }
        
        # Initialize counters for each breathphase
        pool_counters = {
            'inhale': 0,
            'exhale': 0
        }
        
        # Final list to store assigned trials in scaffold order
        final_assignments = []
        
        # Assign trials to scaffold rows, maintaining the original order
        for _, row in scaffold_df.iterrows():
            breathphase = row['breathphase'].lower()
            
            # Get the next trial from the appropriate pool
            if pool_counters[breathphase] < len(phase_to_pool[breathphase]):
                trial = phase_to_pool[breathphase][pool_counters[breathphase]]
                pool_counters[breathphase] += 1
                
                # Combine scaffold information with trial information
                combined = trial.copy()
                for col in row.index:
                    combined[col] = row[col]
                
                final_assignments.append(combined)
        
        # Create final dataframe and sort by scaffold order
        final_df = pd.DataFrame(final_assignments).sort_values(by='milliseconds').reset_index(drop=True)
        final_df['trial_number'] = range(1, len(final_df) + 1)
        
        # Compute additional timing information
        final_df['jittered_ms'] = final_df['milliseconds'] + final_df['jitter_ms']
        final_df['looming_stimulus_timestamp'] = final_df['jittered_ms'].apply(convert_ms_to_timestamp)
        
        # Add SOA timing for non-catch trials
        non_catch_mask = final_df['trial_type'] != 'catch'
        final_df.loc[non_catch_mask, 'soa_ms'] = final_df.loc[non_catch_mask, 'jittered_ms'] + final_df.loc[non_catch_mask, 'soa_value_ms']
        final_df.loc[non_catch_mask, 'tactile_stimulus_timestamp'] = final_df.loc[non_catch_mask, 'soa_ms'].apply(convert_ms_to_timestamp)
        
        # Verify the assignment respects the breathphase constraints
        self._validate_breathphase_assignment(final_df)
        
        return final_df

    def _validate_breathphase_assignment(self, final_df):
        """Verify that the trial assignment respects breathphase constraints."""
        if not self.config['debug_mode']:
            return
            
        print("\n=== BREATHPHASE ASSIGNMENT VALIDATION ===")
        
        # Check for inhalation trials assigned to inhale
        inh_trials = final_df[final_df['trial_type'] == 'inhalation']
        inh_correct = (inh_trials['breathphase'].str.lower() == 'inhale').all()
        print(f"Inhalation trials on inhale breathphase: {inh_correct}")
        
        # Check for exhalation trials assigned to exhale
        exh_trials = final_df[final_df['trial_type'] == 'exhalation']
        exh_correct = (exh_trials['breathphase'].str.lower() == 'exhale').all()
        print(f"Exhalation trials on exhale breathphase: {exh_correct}")
        
        # Count distribution of baseline and catch trials across breathphases
        baseline_inhale = ((final_df['trial_type'] == 'baseline') & 
                         (final_df['breathphase'].str.lower() == 'inhale')).sum()
        baseline_exhale = ((final_df['trial_type'] == 'baseline') & 
                          (final_df['breathphase'].str.lower() == 'exhale')).sum()
        catch_inhale = ((final_df['trial_type'] == 'catch') & 
                       (final_df['breathphase'].str.lower() == 'inhale')).sum()
        catch_exhale = ((final_df['trial_type'] == 'catch') & 
                       (final_df['breathphase'].str.lower() == 'exhale')).sum()
        
        print(f"Baseline trials: {baseline_inhale} on inhale, {baseline_exhale} on exhale")
        print(f"Catch trials: {catch_inhale} on inhale, {catch_exhale} on exhale")
        
        # Verify breathphase cycle is maintained
        prev_phase = None
        cycle_maintained = True
        expected_cycle = ['inhale', 'exhale'] * (len(final_df) // 2 + 1)
        
        for i, phase in enumerate(final_df['breathphase'].str.lower()):
            if i < len(expected_cycle) and phase != expected_cycle[i]:
                cycle_maintained = False
                print(f"Breathphase cycle disrupted at index {i}: expected {expected_cycle[i]}, got {phase}")
                break
        
        print(f"Breathphase cycle maintained: {cycle_maintained}")

    # ------------------------------------------------------------------
    # Finalize the design CSV: Retain the scaffold column order and append new columns.
    # Also, for baseline trials remove looming stimulus info; for catch trials remove tactile stimulus info.
    # ------------------------------------------------------------------
    def finalize_design_csv(self, design_df):
        baseline_mask = (design_df['trial_type'] == 'baseline')
        design_df.loc[baseline_mask, 'stimulus_type'] = None
        design_df.loc[baseline_mask, 'looming_stimulus_timestamp'] = None
        
        catch_mask = (design_df['trial_type'] == 'catch')
        design_df.loc[catch_mask, 'tactile_stimulus_timestamp'] = None
        
        design_df['expected_response'] = ~design_df['trial_type'].isin(['catch'])
        
        scaffold_cols = self.scaffold_order if hasattr(self, 'scaffold_order') else ['timestamp', 'milliseconds', 'sample_index', 'description', 'breathphase']
        new_cols = ['participant_id', 'trial_number', 'trial_type', 'stimulus_type', 'soa_value_ms', 'jitter_ms', 'is_tactile', 'looming_stimulus_timestamp', 'tactile_stimulus_timestamp', 'soa_ms', 'jittered_ms', 'expected_response']
        final_order = scaffold_cols + [col for col in new_cols if col not in scaffold_cols and col in design_df.columns]
        remaining = [col for col in design_df.columns if col not in final_order]
        final_order.extend(remaining)
        design_df = design_df[final_order]
        
        self._validate_design(design_df)
        return design_df

    # ------------------------------------------------------------------
    # Validation: Print distributions and sequence properties.
    # ------------------------------------------------------------------
    def _validate_design(self, design_df):
        """Perform comprehensive validation and counterbalancing checks on the design."""
        if not self.config['debug_mode']:
            return
            
        print("\n=== DESIGN VALIDATION ===")
        
        # 1. Overall trial count summary
        total_trials = len(design_df)
        trial_counts = design_df['trial_type'].value_counts().to_dict()
        
        print(f"TOTAL TRIALS: {total_trials}")
        print("Trial type distribution:")
        for trial_type, count in sorted(trial_counts.items()):
            percent = (count / total_trials) * 100
            print(f"  {trial_type.upper()}: {count} trials ({percent:.1f}%)")
        
        # 2. Detailed breakdown by trial type
        print("\nStimulus distribution by trial type:")
        for trial_type in sorted(design_df['trial_type'].unique()):
            if trial_type == 'baseline':
                continue
            subset = design_df[design_df['trial_type'] == trial_type]
            stim_counts = subset['stimulus_type'].value_counts().to_dict()
            print(f"  {trial_type.upper()}:")
            for stim, count in sorted(stim_counts.items()):
                percent = (count / len(subset)) * 100
                print(f"    {stim}: {count} trials ({percent:.1f}%)")
        
        # 3. SOA distribution checks
        print("\nSOA distribution by trial type:")
        for trial_type in sorted(design_df['trial_type'].unique()):
            if trial_type == 'baseline':
                continue
            subset = design_df[design_df['trial_type'] == trial_type]
            soa_counts = subset['soa_value_ms'].value_counts().to_dict()
            print(f"  {trial_type.upper()}:")
            for soa, count in sorted(soa_counts.items()):
                percent = (count / len(subset)) * 100
                print(f"    {soa} ms: {count} trials ({percent:.1f}%)")
        
        # 4. Jitter distribution
        print("\nJitter distribution:")
        jitter_counts = design_df['jitter_ms'].value_counts().to_dict()
        for jitter, count in sorted(jitter_counts.items()):
            percent = (count / total_trials) * 100
            print(f"  {jitter} ms: {count} trials ({percent:.1f}%)")
        
        # 5. Breath phase checks
        print("\nBreath phase distribution:")
        if 'breathphase' in design_df.columns:
            phase_counts = design_df['breathphase'].value_counts().to_dict()
            for phase, count in sorted(phase_counts.items()):
                percent = (count / total_trials) * 100
                print(f"  {phase}: {count} trials ({percent:.1f}%)")
        
        # 6. Cross-tabulation of trial types by breath phase
        print("\nTrial types by breath phase (COUNTERBALANCE CHECK):")
        if 'breathphase' in design_df.columns:
            trial_by_phase = pd.crosstab(
                design_df['trial_type'], 
                design_df['breathphase']
            )
            print(trial_by_phase)
            
            # Calculate chi-square statistic for independence (excluding mandatory assignments)
            try:
                # Create subset without inhalation/exhalation (which are deterministically assigned)
                flex_df = design_df[design_df['trial_type'].isin(['baseline', 'catch'])]
                if len(flex_df) > 0:
                    flex_table = pd.crosstab(flex_df['trial_type'], flex_df['breathphase'])
                    chi2 = 0
                    row_sums = flex_table.sum(axis=1)
                    col_sums = flex_table.sum(axis=0) 
                    total = flex_table.sum().sum()
                    
                    for i in flex_table.index:
                        for j in flex_table.columns:
                            observed = flex_table.loc[i, j]
                            expected = row_sums[i] * col_sums[j] / total
                            if expected > 0:
                                chi2 += ((observed - expected) ** 2) / expected
                    
                    print(f"\nChi-square for baseline/catch distribution: {chi2:.2f}")
                    print(f"Lower values indicate better balance between breathphases.")
            except Exception as e:
                print(f"Error calculating chi-square: {e}")
        
        # 7. Consecutive trials analysis
        trial_sequence = design_df['trial_type'].tolist()
        consecutive_counts = {}
        current_type = trial_sequence[0]
        current_count = 1
        for t in trial_sequence[1:]:
            if t == current_type:
                current_count += 1
            else:
                key = f"{current_type}_{current_count}"
                consecutive_counts[key] = consecutive_counts.get(key, 0) + 1
                current_type = t
                current_count = 1
        key = f"{current_type}_{current_count}"
        consecutive_counts[key] = consecutive_counts.get(key, 0) + 1
        
        print("\nRuns of consecutive trial types:")
        for key, count in sorted(consecutive_counts.items()):
            print(f"  {key}: {count}")
        
        # 8. SOA and stimulus type correlation check
        print("\nSOA-by-stimulus-type COUNTERBALANCE CHECK:")
        non_baseline = design_df[design_df['trial_type'] != 'baseline']
        if len(non_baseline) > 0:
            soa_by_stim = pd.crosstab(
                non_baseline['soa_value_ms'], 
                non_baseline['stimulus_type']
            )
            print(soa_by_stim)
            
            # Calculate coefficient of variation as a balance metric
            cv_values = []
            for _, row in soa_by_stim.iterrows():
                if row.sum() > 0:
                    cv = row.std() / row.mean() if row.mean() > 0 else 0
                    cv_values.append(cv)
            
            mean_cv = sum(cv_values) / len(cv_values) if cv_values else 0
            print(f"\nCoefficient of variation across SOA-stimulus combinations: {mean_cv:.4f}")
            print(f"Lower values indicate better counterbalancing.")
        
        # 9. Breathphase alternation check
        if 'breathphase' in design_df.columns:
            phases = design_df.sort_values('milliseconds')['breathphase'].str.lower().tolist()
            alternation_violations = 0
            for i in range(1, len(phases)):
                if phases[i] == phases[i-1]:
                    alternation_violations += 1
            
            print(f"\nBreathphase alternation violations: {alternation_violations}")
            print(f"This should be 0 - the inhale/exhale cycle must be maintained.")
        
        # 10. Timing analysis
        if 'jittered_ms' in design_df.columns:
            time_diffs = []
            sorted_df = design_df.sort_values('jittered_ms')
            for i in range(1, len(sorted_df)):
                diff = sorted_df.iloc[i]['jittered_ms'] - sorted_df.iloc[i-1]['jittered_ms']
                time_diffs.append(diff)
            
            if time_diffs:
                min_diff = min(time_diffs)
                max_diff = max(time_diffs)
                avg_diff = sum(time_diffs) / len(time_diffs)
                print(f"\nStimulus timing intervals:")
                print(f"  Minimum interval: {min_diff:.1f} ms")
                print(f"  Maximum interval: {max_diff:.1f} ms")
                print(f"  Average interval: {avg_diff:.1f} ms")
    
    # ------------------------------------------------------------------
    # Process a single participant: generate design, assign slots, finalize, and save CSV.
    # ------------------------------------------------------------------
    def process_participant(self, participant_id):
        try:
            if self.config['debug_mode']:
                print(f"\n=== Processing participant {participant_id} ===")
            design_df = self.generate_counterbalanced_design(participant_id)
            design_df = self.assign_breath_holds(design_df)
            design_df = self.finalize_design_csv(design_df)
            out_path = os.path.join(self.experiment_log_dir, f"participant_{participant_id}_design.csv")
            design_df.to_csv(out_path, index=False)
            if self.config['debug_mode']:
                print(f"Saved participant {participant_id} design to {out_path}")
            return True, out_path
        except Exception as e:
            print(f"Error processing participant {participant_id}: {e}")
            traceback.print_exc()
            return False, None
    
    # ------------------------------------------------------------------
    # Run design generation for multiple participants.
    # ------------------------------------------------------------------
    def run(self, participant_ids=None, num_participants=None):
        if participant_ids is None and num_participants is None:
            num_participants = self.config.get('num_participants', 5)
        if participant_ids is None:
            participant_ids = list(range(num_participants))
        if self.config['debug_mode']:
            print("=== Starting PPSDesignGenerator ===")
            print(f"Repetitions: {self.config['repetitions']}")
            print(f"Participants: {participant_ids}")
            print(f"Box breathing settings: {self.config['box_breathing']}")
        results = {}
        for pid in participant_ids:
            success, path = self.process_participant(pid)
            results[pid] = {'success': success, 'csv_path': path}
        
        # Summary of all participants
        if self.config['debug_mode'] and len(results) > 1:
            print("\n=== CROSS-PARTICIPANT VALIDATION ===")
            all_success = all(info['success'] for info in results.values())
            print(f"All participants generated successfully: {all_success}")
        
        return results

# ----------------------------------------------------------------------
# AUDIO GENERATION PLACEHOLDER FUNCTION
# ----------------------------------------------------------------------
def generate_audio_from_design(design_csv_path, output_dir, base_audio_path=None):
    """
    Generate audio files from a design CSV.
    (Placeholder function – implement audio generation as needed.)
    """
    print(f"Audio generation from {design_csv_path} not implemented yet")
    return None, None

# ----------------------------------------------------------------------
# MAIN FUNCTION
# ----------------------------------------------------------------------
def main():
    """Main function to run the design generator."""
    # Print configuration summary
    print("\n===== PPS EXPERIMENT DESIGN GENERATOR =====")
    print(f"Repetitions (trials per SOA per condition): {CONFIG['repetitions']}")
    print(f"SOA Conditions: {CONFIG['soa_conditions_ms']}")
    print(f"Jitter Options: {CONFIG['jitter_options_ms']}")
    print(f"Looming Stimuli: {LOOMING_STIMULI}")
    print(f"Number of Participants: {CONFIG['num_participants']}")
    
    # Calculate expected trial counts
    reps = CONFIG['repetitions']
    soa_count = len(CONFIG['soa_conditions_ms'])
    stim_count = len(LOOMING_STIMULI)
    
    trials_per_condition = reps * soa_count
    enabled_conditions = [cond for cond, enabled in CONFIG['included_conditions'].items() if enabled]
    total_base_trials = trials_per_condition * len(enabled_conditions)
    catch_trials = int(np.ceil(total_base_trials * CONFIG['catch_trial_percentage'] / 100))
    
    print("\n===== EXPECTED TRIAL COUNTS =====")
    print(f"Trials per condition: {reps} trials per SOA × {soa_count} SOAs = {trials_per_condition}")
    for cond in enabled_conditions:
        print(f"  {cond.upper()}: {trials_per_condition} trials")
    print(f"Catch trials ({CONFIG['catch_trial_percentage']}%): {catch_trials}")
    print(f"TOTAL TRIALS PER PARTICIPANT: {total_base_trials + catch_trials}")
    
    # Run the generator
    generator = PPSDesignGenerator(CONFIG)
    results = generator.run()
    
    # Summarize results
    print("\n===== GENERATION SUMMARY =====")
    success_count = sum(1 for info in results.values() if info['success'])
    print(f"Successfully generated designs for {success_count}/{len(results)} participants.")
    
    for pid, info in results.items():
        if info['success']:
            print(f"Participant {pid}: CSV generated at {info['csv_path']}")
        else:
            print(f"Participant {pid}: ERROR - no CSV generated.")
    
    print("\nTo generate audio files from these designs, implement the audio generation function.")

if __name__ == "__main__":
    main()


===== PPS EXPERIMENT DESIGN GENERATOR =====
Repetitions (trials per SOA per condition): 12
SOA Conditions: [190, 400, 700, 1000, 1500]
Jitter Options: [100, 200, 300, 400, 500]
Looming Stimuli: ['blue', 'brown', 'pink', 'white']
Number of Participants: 100

===== EXPECTED TRIAL COUNTS =====
Trials per condition: 12 trials per SOA × 5 SOAs = 60
  INHALATION: 60 trials
  EXHALATION: 60 trials
  BASELINE: 60 trials
Catch trials (10%): 18
TOTAL TRIALS PER PARTICIPANT: 198
Trial count calculation:
  inhalation: 60 trials
  exhalation: 60 trials
  baseline: 60 trials
  catch: 18 trials
  Total: 198 trials
Initialized PPSDesignGenerator with:
  - Repetitions: 12
  - Conditions: ['inhalation', 'exhalation', 'baseline']
  - SOA values: [190, 400, 700, 1000, 1500]
  - Total trials per participant: 198
  - Output directory: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog
=== Starting PPSDesignGenerator ===
Repetitions: 12
Participants: [0, 1, 2, 3, 4, 5

Traceback (most recent call last):
  File "C:\Users\cogpsy-vrlab\AppData\Local\Temp\ipykernel_5360\1058900113.py", line 735, in process_participant
    design_df.to_csv(out_path, index=False)
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\site-packages\pandas\util\_decorators.py", line 333, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\site-packages\pandas\core\generic.py", line 3967, in to_csv
    return DataFrameRenderer(formatter).to_csv(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\site-packages\pandas\io\formats\format.py", line 1014, in to_csv
    csv_formatter.save()
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\site-packages\pandas\io\formats\csvs.py", line 251, in save
    with get_handle(
         ^^^^^^^^^^^
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\site-packages\pandas\io\common.py", line 873, in get_handle
    handle = open(
             ^^^^^
PermissionEr


Breathphase alternation violations: 0
This should be 0 - the inhale/exhale cycle must be maintained.

Stimulus timing intervals:
  Minimum interval: 7600.0 ms
  Maximum interval: 8500.0 ms
  Average interval: 8003.6 ms
Error processing participant 1: [Errno 13] Permission denied: 'C:\\Users\\cogpsy-vrlab\\Documents\\GitHub\\BreathingSpace\\Level1_AudioGeneration\\ExperimentLog\\participant_1_design.csv'

=== Processing participant 2 ===

Trial distribution:
Total trials: 198
Total baseline: 60
  baseline_190: 12
  baseline_400: 12
  baseline_700: 12
  baseline_1000: 12
  baseline_1500: 12
Total catch: 18
  catch_trials: 18 (no tactile stimulus, no SOA)
Total exhalation: 60
  exhalation_190: 12
  exhalation_400: 12
  exhalation_700: 12
  exhalation_1000: 12
  exhalation_1500: 12
Total inhalation: 60
  inhalation_190: 12
  inhalation_400: 12
  inhalation_700: 12
  inhalation_1000: 12
  inhalation_1500: 12

Stimulus type distribution by trial type:
  catch: {'brown': 5, 'blue': 5, 'pink'

In [2]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
PPS Design Generator

This script generates participant‐specific experimental design CSVs for a 
Peripersonal Space (PPS) experiment. Each design CSV is built upon a 
"scaffold" CSV that contains the retention slot (breathhold) timestamps 
from a box‐breathing audio file. If needed, additional retention slots are 
appended (using fixed 8‑second increments) so that every trial can be assigned 
to a specific retention slot. Then, a counterbalanced set of trial conditions 
(e.g., trial type, looming stimulus type, SOA, jitter) is allocated to each 
scaffold row according to the following rules:
  
  - Inhalation trials (looming cue + tactile) are only assigned to scaffold rows 
    whose breathphase is "inhale."
  - Exhalation trials (looming cue + tactile) are only assigned to scaffold rows 
    whose breathphase is "exhale."
  - Baseline (tactile only; looming = NA) and catch (looming only; no tactile) trials 
    are allocated from the remaining pool in a random, counterbalanced manner 
    between the inhale and exhale rows.
    
The final output retains the original column order from the scaffold, with new 
trial‐specific variables appended as new columns.

Author: AI Assistant
"""

import os
import numpy as np
import pandas as pd
import random
import traceback

# ----------------------------------------------------------------------
# Helper function: Convert milliseconds to "MM:SS.s" format
# ----------------------------------------------------------------------
def convert_ms_to_timestamp(ms):
    m = int(ms // 60000)
    s = (ms % 60000) / 1000.0
    return f"{m:02}:{s:.1f}"

# ----------------------------------------------------------------------
# CONFIGURATION PARAMETERS
# ----------------------------------------------------------------------
CONFIG = {
    'repetitions': 12,           # Number of trials per SOA condition
    'num_participants': 100,     # Default number of participants
    'practice_trials': 8,        # Number of practice trials at the beginning
    'included_conditions': {
        'inhalation': True,
        'exhalation': True,
        'baseline': True,
    },
    'catch_trial_percentage': 10,
    'soa_conditions_ms': [190, 400, 700, 1000, 1500],
    'jitter_options_ms': [100, 200, 300, 400, 500],
    'debug_mode': True,
    'paths': {
        # Directory where participant-specific CSV files will be saved
        'experiment_log_dir': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog",
    },
    # Path to the original scaffold CSV containing retention (breathhold) timestamps.
    'skeleton_csv_path': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_timestamps.csv",
    'box_breathing': {
        'cycle_duration_sec': 8,      # Duration of one breath cycle in seconds
        # Intro/outro are not used here because the scaffold drives timing.
        'intro_duration_sec': 60,
        'outro_duration_sec': 30,
        'sample_rate': 48000,         # Audio sample rate (samples per second)
    }
}

# Noise types for looming stimuli (can be any labels)
LOOMING_STIMULI = ['blue', 'brown', 'pink', 'white']

# ----------------------------------------------------------------------
# CLASS: PPSDesignGenerator
# ----------------------------------------------------------------------
class PPSDesignGenerator:
    """Generates participant-specific CSV design files for PPS experiments."""
    
    def __init__(self, config=None):
        self.config = config or CONFIG
        
        # Prepare output directory.
        self.experiment_log_dir = self.config['paths']['experiment_log_dir']
        os.makedirs(self.experiment_log_dir, exist_ok=True)
        
        # Calculate trial counts.
        # For each condition (inhalation, exhalation, baseline): count = (# noise types) × (# SOA) × (repetitions).
        # Then, add 10% catch trials.
        self.trial_counts = self._calculate_trial_counts()
        self.conditions = [cond for cond, inc in self.config['included_conditions'].items() if inc]
        
        # Set seed for reproducibility.
        self.base_seed = 42
        
        if self.config['debug_mode']:
            total_trials = sum(self.trial_counts.values())
            print("Initialized PPSDesignGenerator with:")
            print(f"  - Repetitions: {self.config['repetitions']}")
            print(f"  - Practice trials: {self.config['practice_trials']}")
            print(f"  - Conditions: {self.conditions}")
            print(f"  - SOA values: {self.config['soa_conditions_ms']}")
            print(f"  - Total trials per participant: {total_trials} + {self.config['practice_trials']} practice = {total_trials + self.config['practice_trials']}")
            print(f"  - Output directory: {self.experiment_log_dir}")
    
    def _calculate_trial_counts(self):
        """
        Calculate the number of trials for each condition, using the repetitions parameter
        to determine how many trials per SOA per condition.
        """
        soa_conditions = self.config['soa_conditions_ms']
        repetitions = self.config['repetitions']  # This now directly controls trials per SOA
        catch_percentage = self.config['catch_trial_percentage']
        
        trial_counts = {}
        for condition, include in self.config['included_conditions'].items():
            if include:
                # Each condition gets repetitions × number of SOAs trials
                trial_counts[condition] = len(soa_conditions) * repetitions
        
        regular_trials = sum(trial_counts.values())
        catch_trials = int(np.ceil(regular_trials * catch_percentage / 100))
        trial_counts['catch'] = catch_trials
        
        if self.config['debug_mode']:
            print("Trial count calculation:")
            for cond, count in trial_counts.items():
                print(f"  {cond}: {count} trials")
            print(f"  Total: {sum(trial_counts.values())} trials")
        
        return trial_counts

    # ------------------------------------------------------------------
    # Load the scaffold CSV and extend it if needed.
    # Retain the original column order.
    # ------------------------------------------------------------------
    def load_retention_slots(self, required_slots):
        """
        Load the scaffold CSV containing retention slots and extend it if needed.
        Handles different column naming variations by standardizing them.
        """
        skeleton_path = self.config['skeleton_csv_path']
        try:
            # Try to read the CSV, handling different separators
            skeleton_df = pd.read_csv(skeleton_path, sep=None, engine='python')
            if len(skeleton_df.columns) == 1:
                skeleton_df = pd.read_csv(skeleton_path, delimiter="\t")
            
            # Normalize column names (strip whitespace and convert to lowercase)
            skeleton_df.columns = [col.strip().lower() for col in skeleton_df.columns]
            self.scaffold_order = skeleton_df.columns.tolist()
            
            if self.config['debug_mode']:
                print("Scaffold CSV columns:", self.scaffold_order)
                
            # Determine timestamp column name - could be 'timestamp' or 'retentionslot_timestamp'
            timestamp_col = None
            if 'timestamp' in skeleton_df.columns:
                timestamp_col = 'timestamp'
            elif 'retentionslot_timestamp' in skeleton_df.columns:
                timestamp_col = 'retentionslot_timestamp'
            
            # If neither standard name exists, try to find a column that might contain timestamp info
            if timestamp_col is None:
                for col in skeleton_df.columns:
                    if 'time' in col.lower() and not col.lower() == 'milliseconds':
                        timestamp_col = col
                        break
            
            if timestamp_col is None:
                raise ValueError("Could not find timestamp column in scaffold CSV.")
                
            # Standardize the timestamp column name to 'retentionslot_timestamp'
            if timestamp_col != 'retentionslot_timestamp':
                skeleton_df = skeleton_df.rename(columns={timestamp_col: 'retentionslot_timestamp'})
                if timestamp_col in self.scaffold_order:
                    self.scaffold_order[self.scaffold_order.index(timestamp_col)] = 'retentionslot_timestamp'
                
        except Exception as e:
            raise ValueError(f"Error reading scaffold CSV from {skeleton_path}: {e}")
        
        current_slots = len(skeleton_df)
        required_slots = int(required_slots)
        sample_rate = self.config['box_breathing']['sample_rate']
        cycle_sec = self.config['box_breathing']['cycle_duration_sec']
        
        if self.config['debug_mode']:
            print(f"Loaded scaffold CSV with {current_slots} retention slots.")
            # Display the last few existing timestamps
            try:
                print(f"Last timestamp in skeleton CSV: {skeleton_df.iloc[-1]['retentionslot_timestamp']} ({skeleton_df.iloc[-1]['milliseconds']} ms)")
            except Exception as e:
                print(f"Could not display last timestamp: {e}")
        
        if current_slots < required_slots:
            last_row = skeleton_df.iloc[-1]
            last_ms = float(last_row['milliseconds'])
            last_sample_index = int(last_row['sample_index'])
            new_rows = []
            
            # Get the last breathphase to ensure proper alternation
            last_breathphase = last_row['breathphase'].lower()
            
            for i in range(required_slots - current_slots):
                # Increment by 8 seconds (8000 ms) for each new row
                new_ms = last_ms + (cycle_sec * 1000) * (i + 1)
                new_sample_index = last_sample_index + int(cycle_sec * sample_rate * (i + 1))
                new_timestamp = convert_ms_to_timestamp(new_ms)
                
                # Alternate breathphase: if last was 'exhale', next is 'inhale' and vice versa
                if i == 0:  # First new row
                    new_breathphase = 'inhale' if last_breathphase == 'exhale' else 'exhale'
                else:  # Subsequent rows
                    prev_phase = new_rows[i-1]['breathphase'].lower()
                    new_breathphase = 'inhale' if prev_phase == 'exhale' else 'exhale'
                
                new_description = f"Additional hold {i+1}"
                
                # Create the new row with the correct timestamp column name
                new_row = {
                    'retentionslot_timestamp': new_timestamp,
                    'milliseconds': new_ms,
                    'sample_index': new_sample_index,
                    'description': new_description,
                    'breathphase': new_breathphase
                }
                new_rows.append(new_row)
                
                if self.config['debug_mode'] and i < 5:  # Show first few new rows
                    print(f"Adding new row: {new_timestamp} ({new_ms} ms) - {new_breathphase}")
            
            if new_rows:
                new_rows_df = pd.DataFrame(new_rows)
                # Ensure new_rows_df has all columns from skeleton_df
                for col in skeleton_df.columns:
                    if col not in new_rows_df.columns:
                        new_rows_df[col] = None
                
                # Concatenate with only the columns from the original DataFrame
                skeleton_df = pd.concat([skeleton_df, new_rows_df[skeleton_df.columns]], ignore_index=True)
            
            if self.config['debug_mode']:
                print(f"Extended scaffold CSV to {len(skeleton_df)} retention slots.")
                print(f"New timestamp range: {skeleton_df['retentionslot_timestamp'].iloc[0]} to {skeleton_df['retentionslot_timestamp'].iloc[-1]}")
        
        if len(skeleton_df) > required_slots:
            skeleton_df = skeleton_df.iloc[:required_slots]
        
        return skeleton_df

    # ------------------------------------------------------------------
    # Generate counterbalanced stimulus design (trial conditions)
    # ------------------------------------------------------------------
    def generate_counterbalanced_design(self, participant_id):
        """
        Generate a counterbalanced design for a participant with trials per SOA
        determined by the repetitions parameter in the config.
        
        Catch trials don't have SOA values assigned since they don't have tactile stimuli.
        """
        random.seed(self.base_seed + participant_id)
        np.random.seed(self.base_seed + participant_id)
        
        soa_conditions = self.config['soa_conditions_ms']
        jitter_options = self.config['jitter_options_ms']
        stimulus_types = LOOMING_STIMULI
        repetitions = self.config['repetitions']  # Trials per SOA per condition
        
        all_trials = []
        
        # Generate main conditions: inhalation, exhalation, and baseline
        for condition, count in self.trial_counts.items():
            if condition == 'catch' or count == 0:
                continue
            
            # For each SOA, generate exactly 'repetitions' trials
            for soa in soa_conditions:
                # Calculate how many trials per stimulus type (as evenly as possible)
                trials_per_soa = repetitions
                trials_per_stim = trials_per_soa // len(stimulus_types)  # Integer division
                remainder = trials_per_soa % len(stimulus_types)
                
                # Distribute trials across stimulus types
                stim_trials = []
                for i, stim_type in enumerate(stimulus_types):
                    # Add extra trial for remainder distribution
                    extra = 1 if i < remainder else 0
                    for _ in range(trials_per_stim + extra):
                        stim_trials.append(stim_type)
                
                # Shuffle the stimulus types
                random.shuffle(stim_trials)
                
                # Create trials for this SOA
                for stim_type in stim_trials:
                    all_trials.append({
                        'participant_id': participant_id,
                        'trial_type': condition,
                        'stimulus_type': stim_type,
                        'soa_value_ms': soa,
                        'jitter_ms': random.choice(jitter_options),
                        'is_tactile': True  # For these main trials, tactile is present.
                    })
        
        # Add catch trials (looming only, no tactile) - no SOA values needed
        catch_count = self.trial_counts.get('catch', 0)
        if catch_count > 0:
            # Distribute catch trials evenly across stimulus types only
            catch_trials = []
            stim_counts = {stim: 0 for stim in stimulus_types}
            
            for _ in range(catch_count):
                # Choose the stimulus type with the lowest count
                min_stim = min(stim_counts, key=stim_counts.get)
                
                catch_trials.append({
                    'participant_id': participant_id,
                    'trial_type': 'catch',
                    'stimulus_type': min_stim,
                    'soa_value_ms': None,  # No SOA for catch trials
                    'jitter_ms': random.choice(jitter_options),
                    'is_tactile': False
                })
                
                stim_counts[min_stim] += 1
            
            # Shuffle catch trials and add to all trials
            random.shuffle(catch_trials)
            all_trials.extend(catch_trials)
        
        # Shuffle all trials and assign trial numbers
        all_trials_df = pd.DataFrame(all_trials)
        all_trials_df = all_trials_df.sample(frac=1, random_state=participant_id*100).reset_index(drop=True)
        all_trials_df['trial_number'] = range(1, len(all_trials_df) + 1)
        
        if self.config['debug_mode']:
            total_trials = len(all_trials_df)
            print("\nTrial distribution:")
            print(f"Total trials: {total_trials}")
            
            # Count by trial type
            for trial_type in sorted(all_trials_df['trial_type'].unique()):
                type_df = all_trials_df[all_trials_df['trial_type'] == trial_type]
                print(f"Total {trial_type}: {len(type_df)}")
                
                # Count by SOA for non-baseline and non-catch trial types
                if trial_type not in ['baseline', 'catch']:
                    for soa in sorted(soa_conditions):
                        soa_count = len(type_df[type_df['soa_value_ms'] == soa])
                        print(f"  {trial_type}_{soa}: {soa_count}")
                elif trial_type == 'baseline':
                    # For baseline, show SOA distribution (though not used for looming)
                    for soa in sorted(soa_conditions):
                        soa_count = len(type_df[type_df['soa_value_ms'] == soa])
                        print(f"  baseline_{soa}: {soa_count}")
                elif trial_type == 'catch':
                    # For catch trials, show stimulus distribution
                    catch_count = len(type_df)
                    print(f"  catch_trials: {catch_count} (no tactile stimulus, no SOA)")
            
            # Show stimulus type distribution
            print("\nStimulus type distribution by trial type:")
            for trial_type in sorted(all_trials_df['trial_type'].unique()):
                if trial_type == 'baseline':
                    continue
                subset = all_trials_df[all_trials_df['trial_type'] == trial_type]
                print(f"  {trial_type}: {subset['stimulus_type'].value_counts().to_dict()}")
        
        return all_trials_df

    # ------------------------------------------------------------------
    # Assign scaffold retention slots to trials while strictly maintaining the breathphase cycle.
    # ------------------------------------------------------------------
    def assign_breath_holds(self, design_df, participant_id):
        """
        Assign scaffold retention slots to trials while strictly maintaining the breathphase cycle.
        
        Key requirements:
        1. Breathphase cycle (inhale-exhale-inhale-exhale) must remain intact
        2. Inhalation trials must go to inhale breathphase slots
        3. Exhalation trials must go to exhale breathphase slots
        4. Baseline and catch trials must be equally distributed between inhale and exhale breathphases
        5. The first N slots (where N is practice_trials) are reserved for practice trials
        """
        # Account for practice trials in required slots calculation
        practice_trials = self.config.get('practice_trials', 0)
        required_slots = len(design_df) + practice_trials
        scaffold_df = self.load_retention_slots(required_slots)
        
        # Ensure scaffold is properly sorted by timestamp/milliseconds
        scaffold_df = scaffold_df.sort_values(by='milliseconds').reset_index(drop=True)
        
        # Reserve the first N slots for practice trials
        practice_scaffold = scaffold_df.iloc[:practice_trials].copy() if practice_trials > 0 else pd.DataFrame()
        regular_scaffold = scaffold_df.iloc[practice_trials:].copy()
        
        # Create practice trials (empty placeholder trials)
        practice_trial_list = []
        if practice_trials > 0:
            for i, (_, row) in enumerate(practice_scaffold.iterrows()):
                breathphase = row['breathphase'].lower()
                practice_trial = {
                    'participant_id': participant_id,
                    'trial_number': i + 1,  # Start numbering from 1
                    'trial_type': 'practice',
                    'stimulus_type': None,
                    'soa_value_ms': None,
                    'jitter_ms': random.choice(self.config['jitter_options_ms']),  # Add jitter for realistic timing
                    'is_tactile': False
                }
                # Add all scaffold columns
                for col in row.index:
                    practice_trial[col] = row[col]
                
                practice_trial_list.append(practice_trial)
        
        # Partition design trials by trial type
        pool_inh = [d for d in design_df.to_dict('records') if d['trial_type'] == 'inhalation']
        pool_exh = [d for d in design_df.to_dict('records') if d['trial_type'] == 'exhalation']
        pool_baseline = [d for d in design_df.to_dict('records') if d['trial_type'] == 'baseline']
        pool_catch = [d for d in design_df.to_dict('records') if d['trial_type'] == 'catch']
        
        # Count scaffold slots by breathphase
        inhale_slots = regular_scaffold[regular_scaffold['breathphase'].str.lower() == 'inhale']
        exhale_slots = regular_scaffold[regular_scaffold['breathphase'].str.lower() == 'exhale']
        n_inhale_slots = len(inhale_slots)
        n_exhale_slots = len(exhale_slots)
        
        if self.config['debug_mode']:
            print(f"Scaffold: {n_inhale_slots} inhale slots, {n_exhale_slots} exhale slots")
            print(f"Trials: {len(pool_inh)} inhalation, {len(pool_exh)} exhalation, " 
                  f"{len(pool_baseline)} baseline, {len(pool_catch)} catch")
            print(f"Practice trials: {practice_trials}")
        
        # Verify we have enough slots for mandatory trials
        if len(pool_inh) > n_inhale_slots:
            raise ValueError(f"Not enough inhale slots ({n_inhale_slots}) for inhalation trials ({len(pool_inh)})")
        if len(pool_exh) > n_exhale_slots:
            raise ValueError(f"Not enough exhale slots ({n_exhale_slots}) for exhalation trials ({len(pool_exh)})")
        
        # Calculate remaining slots per breathphase
        remaining_inhale = n_inhale_slots - len(pool_inh)
        remaining_exhale = n_exhale_slots - len(pool_exh)
        
        # Total baseline+catch trials
        total_flex_trials = len(pool_baseline) + len(pool_catch)
        
        # Verify we have enough slots for all trials
        if remaining_inhale + remaining_exhale != total_flex_trials:
            raise ValueError(f"Mismatch between available slots ({remaining_inhale + remaining_exhale}) "
                            f"and flexible trials ({total_flex_trials})")
        
        # Ensure equal distribution of baseline and catch trials between breathphases
        
        # Shuffle baseline and catch trials
        random.shuffle(pool_baseline)
        random.shuffle(pool_catch)
        
        # Calculate how many of each should go to each breathphase
        baseline_inhale = len(pool_baseline) // 2
        baseline_exhale = len(pool_baseline) - baseline_inhale
        
        catch_inhale = len(pool_catch) // 2
        catch_exhale = len(pool_catch) - catch_inhale
        
        # Create inhale and exhale flexible trial pools
        inhale_flexible = pool_baseline[:baseline_inhale] + pool_catch[:catch_inhale]
        exhale_flexible = pool_baseline[baseline_inhale:] + pool_catch[catch_inhale:]
        
        random.shuffle(inhale_flexible)
        random.shuffle(exhale_flexible)
        
        if self.config['debug_mode']:
            print(f"Distributing flexible trials:")
            print(f"  Baseline: {baseline_inhale} to inhale, {baseline_exhale} to exhale")
            print(f"  Catch: {catch_inhale} to inhale, {catch_exhale} to exhale")
        
        # Create final allocation pools
        inhale_pool = pool_inh + inhale_flexible
        exhale_pool = pool_exh + exhale_flexible
        
        # Shuffle within each pool to randomize order
        random.shuffle(inhale_pool)
        random.shuffle(exhale_pool)
        
        # Create a dictionary mapping breathphase to trial pool
        phase_to_pool = {
            'inhale': inhale_pool,
            'exhale': exhale_pool
        }
        
        # Initialize counters for each breathphase
        pool_counters = {
            'inhale': 0,
            'exhale': 0
        }
        
        # Final list to store assigned trials in scaffold order
        regular_assignments = []
        
        # Assign trials to regular scaffold rows, maintaining the original order
        for _, row in regular_scaffold.iterrows():
            breathphase = row['breathphase'].lower()
            
            # Get the next trial from the appropriate pool
            if pool_counters[breathphase] < len(phase_to_pool[breathphase]):
                trial = phase_to_pool[breathphase][pool_counters[breathphase]]
                pool_counters[breathphase] += 1
                
                # Combine scaffold information with trial information
                combined = trial.copy()
                for col in row.index:
                    combined[col] = row[col]
                
                regular_assignments.append(combined)
        
        # Combine practice trials and regular trials
        final_assignments = practice_trial_list + regular_assignments
        
        # Create final dataframe and sort by scaffold order
        final_df = pd.DataFrame(final_assignments).sort_values(by='milliseconds').reset_index(drop=True)
        
        # Renumber trials from 1 to N to ensure sequential numbering
        final_df['trial_number'] = range(1, len(final_df) + 1)
        
        # Compute additional timing information
        final_df['jittered_ms'] = final_df['milliseconds'] + final_df['jitter_ms'].fillna(0)
        
        # Add looming stimulus timestamp for non-baseline trials
        looming_mask = ~final_df['trial_type'].isin(['baseline', 'practice'])
        final_df.loc[looming_mask, 'looming_stimulus_timestamp'] = final_df.loc[looming_mask, 'jittered_ms'].apply(convert_ms_to_timestamp)
        
        # Add SOA timing for non-catch, non-practice trials
        non_catch_mask = ~final_df['trial_type'].isin(['catch', 'practice'])
        final_df.loc[non_catch_mask, 'soa_ms'] = final_df.loc[non_catch_mask, 'jittered_ms'] + final_df.loc[non_catch_mask, 'soa_value_ms'].fillna(0)
        final_df.loc[non_catch_mask, 'tactile_stimulus_timestamp'] = final_df.loc[non_catch_mask, 'soa_ms'].apply(convert_ms_to_timestamp)
        
        # Verify the assignment respects the breathphase constraints
        self._validate_breathphase_assignment(final_df)
        
        return final_df

    def _validate_breathphase_assignment(self, final_df):
        """Verify that the trial assignment respects breathphase constraints."""
        if not self.config['debug_mode']:
            return
            
        print("\n=== BREATHPHASE ASSIGNMENT VALIDATION ===")
        
        # Skip practice trials for validation
        non_practice_df = final_df[final_df['trial_type'] != 'practice']
        
        # Check for inhalation trials assigned to inhale
        inh_trials = non_practice_df[non_practice_df['trial_type'] == 'inhalation']
        inh_correct = (inh_trials['breathphase'].str.lower() == 'inhale').all() if len(inh_trials) > 0 else True
        print(f"Inhalation trials on inhale breathphase: {inh_correct}")
        
        # Check for exhalation trials assigned to exhale
        exh_trials = non_practice_df[non_practice_df['trial_type'] == 'exhalation']
        exh_correct = (exh_trials['breathphase'].str.lower() == 'exhale').all() if len(exh_trials) > 0 else True
        print(f"Exhalation trials on exhale breathphase: {exh_correct}")
        
        # Count distribution of baseline and catch trials across breathphases
        baseline_inhale = ((non_practice_df['trial_type'] == 'baseline') & 
                         (non_practice_df['breathphase'].str.lower() == 'inhale')).sum()
        baseline_exhale = ((non_practice_df['trial_type'] == 'baseline') & 
                          (non_practice_df['breathphase'].str.lower() == 'exhale')).sum()
        catch_inhale = ((non_practice_df['trial_type'] == 'catch') & 
                       (non_practice_df['breathphase'].str.lower() == 'inhale')).sum()
        catch_exhale = ((non_practice_df['trial_type'] == 'catch') & 
                       (non_practice_df['breathphase'].str.lower() == 'exhale')).sum()
        
        print(f"Baseline trials: {baseline_inhale} on inhale, {baseline_exhale} on exhale")
        print(f"Catch trials: {catch_inhale} on inhale, {catch_exhale} on exhale")
        
        # Verify breathphase cycle is maintained
        prev_phase = None
        cycle_maintained = True
        expected_cycle = ['inhale', 'exhale'] * (len(final_df) // 2 + 1)
        
        for i, phase in enumerate(final_df['breathphase'].str.lower()):
            if i < len(expected_cycle) and phase != expected_cycle[i]:
                cycle_maintained = False
                print(f"Breathphase cycle disrupted at index {i}: expected {expected_cycle[i]}, got {phase}")
                break
        
        print(f"Breathphase cycle maintained: {cycle_maintained}")
        
        # Count practice trials
        practice_count = (final_df['trial_type'] == 'practice').sum()
        print(f"Practice trials: {practice_count}")

    # ------------------------------------------------------------------
    # Finalize the design CSV: Retain the scaffold column order and append new columns.
    # Also, for baseline trials remove looming stimulus info; for catch trials remove tactile stimulus info.
    # ------------------------------------------------------------------
    def finalize_design_csv(self, design_df):
        baseline_mask = (design_df['trial_type'] == 'baseline')
        design_df.loc[baseline_mask, 'stimulus_type'] = None
        design_df.loc[baseline_mask, 'looming_stimulus_timestamp'] = None
        
        catch_mask = (design_df['trial_type'] == 'catch')
        design_df.loc[catch_mask, 'tactile_stimulus_timestamp'] = None
        
        practice_mask = (design_df['trial_type'] == 'practice')
        design_df.loc[practice_mask, 'stimulus_type'] = None
        design_df.loc[practice_mask, 'looming_stimulus_timestamp'] = None
        design_df.loc[practice_mask, 'tactile_stimulus_timestamp'] = None
        
        # Set expected_response based on trial type
        design_df['expected_response'] = ~design_df['trial_type'].isin(['catch', 'practice'])
        
        scaffold_cols = self.scaffold_order if hasattr(self, 'scaffold_order') else ['timestamp', 'milliseconds', 'sample_index', 'description', 'breathphase']
        new_cols = ['participant_id', 'trial_number', 'trial_type', 'stimulus_type', 'soa_value_ms', 'jitter_ms', 'is_tactile', 'looming_stimulus_timestamp', 'tactile_stimulus_timestamp', 'soa_ms', 'jittered_ms', 'expected_response']
        final_order = scaffold_cols + [col for col in new_cols if col not in scaffold_cols and col in design_df.columns]
        remaining = [col for col in design_df.columns if col not in final_order]
        final_order.extend(remaining)
        design_df = design_df[final_order]
        
        self._validate_design(design_df)
        return design_df

    # ------------------------------------------------------------------
    # Validation: Print distributions and sequence properties.
    # ------------------------------------------------------------------
    def _validate_design(self, design_df):
        """Perform comprehensive validation and counterbalancing checks on the design."""
        if not self.config['debug_mode']:
            return
            
        print("\n=== DESIGN VALIDATION ===")
        
        # Separate practice trials from experimental trials
        practice_df = design_df[design_df['trial_type'] == 'practice']
        exp_df = design_df[design_df['trial_type'] != 'practice']
        
        # 1. Overall trial count summary
        total_trials = len(design_df)
        practice_count = len(practice_df)
        experimental_count = len(exp_df)
        
        trial_counts = exp_df['trial_type'].value_counts().to_dict()
        
        print(f"TOTAL TRIALS: {total_trials} (Practice: {practice_count}, Experimental: {experimental_count})")
        print("Trial type distribution (excluding practice):")
        for trial_type, count in sorted(trial_counts.items()):
            percent = (count / experimental_count) * 100
            print(f"  {trial_type.upper()}: {count} trials ({percent:.1f}%)")
        
        # 2. Detailed breakdown by trial type
        print("\nStimulus distribution by trial type:")
        for trial_type in sorted(exp_df['trial_type'].unique()):
            if trial_type == 'baseline':
                continue
            subset = exp_df[exp_df['trial_type'] == trial_type]
            stim_counts = subset['stimulus_type'].value_counts().to_dict()
            print(f"  {trial_type.upper()}:")
            for stim, count in sorted(stim_counts.items()):
                percent = (count / len(subset)) * 100
                print(f"    {stim}: {count} trials ({percent:.1f}%)")
        
        # 3. SOA distribution checks
        print("\nSOA distribution by trial type:")
        for trial_type in sorted(exp_df['trial_type'].unique()):
            if trial_type == 'baseline':
                continue
            subset = exp_df[exp_df['trial_type'] == trial_type]
            soa_counts = subset['soa_value_ms'].value_counts().to_dict()
            print(f"  {trial_type.upper()}:")
            for soa, count in sorted(soa_counts.items()):
                percent = (count / len(subset)) * 100
                print(f"    {soa} ms: {count} trials ({percent:.1f}%)")
        
        # 4. Jitter distribution
        print("\nJitter distribution (excluding practice):")
        jitter_counts = exp_df['jitter_ms'].value_counts().to_dict()
        for jitter, count in sorted(jitter_counts.items()):
            percent = (count / experimental_count) * 100
            print(f"  {jitter} ms: {count} trials ({percent:.1f}%)")
        
        # 5. Breath phase checks for experimental trials
        print("\nBreath phase distribution (excluding practice):")
        if 'breathphase' in exp_df.columns:
            phase_counts = exp_df['breathphase'].value_counts().to_dict()
            for phase, count in sorted(phase_counts.items()):
                percent = (count / experimental_count) * 100
                print(f"  {phase}: {count} trials ({percent:.1f}%)")
        
        # 6. Cross-tabulation of trial types by breath phase
        print("\nTrial types by breath phase (COUNTERBALANCE CHECK):")
        if 'breathphase' in exp_df.columns:
            trial_by_phase = pd.crosstab(
                exp_df['trial_type'], 
                exp_df['breathphase']
            )
            print(trial_by_phase)
            
            # Calculate chi-square statistic for independence (excluding mandatory assignments)
            try:
                # Create subset without inhalation/exhalation (which are deterministically assigned)
                flex_df = exp_df[exp_df['trial_type'].isin(['baseline', 'catch'])]
                if len(flex_df) > 0:
                    flex_table = pd.crosstab(flex_df['trial_type'], flex_df['breathphase'])
                    chi2 = 0
                    row_sums = flex_table.sum(axis=1)
                    col_sums = flex_table.sum(axis=0) 
                    total = flex_table.sum().sum()
                    
                    for i in flex_table.index:
                        for j in flex_table.columns:
                            observed = flex_table.loc[i, j]
                            expected = row_sums[i] * col_sums[j] / total
                            if expected > 0:
                                chi2 += ((observed - expected) ** 2) / expected
                    
                    print(f"\nChi-square for baseline/catch distribution: {chi2:.2f}")
                    print(f"Lower values indicate better balance between breathphases.")
            except Exception as e:
                print(f"Error calculating chi-square: {e}")
        
        # 7. Consecutive trials analysis (excluding practice)
        trial_sequence = exp_df['trial_type'].tolist()
        consecutive_counts = {}
        
        if trial_sequence:
            current_type = trial_sequence[0]
            current_count = 1
            for t in trial_sequence[1:]:
                if t == current_type:
                    current_count += 1
                else:
                    key = f"{current_type}_{current_count}"
                    consecutive_counts[key] = consecutive_counts.get(key, 0) + 1
                    current_type = t
                    current_count = 1
            key = f"{current_type}_{current_count}"
            consecutive_counts[key] = consecutive_counts.get(key, 0) + 1
            
            print("\nRuns of consecutive trial types (excluding practice):")
            for key, count in sorted(consecutive_counts.items()):
                print(f"  {key}: {count}")
        
        # 8. SOA and stimulus type correlation check (excluding practice)
        print("\nSOA-by-stimulus-type COUNTERBALANCE CHECK (excluding practice):")
        non_baseline = exp_df[exp_df['trial_type'] != 'baseline']
        if len(non_baseline) > 0:
            soa_by_stim = pd.crosstab(
                non_baseline['soa_value_ms'], 
                non_baseline['stimulus_type']
            )
            print(soa_by_stim)
            
            # Calculate coefficient of variation as a balance metric
            cv_values = []
            for _, row in soa_by_stim.iterrows():
                if row.sum() > 0:
                    cv = row.std() / row.mean() if row.mean() > 0 else 0
                    cv_values.append(cv)
            
            mean_cv = sum(cv_values) / len(cv_values) if cv_values else 0
            print(f"\nCoefficient of variation across SOA-stimulus combinations: {mean_cv:.4f}")
            print(f"Lower values indicate better counterbalancing.")
        
        # 9. Breathphase alternation check
        if 'breathphase' in design_df.columns:
            phases = design_df.sort_values('milliseconds')['breathphase'].str.lower().tolist()
            alternation_violations = 0
            for i in range(1, len(phases)):
                if phases[i] == phases[i-1]:
                    alternation_violations += 1
            
            print(f"\nBreathphase alternation violations: {alternation_violations}")
            print(f"This should be 0 - the inhale/exhale cycle must be maintained.")
        
        # 10. Timing analysis
        if 'jittered_ms' in design_df.columns:
            time_diffs = []
            sorted_df = design_df.sort_values('jittered_ms')
            for i in range(1, len(sorted_df)):
                diff = sorted_df.iloc[i]['jittered_ms'] - sorted_df.iloc[i-1]['jittered_ms']
                time_diffs.append(diff)
            
            if time_diffs:
                min_diff = min(time_diffs)
                max_diff = max(time_diffs)
                avg_diff = sum(time_diffs) / len(time_diffs)
                print(f"\nStimulus timing intervals:")
                print(f"  Minimum interval: {min_diff:.1f} ms")
                print(f"  Maximum interval: {max_diff:.1f} ms")
                print(f"  Average interval: {avg_diff:.1f} ms")
        
        # 11. Practice trials validation
        if not practice_df.empty:
            print("\nPractice trials analysis:")
            practice_phase_counts = practice_df['breathphase'].value_counts().to_dict() if 'breathphase' in practice_df.columns else {}
            print(f"  Total practice trials: {len(practice_df)}")
            for phase, count in sorted(practice_phase_counts.items()):
                print(f"  {phase}: {count} trials")
            
            print(f"  Practice trial numbers: {practice_df['trial_number'].tolist()}")
            print(f"  Practice trials have jitter: {'jitter_ms' in practice_df.columns and not practice_df['jitter_ms'].isnull().all()}")
    
    # ------------------------------------------------------------------
    # Process a single participant: generate design, assign slots, finalize, and save CSV.
    # ------------------------------------------------------------------
    def process_participant(self, participant_id):
        try:
            if self.config['debug_mode']:
                print(f"\n=== Processing participant {participant_id} ===")
            design_df = self.generate_counterbalanced_design(participant_id)
            design_df = self.assign_breath_holds(design_df, participant_id)
            design_df = self.finalize_design_csv(design_df)
            out_path = os.path.join(self.experiment_log_dir, f"participant_{participant_id}_design.csv")
            design_df.to_csv(out_path, index=False)
            if self.config['debug_mode']:
                print(f"Saved participant {participant_id} design to {out_path}")
            return True, out_path
        except Exception as e:
            print(f"Error processing participant {participant_id}: {e}")
            traceback.print_exc()
            return False, None
    
    # ------------------------------------------------------------------
    # Run design generation for multiple participants.
    # ------------------------------------------------------------------
    def run(self, participant_ids=None, num_participants=None):
        if participant_ids is None and num_participants is None:
            num_participants = self.config.get('num_participants', 5)
        if participant_ids is None:
            participant_ids = list(range(num_participants))
        if self.config['debug_mode']:
            print("=== Starting PPSDesignGenerator ===")
            print(f"Repetitions: {self.config['repetitions']}")
            print(f"Practice trials: {self.config['practice_trials']}")
            print(f"Participants: {participant_ids}")
            print(f"Box breathing settings: {self.config['box_breathing']}")
        results = {}
        for pid in participant_ids:
            success, path = self.process_participant(pid)
            results[pid] = {'success': success, 'csv_path': path}
        
        # Summary of all participants
        if self.config['debug_mode'] and len(results) > 1:
            print("\n=== CROSS-PARTICIPANT VALIDATION ===")
            all_success = all(info['success'] for info in results.values())
            print(f"All participants generated successfully: {all_success}")
        
        return results

# ----------------------------------------------------------------------
# AUDIO GENERATION PLACEHOLDER FUNCTION
# ----------------------------------------------------------------------
def generate_audio_from_design(design_csv_path, output_dir, base_audio_path=None):
    """
    Generate audio files from a design CSV.
    (Placeholder function – implement audio generation as needed.)
    """
    print(f"Audio generation from {design_csv_path} not implemented yet")
    return None, None

# ----------------------------------------------------------------------
# MAIN FUNCTION
# ----------------------------------------------------------------------
def main():
    """Main function to run the design generator."""
    # Print configuration summary
    print("\n===== PPS EXPERIMENT DESIGN GENERATOR =====")
    print(f"Repetitions (trials per SOA per condition): {CONFIG['repetitions']}")
    print(f"Practice trials at beginning: {CONFIG['practice_trials']}")
    print(f"SOA Conditions: {CONFIG['soa_conditions_ms']}")
    print(f"Jitter Options: {CONFIG['jitter_options_ms']}")
    print(f"Looming Stimuli: {LOOMING_STIMULI}")
    print(f"Number of Participants: {CONFIG['num_participants']}")
    
    # Calculate expected trial counts
    reps = CONFIG['repetitions']
    soa_count = len(CONFIG['soa_conditions_ms'])
    
    trials_per_condition = reps * soa_count
    enabled_conditions = [cond for cond, enabled in CONFIG['included_conditions'].items() if enabled]
    total_base_trials = trials_per_condition * len(enabled_conditions)
    catch_trials = int(np.ceil(total_base_trials * CONFIG['catch_trial_percentage'] / 100))
    practice_trials = CONFIG['practice_trials']
    
    print("\n===== EXPECTED TRIAL COUNTS =====")
    print(f"Trials per condition: {reps} trials per SOA × {soa_count} SOAs = {trials_per_condition}")
    for cond in enabled_conditions:
        print(f"  {cond.upper()}: {trials_per_condition} trials")
    print(f"Catch trials ({CONFIG['catch_trial_percentage']}%): {catch_trials}")
    print(f"Practice trials: {practice_trials}")
    print(f"TOTAL TRIALS PER PARTICIPANT: {total_base_trials + catch_trials + practice_trials}")
    
    # Run the generator
    generator = PPSDesignGenerator(CONFIG)
    results = generator.run()
    
    # Summarize results
    print("\n===== GENERATION SUMMARY =====")
    success_count = sum(1 for info in results.values() if info['success'])
    print(f"Successfully generated designs for {success_count}/{len(results)} participants.")
    
    for pid, info in results.items():
        if info['success']:
            print(f"Participant {pid}: CSV generated at {info['csv_path']}")
        else:
            print(f"Participant {pid}: ERROR - no CSV generated.")
    
    print("\nTo generate audio files from these designs, implement the audio generation function.")

if __name__ == "__main__":
    main()


===== PPS EXPERIMENT DESIGN GENERATOR =====
Repetitions (trials per SOA per condition): 12
Practice trials at beginning: 8
SOA Conditions: [190, 400, 700, 1000, 1500]
Jitter Options: [100, 200, 300, 400, 500]
Looming Stimuli: ['blue', 'brown', 'pink', 'white']
Number of Participants: 100

===== EXPECTED TRIAL COUNTS =====
Trials per condition: 12 trials per SOA × 5 SOAs = 60
  INHALATION: 60 trials
  EXHALATION: 60 trials
  BASELINE: 60 trials
Catch trials (10%): 18
Practice trials: 8
TOTAL TRIALS PER PARTICIPANT: 206
Trial count calculation:
  inhalation: 60 trials
  exhalation: 60 trials
  baseline: 60 trials
  catch: 18 trials
  Total: 198 trials
Initialized PPSDesignGenerator with:
  - Repetitions: 12
  - Practice trials: 8
  - Conditions: ['inhalation', 'exhalation', 'baseline']
  - SOA values: [190, 400, 700, 1000, 1500]
  - Total trials per participant: 198 + 8 practice = 206
  - Output directory: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\Expe

Traceback (most recent call last):
  File "C:\Users\cogpsy-vrlab\AppData\Local\Temp\ipykernel_5360\1204939600.py", line 817, in process_participant
    design_df.to_csv(out_path, index=False)
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\site-packages\pandas\util\_decorators.py", line 333, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\site-packages\pandas\core\generic.py", line 3967, in to_csv
    return DataFrameRenderer(formatter).to_csv(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\site-packages\pandas\io\formats\format.py", line 1014, in to_csv
    csv_formatter.save()
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\site-packages\pandas\io\formats\csvs.py", line 251, in save
    with get_handle(
         ^^^^^^^^^^^
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\site-packages\pandas\io\common.py", line 873, in get_handle
    handle = open(
             ^^^^^
PermissionEr

  INHALATION:
    190.0 ms: 12 trials (20.0%)
    400.0 ms: 12 trials (20.0%)
    700.0 ms: 12 trials (20.0%)
    1000.0 ms: 12 trials (20.0%)
    1500.0 ms: 12 trials (20.0%)

Jitter distribution (excluding practice):
  100 ms: 40 trials (20.2%)
  200 ms: 51 trials (25.8%)
  300 ms: 32 trials (16.2%)
  400 ms: 39 trials (19.7%)
  500 ms: 36 trials (18.2%)

Breath phase distribution (excluding practice):
  exhale: 99 trials (50.0%)
  inhale: 99 trials (50.0%)

Trial types by breath phase (COUNTERBALANCE CHECK):
breathphase  exhale  inhale
trial_type                 
baseline         30      30
catch             9       9
exhalation       60       0
inhalation        0      60

Chi-square for baseline/catch distribution: 0.00
Lower values indicate better balance between breathphases.

Runs of consecutive trial types (excluding practice):
  baseline_1: 27
  baseline_2: 11
  baseline_3: 1
  baseline_4: 2
  catch_1: 16
  catch_2: 1
  exhalation_1: 60
  inhalation_1: 60

SOA-by-stimulus-typ

Traceback (most recent call last):
  File "C:\Users\cogpsy-vrlab\AppData\Local\Temp\ipykernel_5360\1204939600.py", line 817, in process_participant
    design_df.to_csv(out_path, index=False)
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\site-packages\pandas\util\_decorators.py", line 333, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\site-packages\pandas\core\generic.py", line 3967, in to_csv
    return DataFrameRenderer(formatter).to_csv(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\site-packages\pandas\io\formats\format.py", line 1014, in to_csv
    csv_formatter.save()
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\site-packages\pandas\io\formats\csvs.py", line 251, in save
    with get_handle(
         ^^^^^^^^^^^
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\site-packages\pandas\io\common.py", line 873, in get_handle
    handle = open(
             ^^^^^
PermissionEr


Stimulus timing intervals:
  Minimum interval: 7600.0 ms
  Maximum interval: 8400.0 ms
  Average interval: 8002.0 ms

Practice trials analysis:
  Total practice trials: 8
  exhale: 4 trials
  inhale: 4 trials
  Practice trial numbers: [1, 2, 3, 4, 5, 6, 7, 8]
  Practice trials have jitter: True
Saved participant 95 design to C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog\participant_95_design.csv

=== Processing participant 96 ===

Trial distribution:
Total trials: 198
Total baseline: 60
  baseline_190: 12
  baseline_400: 12
  baseline_700: 12
  baseline_1000: 12
  baseline_1500: 12
Total catch: 18
  catch_trials: 18 (no tactile stimulus, no SOA)
Total exhalation: 60
  exhalation_190: 12
  exhalation_400: 12
  exhalation_700: 12
  exhalation_1000: 12
  exhalation_1500: 12
Total inhalation: 60
  inhalation_190: 12
  inhalation_400: 12
  inhalation_700: 12
  inhalation_1000: 12
  inhalation_1500: 12

Stimulus type distribution by trial type:
 